# SCEG v9 · ACL18
Same code as `SCEG_v9_INDIA` with `DATASET = "acl18"`. The v8 notebook text below is unchanged except for the `🔧 [v9]` edits listed in the change log.


> **v10.** Identical to v9.2 plus a new §21 at the end: an audit of the RankIC implementation, bottleneck diagnostics, and a pre-registered, validation-only, one-factor-at-a-time ranking study (SCEG-Rank) with ablations and ranking baselines. Nothing before §21 changed, and every cached prediction is reused. With your existing cache, re-running trains only the §21 arms (ACL18 ≈ 20–40 min, FNSPID ≈ 1.5–3 h on a T4, resumable). The §21 cells are identical in the ACL18 and FNSPID notebooks.


> **v9.2.** Identical to v9.1 plus a new §19 at the end (evidence audit, P4 replication on FNSPID, model-free propagation event study). Nothing before §19 changed; SCEG-R (§16) is byte-identical (spec `0bfe3098df8c`). With your existing cache, re-running trains only: the §16–§18 arms not yet cached, and on FNSPID the 15 runs of the P4 replication arm. §19.1–19.3 and §19.5 only read cached predictions and prices.


> **v9.1.** Same code as `SCEG_v9_1_INDIA` with `DATASET = "acl18"`. For this dataset every v9.1 change is a no-op (all-True evaluation mask) except the new §18 (supplementary baselines and metrics), which reuses the cached §14–§17 runs. Your cached v9 results stay valid: re-running only trains the §18 arms.


---
## v9 change log (on top of v8; identical code for ACL18, FNSPID and India)
| # | Change | Effect on ACL18 / FNSPID numbers |
|---|---|---|
| 1 | §0.1 frozen confirmation protocol, hashed *before data load*; refuses to run if changed | none |
| 2 | India data layer (§1.0 universe/prices, §1.4 audit, §2.0 BSE filings), guarded by `DATASET == "india"` | none (no-op) |
| 3 | Generic evaluation mask `EVAL_MASK` (India: point-in-time index members), market close in local time zone | none (all-True mask; 16:00 ET) |
| 4 | §15.9 / §16.5 read all three dataset caches | none |
| 5 | §16.0 asserts the SCEG-R specification equals v8 (`0bfe3098df8c`) | none |
| 6 | §17 cross-market confirmation: XM arms, 8 primary + 6 secondary hypotheses, TOST equivalence, market vs stock-specific decomposition, cross-market table | new results only |
| 7 | Empty-event guard in the legacy §3 cell (India has no v2 shards) | none |
| 8 | India test predictions locked until the FNSPID run of this code exists (`require_shakedown`) | none |


# SCEG v2 — Structured Event Graphs with Regime-Conditioned Propagation
### Next-day cross-sectional return prediction on FNSPID (NASDAQ-100, 2018–2023)
#### 🔧 FINAL-ITERATION NOTEBOOK. One source, two datasets: set `DATASET` in the configuration cell to `"acl18"` (StockNet benchmark) or `"fnspid"` (NASDAQ-100 news, walk-forward). Sections 3–13 (the v2 'before' results) run for ACL18 only; sections 14–15 run for both.

> **How to run (Colab):** Runtime → Change runtime type → **T4 GPU** → Save. Then Runtime → **Run all**.
> Allow Google Drive access when asked (it keeps the extraction progress safe). If the session disconnects,
> reconnect and **Run all** again: the notebook continues from where it stopped.

**Research hypothesis (unchanged since the original SCEG proposal).** Next-day returns depend on
(i) *what happened* to a firm (structured news events), (ii) *how that information travels* along
directed inter-firm dependencies, and (iii) *the market state in which it travels*.

**What v1 showed and what v2 changes.** In v1 the regime was a firm-invariant one-hot *input*. That
can only shift all predictions up or down together; it cannot change *how* information propagates
between firms. So v1 could not test clause (iii). v2 implements the mechanism the original proposal
described:

| # | v1 (previous notebook) | v2 (this notebook) | Why |
|---|---|---|---|
| 1 | Regime = one-hot input feature | **Regime-conditioned message passing**: K relation-specific propagation experts mixed by the *filtered* HMM posterior π_t | Tests whether market state changes *propagation* (original SCEG Eq. 7) |
| 2 | Messages carry all neighbour features equally | **Event-gated messages**: a neighbour's message is scaled by a learned, regime-specific gate on *its own* event representation | "Events propagate to related firms" is the core SCEG idea |
| 3 | Union of correlation + DYNOTEARS, binarised, lags merged | **Multi-relational graph**: separate contemporaneous (DYNOTEARS W), **lead–lag** (DYNOTEARS A, lags 1–2) and correlation relations, weighted and row-normalised; sparsified | Keeps the *direction and lag* information that motivated DYNOTEARS |
| 4 | 20 tickers with price series ending in 2020 kept as stale nodes; DYNOTEARS on forward-filled returns | Universe = tickers with ≥95 % coverage that reach the end of the sample (73). Missing returns are *not* forward-filled | Removes graph contamination |
| 5 | 4-state HMM on 5 daily means, K fixed, hard labels, ~2.8-day spells | HMM on backward-looking market-state features (return, 20-d momentum, 10-d volatility, breadth, dispersion, range); **sticky** transition prior; **K by BIC**; **filtered posterior** (forward algorithm, no look-ahead) | Regimes should be persistent states, and soft posteriors are needed for mixing |
| 6 | Target = raw next-day return (the model spent capacity on market timing) | Target = **cross-sectionally demeaned** next-day return; loss = Huber + IC (correlation) term | The hypothesis is about *relative* firm responses; IC is unchanged by demeaning |
| 7 | Single 70/15/15 split, test = 2023 only (1 crisis day) | **Walk-forward, 3 folds** (test 2021, 2022, 2023; expanding train; 6-month validation); every fitted object is re-estimated per fold on that fold's training dates | More test days (≈750) and a 2022 bear year in test |
| 8 | 1 seed; naive baselines only | **5 seeds**, seed-ensembles; learned baselines **MLP, GRU, LightGBM**; naive momentum/reversal | Addresses the main reviewer objections |
| 9 | No raw-text control; no placebo | **Raw-text control**, **regime placebo** (posterior permuted across days), **random graph** of matched density | Isolates structure vs text, and real regimes vs extra parameters |
| 10 | IC skipped days with constant predictions | Constant-prediction days count as IC = 0, and coverage is reported; paired Newey–West tests on daily IC differences | Removes the price-only artefact |
| 11 | 1-day label overlap at split boundaries | A sample needs its target day inside the same split (purge) | Removes the residual overlap |

**Leakage guarantees (all per fold).** News aligned at 16:00 ET (date-only → next trading day).
PCA, scalers, TF-IDF (fallback encoder), HMM parameters, HMM K, regime names, correlation graph,
DYNOTEARS graph and LightGBM early stopping use training/validation dates only. Regime posteriors are
*filtered* (use observations ≤ t). Windows and targets never cross split boundaries.

---
## ACL18 (StockNet) benchmark adaptation — change log
**Same SCEG model, new dataset.** Only the data layer and the benchmark evaluation protocol were
changed. Every modified cell starts with a `🔧 [ACL18]` comment that says why it changed.

| Cell | Change | Type |
|---|---|---|
| Config | ACL18 paths, date window, benchmark split, label rule, tweet cap, alignment mode | load + protocol |
| §1 prices | Downloads the public StockNet release from GitHub (sparse clone, checksum-verified); `price/raw/*.csv` loader; StockNet 88-stock list | load |
| §1.2 tensors | Benchmark sample mask (movement threshold −0.5 % / +0.55 %, 5-day price window) and binary label; regression target = raw next-day return | protocol |
| §2 news | StockNet `tweet/preprocessed` loader; tweet → trading-day alignment (benchmark or strict); duplicate handling for tweets | load (+ protocol for alignment) |
| §3 extraction | Smoke-test fallback only (`ALLOW_RAW_FALLBACK`, default off). Prompt/schema unchanged | load |
| §5 splits | Benchmark boundaries; sample assigned by its **target** day | protocol |
| §6.3 / §8 | Target tensor name `YT` instead of `YXS` (value is the raw return under the benchmark) | protocol |
| §7.1 grid | Adds the "DYNOTEARS only" ablation from v1 (config only, no new component) | experiment preserved |
| §10 | ACC / MCC on benchmark samples, mean ± sd over seeds; majority-class baseline | protocol |

**Unchanged:** six-field event schema and prompt, extraction model, BGE embeddings, firm-day pooling,
PCA, bounded forward-fill + staleness features, DYNOTEARS, correlation graph, relation handling, HMM,
SCEG-RC architecture (GCN-style relational propagation, event gate, regime conditioning, Transformer,
head), Huber + IC loss, AdamW / cosine / clipping, early stopping on validation IC, seeds, baselines and
the rest of the ablation grid.

---
## 0 · Setup and configuration

In [1]:
import sys, os
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # 🐞 fix: no "-U" — upgrading Colab's preinstalled libraries can break them until a runtime restart.
    get_ipython().system('pip -q install hmmlearn sentence-transformers lightgbm accelerate yfinance')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.0/166.0 kB 7.7 MB/s eta 0:00:00


In [2]:
import gc, json, re, glob, zipfile, random, warnings, time, math
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

CONFIG = {
    # ---------- data window ----------
    "START_DATE": "2018-01-01",
    "END_DATE":   "2023-12-31",
    "UNIVERSE_MODE": "nasdaq100",      # "nasdaq100" (default, ~73 firms) | "sp500" (broader cross-section, see §1)
    "NEWS_SOURCE": "mirror",           # "mirror" (NASDAQ-100 subset) | "all_external" (5.7 GB, needed for sp500)
    "NEWS_CHUNKSIZE": 200_000,
    "NEWS_CUTOFF_HOUR_ET": 16,
    "MAX_ARTICLES_PER_TICKER_DAY": 2,
    "UNIVERSE_MIN_COVERAGE": 0.95,     # NEW: fraction of listed trading days with a price row

    # ---------- event extraction (unchanged: prompt + six-field schema) ----------
    "USE_LLM_EVENTS": True,
    "EXTRACT_MODEL": "Qwen/Qwen2.5-1.5B-Instruct",
    "EXTRACT_BATCH_SIZE": 32,
    "EXTRACT_SHARD_SIZE": 1_000,     # save progress to Drive every 1,000 tweets
    "EXTRACT_MAX_ARTICLES": None,

    # ---------- representation ----------
    "EVENT_ENCODER": "bge",            # "bge" (paper) | "tfidf" (offline fallback, no HF access)
    "EMBED_MODEL": "BAAI/bge-base-en-v1.5",
    "EMBED_BATCH_SIZE": 128,
    "PCA_COMPONENTS": 20,
    "FFILL_LIMIT": 5,
    "NEWS_AGE_CAP": 10,

    # ---------- regimes ----------
    "HMM_K_GRID": [2, 3, 4],           # K chosen by BIC on training dates
    "HMM_STICKY": 50.0,                # Dirichlet self-transition pseudo-count (persistence)
    "HMM_RESTARTS": 5,

    # ---------- graph ----------
    "TOP_K_CORR": 5,
    "DYNOTEARS_P": 2,
    "DYNOTEARS_LAMBDA": 0.05,          # L1 on contemporaneous W
    "DYNOTEARS_LAMBDA_A": 0.01,        # L1 on lagged A (lead-lag effects are weaker -> lighter penalty)
    "DYNOTEARS_MAX_ITER": 20,
    "DYNOTEARS_W_THRESHOLD": 0.05,
    "MAX_IN_DEGREE": 10,               # keep the strongest incoming edges per node and relation

    # ---------- walk-forward folds: (train_end, val_end, test_end); train starts at the beginning ----------
    "FOLDS": [("2020-06-30", "2020-12-31", "2021-12-31"),
              ("2021-06-30", "2021-12-31", "2022-12-31"),
              ("2022-06-30", "2022-12-31", "2023-12-31")],

    # ---------- model / training ----------
    "WINDOW": 5,
    "HIDDEN": 64,
    "NHEAD": 4,
    "N_TRANSFORMER_LAYERS": 2,
    "DROPOUT": 0.2,
    "LR": 5e-4,
    "WEIGHT_DECAY": 1e-4,
    "EPOCHS": 40,
    "PATIENCE": 8,
    "BATCH_DAYS": 16,
    "IC_LOSS_WEIGHT": 0.5,
    "SEEDS": [0, 1, 2, 3, 4],
    "COST_BPS": 5,

    # ---------- infrastructure ----------
    "USE_DRIVE": True,
    "CACHE_DIR": "/content/drive/MyDrive/MinorProject/fnspid_cache",
    "QUICK": False,                    # True -> 1 seed, fewer epochs (smoke test)
}
# 🔧 [ACL18] dataset / benchmark overrides. WHY: new data source and the published StockNet protocol
# (Xu & Cohen, ACL 2018; github.com/yumoxu/stocknet-code, src/config.yml and src/DataPipe.py).
DATASET = os.environ.get("SCEG_DATASET", "acl18")   # ◀◀ "acl18" | "fnspid" | "india"
# ◀◀ RUN_PHASE (free-Colab friendly; every phase resumes from the Drive cache):
#   "data" = download and align the data on a CPU runtime (uses no GPU quota), then stop with a message
#   "prep" = + LLM event extraction, embeddings, graphs, regimes on a T4; stops before ANY test prediction
#   "all"  = everything (India test predictions also need the FNSPID shake-down file, see §0.1)
RUN_PHASE = os.environ.get("SCEG_PHASE", "all")
class PhaseComplete(Exception): pass
RUN_LEGACY = DATASET == "acl18"                     # v2 sections 3–13 (FNSPID v2 results come from the earlier FNSPID notebook)
_ACL18_OVERRIDES = ({
    "DATASET": "acl18",
    # 🔧 [ACL18] public source: the official StockNet release on GitHub (Xu & Cohen, 2018), downloaded at
    # run time, so no local/Drive copy is needed. Set ACL18_ROOT to a folder to use an existing copy instead.
    "ACL18_REPO": "https://github.com/yumoxu/stocknet-dataset.git",
    "ACL18_ROOT": "/content/stocknet-dataset",
    "UNIVERSE_MODE": "acl18",
    "START_DATE": "2013-06-01",        # price history for feature/regime warm-up only; no sample before 2014
    "END_DATE":   "2016-01-15",        # lets y_next of the last test feature day exist
    "TRAIN_START": "2014-01-01",       # StockNet train_start_date
    # StockNet: train [2014-01-01, 2015-08-01), dev [2015-08-01, 2015-10-01), test [2015-10-01, 2016-01-01)
    "FOLDS": [("2015-07-31", "2015-09-30", "2015-12-31")],
    "SPLIT_RULE": "target_day",        # StockNet assigns a sample to the split of its target date
    "TARGET_MODE": "raw",              # benchmark label = direction of the raw adjusted-close movement
    "MV_THRESHOLDS": (-0.005, 0.0055), # StockNet discards -0.5% <= movement < 0.55%
    "BENCH_WINDOW_CAL_DAYS": 5,        # StockNet max_n_days (price-window availability check)
    "MIN_TARGETS_PER_DAY": 1,          # keep every benchmark sample (v2 default 10 would drop some)
    "NEWS_ALIGNMENT": "benchmark",     # "benchmark" (StockNet calendar-day rule) | "strict" (16:00 ET cut-off)
    "MAX_ARTICLES_PER_TICKER_DAY": 30, # StockNet max_n_msgs (tweets per stock-day)
    "ALLOW_RAW_FALLBACK": False,       # smoke tests without LLM shards only; never for reported results
    "CACHE_DIR": "/content/drive/MyDrive/MinorProject/acl18_cache",
})
# 🔧 [v9 · INDIA] external confirmation dataset: point-in-time NIFTY-50, BSE filings, strict 15:30 IST, 5 walk-forward
# folds (tests 2020-2024). Every value here is copied into the frozen protocol (§0.1).
_INDIA_OVERRIDES = {
    "DATASET": "india", "UNIVERSE_MODE": "nifty50_pit", "NEWS_SOURCE": "bse_announcements",
    "START_DATE": "2015-06-01",        # warm-up for rolling features / regimes; no sample before TRAIN_START
    "END_DATE":   "2024-12-31",
    "TRAIN_START": "2016-01-01",       # BSE dissemination timestamps are complete from Nov 2015
    "FOLDS": [("2019-06-30", "2019-12-31", "2020-12-31"), ("2020-06-30", "2020-12-31", "2021-12-31"),
              ("2021-06-30", "2021-12-31", "2022-12-31"), ("2022-06-30", "2022-12-31", "2023-12-31"),
              ("2023-06-30", "2023-12-31", "2024-12-31")],
    "SPLIT_RULE": "embargo",           # whole input window and target inside the split (same as FNSPID)
    "MARKET_TZ": "Asia/Kolkata", "CLOSE_MINUTES": 15 * 60 + 30,     # NSE close 15:30 IST
    "MAX_ARTICLES_PER_TICKER_DAY": None,  # set in §2.0 by the label-free rule in the protocol
    "UNIVERSE_MIN_COVERAGE": 0.95, "MIN_PRICE_ROWS": 250,
    "CONSTITUENTS_REPO": "AMP4010/Historical_Nifty_50_Constituent_Weights_20Y",
    "CONSTITUENTS_REV": "c15498388ae4d346f42291c4e5937235e56ae67c",
    "DISSEM_FALLBACK_BUFFER_MIN": 120, "SHORT_SESSION_REL_VOL": 0.25, "MIN_DAY_COVERAGE": 0.5,
    "REQUIRE_FNSPID_SHAKEDOWN": True,
    "SYNTH": bool(os.environ.get("SCEG_INDIA_SYNTH")),   # synthetic pipeline check only (never reportable)
    "CACHE_DIR": "/content/drive/MyDrive/MinorProject/india_cache",
}
if DATASET == "acl18":
    CONFIG.update(_ACL18_OVERRIDES)
elif DATASET == "india":
    CONFIG.update(_INDIA_OVERRIDES)
else:
    CONFIG["DATASET"] = "fnspid"            # FNSPID defaults above: 2018–2023, NASDAQ-100, 3 walk-forward folds, strict 16:00 ET
# 🔧 [ACL18] Google Drive is optional: it only keeps caches (LLM-extraction shards, embeddings, predictions)
# across Colab disconnects. With USE_DRIVE=False everything lives in /content (lost when the runtime resets).
if IN_COLAB and not CONFIG["USE_DRIVE"]:
    CONFIG["CACHE_DIR"] = f"/content/{DATASET}_cache"
if not IN_COLAB:                       # local / offline execution
    CONFIG.update(USE_DRIVE=False, CACHE_DIR=os.environ.get("SCEG_CACHE", f"./{DATASET}_cache"),
                  EVENT_ENCODER=os.environ.get("SCEG_ENCODER", "tfidf"),
                  ALLOW_RAW_FALLBACK=bool(os.environ.get("SCEG_RAW_FALLBACK")),
                  USE_LLM_EVENTS=not bool(os.environ.get("SCEG_RAW_FALLBACK")))
    if DATASET == "acl18":
        CONFIG.update(ACL18_ROOT=os.environ.get("SCEG_ACL18_ROOT", CONFIG["ACL18_ROOT"]),
                      NEWS_ALIGNMENT=os.environ.get("SCEG_ALIGN", CONFIG["NEWS_ALIGNMENT"]))
    if os.environ.get("SCEG_DYNO_ITER"):     # smoke tests only (changes the graph cache key)
        CONFIG["DYNOTEARS_MAX_ITER"] = int(os.environ["SCEG_DYNO_ITER"])
if DATASET == "india" and CONFIG["SYNTH"]:   # offline synthetic check: own cache, last two folds, TF-IDF encoder
    CONFIG.update(CACHE_DIR=os.environ.get("SCEG_CACHE", "./india_synth_cache"), FOLDS=CONFIG["FOLDS"][-2:],
                  EVENT_ENCODER="tfidf", USE_DRIVE=False)
if CONFIG["QUICK"] or os.environ.get("SCEG_QUICK"):
    CONFIG.update(SEEDS=[0], EPOCHS=15, PATIENCE=5)
if os.environ.get("SCEG_SEEDS"):
    CONFIG["SEEDS"] = [int(x) for x in os.environ["SCEG_SEEDS"].split(",")]
print(json.dumps({k: str(v) for k, v in CONFIG.items()}, indent=1))

{
 "START_DATE": "2013-06-01",
 "END_DATE": "2016-01-15",
 "UNIVERSE_MODE": "acl18",
 "NEWS_SOURCE": "mirror",
 "NEWS_CHUNKSIZE": "200000",
 "NEWS_CUTOFF_HOUR_ET": "16",
 "MAX_ARTICLES_PER_TICKER_DAY": "30",
 "UNIVERSE_MIN_COVERAGE": "0.95",
 "USE_LLM_EVENTS": "True",
 "EXTRACT_MODEL": "Qwen/Qwen2.5-1.5B-Instruct",
 "EXTRACT_BATCH_SIZE": "32",
 "EXTRACT_SHARD_SIZE": "1000",
 "EXTRACT_MAX_ARTICLES": "None",
 "EVENT_ENCODER": "bge",
 "EMBED_MODEL": "BAAI/bge-base-en-v1.5",
 "EMBED_BATCH_SIZE": "128",
 "PCA_COMPONENTS": "20",
 "FFILL_LIMIT": "5",
 "NEWS_AGE_CAP": "10",
 "HMM_K_GRID": "[2, 3, 4]",
 "HMM_STICKY": "50.0",
 "HMM_RESTARTS": "5",
 "TOP_K_CORR": "5",
 "DYNOTEARS_P": "2",
 "DYNOTEARS_LAMBDA": "0.05",
 "DYNOTEARS_LAMBDA_A": "0.01",
 "DYNOTEARS_MAX_ITER": "20",
 "DYNOTEARS_W_THRESHOLD": "0.05",
 "MAX_IN_DEGREE": "10",
 "FOLDS": "[('2015-07-31', '2015-09-30', '2015-12-31')]",
 "WINDOW": "5",
 "HIDDEN": "64",
 "NHEAD": "4",
 "N_TRANSFORMER_LAYERS": "2",
 "DROPOUT": "0.2",
 "LR": "0.0

In [3]:
if CONFIG["USE_DRIVE"]:
    from google.colab import drive
    drive.mount("/content/drive")
CACHE = CONFIG["CACHE_DIR"]
os.makedirs(f"{CACHE}/event_shards", exist_ok=True)
os.makedirs(f"{CACHE}/v2", exist_ok=True)
WORK = "/content/fnspid_work" if IN_COLAB else "./fnspid_work"
os.makedirs(WORK, exist_ok=True)
print("cache:", CACHE)

Mounted at /content/drive
cache: /content/drive/MyDrive/MinorProject/acl18_cache


### 0.1 · Frozen confirmation protocol (hashed before any data are loaded)


In [4]:
# ===== 0.1 · FROZEN CONFIRMATION PROTOCOL (written and hashed before any data are loaded) =====
# One source of truth for the cross-market study. §17 reads its hypotheses, arms and decision rules from this dict,
# so the analysis that is run is, by construction, the analysis that was registered. "<BASE>" / "<FINAL>" are
# resolved at run time to the SCEG-R arms that §16 selects on VALIDATION data (rules below), never on test data.
import hashlib as _hl
IS_INDIA = CONFIG.get("DATASET") == "india"
SYNTH = bool(CONFIG.get("SYNTH"))
EXPECTED_SCEGR_SPEC_SHA = "0bfe3098df8c"      # sha1[:12] of the v8 §16 R_SPEC (SCEG-R pre-registration); asserted in §16.0

PROTOCOL_CORE = {
    "study": "Cross-market confirmation of SCEG mechanisms (ACL18 / FNSPID / India-NSE)",
    "frozen_model": "SCEG-R (v8 §16), specification sha " + EXPECTED_SCEGR_SPEC_SHA + "; no architectural change for any dataset",
    "hyperparameters": "ACL18-validation selection transferred unchanged: sceg/gru dropout 0.3, lambda_rank 0.25; mlp dropout 0.3, "
                       "lambda_rank 1.0; window 5, hidden 64, lr 1e-3, wd 1e-4, 60 epochs, patience 10, 16 days/batch; seeds 0-4",
    "validation_selection_rules": {
        "price_set": "SCEG-R rule_price (v8)", "graph_prior": "SCEG-R rule_graph (v8)",
        "early_stopping": "(val AUC - 0.5) + val IC for SCEG-R and XM arms; val AUC for v3 variants (v8)",
        "hmm_K": "BIC over {2,3,4} on training dates; sticky prior 50; filtered posterior",
    },
    "xm_arms": {   # evaluation arms: one mechanism changed relative to the SCEG-R base (stage 1) or its frozen encoder (stage 2)
        "stage1": {"XM | price only": {"use_raw": False, "use_struct": False},
                   "XM | structured only": {"use_raw": False},
                   "XM | regime shuffled": {"placebo": True},
                   "XM | no regime": {"regime": "none"}},
        "stage2": {"XM + lead-lag (lag1, lag2)": {"rels": ["lag1", "lag2"]},
                   "XM + random (matched lead-lag)": {"rels": ["lag1", "lag2"], "random_graph": True},
                   "XM + random (matched sector)": {"rels": ["sector"], "random_sector": True}},
    },
    "primary_metric": "daily cross-sectional Pearson IC of the stock-specific score with the next-day return, over valid "
                      "(and, for India, point-in-time NIFTY-50 member) firm-days; seed-paired; pooled over test folds",
    "secondary_metrics": ["RankIC", "ACC", "MCC", "AUC", "F1", "ACC_market (day-level common part)",
                          "DirAcc_stock (stock-specific direction vs cross-sectional mean)", "IC on large-move subset"],
    "hypotheses": {
        "primary": [   # Holm across these 8 (primary: p_IC); two-sided alpha 0.05
            ["H1", "SCEG-R final beats majority class", "<FINAL>", "majority class"],
            ["H2", "text adds to price", "<BASE>", "XM | price only"],
            ["H3", "structured events add to raw text", "<BASE>", "SCEG-R | raw only"],
            ["H4a", "DYNOTEARS graph adds to no graph", "SCEG-R + DYNOTEARS", "SCEG-R heads retrained (no graph)"],
            ["H4b", "DYNOTEARS beats matched random graph", "SCEG-R + DYNOTEARS", "SCEG-R + random (matched)"],
            ["H4c", "sector graph adds to no graph", "SCEG-R + sector", "SCEG-R heads retrained (no graph)"],
            ["H4d", "sector beats matched random graph", "SCEG-R + sector", "XM + random (matched sector)"],
            ["H5", "real regimes beat shuffled regimes", "<BASE>", "XM | regime shuffled"]],
        "secondary": [  # Holm across these 6
            ["S1", "raw text adds to price", "SCEG-R | raw only", "XM | price only"],
            ["S2", "regime information adds (vs no regime)", "<BASE>", "XM | no regime"],
            ["S3", "lead-lag graph adds to no graph", "XM + lead-lag (lag1, lag2)", "SCEG-R heads retrained (no graph)"],
            ["S4", "lead-lag beats matched random graph", "XM + lead-lag (lag1, lag2)", "XM + random (matched lead-lag)"],
            ["S5", "sector (economic) vs DYNOTEARS (learned) prior", "SCEG-R + sector", "SCEG-R + DYNOTEARS"],
            ["S6", "structured-only vs raw-only text", "XM | structured only", "SCEG-R | raw only"]],
        "exploratory": ["graph gain (H4a, H4c, S3) by filtered HMM regime and by test year, with a Bear-minus-Bull contrast",
                        "market-direction vs stock-specific decomposition of every primary comparison",
                        "v3 claims C1-C18 (incl. C6/C7 regime-conditioned propagation) via the v8 families"],
    },
    "tests": {"paired": "seed-paired (mean over seeds of per-seed differences), pooled test days",
              "IC": "Newey-West t (5 lags) on the daily seed-averaged IC difference",
              "ACC_MCC": "day-block bootstrap, block 5, 2000 resamples, percentile 95% CI",
              "multiplicity": "Holm within each family (primary, secondary); exploratory results are uncorrected and labelled",
              "equivalence": "TOST on daily dIC with NW SE, margin delta_IC = 0.01, Holm within family; 90% CI reported",
              "alpha": 0.05},
    "classification": {"positive": "Holm p_IC < 0.05 and dIC > 0", "negative": "Holm p_IC < 0.05 and dIC < 0",
                       "equivalent": "not positive/negative and Holm TOST p < 0.05 (|dIC| < 0.01)",
                       "inconclusive": "otherwise"},
    "generalization_rule": {
        "GENERALIZES (+)": "positive in India AND positive in >=1 of ACL18/FNSPID AND negative nowhere",
        "India-specific (+)": "positive in India only",
        "does not transfer": "positive in ACL18 or FNSPID, equivalent or inconclusive in India",
        "consistent null": "equivalent in India AND equivalent in FNSPID (ACL18 may be inconclusive: 64 test days)",
        "harmful": "negative in India", "conflicting": "positive and negative in different datasets",
        "inconclusive": "otherwise",
        "placebo flag": "a graph/regime gain vs 'none' that is not also positive vs its matched placebo is reported as "
                        "'fails matched placebo' (gain not attributable to the specific structure)",
    },
    "run_order": "FNSPID v9 through §16 (code shakedown) BEFORE any India test prediction; India test predictions are "
                 "gated in code (REQUIRE_FNSPID_SHAKEDOWN). ACL18 test was inspected for v3 variants: not independent.",
}
PROTOCOL_DATA = {
    "acl18": {"source": "StockNet release (commit 330708b)", "splits": "StockNet train/dev/test by target date",
              "alignment": "strict 16:00 ET (primary); benchmark reported", "universe": "87 StockNet stocks"},
    "fnspid": {"source": "FNSPID prices + NASDAQ-100 news mirror", "splits": "3 walk-forward folds, tests 2021/2022/2023",
               "alignment": "strict 16:00 ET; date-only rows -> next trading day", "universe": "73 NASDAQ-100 firms (coverage rule)"},
    "india": {
        "universe": "point-in-time NIFTY-50: every stock with weight > 0 in the constituent file on any row between "
                    "START_DATE and END_DATE (node set); daily evaluation/loss mask = membership as of the last file row "
                    "dated <= t (non-anticipative)",
        "constituents": f"{CONFIG.get('CONSTITUENTS_REPO')} @ {CONFIG.get('CONSTITUENTS_REV')} (weights.csv, sectors.csv; "
                        "sourced from niftyindices.com reports; CC BY-NC-SA 4.0)",
        "prices": "Yahoo Finance .NS daily OHLCV + Adj Close (candidate symbols per firm, fixed list); fallback when no "
                  "Yahoo series: BSE daily CSV (raw) adjusted for bonus/split ex-dates from BSE corporate actions; "
                  "dividends not adjusted in the fallback (disclosed)",
        "calendar": "NSE sessions from the price panel; drop Muhurat sessions (fixed list), weekend sessions except full "
                    "sessions 2020-02-01 and 2024-01-20, days where < 50% of alive firms have a row, and days whose "
                    "median relative volume (vs each firm's trailing 20-day median, past only) < 0.25 (short sessions); "
                    "firm-days with zero volume are treated as missing",
        "text": "BSE corporate announcements (exchange filings) for the node set; text = NEWSSUB + '. ' + body (MORE, "
                "else HEADLINE), HTML stripped, <= 1500 chars; PDFs not used",
        "timestamp": "DissemDT (BSE dissemination time, IST); if missing, NEWS_DT + 120 min (max observed lag 68 min)",
        "alignment": "strict 15:30 IST: item at/after 15:30 on day d or on a non-trading day -> next trading day",
        "dedupe_cap": "drop exact duplicates of (firm, full text) keeping the earliest; per firm-day keep the earliest k "
                      "items, k = min(10, max(2, ceil(q99 of items per firm-day on first-fold training dates))) (label-free)",
        "prompt": "v3 prompt with the noun 'tweet' -> 'stock-exchange filing' (same precedent as ACL18 -> FNSPID)",
        "splits": "5 walk-forward folds: train from 2016-01-01 expanding; 6-month validation; tests 2020, 2021, 2022, "
                  "2023, 2024; whole window + target inside the split (embargo + purge)",
        "sector_graph": "same-sector edges from sectors.csv (external; no data from any split)",
        "amendment_A1_point_in_time_nodes": (
            "v9.1, before any India data were loaded: at day t the node set is the index members at t. Non-member "
            "firm-days are zeroed in every input window (the whole window of a sample dated t uses members at t), "
            "excluded from same-day cross-sectional price features, and excluded from the returns used to fit the "
            "DYNOTEARS / correlation graphs. Reason: the v9 node set ('ever a member 2015-2024') revealed future index "
            "inclusion to the model through inputs and graphs."),
    },
}
PROTOCOL = {"core": PROTOCOL_CORE, "data": PROTOCOL_DATA[CONFIG["DATASET"]], "dataset": CONFIG["DATASET"],
            "config": {k: str(v) for k, v in CONFIG.items() if k not in ("CACHE_DIR", "USE_DRIVE", "QUICK", "REQUIRE_FNSPID_SHAKEDOWN")}}
_ptxt = json.dumps(PROTOCOL, sort_keys=True, default=str)
CORE_SHA = _hl.sha256(json.dumps(PROTOCOL_CORE, sort_keys=True).encode()).hexdigest()[:16]
PROTOCOL_SHA = _hl.sha256(_ptxt.encode()).hexdigest()[:16]
PROTOCOL_PATH = f"{CACHE}/confirmation_protocol_frozen.json"
_test_preds = glob.glob(f"{CACHE}/v3/preds_*/*.npz")       # any test prediction of this dataset
if os.path.exists(PROTOCOL_PATH) and json.load(open(PROTOCOL_PATH))["sha"] != PROTOCOL_SHA:
    _old = json.load(open(PROTOCOL_PATH))
    # 🔧 [v9.1] an amendment is allowed only while no test prediction exists; the old protocol is archived, not deleted
    assert not _test_preds, (
        "The frozen protocol on disk differs from this notebook's protocol and test predictions already exist "
        f"({len(_test_preds)} files). Changing the protocol now is not allowed. On disk: {_old['sha']} "
        f"(frozen {_old['frozen_at']}); now: {PROTOCOL_SHA}.")
    _arch = PROTOCOL_PATH.replace(".json", f".superseded_{_old['sha']}.json")
    json.dump(_old, open(_arch, "w"), indent=1, default=str)
    json.dump({"sha": PROTOCOL_SHA, "core_sha": CORE_SHA, "frozen_at": time.strftime("%Y-%m-%d %H:%M:%S"),
               "amends": {"sha": _old["sha"], "frozen_at": _old["frozen_at"], "archived_as": os.path.basename(_arch),
                          "test_predictions_at_amendment": 0},
               "protocol": PROTOCOL}, open(PROTOCOL_PATH, "w"), indent=1, default=str)
    print(f"PROTOCOL AMENDED before any test prediction: {_old['sha']} -> {PROTOCOL_SHA} (old copy kept: {_arch})")
elif os.path.exists(PROTOCOL_PATH):
    _old = json.load(open(PROTOCOL_PATH))
    print(f"protocol verified against the frozen copy (frozen {_old['frozen_at']})")
else:
    json.dump({"sha": PROTOCOL_SHA, "core_sha": CORE_SHA, "frozen_at": time.strftime("%Y-%m-%d %H:%M:%S"),
               "protocol": PROTOCOL}, open(PROTOCOL_PATH, "w"), indent=1, default=str)
    print("protocol frozen now (no data has been loaded in this session yet)")
print(f"PROTOCOL sha256[:16] = {PROTOCOL_SHA} | core (identical on every dataset) = {CORE_SHA} | {PROTOCOL_PATH}")

def require_shakedown(stage):
    """Gate before any India TEST prediction: the identical code must first have completed on FNSPID (shakedown), so a
    bug found later cannot force a re-run after India test labels have been seen."""
    if not IS_INDIA or SYNTH or not CONFIG.get("REQUIRE_FNSPID_SHAKEDOWN", True):
        return
    root = os.path.dirname(CACHE.rstrip("/"))
    need = f"{root}/fnspid_cache/v3/scegr_headline_fnspid_bge_v3.csv"
    assert os.path.exists(need), (
        f"[{stage}] India test predictions are locked until FNSPID has run through §16 with this code (file missing: "
        f"{need}). Run SCEG_v9_FNSPID first. Override only by setting CONFIG['REQUIRE_FNSPID_SHAKEDOWN']=False and "
        "disclosing it: a later code fix would then require re-running India after its test labels were seen.")
    print(f"[{stage}] shakedown gate passed: {need}")
if SYNTH:
    print("\n" + "!" * 100 + "\nSYNTHETIC DATA MODE (pipeline check only). Nothing produced in this mode is a result.\n" + "!" * 100)


protocol verified against the frozen copy (frozen 2026-09-29 11:35:39)
PROTOCOL sha256[:16] = 5d21ac233a966618 | core (identical on every dataset) = b92d3e6827d1f8e9 | /content/drive/MyDrive/MinorProject/acl18_cache/confirmation_protocol_frozen.json


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as Fnn
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.use_deterministic_algorithms(True, warn_only=True)
torch.backends.cudnn.benchmark = False

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seed(0)
print("device:", DEVICE)
if IN_COLAB and DEVICE.type != "cuda" and RUN_PHASE != "data":   # 🔧 [v9] data phase runs on CPU   # 🐞 fix: fail early instead of running the LLM for days on CPU
    raise RuntimeError("No GPU. In Colab: Runtime -> Change runtime type -> T4 GPU -> Save, then Runtime -> Run all.")

device: cuda


---
## 1 · Universe and prices

### 1.0 · [INDIA] universe, sectors and prices (point in time; no-op for ACL18 / FNSPID)


In [6]:
# ===== 1.0 · [INDIA] point-in-time NIFTY-50 universe, sectors and prices (no-op for ACL18 / FNSPID) =====
# Every rule in this cell is fixed in the frozen protocol (§0.1). Nothing here reads a return or a label: the
# calendar and data-quality rules use dates, row counts and past volumes only.
import io, html as _html, datetime as _dt, math as _math

# NSE symbol (as in the constituent file) -> (Yahoo candidates in order, BSE scrip code). Scrip codes were checked
# against the BSE master list (api.bseindia.com ListofScripData) on 2026-09-28; the loader re-checks them when BSE is reachable.
FIRM_MAP = {
 "ACC": (["ACC.NS"], "500410"), "ADANIENT": (["ADANIENT.NS"], "512599"), "ADANIPORTS": (["ADANIPORTS.NS"], "532921"),
 "AMBUJACEM": (["AMBUJACEM.NS"], "500425"), "APOLLOHOSP": (["APOLLOHOSP.NS"], "508869"), "ASIANPAINT": (["ASIANPAINT.NS"], "500820"),
 "AUROPHARMA": (["AUROPHARMA.NS"], "524804"), "AXISBANK": (["AXISBANK.NS"], "532215"), "BAJAJAUTO": (["BAJAJ-AUTO.NS"], "532977"),
 "BAJAJFINSV": (["BAJAJFINSV.NS"], "532978"), "BAJFINANCE": (["BAJFINANCE.NS"], "500034"), "BANKBARODA": (["BANKBARODA.NS"], "532134"),
 "BEL": (["BEL.NS"], "500049"), "BHARTIARTL": (["BHARTIARTL.NS"], "532454"), "BHEL": (["BHEL.NS"], "500103"),
 "BOSCHLTD": (["BOSCHLTD.NS"], "500530"), "BPCL": (["BPCL.NS"], "500547"), "BRITANNIA": (["BRITANNIA.NS"], "500825"),
 "CAIRN": (["CAIRN.NS"], "532792"), "CIPLA": (["CIPLA.NS"], "500087"), "COALINDIA": (["COALINDIA.NS"], "533278"),
 "DIVISLAB": (["DIVISLAB.NS"], "532488"), "DRREDDY": (["DRREDDY.NS"], "500124"), "EICHERMOT": (["EICHERMOT.NS"], "505200"),
 "GAIL": (["GAIL.NS"], "532155"), "GRASIM": (["GRASIM.NS"], "500300"), "HCLTECH": (["HCLTECH.NS"], "532281"),
 "HDFC": (["HDFC.NS"], "500010"), "HDFCBANK": (["HDFCBANK.NS"], "500180"), "HDFCLIFE": (["HDFCLIFE.NS"], "540777"),
 "HEROMOTOCO": (["HEROMOTOCO.NS"], "500182"), "HINDALCO": (["HINDALCO.NS"], "500440"), "HINDPETRO": (["HINDPETRO.NS"], "500104"),
 "HINDUNILVR": (["HINDUNILVR.NS"], "500696"), "IBULHSGFIN": (["SAMMAANCAP.NS", "IBULHSGFIN.NS"], "535789"),
 "ICICIBANK": (["ICICIBANK.NS"], "532174"), "IDEA": (["IDEA.NS"], "532822"), "INDUSINDBK": (["INDUSINDBK.NS"], "532187"),
 "INFRATEL": (["INDUSTOWER.NS", "INFRATEL.NS"], "534816"), "INFY": (["INFY.NS"], "500209"), "IOC": (["IOC.NS"], "530965"),
 "ITC": (["ITC.NS"], "500875"), "JSWSTEEL": (["JSWSTEEL.NS"], "500228"), "KOTAKBANK": (["KOTAKBANK.NS"], "500247"),
 "LT": (["LT.NS"], "500510"), "LTIM": (["LTIM.NS", "LTM.NS"], "540005"), "LUPIN": (["LUPIN.NS"], "500257"),
 "M&M": (["M&M.NS"], "500520"), "MARUTI": (["MARUTI.NS"], "532500"), "NESTLEIND": (["NESTLEIND.NS"], "500790"),
 "NMDC": (["NMDC.NS"], "526371"),   # 🔧 [v9.1] in the constituent file; BSE code [AUTHOR VERIFICATION REQUIRED]
 "NTPC": (["NTPC.NS"], "532555"), "ONGC": (["ONGC.NS"], "500312"), "PNB": (["PNB.NS"], "532461"),
 "POWERGRID": (["POWERGRID.NS"], "532898"), "RELIANCE": (["RELIANCE.NS"], "500325"), "SBILIFE": (["SBILIFE.NS"], "540719"),
 "SBIN": (["SBIN.NS"], "500112"), "SHREECEM": (["SHREECEM.NS"], "500387"), "SHRIRAMFIN": (["SHRIRAMFIN.NS", "SRTRANSFIN.NS"], "511218"),
 "SUNPHARMA": (["SUNPHARMA.NS"], "524715"), "TATACONSUM": (["TATACONSUM.NS"], "500800"),
 "TATAMOTORS": (["TMPV.NS", "TATAMOTORS.NS"], "500570"), "TATAPOWER": (["TATAPOWER.NS"], "500400"),
 "TATASTEEL": (["TATASTEEL.NS"], "500470"), "TCS": (["TCS.NS"], "532540"), "TECHM": (["TECHM.NS"], "532755"),
 "TITAN": (["TITAN.NS"], "500114"), "TRENT": (["TRENT.NS"], "500251"), "ULTRACEMCO": (["ULTRACEMCO.NS"], "532538"),
 "UPL": (["UPL.NS"], "512070"), "VEDL": (["VEDL.NS"], "500295"), "WIPRO": (["WIPRO.NS"], "507685"),
 "YESBANK": (["YESBANK.NS"], "532648"), "ZEEL": (["ZEEL.NS"], "505537"),
}
# Muhurat (Diwali one-hour) sessions and full weekend sessions (market-structure rules, fixed in the protocol).
MUHURAT = ["2015-11-11", "2016-10-30", "2017-10-19", "2018-11-07", "2019-10-27", "2020-11-14", "2021-11-04",
           "2022-10-24", "2023-11-12", "2024-11-01"]
FULL_WEEKEND_SESSIONS = ["2020-02-01", "2024-01-20"]           # Union Budget 2020 (Sat); full session Sat 20 Jan 2024

RAW = f"{CACHE}/india_raw"; os.makedirs(RAW, exist_ok=True)
DATA_MANIFEST = {}                                              # everything written here is hashed into the run manifest
def _sha_file(p):
    h = _hl.sha256()
    with open(p, "rb") as fh:
        for b in iter(lambda: fh.read(1 << 20), b""): h.update(b)
    return h.hexdigest()[:16]

# ---------- BSE access: Colab first; if BSE refuses Colab's IP, use the browser harvester file (see §2.0) ----------
_BSE_HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) "
                              "Chrome/124.0 Safari/537.36",
                "Accept": "application/json, text/plain, */*", "Referer": "https://www.bseindia.com/",
                "Origin": "https://www.bseindia.com"}
_BSE = {"session": None, "ok": None}
HARVEST_PATH = f"{RAW}/bse_harvest.json"                       # written by the browser harvester, if used
_HARVEST = None
def bse_harvest():
    """All browser-harvest files (bse_harvest.json + any top-ups bse_harvest_*.json), merged. 🔧 [v9.1]"""
    global _HARVEST
    files = sorted(glob.glob(f"{RAW}/bse_harvest*.json"))
    if _HARVEST is None and files:
        _HARVEST = {"ann": [], "prices": {}, "ca": {}, "skipped": []}
        for p in files:
            h = json.load(open(p)); DATA_MANIFEST[os.path.basename(p)] = _sha_file(p)
            _HARVEST["ann"] += h.get("ann", []); _HARVEST["prices"].update(h.get("prices", {}) or {})
            _HARVEST["ca"].update(h.get("ca", {}) or {}); _HARVEST["skipped"] += h.get("skipped", []) or []
        print(f"using {len(files)} browser-harvest file(s): {len(_HARVEST['ann']):,} announcements, "
              f"{len(_HARVEST['prices'])} BSE price series")
    return _HARVEST

def bse_get(url, params, kind="json", tries=4):
    import requests
    if _BSE["ok"] is False: return None
    if _BSE["session"] is None:
        _BSE["session"] = requests.Session(); _BSE["session"].headers.update(_BSE_HEADERS)
    for k in range(tries):
        try:
            r = _BSE["session"].get(url, params=params, timeout=30)
            if r.status_code == 200 and r.content:
                _BSE["ok"] = True
                return r.json() if kind == "json" else r.text
        except Exception as e:
            err = e
        time.sleep(1.5 * (k + 1))
    if _BSE["ok"] is None:
        _BSE["ok"] = False
        print("BSE API not reachable from this runtime. Use the browser harvester (§2.0) and put bse_harvest.json in", RAW)
    return None

def bse_prices(code, start, end):
    """Raw daily OHLCV from BSE for one scrip, adjusted for bonus/split ex-dates (dividends not adjusted)."""
    H = bse_harvest()
    if H is not None and code in H.get("prices", {}):
        txts = H["prices"][code]
    else:
        txts = []
        for y in range(pd.Timestamp(start).year, pd.Timestamp(end).year + 1):
            t = bse_get("https://api.bseindia.com/BseIndiaAPI/api/StockPriceCSVDownload/w",
                        dict(pageType=0, rbType="D", Scode=code, FDates=f"01/01/{y}", TDates=f"31/12/{y}", Seg="C"), kind="text")
            if t: txts.append(t)
    frames = []
    for t in txts:
        try:
            d = pd.read_csv(io.StringIO(t))
            if len(d): frames.append(d)
        except Exception:
            pass
    if not frames: return None
    d = pd.concat(frames).drop_duplicates()
    d.columns = [c.strip() for c in d.columns]
    d = pd.DataFrame({"date": pd.to_datetime(d["Date"], format="%d-%B-%Y", errors="coerce"), "open": d["Open Price"],
                      "high": d["High Price"], "low": d["Low Price"], "close": d["Close Price"], "volume": d["No.of Shares"]})
    d = d.dropna(subset=["date"]).sort_values("date").drop_duplicates("date")
    fac = np.ones(len(d))
    for ex, f in bse_corporate_factors(code, start, end):
        fac[(d["date"] < ex).values] *= f                          # prices before the ex-date are scaled down
    for c in ("open", "high", "low", "close"):
        d[c] = pd.to_numeric(d[c], errors="coerce")
    d["adj_close"] = d["close"] * fac
    for c in ("open", "high", "low", "close"): d[c] = d[c] * fac  # consistent intraday ratios after adjustment
    d["adj_close"] = d["close"]
    return d

def bse_corporate_factors(code, start, end):
    H = bse_harvest()
    rows = H.get("ca", {}).get(code) if H is not None else None
    if rows is None:
        rows = bse_get("https://api.bseindia.com/BseIndiaAPI/api/DefaultData/w",
                       dict(Fdate=pd.Timestamp(start).strftime("%Y%m%d"), Purposecode="", TDate=pd.Timestamp(end).strftime("%Y%m%d"),
                            ddlcategorys="E", ddlindustrys="", scripcode=code, segment=0, strSearch="S")) or []
    out = []
    for r in rows:
        p = str(r.get("Purpose", "")); ex = pd.to_datetime(r.get("Ex_date"), format="%d %b %Y", errors="coerce")
        if pd.isna(ex): continue
        m = re.search(r"Bonus issue\s*(\d+)\s*:\s*(\d+)", p, re.I)
        if m:
            a, b = float(m.group(1)), float(m.group(2)); out.append((ex, b / (a + b))); continue
        m = re.search(r"Split.*?(\d+(?:\.\d+)?)\D+?(\d+(?:\.\d+)?)", p, re.I)
        if m and float(m.group(1)) > 0:
            out.append((ex, float(m.group(2)) / float(m.group(1))))
    return out

# 🔧 [v9.1] Yahoo loader. The edited copy passed show_errors=False (removed from yfinance), so every call raised a
# TypeError that was swallowed. Here: no removed arguments, flat columns, a Ticker.history fallback, back-off for rate
# limits, every failure logged with its exception, and a per-ticker cache so a partial download is never lost.
YAHOO_LOG = {}
_PX_COLS = ["date", "open", "high", "low", "close", "adj_close", "volume"]
def _yf_frame(d):
    if d is None or len(d) == 0: return None
    d = d.copy()
    if isinstance(d.columns, pd.MultiIndex): d.columns = d.columns.get_level_values(0)
    d = d.reset_index()
    d = d.rename(columns={"Date": "date", "Datetime": "date", "Open": "open", "High": "high", "Low": "low", "Close": "close",
                          "Adj Close": "adj_close", "Volume": "volume"})
    if not set(_PX_COLS) <= set(d.columns): return None       # never substitute unadjusted closes silently
    dt = pd.to_datetime(d["date"])
    if getattr(dt.dt, "tz", None) is not None: dt = dt.dt.tz_localize(None)   # keep the exchange-local calendar date
    d["date"] = dt.dt.normalize()
    return d[_PX_COLS].dropna(subset=["close", "adj_close"])

def yahoo_prices(cands, start, end):
    import yfinance as yf, inspect
    end_x = (pd.Timestamp(end) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")      # yfinance's end date is exclusive
    kw = dict(start=start, end=end_x, auto_adjust=False, actions=False, progress=False, threads=False)
    if "multi_level_index" in inspect.signature(yf.download).parameters: kw["multi_level_index"] = False
    os.makedirs(f"{RAW}/yahoo", exist_ok=True)
    for sym in cands:
        cp = f"{RAW}/yahoo/{sym.replace('&', '_')}.parquet"
        if os.path.exists(cp):
            return pd.read_parquet(cp), sym
        d, errs, empty = None, [], 0
        for k in range(4):
            try:
                d = _yf_frame(yf.download(sym, **kw))
                if d is None:
                    d = _yf_frame(yf.Ticker(sym).history(start=start, end=end_x, auto_adjust=False, actions=False))
                if d is not None and len(d): break
                d = None; empty += 1; errs.append("empty response")
                if empty >= 2: break                               # delisted / unknown symbol: do not hammer Yahoo
            except Exception as e:
                errs.append(f"{type(e).__name__}: {str(e)[:150]}")
            time.sleep(min(60, 5 * 2 ** k))                        # rate limits need long pauses
        YAHOO_LOG[sym] = "ok" if d is not None else (errs[-1] if errs else "no data")
        if d is None or len(d) < CONFIG["MIN_PRICE_ROWS"]: continue
        d.to_parquet(cp, index=False)
        return d, sym
    return None, None

def manual_prices(sym):
    """Optional user-supplied daily prices: india_raw/manual_prices/<SYMBOL>.csv with columns date, open, high, low,
    close, volume and, if available, adj_close (else close, disclosed as unadjusted). Used only when Yahoo has no
    series (delisted firms such as HDFC Ltd or Cairn India)."""
    p = f"{RAW}/manual_prices/{sym}.csv"
    if not os.path.exists(p): return None, None
    d = pd.read_csv(p); d.columns = [str(c).strip().lower().replace(" ", "_") for c in d.columns]
    d = d.rename(columns={"adj_close_price": "adj_close", "close_price": "close", "open_price": "open", "high_price": "high",
                          "low_price": "low", "no._of_shares": "volume", "total_traded_quantity": "volume"})
    tag = "manual"
    if "adj_close" not in d: d["adj_close"] = d["close"]; tag = "manual_unadjusted"
    d["date"] = pd.to_datetime(d["date"], dayfirst=True, errors="coerce").dt.normalize()
    for c in _PX_COLS[1:]: d[c] = pd.to_numeric(d[c].astype(str).str.replace(",", ""), errors="coerce")
    DATA_MANIFEST[f"manual_prices/{sym}.csv"] = _sha_file(p)
    return d[_PX_COLS].dropna(subset=["date", "close"]).sort_values("date"), tag

def load_constituents():
    if SYNTH: return synth_constituents()
    from huggingface_hub import hf_hub_download
    out = {}
    for f in ("weights.csv", "sectors.csv"):
        p = hf_hub_download(CONFIG["CONSTITUENTS_REPO"], f, repo_type="dataset", revision=CONFIG["CONSTITUENTS_REV"],
                            local_dir=f"{RAW}/constituents")
        DATA_MANIFEST[f] = _sha_file(p); out[f] = pd.read_csv(p)
    return out["weights.csv"], out["sectors.csv"]

def membership_tensor(days, symbols):
    """MEMBER[t, i] = stock i has weight > 0 in the last constituent-file row dated <= day t (point in time)."""
    Wt = CONST_W.set_index("DATE").sort_index()
    pos = np.searchsorted(Wt.index.values, np.asarray(days, dtype="datetime64[ns]"), side="right") - 1
    assert (pos >= 0).all(), "a trading day precedes the first constituent row"
    M = np.zeros((len(days), len(symbols)), bool)
    for j, s in enumerate(symbols):
        if s in Wt.columns: M[:, j] = Wt[s].values[pos] > 0
    return M

def clean_calendar(pr):
    """Market-structure rules (label-free): Muhurat / weekend special sessions, thin-coverage days, short sessions,
    zero-volume firm-days. Returns the cleaned panel and an audit dict."""
    audit = {}
    pr = pr.copy(); pr["date"] = pd.to_datetime(pr["date"]).dt.normalize()
    z = (pr["volume"].fillna(0) <= 0); audit["zero_volume_firm_days_dropped"] = int(z.sum()); pr = pr[~z]
    d = pd.DatetimeIndex(sorted(pr["date"].unique()))
    drop = set(pd.to_datetime(MUHURAT)) & set(d)
    wk = [x for x in d if x.weekday() >= 5 and x not in set(pd.to_datetime(FULL_WEEKEND_SESSIONS))]
    drop |= set(wk)
    life = pr.groupby("Stock_symbol")["date"].agg(["min", "max"])
    cnt = pr.groupby("date")["Stock_symbol"].nunique()
    alive = pd.Series([int(((life["min"] <= x) & (life["max"] >= x)).sum()) for x in cnt.index], index=cnt.index)
    thin = set(cnt.index[(cnt / alive.clip(lower=1)) < CONFIG["MIN_DAY_COVERAGE"]])
    pr = pr.sort_values(["Stock_symbol", "date"])
    lv = np.log1p(pr["volume"].astype(float))
    base = lv.groupby(pr["Stock_symbol"]).transform(lambda x: x.shift(1).rolling(20, min_periods=10).median())
    rel = np.exp(lv - base)                                        # volume relative to the firm's own past 20 days
    short = rel.groupby(pr["date"]).median()
    short = set(short.index[short < CONFIG["SHORT_SESSION_REL_VOL"]])
    audit.update(muhurat_dropped=sorted(str(x.date()) for x in set(pd.to_datetime(MUHURAT)) & set(d)),
                 weekend_dropped=sorted(str(x.date()) for x in wk), thin_days_dropped=sorted(str(x.date()) for x in thin),
                 short_sessions_detected=sorted(str(x.date()) for x in short))
    drop |= thin | short
    audit["days_dropped_total"] = len(drop)
    return pr[~pr["date"].isin(drop)].reset_index(drop=True), audit

# ---------------- synthetic data (pipeline check only; never reportable) ----------------
def synth_constituents():
    rng = np.random.default_rng(7); tick = list(FIRM_MAP)[:24]
    dates = pd.date_range("2008-01-31", "2024-12-31", freq="ME")
    W = pd.DataFrame(0.0, index=dates, columns=tick)
    members = set(tick[:20])
    for d in dates:
        if d.month in (3, 9) and d.year >= 2016:                 # semi-annual reshuffle: swap one stock
            out = sorted(members)[rng.integers(len(members))]; inn = sorted(set(tick) - members)[rng.integers(4)]
            members = (members - {out}) | {inn}
        W.loc[d, sorted(members)] = 100.0 / len(members)
    W = W.reset_index().rename(columns={"index": "DATE"}); W["DATE"] = W["DATE"].dt.strftime("%Y-%m-%d")
    secs = ["FINANCIAL SERVICES", "INFORMATION TECHNOLOGY", "ENERGY", "FMCG", "AUTOMOTIVE", "METALS"]
    S = pd.DataFrame({"STOCK": tick, "SECTOR": [secs[i % len(secs)] for i in range(len(tick))]})
    return W, S

def synth_days():
    days = pd.bdate_range(CONFIG["START_DATE"], CONFIG["END_DATE"])
    days = days[~days.isin(pd.to_datetime(["2016-01-26", "2017-08-15", "2020-10-02", "2023-08-15"]))]
    return days.append(pd.DatetimeIndex(pd.to_datetime(["2020-11-14"]))).sort_values()   # a Saturday Muhurat row

def synth_prices(symbols, sectors):
    rng = np.random.default_rng(11)
    days = synth_days()
    secs = sorted(set(sectors.values())); T_ = len(days)
    vol = np.where((days >= "2020-02-20") & (days <= "2020-05-31"), 0.03, 0.012)
    mkt = rng.normal(0.0004, 1, T_) * vol; sf = {s: rng.normal(0, 0.006, T_) for s in secs}
    rows = []
    for s in symbols:
        r = 0.9 * mkt + sf[sectors[s]] + rng.normal(0, 0.012, T_)
        r = np.clip(r + SYNTH_SIGNAL.get(s, np.zeros(T_ + 400))[:T_], -0.2, 0.2)
        c = 100 * np.exp(np.cumsum(r)); o = c * np.exp(rng.normal(0, 0.004, T_))
        v = rng.lognormal(12, 0.3, T_); v[days == pd.Timestamp("2020-11-14")] *= 0.05
        rows.append(pd.DataFrame({"date": days, "open": o, "high": np.maximum(o, c) * 1.005, "low": np.minimum(o, c) * 0.995,
                                  "close": c, "adj_close": c, "volume": v, "Stock_symbol": s}))
    return pd.concat(rows, ignore_index=True)

SYNTH_SIGNAL = {}
def synth_announcements(symbols):
    """BSE-format rows; a 'positive'/'negative' filing moves the firm's NEXT-day return by +-0.4 % (a planted signal)."""
    rng = np.random.default_rng(5); recs = []
    days = synth_days()
    subj = [("Outcome of Board Meeting", "Result"), ("Announcement under Regulation 30 (LODR)-Press Release", "Company Update"),
            ("Trading Window-XBRL", "Others"), ("Newspaper Publication", "Company Update"), ("Credit Rating", "Company Update")]
    for s in symbols:
        code = FIRM_MAP[s][1]; sig = np.zeros(len(days) + 400)
        for k in np.where(rng.random(len(days)) < 0.25)[0]:
            ts = days[k] + pd.Timedelta(minutes=int(rng.integers(8 * 60, 22 * 60)))
            pol = rng.choice(["positive", "negative", "neutral"], p=[0.3, 0.3, 0.4]); sb, cat = subj[rng.integers(len(subj))]
            body = {"positive": "The company reports strong growth and wins a large order",
                    "negative": "The company reports a loss and a regulatory penalty",
                    "neutral": "The company informs about a scheduled meeting"}[pol] + f" (ref {rng.integers(10**6)})"
            recs.append(dict(SCRIP_CD=int(code), NEWSID=f"{s}-{k}", NEWSSUB=sb, HEADLINE=body[:80], MORE=body,
                             CATEGORYNAME=cat, SUBCATNAME="General", NEWS_DT=ts.strftime("%Y-%m-%dT%H:%M:%S"),
                             DissemDT=(ts + pd.Timedelta(seconds=5)).strftime("%Y-%m-%dT%H:%M:%S") if ts.year > 2015 else None,
                             DT_TM=ts.strftime("%Y-%m-%dT%H:%M:%S")))
            after = ts.hour * 60 + ts.minute >= CONFIG["CLOSE_MINUTES"]
            j = k + 1 + int(after)                                   # first return that can react to the filing
            if pol != "neutral" and j < len(sig): sig[j] += 0.004 if pol == "positive" else -0.004
        SYNTH_SIGNAL[s] = sig
    return pd.DataFrame(recs)

# ---------------- browser harvester (fallback when BSE refuses Colab's IP) ----------------
BSE_HARVESTER_JS = r"""
// ==== SCEG India: BSE harvester v2. Use ONLY if the notebook says the BSE API is unreachable from Colab. ====
// 1) Open https://www.bseindia.com/corporates/ann.html in Chrome/Edge  2) F12 -> Console  3) paste this, Enter.
// It fetches the announcements (and, where BSE allows it, fallback prices / corporate actions) and saves
// bse_harvest.json (~30-60 min). Upload that file to Drive: MinorProject/india_cache/india_raw/bse_harvest.json
// v2: the price step can no longer crash the run. BSE's price-CSV endpoint sends two Content-Disposition headers,
// which Chrome refuses (net::ERR_RESPONSE_HEADERS_MULTIPLE_CONTENT_DISPOSITION); those prices are skipped and the
// notebook then drops the affected firm(s) with a "NO PRICE DATA" message. If anything fails, type saveBSE() in
// the console to download whatever has been collected so far.
(async () => {
  const SCRIPS = __SCRIPS__, PRICE_SCRIPS = __PRICE_SCRIPS__, Y0 = __Y0__, Y1 = __Y1__;
  const sleep = ms => new Promise(r => setTimeout(r, ms));
  const get = async (u, kind, tries = 6) => { for (let k = 0; k < tries; k++) { try { const r = await fetch(u);
      if (r.ok) return kind === "text" ? await r.text() : await r.json(); } catch (e) {} if (k + 1 < tries) await sleep(2000 * (k + 1)); }
    throw new Error("failed: " + u); };
  const ann = [], prices = {}, ca = {}, skipped = [], keep = ["SCRIP_CD","NEWSID","NEWSSUB","HEADLINE","MORE","CATEGORYNAME",
    "SUBCATNAME","NEWS_DT","DissemDT","DT_TM"];
  window.saveBSE = () => { const blob = new Blob([JSON.stringify({ann, prices, ca, skipped, created: new Date().toISOString()})],
      {type: "application/json"}); const a = document.createElement("a"); a.href = URL.createObjectURL(blob);
    a.download = "bse_harvest.json"; a.click(); console.log("saved bse_harvest.json:", ann.length, "announcements,",
      Object.keys(prices).length, "price series, skipped:", skipped.join(", ") || "none"); };
  try {
    for (const code of SCRIPS) { for (let y = Y0; y <= Y1; y++) { let got = 0, total = null;
        for (let p = 1; p < 400; p++) {
          const j = await get(`https://api.bseindia.com/BseIndiaAPI/api/AnnSubCategoryGetData/w?pageno=${p}&strCat=-1&strPrevDate=${y}0101&strScrip=${code}&strSearch=P&strToDate=${y}1231&strType=C&subcategory=-1`);
          const t = j.Table || []; total = ((j.Table1 || [{}])[0] || {}).ROWCNT ?? total;
          t.forEach(x => { const o = {}; keep.forEach(k => o[k] = x[k] ?? null); o.YEAR = y; o.ROWCNT = total; ann.push(o); });
          got += t.length; if (t.length < 50 || (total && got >= total)) break; await sleep(200); }
        await sleep(200); }
      console.log("scrip", code, "announcements so far", ann.length); }
    console.log("announcements complete:", ann.length, "- now trying the optional price series");
    for (const code of PRICE_SCRIPS) {           // optional: never fatal, one attempt per year, stop at the first refusal
      const txts = [];
      for (let y = Y0; y <= Y1; y++) {
        try { txts.push(await get(`https://api.bseindia.com/BseIndiaAPI/api/StockPriceCSVDownload/w?pageType=0&rbType=D&Scode=${code}&FDates=01/01/${y}&TDates=31/12/${y}&Seg=C`, "text", 1)); }
        catch (e) { break; } }
      if (txts.length === Y1 - Y0 + 1) prices[code] = txts; else { skipped.push(code); console.warn("price series skipped for", code, "(BSE response refused by the browser)"); continue; }
      try { ca[code] = await get(`https://api.bseindia.com/BseIndiaAPI/api/DefaultData/w?Fdate=${Y0}0101&Purposecode=&TDate=${Y1}1231&ddlcategorys=E&ddlindustrys=&scripcode=${code}&segment=0&strSearch=S`, "json", 3); }
      catch (e) { delete prices[code]; skipped.push(code); console.warn("corporate actions failed for", code, "- price series dropped"); } }
  } catch (e) { console.error("stopped early:", e.message, "- saving what was collected; re-run to complete"); }
  window.saveBSE();
})();
"""
def harvester_script(price_codes=None, codes=None, outfile="bse_harvest.json"):
    """Browser harvester for all node-set announcements plus BSE prices/corporate actions of the fallback scrips.
    🔧 [v9.1] codes / outfile: a top-up harvester for only the scrips missing from the existing harvest."""
    top_up = codes is not None
    codes = codes if top_up else [FIRM_MAP[s][1] for s in INDIA_UNIVERSE]
    fallback = set(price_codes or []) | {FIRM_MAP[s][1] for s in INDIA_UNIVERSE if s in ("HDFC", "CAIRN")}
    if "INDIA_PRICES_RAW" in globals():
        fallback |= {src.split(":")[1] for src in INDIA_PRICES_RAW["source"].unique() if src.startswith("bse:")}
    fallback = sorted(fallback) if not top_up else sorted(set(price_codes or []))
    return (BSE_HARVESTER_JS.replace("__SCRIPS__", json.dumps(codes)).replace("__PRICE_SCRIPS__", json.dumps(fallback))
            .replace('a.download = "bse_harvest.json"', f'a.download = "{outfile}"')
            .replace("__Y0__", str(pd.Timestamp(CONFIG["START_DATE"]).year)).replace("__Y1__", str(pd.Timestamp(CONFIG["END_DATE"]).year)))

# ---------------- run (India only) ----------------
if IS_INDIA:
    CONST_W, CONST_S = load_constituents()
    CONST_W["DATE"] = pd.to_datetime(CONST_W["DATE"])
    _Wt = CONST_W.set_index("DATE").sort_index()
    _first = _Wt.index[_Wt.index <= pd.Timestamp(CONFIG["START_DATE"])]
    _rows = _Wt.loc[(_Wt.index >= (_first[-1] if len(_first) else _Wt.index[0])) & (_Wt.index <= pd.Timestamp(CONFIG["END_DATE"]))]
    INDIA_UNIVERSE = sorted(c for c in _rows.columns if (_rows[c] > 0).any())
    INDIA_SECTOR = {str(a).strip(): str(b).strip().title() for a, b in zip(CONST_S["STOCK"], CONST_S["SECTOR"])}
    missing_map = [s for s in INDIA_UNIVERSE if s not in FIRM_MAP]
    assert not missing_map, f"no ticker mapping for {missing_map}: extend FIRM_MAP (data-format issue, not a model change)"
    print(f"point-in-time NIFTY-50 node set: {len(INDIA_UNIVERSE)} stocks ever in the index {CONFIG['START_DATE']}..{CONFIG['END_DATE']}")
    PRICE_PATH = f"{RAW}/prices_{'synth' if SYNTH else 'real'}.parquet"
    if os.path.exists(PRICE_PATH):
        INDIA_PRICES_RAW = pd.read_parquet(PRICE_PATH); print("loaded cached prices", INDIA_PRICES_RAW.shape)
    else:
        if SYNTH:
            synth_announcements(INDIA_UNIVERSE)                    # plants the signal used by synth_prices
            INDIA_PRICES_RAW = synth_prices(INDIA_UNIVERSE, INDIA_SECTOR).assign(source="synthetic")
        else:
            frames, blocked = [], []
            for s in INDIA_UNIVERSE:
                cands, code = FIRM_MAP[s]
                d, used = yahoo_prices(cands, CONFIG["START_DATE"], CONFIG["END_DATE"])
                src = f"yahoo:{used}"
                if d is None:                                      # 🔧 [v9.1] optional manual file (delisted firms)
                    d, tag = manual_prices(s); src = f"{tag}:{s}"
                if d is None:
                    d = bse_prices(code, CONFIG["START_DATE"], CONFIG["END_DATE"]); src = f"bse:{code}"
                    if d is None and _BSE["ok"] is False and bse_harvest() is None: blocked.append(code)
                if d is None or len(d) < 250:
                    print(f"  NO PRICE DATA: {s} (candidates {cands}, BSE {code})", flush=True); continue
                frames.append(d.assign(Stock_symbol=s, source=src)); print(f"  {s:11s} {src:22s} {len(d):5d} rows", flush=True)
            if blocked:   # do not cache an incomplete panel: get the BSE series through the browser harvester first
                open(f"{RAW}/bse_harvester.js", "w").write(harvester_script(blocked))
                raise RuntimeError(f"No Yahoo series and BSE unreachable for scrips {blocked}. Run the browser harvester "
                                   f"saved at {RAW}/bse_harvester.js (instructions inside), upload bse_harvest.json to "
                                   f"{RAW}, then Run all again.")
            # 🔧 [v9.1] never cache an incomplete panel: a Yahoo outage / rate limit must not become survivorship bias
            if len(frames) < 0.8 * len(INDIA_UNIVERSE):
                print("Yahoo log (last error per symbol):", json.dumps({k: v for k, v in YAHOO_LOG.items() if v != "ok"}, indent=1))
                raise RuntimeError(f"Only {len(frames)} of {len(INDIA_UNIVERSE)} firms have prices. Nothing was cached "
                                   "(successful tickers are kept in india_raw/yahoo/). Most likely Yahoo rate-limited this "
                                   "runtime: wait 15-30 min (or Runtime -> Disconnect and delete runtime) and run again.")
            INDIA_PRICES_RAW = pd.concat(frames, ignore_index=True)
        INDIA_PRICES_RAW.to_parquet(PRICE_PATH, index=False)
    DATA_MANIFEST[os.path.basename(PRICE_PATH)] = _sha_file(PRICE_PATH)
    INDIA_PRICES, CAL_AUDIT = clean_calendar(INDIA_PRICES_RAW)
    INDIA_PRICES = INDIA_PRICES[(INDIA_PRICES["date"] >= CONFIG["START_DATE"]) & (INDIA_PRICES["date"] <= CONFIG["END_DATE"])]
    print("calendar audit:", json.dumps({k: (v if not isinstance(v, list) or len(v) <= 12 else v[:12] + [f"... +{len(v) - 12}"])
                                         for k, v in CAL_AUDIT.items()}, indent=1))
    print("price sources:", INDIA_PRICES_RAW.groupby("Stock_symbol")["source"].first().str.split(":").str[0].value_counts().to_dict())
    _nop = sorted(set(INDIA_UNIVERSE) - set(INDIA_PRICES_RAW["Stock_symbol"]))
    print("node-set firms WITHOUT prices (residual survivorship bias; see §1.4):", _nop or "none")


In [7]:
NASDAQ_100 = [
    "AAPL","ABNB","ADBE","ADI","ADP","ADSK","AEP","AMAT","AMD","AMGN","AMZN","ANSS","ASML","AVGO",
    "AZN","BIIB","BKNG","BKR","CDNS","CDW","CEG","CHTR","CMCSA","COST","CPRT","CRWD","CSCO","CSGP",
    "CSX","CTAS","CTSH","DDOG","DLTR","DXCM","EA","EBAY","ENPH","EXC","FANG","FAST","FTNT","GEHC",
    "GFS","GILD","GOOG","GOOGL","HON","IDXX","ILMN","INTC","INTU","ISRG","JD","KDP","KHC","KLAC",
    "LCID","LRCX","LULU","MAR","MCHP","MDLZ","MELI","META","MNST","MRNA","MRVL","MSFT","MU","NFLX",
    "NVDA","NXPI","ODFL","ON","ORLY","PANW","PAYX","PCAR","PDD","PEP","PYPL","QCOM","REGN","ROST",
    "SBUX","SGEN","SIRI","SNPS","TEAM","TMUS","TSLA","TTD","TTWO","TXN","VRSK","VRTX","WBA","WBD",
    "WDAY","XEL","ZM","ZS",
]
def sp500_tickers():
    """Present-day S&P 500 list (survivorship bias - disclose). Needs internet (Colab)."""
    try:
        t = pd.read_html("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies")[0]
        return sorted(set(t["Symbol"].astype(str).str.replace(".", "-", regex=False)))
    except Exception as e:
        print("could not fetch the S&P 500 list, falling back to NASDAQ-100:", repr(e)); return None

# 🔧 [ACL18] StockNet target stocks (stocknet-code src/config.yml). WHY: benchmark universe.
ACL18_STOCKS = ['XOM','RDS-B','PTR','CVX','TOT','BP','BHP','SNP','SLB','BBL',
                'AAPL','PG','BUD','KO','PM','TM','PEP','UN','UL','MO',
                'JNJ','PFE','NVS','UNH','MRK','AMGN','MDT','ABBV','SNY','CELG',
                'AMZN','BABA','WMT','CMCSA','HD','DIS','MCD','CHTR','UPS','PCLN',
                'NEE','DUK','D','SO','NGG','AEP','PCG','EXC','SRE','PPL',
                'IEP','HRG','CODI','REX','SPLP','PICO','AGFS','GMRE',
                'BCH','BSAC','BRK-A','JPM','WFC','BAC','V','C','HSBC','MA',
                'GE','MMM','BA','HON','UTX','LMT','CAT','GD','DHR','ABB',
                'GOOG','MSFT','FB','T','CHL','ORCL','TSM','VZ','INTC','CSCO']
if CONFIG["UNIVERSE_MODE"] == "nifty50_pit":      # 🔧 [INDIA] point-in-time NIFTY-50 node set (§1.0)
    UNIVERSE = list(INDIA_UNIVERSE)
elif CONFIG["UNIVERSE_MODE"] == "acl18":
    UNIVERSE = sorted(ACL18_STOCKS)
elif CONFIG["UNIVERSE_MODE"] == "sp500":
    # A broader cross-section raises statistical power (daily IC noise ~ 1/sqrt(N)) and adds cross-sector
    # dependencies. COST: ~5-10x more articles to extract (resumable shards: run the extraction cell over
    # several sessions) and a slower DYNOTEARS fit (N^2 parameters). Use NEWS_SOURCE="all_external".
    UNIVERSE = sp500_tickers() or sorted(set(NASDAQ_100))
    if CONFIG["NEWS_SOURCE"] == "mirror":
        CONFIG["NEWS_SOURCE"] = "all_external"
else:
    UNIVERSE = sorted(set(NASDAQ_100))
print(CONFIG["UNIVERSE_MODE"], "universe requested:", len(UNIVERSE), "tickers")

PRICE_CACHE = (f"{CACHE}/prices_{CONFIG['START_DATE'][:4]}_{CONFIG['END_DATE'][:4]}.parquet"
               if CONFIG["UNIVERSE_MODE"] == "nasdaq100" else
               f"{CACHE}/prices_{CONFIG['UNIVERSE_MODE']}_{CONFIG['START_DATE'][:4]}_{CONFIG['END_DATE'][:4]}.parquet")

def _norm_price_cols(d):
    d = d.rename(columns={c: str(c).strip().lower() for c in d.columns})
    for cand in ("adj close", "adj_close", "adjclose", "adjusted close"):
        if cand in d.columns:
            d = d.rename(columns={cand: "adj_close"}); break
    if "date" not in d.columns:
        d = d.rename(columns={d.columns[0]: "date"})
    return d

def fetch_stocknet(root):
    """🔧 [ACL18] Download the public StockNet dataset (only price/ and tweet/preprocessed/ are needed)
    and verify it against checksums of the official files."""
    import subprocess, hashlib, shutil
    need = [f"{root}/price/raw", f"{root}/price/preprocessed", f"{root}/tweet/preprocessed"]
    if not all(os.path.isdir(p) for p in need):
        if os.path.exists(root): shutil.rmtree(root)
        try:   # sparse clone: skips tweet/raw (the largest folder, not used by the benchmark)
            subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
                            CONFIG["ACL18_REPO"], root], check=True)
            subprocess.run(["git", "-C", root, "sparse-checkout", "set", "price", "tweet/preprocessed"], check=True)
        except Exception as e:  # fallback: full archive
            print("sparse clone failed, downloading the full archive:", repr(e))
            import urllib.request
            zp = f"{WORK}/stocknet.zip"
            urllib.request.urlretrieve("https://github.com/yumoxu/stocknet-dataset/archive/refs/heads/master.zip", zp)
            with zipfile.ZipFile(zp) as zf: zf.extractall(WORK)
            shutil.move(f"{WORK}/stocknet-dataset-master", root); os.remove(zp)
    md5 = lambda p: hashlib.md5(open(p, "rb").read()).hexdigest()
    checks = {"price/raw/AAPL.csv": "3a0010e6c6a6415918561b89f31ebe27",
              "price/preprocessed/AAPL.txt": "a92e459ecc3c09592d15ed813fa47340"}
    for rel, h in checks.items():
        assert md5(f"{root}/{rel}") == h, f"checksum mismatch for {rel}: not the official StockNet release"
    n_px = len(os.listdir(f"{root}/price/raw")); n_tw = len(os.listdir(f"{root}/tweet/preprocessed"))
    assert n_px == 88 and n_tw == 87, (n_px, n_tw)
    try:
        rev = subprocess.run(["git", "-C", root, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    except Exception:
        rev = "n/a"
    print(f"StockNet dataset ready at {root} | commit {rev or 'n/a'} | {n_px} price files, {n_tw} tweet folders")

if CONFIG.get("DATASET") == "india":              # 🔧 [INDIA] calendar-cleaned prices (§1.0)
    prices_raw = INDIA_PRICES[["date", "open", "high", "low", "close", "adj_close", "volume", "Stock_symbol"]].copy()
    print("India prices:", prices_raw.shape, "|", prices_raw["Stock_symbol"].nunique(), "tickers")
elif CONFIG.get("DATASET") == "acl18":
    fetch_stocknet(CONFIG["ACL18_ROOT"])
    # 🔧 [ACL18] StockNet price/raw/<TICKER>.csv: Date,Open,High,Low,Close,Adj Close,Volume (Yahoo Finance).
    # WHY: new source. Mapped onto the same columns the v2 pipeline expects.
    frames = []
    for tkr in UNIVERSE:
        fp = f"{CONFIG['ACL18_ROOT']}/price/raw/{tkr}.csv"
        if not os.path.exists(fp): print("  no price file:", tkr); continue
        d = _norm_price_cols(pd.read_csv(fp))
        d = d[["date", "open", "high", "low", "close", "adj_close", "volume"]].copy(); d["Stock_symbol"] = tkr
        frames.append(d)
    prices_raw = pd.concat(frames, ignore_index=True)
    prices_raw["date"] = pd.to_datetime(prices_raw["date"])
    prices_raw = prices_raw[(prices_raw["date"] >= CONFIG["START_DATE"]) &
                            (prices_raw["date"] <= CONFIG["END_DATE"])].reset_index(drop=True)
    print("StockNet prices:", prices_raw.shape, "|", prices_raw["Stock_symbol"].nunique(), "tickers")
elif os.path.exists(PRICE_CACHE):
    prices_raw = pd.read_parquet(PRICE_CACHE)
    print("loaded cached prices:", prices_raw.shape)
else:
    from huggingface_hub import hf_hub_download
    zip_path = hf_hub_download(repo_id="Zihan1004/FNSPID", filename="Stock_price/full_history.zip",
                               repo_type="dataset", local_dir=WORK)
    frames = []
    with zipfile.ZipFile(zip_path) as zf:
        index = {os.path.splitext(os.path.basename(n))[0].upper(): n
                 for n in zf.namelist() if n.lower().endswith(".csv")}
        for tkr in UNIVERSE:
            if tkr not in index: continue
            with zf.open(index[tkr]) as fh:
                d = _norm_price_cols(pd.read_csv(fh))
            if not {"date","open","high","low","close","volume"}.issubset(d.columns): continue
            if "adj_close" not in d.columns: d["adj_close"] = d["close"]
            d = d[["date","open","high","low","close","adj_close","volume"]].copy()
            d["Stock_symbol"] = tkr
            frames.append(d)
    prices_raw = pd.concat(frames, ignore_index=True)
    prices_raw["date"] = pd.to_datetime(prices_raw["date"], errors="coerce", utc=True).dt.tz_localize(None)
    prices_raw = prices_raw.dropna(subset=["date"])
    prices_raw = prices_raw[(prices_raw["date"] >= CONFIG["START_DATE"]) &
                            (prices_raw["date"] <= CONFIG["END_DATE"])].reset_index(drop=True)
    prices_raw.to_parquet(PRICE_CACHE, index=False)
    os.remove(zip_path)
prices_raw["date"] = pd.to_datetime(prices_raw["date"])

acl18 universe requested: 88 tickers
StockNet dataset ready at /content/stocknet-dataset | commit 330708b5ddc359961078bef469f43f48992fd6e4 | 88 price files, 87 tweet folders
StockNet prices: (56894, 8) | 87 tickers


### 1.1 Price features, universe coverage filter, targets
Every feature at day *t* uses data up to the close of *t*. `y_next` is the next trading day's
close-to-close adjusted return. The **universe filter** removes series that stop early in the FNSPID
archive (20 NASDAQ-100 tickers end in 2020): a ticker is kept if its series reaches the end of the
sample and has a price row on ≥95 % of the trading days since its first observation.

In [8]:
px = prices_raw.sort_values(["Stock_symbol", "date"]).reset_index(drop=True)
g = px.groupby("Stock_symbol", group_keys=False)
px["Return"]     = g["adj_close"].pct_change(fill_method=None)
px["RetOC"]      = (px["close"] - px["open"]) / px["open"]
px["Volatility"] = (px["high"] - px["low"]) / px["open"]
px["LogVolume"]  = np.log1p(px["volume"])
px["SpreadRel"]  = (px["high"] - px["low"]) / px["close"]
px["LagReturn"]  = g["Return"].shift(1)
px["LagVolume"]  = g["LogVolume"].shift(1)
px["MA5Ratio"]   = px["close"] / g["close"].transform(lambda x: x.rolling(5).mean()) - 1
px["MA10Ratio"]  = px["close"] / g["close"].transform(lambda x: x.rolling(10).mean()) - 1
px["Vol5"]       = g["Return"].transform(lambda x: x.rolling(5).std())
px["Above50"]    = (px["close"] > g["close"].transform(lambda x: x.rolling(50, min_periods=20).mean())).astype(float)
PRICE_FEATURES = ["Return","RetOC","Volatility","LogVolume","SpreadRel",
                  "LagReturn","LagVolume","MA5Ratio","MA10Ratio","Vol5"]
px["y_next"] = px.groupby("Stock_symbol")["Return"].shift(-1)
px = px.replace([np.inf, -np.inf], np.nan).dropna(subset=PRICE_FEATURES).reset_index(drop=True)
px = px.rename(columns={"date": "Date"})

TRADING_DAYS = pd.DatetimeIndex(sorted(px["Date"].unique()))
pos_of_day = {d: i for i, d in enumerate(TRADING_DAYS)}
span = px.groupby("Stock_symbol")["Date"].agg(["min", "max", "count"])
span["listed_days"] = [pos_of_day[b] - pos_of_day[a] + 1 for a, b in zip(span["min"], span["max"])]
span["coverage"] = span["count"] / span["listed_days"]
span["reaches_end"] = span["max"] >= TRADING_DAYS[-5]
# 🔧 [INDIA] point-in-time universe: firms may leave the index or be delisted, so "reaches the end" is not required
# (membership is enforced per day by EVAL_MASK); a minimum history is. ACL18 / FNSPID: unchanged.
_india = CONFIG.get("DATASET") == "india"
keep = span[(span["coverage"] >= CONFIG["UNIVERSE_MIN_COVERAGE"]) & (span["reaches_end"] | _india)
            & (span["count"] >= CONFIG.get("MIN_PRICE_ROWS", 0))].index
dropped = sorted(set(span.index) - set(keep))
px = px[px["Stock_symbol"].isin(keep)].reset_index(drop=True)
SYMBOLS = sorted(keep); N = len(SYMBOLS); T = len(TRADING_DAYS)
sym_idx = {s: i for i, s in enumerate(SYMBOLS)}
print(f"tickers loaded {len(span)} | kept {N} | dropped {len(dropped)}: {dropped}")
print(f"trading days {T}: {TRADING_DAYS[0].date()} -> {TRADING_DAYS[-1].date()}")
if CONFIG.get("DATASET") == "india":             # 🔧 [INDIA] MEMBER[t, i]: NIFTY-50 member on day t (point in time)
    MEMBER_TN = membership_tensor(TRADING_DAYS, SYMBOLS)
    print(f"point-in-time members per day (median): {int(np.median(MEMBER_TN.sum(1)))}")


tickers loaded 87 | kept 87 | dropped 0: []
trading days 653: 2013-06-14 -> 2016-01-15


### 1.2 Dense day × firm tensors

In [9]:
ti = px["Date"].map(pos_of_day).values
ni = px["Stock_symbol"].map(sym_idx).values
P = np.zeros((T, N, len(PRICE_FEATURES)), np.float32)
P[ti, ni] = px[PRICE_FEATURES].values.astype(np.float32)
ROW = np.zeros((T, N), bool); ROW[ti, ni] = True                  # a price row exists
Y = np.full((T, N), np.nan, np.float32); Y[ti, ni] = px["y_next"].values
RET = np.full((T, N), np.nan, np.float32); RET[ti, ni] = px["Return"].values
ABOVE = np.full((T, N), np.nan, np.float32); ABOVE[ti, ni] = px["Above50"].values
YMASK = ROW & np.isfinite(Y)
# 🔧 [v9] evaluation universe. India: point-in-time NIFTY-50 members only (loss, metrics, market state). ACL18 / FNSPID:
# every firm-day (all True), i.e. numerically identical to v8.
EVAL_MASK = MEMBER_TN if CONFIG.get("DATASET") == "india" else np.ones((T, N), bool)
YMASK = YMASK & EVAL_MASK

# 🔧 [ACL18] Benchmark sample set and label. WHY: reproduce the StockNet protocol exactly
# (stocknet-code DataPipe._get_prices_and_ts): movement = adjusted-close change of the target day from
# price/preprocessed/<TICKER>.txt; samples with -0.5% <= movement < 0.55% are discarded; the target date
# must be in [2014-01-01, 2016-01-01) and have a complete price window (the trading days in the 4 calendar
# days before the target plus one earlier row). Label = 1 if movement > 1e-7 else 0.
# With these rules the sample counts are 20,339 / 2,555 / 3,720 (train / dev / test), the numbers
# published in the StockNet paper; they are printed and checked below.
YCLS = np.full((T, N), -1, np.int8); MVT = np.full((T, N), np.nan, np.float32)
if CONFIG.get("DATASET") == "acl18":
    import datetime as _dt
    lo_mv, hi_mv = CONFIG["MV_THRESHOLDS"]; bench = {}
    for tkr in SYMBOLS:
        fp = f"{CONFIG['ACL18_ROOT']}/price/preprocessed/{tkr}.txt"
        if not os.path.exists(fp): continue
        rows = [l.rstrip("\n").split("\t") for l in open(fp, encoding="utf-8")]  # descending dates
        dates = [_dt.date.fromisoformat(r[0]) for r in rows]
        for k, (d, r) in enumerate(zip(dates, rows)):
            if not ("2014-01-01" <= d.isoformat() < "2016-01-01"): continue
            mv = float(r[1])
            if lo_mv <= mv < hi_mv: continue
            dmin = d - _dt.timedelta(days=CONFIG["BENCH_WINDOW_CAL_DAYS"] - 1)
            later = [x for x in dates[k + 1:]]                                # earlier dates (descending file)
            n_win = sum(1 for x in later if dmin <= x < d)
            has_extra = any(x < dmin for x in later)
            if not has_extra: continue                                        # len(prices) != T in DataPipe
            bench[(tkr, pd.Timestamp(d))] = mv
    px_ = px.sort_values(["Stock_symbol", "Date"])
    nxt = px_.groupby("Stock_symbol")["Date"].shift(-1)
    for (tkr, t_), nd in zip(zip(px_["Stock_symbol"], px_["Date"]), nxt):
        if pd.isna(nd) or (tkr, nd) not in bench: continue
        t = pos_of_day[t_]
        if t + 1 >= T or TRADING_DAYS[t + 1] != nd: continue                 # next row must be next trading day
        i = sym_idx[tkr]; MVT[t, i] = bench[(tkr, nd)]; YCLS[t, i] = int(bench[(tkr, nd)] > 1e-7)
    BENCH = (YCLS >= 0) & YMASK
    YMASK = BENCH
    tgt_dates = TRADING_DAYS[np.minimum(np.arange(T) + 1, T - 1)]
    for name, (a, b) in {"train": ("2014-01-01", "2015-08-01"), "dev": ("2015-08-01", "2015-10-01"),
                         "test": ("2015-10-01", "2016-01-01")}.items():
        sel = (tgt_dates >= a) & (tgt_dates < b)
        n_b = sum(1 for (_, d) in bench if a <= d.strftime("%Y-%m-%d") < b)
        print(f"benchmark {name:5s}: {int(BENCH[sel].sum()):6,d} samples mapped / {n_b:6,d} in StockNet protocol "
              f"(published: {dict(train=20339, dev=2555, test=3720)[name]:,}) | up-rate {YCLS[sel][BENCH[sel]].mean():.4f}")
    mapped = {(SYMBOLS[i], TRADING_DAYS[t + 1]) for t, i in zip(*np.where(BENCH))}
    missing = sorted(k for k in bench if k not in mapped)
    print(f"benchmark samples not representable: {len(missing)}", [(a, b.date().isoformat()) for a, b in missing[:10]])
    # sanity: movement percent == adjusted-close return used as regression target
    print("max |movement - y_next| on benchmark samples:", float(np.nanmax(np.abs(MVT[BENCH] - Y[BENCH]))))
# cross-sectionally demeaned target (relative performance)
ymean = np.nanmean(np.where(YMASK, Y, np.nan), axis=1, keepdims=True)
YXS = np.where(YMASK, Y - ymean, 0.0).astype(np.float32)
Y0 = np.where(YMASK, Y, 0.0).astype(np.float32)
# 🔧 [ACL18] regression target used by training/evaluation. WHY: the benchmark label is the direction of the
# RAW movement; v2 trained on the cross-sectionally demeaned return, whose sign is not the benchmark label.
# The loss (Huber + IC) and the model are unchanged; IC is identical for YXS and Y0 (per-day shift).
YT = Y0 if CONFIG.get("TARGET_MODE", "xs") == "raw" else YXS
print("tensors:", P.shape, "| valid targets per day (median):", int(np.median(YMASK.sum(1))))

benchmark train: 20,335 samples mapped / 20,339 in StockNet protocol (published: 20,339) | up-rate 0.5074
benchmark dev  :  2,555 samples mapped /  2,555 in StockNet protocol (published: 2,555) | up-rate 0.4458
benchmark test :  3,720 samples mapped /  3,720 in StockNet protocol (published: 3,720) | up-rate 0.5129
benchmark samples not representable: 4 [('AGFS', '2014-12-04'), ('BABA', '2014-09-29'), ('BABA', '2014-10-01'), ('BABA', '2014-10-02')]
max |movement - y_next| on benchmark samples: 5.010515451431274e-07
tensors: (653, 87, 10) | valid targets per day (median): 46


### 1.3 Market-state observations for the regime model (backward-looking only)
Close to the original SCEG proposal (volatility, momentum, breadth, market return), built from the
price panel because VIX is not part of FNSPID.

In [10]:
_MSM = ROW & EVAL_MASK        # 🔧 [v9] India: index members on day t; ACL18/FNSPID: = ROW (unchanged)
retm = np.where(_MSM, RET, np.nan)
mret = np.nanmean(retm, axis=1)
ms = pd.DataFrame({"mret": mret}, index=TRADING_DAYS)
ms["mom20"]   = ms["mret"].rolling(20, min_periods=5).sum()
ms["vol10"]   = ms["mret"].rolling(10, min_periods=5).std()
ms["breadth"] = np.nanmean(np.where(_MSM, ABOVE, np.nan), axis=1)
ms["disp"]    = np.nanstd(retm, axis=1)
ms["range"]   = np.nanmean(np.where(_MSM, P[:, :, PRICE_FEATURES.index("Volatility")], np.nan), axis=1)
MARKET_STATE = ms.bfill().ffill().values.astype(np.float64)   # bfill only affects the first warm-up days
print(ms.describe().T[["mean", "std"]].round(4))

           mean     std
mret     0.0003  0.0080
mom20    0.0077  0.0297
vol10    0.0072  0.0033
breadth  0.5553  0.2169
disp     0.0112  0.0033
range    0.0162  0.0052


### 1.4 · [INDIA] data-integrity and survivorship audit


In [11]:
# ===== 1.4 · [INDIA] data-integrity and survivorship audit (label-free; assertions guard against leakage) =====
if IS_INDIA:
    mem_per_day = (EVAL_MASK & ROW).sum(1)
    exp_members = np.array([(CONST_W.set_index("DATE").sort_index().loc[:d].iloc[-1] > 0).sum() for d in TRADING_DAYS[::50]])
    print(f"firms (nodes) {N} | trading days {T} ({TRADING_DAYS[0].date()} -> {TRADING_DAYS[-1].date()})")
    print(f"point-in-time members with a price row per day: min {mem_per_day.min()} | median {int(np.median(mem_per_day))} | "
          f"max {mem_per_day.max()} (index size in the constituent file: {int(np.median(exp_members))})")
    lost = sorted(set(INDIA_UNIVERSE) - set(SYMBOLS))
    print("index members without usable prices (residual survivorship bias, disclosed):", lost or "none")
    src = INDIA_PRICES_RAW.groupby("Stock_symbol")["source"].first()
    print("non-Yahoo price sources:", src[~src.str.startswith("yahoo")].to_dict() or "none")
    big = px.loc[px["Return"].abs() > 0.25, ["Date", "Stock_symbol", "Return"]]
    print(f"|daily return| > 25 %: {len(big)} firm-days (listed for manual corporate-action check; NOT removed):")
    print(big.sort_values("Date").head(20).to_string(index=False))
    # membership is point in time: the mask on day t equals the file row dated <= t (checked on a sample of days)
    Wt_ = CONST_W.set_index("DATE").sort_index()
    for t in np.linspace(0, T - 1, 25).astype(int):
        row = Wt_.loc[:TRADING_DAYS[t]].iloc[-1]
        assert all(EVAL_MASK[t, sym_idx[s]] == (row.get(s, 0) > 0) for s in SYMBOLS)
    # every target uses the NEXT trading day's adjusted close only; every feature uses data up to day t only
    _m = YMASK[:-1] & ROW[1:]
    assert np.allclose(Y[:-1][_m], RET[1:][_m], atol=1e-6), "y_next must equal the next trading day's return"
    print("audit passed: point-in-time membership, next-day target, calendar rules applied "
          f"({CAL_AUDIT['days_dropped_total']} special/short/thin sessions removed)")


---
## 2 · News: ingestion, point-in-time alignment, deduplication  (🔧 [ACL18] tweet loader added)

### 2.0 · [INDIA] BSE corporate announcements → strict 15:30 IST alignment (no-op for ACL18 / FNSPID)


In [12]:
# ===== 2.0 · [INDIA] BSE corporate announcements -> strict 15:30 IST point-in-time alignment (no-op elsewhere) =====
# Fields used: SCRIP_CD, NEWSSUB (subject), MORE / HEADLINE (body), CATEGORYNAME (description only, never a model
# input), DissemDT (exchange dissemination time, IST, to the millisecond), NEWS_DT (fallback, +120 min buffer).
ANN_COLS = ["SCRIP_CD", "NEWSID", "NEWSSUB", "HEADLINE", "MORE", "CATEGORYNAME", "SUBCATNAME", "NEWS_DT", "DissemDT", "DT_TM"]

def fetch_announcements(codes, y0, y1):
    """One parquet per (scrip, year): resumable; checks the row count BSE reports (ROWCNT)."""
    os.makedirs(f"{RAW}/bse_ann", exist_ok=True); H = bse_harvest(); out, short = [], []
    if H is not None:
        a = pd.DataFrame(H["ann"]); a["SCRIP_CD"] = a["SCRIP_CD"].astype(str)
        a = a.drop_duplicates(["SCRIP_CD", "NEWSID", "YEAR"])            # 🔧 [v9.1] several harvest files may overlap
        missing = [c for c in codes if c not in set(a["SCRIP_CD"])]
        if missing and not globals().get("ALLOW_MISSING_FILINGS", False):
            open(f"{RAW}/bse_harvester_topup.js", "w").write(harvester_script(codes=missing, outfile="bse_harvest_topup.json"))
            raise RuntimeError(
                f"The BSE harvest has no filings for scrips {missing} "
                f"({[s for s in INDIA_UNIVERSE if FIRM_MAP[s][1] in missing]}). A firm without filings would silently "
                f"look like a firm without news. Run the top-up harvester saved at {RAW}/bse_harvester_topup.js "
                "(same steps as before; a few minutes), upload bse_harvest_topup.json to the same folder, and run again.")
        for (c, y), g in a.groupby(["SCRIP_CD", "YEAR"]):
            if g["ROWCNT"].notna().any() and len(g.drop_duplicates("NEWSID")) < int(g["ROWCNT"].max()): short.append((c, y))
        return a[a["SCRIP_CD"].isin(codes)][ANN_COLS], short
    for c in codes:
        for y in range(y0, y1 + 1):
            p = f"{RAW}/bse_ann/{c}_{y}.parquet"
            if os.path.exists(p): out.append(pd.read_parquet(p)); continue
            rows, total, page = [], None, 1
            while page < 400:
                j = bse_get("https://api.bseindia.com/BseIndiaAPI/api/AnnSubCategoryGetData/w",
                            dict(pageno=page, strCat=-1, strPrevDate=f"{y}0101", strScrip=c, strSearch="P",
                                 strToDate=f"{y}1231", strType="C", subcategory=-1))
                if j is None:
                    raise RuntimeError("BSE unreachable: run the browser harvester printed below, upload bse_harvest.json to "
                                       f"{RAW}, then re-run this cell.\n\n" + harvester_script())
                t = j.get("Table") or []; rows += t
                total = ((j.get("Table1") or [{}])[0] or {}).get("ROWCNT", total)
                if len(t) < 50 or (total and len(rows) >= total): break
                page += 1; time.sleep(0.25)
            d = pd.DataFrame(rows).reindex(columns=ANN_COLS).astype({"SCRIP_CD": str})
            if total and len(d.drop_duplicates("NEWSID")) < int(total): short.append((c, y))
            d.to_parquet(p, index=False); out.append(d); time.sleep(0.25)
        print(f"  scrip {c}: done", flush=True)
    return (pd.concat(out, ignore_index=True) if out else pd.DataFrame(columns=ANN_COLS)), short

def _clean(s):
    s = _html.unescape(re.sub(r"<[^>]+>", " ", str(s or ""))); return re.sub(r"\s+", " ", s).strip()

if IS_INDIA:
    open(f"{RAW}/bse_harvester.js", "w").write(harvester_script())        # saved for convenience
    code2sym = {FIRM_MAP[s][1]: s for s in INDIA_UNIVERSE}
    y0, y1 = pd.Timestamp(CONFIG["START_DATE"]).year, pd.Timestamp(CONFIG["END_DATE"]).year
    ANN_PATH = f"{RAW}/announcements_{'synth' if SYNTH else 'real'}.parquet"
    if os.path.exists(ANN_PATH):
        ann = pd.read_parquet(ANN_PATH); short = json.load(open(ANN_PATH + ".short.json"))
    else:
        if SYNTH: ann, short = synth_announcements(INDIA_UNIVERSE).astype({"SCRIP_CD": str}), []
        else: ann, short = fetch_announcements(list(code2sym), y0, y1)
        ann = ann.astype({c: str for c in ANN_COLS if c != "SCRIP_CD"}).replace({"None": None, "nan": None})
        ann.to_parquet(ANN_PATH, index=False); json.dump([list(x) for x in short], open(ANN_PATH + ".short.json", "w"))
    DATA_MANIFEST[os.path.basename(ANN_PATH)] = _sha_file(ANN_PATH)
    ann = ann.drop_duplicates("NEWSID") if ann["NEWSID"].notna().any() else ann
    ann["Stock_symbol"] = ann["SCRIP_CD"].astype(str).map(code2sym)
    ann = ann[ann["Stock_symbol"].isin(sym_idx)].copy()

    # ---- timestamps (IST) ----
    t_news = pd.to_datetime(ann["NEWS_DT"], errors="coerce"); t_dis = pd.to_datetime(ann["DissemDT"], errors="coerce")
    has_dis = t_dis.notna()
    ts_local = t_dis.where(has_dis, t_news + pd.Timedelta(minutes=CONFIG["DISSEM_FALLBACK_BUFFER_MIN"]))
    ts_local = ts_local.where(~(has_dis & t_news.notna() & (t_news > t_dis)), t_news)   # never earlier than NEWS_DT
    ann = ann[ts_local.notna()].copy(); ts_local = ts_local[ts_local.notna()]
    ann["_ts"] = ts_local.dt.tz_localize(CONFIG["MARKET_TZ"]).dt.tz_convert("UTC")
    # ---- strict alignment: at/after 15:30 IST, or a non-trading day -> next trading day ----
    mins = ts_local.dt.hour * 60 + ts_local.dt.minute
    cand = ts_local.dt.normalize() + pd.to_timedelta((mins >= CONFIG["CLOSE_MINUTES"]).astype(int), unit="D")
    posd = np.searchsorted(TRADING_DAYS, cand.values, side="left"); ok = posd < len(TRADING_DAYS)
    ann = ann[ok].copy(); ann["eff_date"] = TRADING_DAYS[posd[ok]]
    # ---- text ----
    body = [_clean(m) if _clean(m) else _clean(h) for m, h in zip(ann["MORE"], ann["HEADLINE"])]
    ann["Article_title"] = [_clean(s) for s in ann["NEWSSUB"]]
    ann["text"] = [(a + ". " + b if b else a)[:1500].strip() for a, b in zip(ann["Article_title"], body)]
    ann = ann[ann["text"].str.len() > 20].sort_values("_ts")
    n0 = len(ann); ann = ann.drop_duplicates(subset=["Stock_symbol", "text"], keep="first")
    # ---- per firm-day cap: label-free rule fixed in the protocol ----
    per_day = ann.groupby(["Stock_symbol", "eff_date"]).size()
    tr_lo, tr_hi = pd.Timestamp(CONFIG["TRAIN_START"]), pd.Timestamp(CONFIG["FOLDS"][0][0])
    q99 = per_day[(per_day.index.get_level_values(1) >= tr_lo) & (per_day.index.get_level_values(1) <= tr_hi)].quantile(0.99)
    CAP = int(min(10, max(2, _math.ceil(q99)))) if np.isfinite(q99) else 2   # no training-period items -> floor
    CONFIG["MAX_ARTICLES_PER_TICKER_DAY"] = CAP
    ann["rank"] = ann.groupby(["Stock_symbol", "eff_date"])["_ts"].rank(method="first")
    INDIA_ARTICLES = ann[ann["rank"] <= CAP].drop(columns="rank").reset_index(drop=True)
    INDIA_ARTICLES = INDIA_ARTICLES[(INDIA_ARTICLES["eff_date"] >= CONFIG["START_DATE"])].reset_index(drop=True)
    INDIA_ARTICLES["art_id"] = np.arange(len(INDIA_ARTICLES))
    DATA_CHOICES = {"article_cap": CAP, "q99_items_per_firm_day_train": float(q99),
                    "incomplete_scrip_years": [list(map(str, x)) for x in short]}
    json.dump(DATA_CHOICES, open(f"{CACHE}/data_choices_india.json", "w"), indent=1)
    # ---- news-layer audit (printed; asserted where a violation would be leakage) ----
    after_close = (mins[ok] >= CONFIG["CLOSE_MINUTES"]).mean()
    print(f"announcements: {n0:,} after cleaning | exact duplicates removed {n0 - len(ann):,} | per-firm-day cap k = {CAP} "
          f"(q99 = {q99:.1f}) | used {len(INDIA_ARTICLES):,}")
    print(f"timestamp source: DissemDT {has_dis.mean():.1%} | NEWS_DT + {CONFIG['DISSEM_FALLBACK_BUFFER_MIN']} min {1 - has_dis.mean():.1%}"
          f" | disseminated at/after 15:30 IST (pushed to next trading day): {after_close:.1%}")
    if short: print(f"WARNING: {len(short)} scrip-years returned fewer rows than BSE's ROWCNT: {short[:10]}")
    print("category mix (description only):", INDIA_ARTICLES["CATEGORYNAME"].value_counts(normalize=True).head(8).round(3).to_dict())
    _loc = INDIA_ARTICLES["_ts"].dt.tz_convert(CONFIG["MARKET_TZ"])
    _close = (INDIA_ARTICLES["eff_date"] + pd.Timedelta(minutes=CONFIG["CLOSE_MINUTES"])).dt.tz_localize(CONFIG["MARKET_TZ"])
    assert (_loc < _close).all(), "an item is used on a feature day whose 15:30 IST close precedes the item"
    assert INDIA_ARTICLES["eff_date"].isin(TRADING_DAYS).all()


In [13]:
# 🔧 [ACL18] News layer. WHY: StockNet provides tweets (tweet/preprocessed/<TICKER>/<YYYY-MM-DD>, one JSON per
# line with keys text [token list], created_at [UTC], user_id_str) instead of FNSPID articles. The FNSPID
# branch below is kept verbatim (indented under `else`) so the notebook still runs on FNSPID.
if CONFIG.get("DATASET") == "india":             # 🔧 [INDIA] BSE filings, strict 15:30 IST (built in §2.0)
    articles = INDIA_ARTICLES.copy()
    print(f"India filings for extraction: {len(articles):,}")
elif CONFIG.get("DATASET") == "acl18":
    TWEET_CACHE = f"{CACHE}/acl18_tweets.parquet"
    if os.path.exists(TWEET_CACHE):
        news = pd.read_parquet(TWEET_CACHE)
    else:
        recs = []
        for tkr in UNIVERSE:
            ddir = f"{CONFIG['ACL18_ROOT']}/tweet/preprocessed/{tkr}"
            if not os.path.isdir(ddir): print("  no tweets:", tkr); continue
            for fname in sorted(os.listdir(ddir)):
                with open(f"{ddir}/{fname}", encoding="utf-8") as fh:
                    for line in fh:
                        m = json.loads(line)
                        if not m.get("text"): continue                     # DataPipe skips empty messages
                        recs.append((tkr, fname, m["created_at"], m.get("user_id_str", ""), " ".join(m["text"])))
        news = pd.DataFrame(recs, columns=["Stock_symbol", "file_date", "created_at", "user", "text"])
        news.to_parquet(TWEET_CACHE, index=False)
    news["_ts"] = pd.to_datetime(news["created_at"], format="%a %b %d %H:%M:%S %z %Y", utc=True)
    news["file_date"] = pd.to_datetime(news["file_date"])
    # tokenised tweets: re-attach cashtags ("$ aapl" -> "$aapl") so the LLM sees the ticker
    news["text"] = news["text"].str.replace(r"\$ (\w)", r"$\1", regex=True).str.slice(0, 1500).str.strip()
    news["Article_title"] = news["text"]
    print(f"tweets: {len(news):,} | stocks with tweets: {news['Stock_symbol'].nunique()} | "
          f"file dates {news['file_date'].min().date()} -> {news['file_date'].max().date()}")

    # ---- tweet -> trading-day alignment (feature day t; target = t+1) ----
    if CONFIG["NEWS_ALIGNMENT"] == "benchmark":
        # StockNet (DataPipe._trading_day_alignment): tweets in calendar file c inform the first trading day
        # AFTER c. In SCEG's convention (features at close of t predict t+1) that is feature day
        # t = last trading day <= c. NOTE: file c includes tweets posted after the 16:00 ET close of c.
        posd = np.searchsorted(TRADING_DAYS, news["file_date"].values, side="right") - 1
        valid = posd >= 0
    else:
        # strict = v2 point-in-time rule on the tweet timestamp: after 16:00 ET or non-trading day -> next day
        ts_et = news["_ts"].dt.tz_convert("America/New_York")
        stated_day = ts_et.dt.normalize().dt.tz_localize(None)
        push = ts_et.dt.hour >= CONFIG["NEWS_CUTOFF_HOUR_ET"]
        cand = stated_day + pd.to_timedelta(push.astype(int), unit="D")
        posd = np.searchsorted(TRADING_DAYS, cand.values, side="left"); valid = posd < len(TRADING_DAYS)
    news = news[valid].copy(); news["eff_date"] = TRADING_DAYS[posd[valid]]
    print("alignment:", CONFIG["NEWS_ALIGNMENT"])

    # ---- duplicates and cap. WHY changed: tweets have no titles; ~17% are exact repeats (retweets) within a
    # stock-day, so duplicates are removed within (stock, feature day, text). Cap = StockNet's 30 msgs/day.
    news = news[news["text"].str.len() > 0].sort_values("_ts")
    n0 = len(news); news = news.drop_duplicates(subset=["Stock_symbol", "eff_date", "text"])
    news["rank"] = news.groupby(["Stock_symbol", "eff_date"])["_ts"].rank(method="first")
    articles = news[news["rank"] <= CONFIG["MAX_ARTICLES_PER_TICKER_DAY"]].drop(columns="rank").reset_index(drop=True)
    articles["art_id"] = np.arange(len(articles))
    print(f"duplicates removed: {n0 - len(news):,} | tweets for extraction: {len(articles):,}")
    del news; gc.collect()
else:
    NEWS_CACHE = (f"{CACHE}/news_{CONFIG['NEWS_SOURCE']}_{CONFIG['START_DATE'][:4]}_{CONFIG['END_DATE'][:4]}.parquet"
                  if CONFIG["UNIVERSE_MODE"] == "nasdaq100" else
                  f"{CACHE}/news_{CONFIG['UNIVERSE_MODE']}_{CONFIG['NEWS_SOURCE']}_{CONFIG['START_DATE'][:4]}_{CONFIG['END_DATE'][:4]}.parquet")
    KEEP_COLS = ["Date","Article_title","Stock_symbol","Publisher","Lsa_summary","Textrank_summary"]

    def _normalise_news(chunk):
        chunk = chunk.rename(columns={c: str(c).strip() for c in chunk.columns})
        ren = {}
        for c in chunk.columns:
            lc = c.lower()
            if lc == "date": ren[c] = "Date"
            elif lc in ("article_title","title","headline"): ren[c] = "Article_title"
            elif lc in ("stock_symbol","symbol","ticker"): ren[c] = "Stock_symbol"
            elif lc == "publisher": ren[c] = "Publisher"
            elif lc == "lsa_summary": ren[c] = "Lsa_summary"
            elif lc == "textrank_summary": ren[c] = "Textrank_summary"
        chunk = chunk.rename(columns=ren)
        for c in KEEP_COLS:
            if c not in chunk.columns: chunk[c] = np.nan
        return chunk[KEEP_COLS]

    def _filter_chunk(chunk, universe_set, lo, hi):
        chunk = _normalise_news(chunk)
        raw = chunk["Date"]
        if raw.dtype == object:
            raw = raw.astype(str).str.replace(r"\s*UTC\s*$", "", regex=True).str.strip()
        ts = pd.to_datetime(raw, errors="coerce", utc=True)
        chunk = chunk.assign(_ts=ts).dropna(subset=["_ts"])
        chunk = chunk[(chunk["_ts"] >= lo) & (chunk["_ts"] <= hi)]
        sym = chunk["Stock_symbol"].astype(str).str.strip().str.upper()
        chunk = chunk[sym.isin(universe_set)].copy()
        chunk["Stock_symbol"] = sym[sym.isin(universe_set)]
        return chunk

    if os.path.exists(NEWS_CACHE):
        news = pd.read_parquet(NEWS_CACHE)
        print("loaded cached news:", news.shape)
    else:
        from huggingface_hub import hf_hub_download, list_repo_files
        lo = pd.Timestamp(CONFIG["START_DATE"], tz="UTC"); hi = pd.Timestamp(CONFIG["END_DATE"], tz="UTC") + pd.Timedelta(days=2)
        out = []
        if CONFIG["NEWS_SOURCE"] == "mirror":
            repo = "benstaf/FNSPID-filtered-nasdaq-100"
            for f in [f for f in list_repo_files(repo, repo_type="dataset") if f.endswith((".parquet", ".csv"))]:
                p = hf_hub_download(repo, f, repo_type="dataset", local_dir=WORK)
                d = pd.read_parquet(p) if f.endswith(".parquet") else pd.read_csv(p, dtype=str, on_bad_lines="skip")
                out.append(_filter_chunk(d, set(UNIVERSE), lo, hi)); del d; gc.collect()
        else:
            # streamed; the full-article column is never read (usecols), so RAM stays small
            fname = ("Stock_news/All_external.csv" if CONFIG["NEWS_SOURCE"] == "all_external"
                     else "Stock_news/nasdaq_exteral_data.csv")
            url = f"https://huggingface.co/datasets/Zihan1004/FNSPID/resolve/main/{fname}"
            want = lambda c: str(c).strip().lower() in {"date", "article_title", "stock_symbol", "publisher",
                                                        "lsa_summary", "textrank_summary"}
            reader = pd.read_csv(url, chunksize=CONFIG["NEWS_CHUNKSIZE"], dtype=str, usecols=want,
                                 on_bad_lines="skip", engine="c", low_memory=False)
            for i, chunk in enumerate(reader):
                out.append(_filter_chunk(chunk, set(UNIVERSE), lo, hi)); del chunk
                if i % 20 == 0:
                    gc.collect(); print(f"  chunk {i:4d} | kept {sum(len(o) for o in out):,}", flush=True)
        news = pd.concat(out, ignore_index=True)
        news.to_parquet(NEWS_CACHE, index=False)
    news = news[news["Stock_symbol"].isin(set(UNIVERSE))].copy()

    # ---- point-in-time alignment: 16:00 ET cut-off, date-only rows -> next trading day ----
    ts_utc = news["_ts"]; ts_et = ts_utc.dt.tz_convert("America/New_York")
    date_only = (ts_utc.dt.hour == 0) & (ts_utc.dt.minute == 0) & (ts_utc.dt.second == 0)
    stated_day = pd.to_datetime(pd.Series(np.where(date_only, ts_utc.dt.normalize().dt.tz_localize(None),
                                                   ts_et.dt.normalize().dt.tz_localize(None)), index=news.index))
    push = date_only | (ts_et.dt.hour >= CONFIG["NEWS_CUTOFF_HOUR_ET"])
    cand = stated_day + pd.to_timedelta(push.astype(int), unit="D")
    posd = np.searchsorted(TRADING_DAYS, cand.values, side="left"); valid = posd < len(TRADING_DAYS)
    news = news.loc[valid].copy(); news["eff_date"] = TRADING_DAYS[posd[valid]]
    print(f"date-only rows: {date_only.mean():.1%} | pushed to next day: {push.mean():.1%}")

    # ---- text, dedupe, cap (identical to v1, so art_id matches the cached extraction shards) ----
    summary = news["Lsa_summary"].fillna(news["Textrank_summary"]).fillna("")
    news["text"] = (news["Article_title"].fillna("").astype(str).str.strip() + ". "
                    + summary.astype(str).str.strip()).str.slice(0, 1500).str.strip()
    news = news[news["text"].str.len() > 20].copy().sort_values("_ts")
    news = news.drop_duplicates(subset=["Stock_symbol", "Article_title"], keep="first")
    news["rank"] = news.groupby(["Stock_symbol", "eff_date"])["_ts"].rank(method="first")
    articles = news[news["rank"] <= CONFIG["MAX_ARTICLES_PER_TICKER_DAY"]].drop(columns="rank").reset_index(drop=True)
    articles["art_id"] = np.arange(len(articles))
    del news; gc.collect()
    print(f"articles for extraction: {len(articles):,}")

tweets: 106,338 | stocks with tweets: 87 | file dates 2014-01-01 -> 2015-12-31
alignment: benchmark
duplicates removed: 14,439 | tweets for extraction: 85,850


In [14]:
if RUN_PHASE == "data":   # 🔧 [v9] end of the CPU data phase
    raise PhaseComplete("Phase 'data' finished: prices, membership and filings are cached in Drive. "
                        "Next: Runtime -> Change runtime type -> T4 GPU, set RUN_PHASE = 'prep', Run all.")
print("data layer complete; phase:", RUN_PHASE)


data layer complete; phase: all


---
## 3 · LLM event extraction (unchanged prompt and six-field schema; resumes from cached shards)

In [15]:
PROMPT = """Extract ONLY the primary event from the article.

Return ONLY valid JSON:
{
"Subject":"",
"Predicate":"",
"Object":"",
"Event Type":"",
"Magnitude":"",
"Temporal Horizon":""
}

Article:
{article}
"""
EMPTY_EVENT = {"Subject":"","Predicate":"","Object":"","Event Type":"","Magnitude":"","Temporal Horizon":""}

def parse_event(decoded):
    m = re.search(r"\{[^{}]*\}", decoded, re.S)
    if not m: return dict(EMPTY_EVENT)
    try: obj = json.loads(m.group())
    except Exception: return dict(EMPTY_EVENT)
    return {k: str(obj.get(k, "") or "") for k in EMPTY_EVENT}

SHARD_DIR = f"{CACHE}/event_shards" if CONFIG["UNIVERSE_MODE"] == "nasdaq100" else f"{CACHE}/event_shards_{CONFIG['UNIVERSE_MODE']}"
if CONFIG.get("DATASET") == "acl18":   # 🔧 [ACL18] art_id depends on the alignment mode (dedupe/cap per feature day)
    SHARD_DIR += f"_{CONFIG['NEWS_ALIGNMENT']}"
os.makedirs(SHARD_DIR, exist_ok=True)
def load_shards():
    fs = sorted(glob.glob(f"{SHARD_DIR}/shard_*.parquet"))
    if not fs: return pd.DataFrame(columns=["art_id"] + list(EMPTY_EVENT))
    return pd.concat([pd.read_parquet(f) for f in fs], ignore_index=True)

done = load_shards()
todo = articles[~articles["art_id"].isin(set(done["art_id"]))]
_eta = len(todo) / (CONFIG["EXTRACT_BATCH_SIZE"] * 0.7) / 3600
print(f"already extracted {len(done):,} | remaining {len(todo):,} | rough ETA {_eta:.1f} h on a T4 "
      "(resumable: re-run this cell in later sessions)")
if not RUN_LEGACY:
    print("(the v2 extraction is not run for FNSPID in this notebook; section 14 runs the v3 extraction)")
if len(todo) and CONFIG["USE_LLM_EVENTS"] and RUN_LEGACY:
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tok = AutoTokenizer.from_pretrained(CONFIG["EXTRACT_MODEL"]); tok.padding_side = "left"
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    lm = AutoModelForCausalLM.from_pretrained(CONFIG["EXTRACT_MODEL"],
            torch_dtype=torch.float16 if DEVICE.type == "cuda" else torch.float32).to(DEVICE).eval()
    B, SH = CONFIG["EXTRACT_BATCH_SIZE"], CONFIG["EXTRACT_SHARD_SIZE"]
    shard_no = len(glob.glob(f"{SHARD_DIR}/shard_*.parquet")); buf = []
    ids, texts = todo["art_id"].tolist(), todo["text"].tolist()
    from tqdm.auto import tqdm   # progress bar (display only)
    for s in tqdm(range(0, len(texts), B), desc="LLM event extraction", unit="batch"):
        chats = [tok.apply_chat_template([{"role": "user", "content": PROMPT.replace("{article}", t)}],
                                         tokenize=False, add_generation_prompt=True) for t in texts[s:s+B]]
        enc = tok(chats, return_tensors="pt", padding=True, truncation=True, max_length=768).to(DEVICE)
        with torch.no_grad():
            out = lm.generate(**enc, max_new_tokens=128, do_sample=False, pad_token_id=tok.pad_token_id)
        for aid, dec in zip(ids[s:s+B], tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)):
            buf.append({"art_id": aid, **parse_event(dec)})
        if len(buf) >= SH:
            pd.DataFrame(buf).to_parquet(f"{SHARD_DIR}/shard_{shard_no:05d}.parquet", index=False); shard_no += 1; buf = []
    if buf: pd.DataFrame(buf).to_parquet(f"{SHARD_DIR}/shard_{shard_no:05d}.parquet", index=False)
    del lm, tok; gc.collect(); torch.cuda.empty_cache()
events = load_shards().drop_duplicates(subset="art_id")

def create_event_text(row):
    return (f"Subject: {row['Subject']}. Predicate: {row['Predicate']}. Object: {row['Object']}. "
            f"Event Type: {row['Event Type']}. Magnitude: {row['Magnitude']}. "
            f"Temporal Horizon: {row['Temporal Horizon']}.")

# 🔧 [ACL18] smoke-test guard. WHY: allows checking the pipeline where the LLM cannot run (no shards yet).
# With ALLOW_RAW_FALLBACK=False (default) nothing changes; results with the fallback are NOT SCEG results.
if len(events) == 0 and CONFIG.get("ALLOW_RAW_FALLBACK"):
    print("WARNING: no extracted events - SMOKE TEST ONLY: using raw text in place of structured events")
    events = articles[["art_id"]].assign(**{k: "" for k in EMPTY_EVENT})
    _fallback_raw = True
else:
    _fallback_raw = False
ev = articles[["art_id", "Stock_symbol", "eff_date", "text"]].merge(events, on="art_id", how="inner")
ev["event_text"] = ev["text"] if (_fallback_raw or len(ev) == 0) else ev.apply(create_event_text, axis=1)   # 🔧 [v9] empty-set guard
ev = ev[ev["Stock_symbol"].isin(sym_idx)].reset_index(drop=True)
fields = ["Subject","Predicate","Object","Event Type","Magnitude","Temporal Horizon"]
print(f"{len(ev):,} events for the {N}-firm universe | empty-field rates:",
      {f: round(float((ev[f].str.strip() == "").mean()), 3) for f in fields})

already extracted 85,850 | remaining 0 | rough ETA 0.0 h on a T4 (resumable: re-run this cell in later sessions)
85,850 events for the 87-firm universe | empty-field rates: {'Subject': 0.098, 'Predicate': 0.154, 'Object': 0.5, 'Event Type': 0.687, 'Magnitude': 0.62, 'Temporal Horizon': 0.89}


---
## 4 · Event representations → day × firm tensors (cached)
Two representations are built from the same articles: **structured** (serialised six-field events,
the SCEG representation) and **raw** (headline + summary, the control). `bge` is the paper setting;
`tfidf` is an offline fallback (TF-IDF + SVD fitted on the first fold's training dates only).

In [16]:
def encode_texts(texts, kind):
    path = f"{CACHE}/v2/emb_{CONFIG['UNIVERSE_MODE']}_{CONFIG.get('NEWS_ALIGNMENT', '')}_{CONFIG['EVENT_ENCODER']}_{kind}.npy"  # 🔧 [ACL18] + alignment tag
    if os.path.exists(path):
        E = np.load(path)
        if len(E) == len(texts): return E
    if CONFIG["EVENT_ENCODER"] == "bge":
        from sentence_transformers import SentenceTransformer
        enc = SentenceTransformer(CONFIG["EMBED_MODEL"], device=str(DEVICE))
        E = enc.encode(texts, batch_size=CONFIG["EMBED_BATCH_SIZE"], convert_to_numpy=True,
                       show_progress_bar=True, normalize_embeddings=True).astype(np.float16)
        del enc; gc.collect(); torch.cuda.empty_cache()
    else:
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.decomposition import TruncatedSVD
        first_train_end = pd.Timestamp(CONFIG["FOLDS"][0][0])
        fit_mask = (ev["eff_date"] <= first_train_end).values           # no look-ahead
        vec = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), min_df=3, sublinear_tf=True)
        Xtr = vec.fit_transform([t for t, m in zip(texts, fit_mask) if m])
        svd = TruncatedSVD(256, random_state=0).fit(Xtr)
        E = svd.transform(vec.transform(texts))
        E = (E / (np.linalg.norm(E, axis=1, keepdims=True) + 1e-9)).astype(np.float16)
    np.save(path, E)
    return E

def to_tensor(E):
    """Mean-pool article vectors per (effective day, firm); bounded forward-fill; staleness features."""
    D = E.shape[1]
    t_idx = ev["eff_date"].map(pos_of_day).values; n_idx = ev["Stock_symbol"].map(sym_idx).values
    V = np.zeros((T, N, D), np.float32); C = np.zeros((T, N), np.float32)
    np.add.at(V, (t_idx, n_idx), E.astype(np.float32)); np.add.at(C, (t_idx, n_idx), 1.0)
    has = C > 0
    V[has] /= C[has][:, None]
    age = np.full((T, N), CONFIG["NEWS_AGE_CAP"], np.float32)
    last = np.full(N, -10**9)
    for t in range(T):
        last = np.where(has[t], t, last)
        a = t - last
        age[t] = np.minimum(a, CONFIG["NEWS_AGE_CAP"])
        fill = (~has[t]) & (a <= CONFIG["FFILL_LIMIT"]) & (last >= 0)
        if fill.any(): V[t, fill] = V[last[fill], np.where(fill)[0]]
    return V.astype(np.float16), has.astype(np.float32), (age / CONFIG["NEWS_AGE_CAP"]).astype(np.float32)

if RUN_LEGACY:   # v2 representations (sections 6–13); the v3 pipeline in section 14 builds its own
    E_struct = encode_texts(ev["event_text"].tolist(), "struct")
    E_raw    = encode_texts(ev["text"].tolist(), "raw")
    V_STRUCT, HASNEWS, NEWSAGE = to_tensor(E_struct)
    V_RAW, _, _ = to_tensor(E_raw)
    del E_struct, E_raw; gc.collect()
    print("event tensor:", V_STRUCT.shape, "| firm-days with same-day news:", f"{HASNEWS[ROW].mean():.1%}")

event tensor: (653, 87, 768) | firm-days with same-day news: 37.1%


---
## 5 · Walk-forward folds
For each fold: train = [start, train_end], validation = (train_end, val_end], test = (val_end, test_end].
Sample *t* uses days t−W+1..t as input and day t+1 as target; it belongs to a split only if all
input days **and** the target day lie in that split (window embargo + one-day purge).

In [17]:
W_ = CONFIG["WINDOW"]
def split_days(lo, hi):
    return np.where((TRADING_DAYS > pd.Timestamp(lo) if lo else True) & (TRADING_DAYS <= pd.Timestamp(hi)))[0]

# 🔧 [ACL18] split rule. WHY: StockNet assigns a sample to the split of its TARGET date, and a sample's price
# window may reach back before the split start (e.g. the first test samples use late-September inputs).
# Using earlier days only as INPUTS is not leakage: every fitted object still uses training dates only, and no
# target of one split is a target of another. v2's stricter rule (whole window inside the split) would drop
# benchmark test samples and break comparability, so it is kept only for SPLIT_RULE="embargo" (FNSPID).
MIN_T = CONFIG.get("MIN_TARGETS_PER_DAY", 10)
def sample_idx(day_ids):
    s = set(day_ids.tolist()); out = []
    rule = CONFIG.get("SPLIT_RULE", "embargo")
    cand = range(T - 1) if rule == "target_day" else day_ids
    for t in cand:
        if t - W_ + 1 < 0 or t + 1 >= T: continue
        inside = (t + 1) in s if rule == "target_day" else (all((t - k) in s for k in range(W_)) and (t + 1) in s)
        if inside and YMASK[t].sum() >= MIN_T:
            out.append(t)
    return np.array(out, int)

FOLDS = []
for k, (tr_end, va_end, te_end) in enumerate(CONFIG["FOLDS"]):
    tr_days = split_days(None, tr_end)
    if CONFIG.get("TRAIN_START"):          # 🔧 [ACL18] benchmark training period starts 2014-01-01 (earlier = warm-up only)
        tr_days = tr_days[TRADING_DAYS[tr_days] >= pd.Timestamp(CONFIG["TRAIN_START"])]
    va_days = split_days(tr_end, va_end); te_days = split_days(va_end, te_end)
    f = dict(name=f"F{k+1}", test_year=pd.Timestamp(te_end).year, tr_days=tr_days, va_days=va_days, te_days=te_days,
             tr=sample_idx(tr_days), va=sample_idx(va_days), te=sample_idx(te_days))
    FOLDS.append(f)
    if CONFIG.get("DATASET") == "acl18":   # 🔧 [ACL18] check the benchmark samples are all present
        print(f"   benchmark samples  train {int(YMASK[f['tr']].sum()):,} | dev {int(YMASK[f['va']].sum()):,} | "
              f"test {int(YMASK[f['te']].sum()):,}   (StockNet: 20,339 / 2,555 / 3,720)")
    print(f"{f['name']}: train {TRADING_DAYS[tr_days[0]].date()}..{TRADING_DAYS[tr_days[-1]].date()} ({len(f['tr'])}) | "
          f"val ..{TRADING_DAYS[va_days[-1]].date()} ({len(f['va'])}) | test {TRADING_DAYS[te_days[0]].date()}.."
          f"{TRADING_DAYS[te_days[-1]].date()} ({len(f['te'])})")

   benchmark samples  train 20,335 | dev 2,555 | test 3,720   (StockNet: 20,339 / 2,555 / 3,720)
F1: train 2014-01-02..2015-07-31 (398) | val ..2015-09-30 (42) | test 2015-10-01..2015-12-31 (64)


---
## 6 · Per-fold fitted structure: regimes, graphs, feature scaling
### 6.1 Regime model — sticky Gaussian HMM, K by BIC, filtered posteriors

In [18]:
from hmmlearn.hmm import GaussianHMM
import logging; logging.getLogger("hmmlearn").setLevel(logging.ERROR)
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.stats import multivariate_normal
from scipy.special import logsumexp

def fit_regimes(tr_days, seed=0):
    sc = StandardScaler().fit(MARKET_STATE[tr_days]); H = sc.transform(MARKET_STATE)
    Htr = H[tr_days]; best = None
    for K in CONFIG["HMM_K_GRID"]:
        prior = np.ones((K, K)) + np.eye(K) * CONFIG["HMM_STICKY"]
        for r in range(CONFIG["HMM_RESTARTS"]):
            m = GaussianHMM(n_components=K, covariance_type="full", n_iter=300, tol=1e-4,
                            random_state=seed * 100 + r, transmat_prior=prior, init_params="mc", params="stmc")
            m.startprob_ = np.full(K, 1.0 / K)
            m.transmat_ = np.full((K, K), 0.05 / max(K - 1, 1)); np.fill_diagonal(m.transmat_, 0.95)
            try:
                m.fit(Htr); ll = m.score(Htr)
            except Exception:
                continue
            n_par = K * (K - 1) + (K - 1) + K * Htr.shape[1] + K * Htr.shape[1] * (Htr.shape[1] + 1) / 2
            bic = -2 * ll + n_par * np.log(len(Htr))
            if best is None or bic < best[0]: best = (bic, K, m)
    bic, K, m = best
    # filtered posterior p(s_t | m_1..t) by the forward algorithm (no look-ahead)
    logB = np.stack([multivariate_normal(m.means_[k], m.covars_[k], allow_singular=True).logpdf(H)
                     for k in range(K)], axis=1)
    logA = np.log(m.transmat_ + 1e-300); la = np.log(m.startprob_ + 1e-300) + logB[0]
    post = np.zeros((T, K)); post[0] = np.exp(la - logsumexp(la))
    for t in range(1, T):
        la = logsumexp(la[:, None] + logA, axis=0) + logB[t]
        la -= logsumexp(la); post[t] = np.exp(la)
    # order states by mean market return on training days (0 = lowest)
    mu = np.array([np.average(MARKET_STATE[tr_days, 0], weights=post[tr_days, k] + 1e-12) for k in range(K)])
    order = np.argsort(mu); post = post[:, order]
    names = {0: "Bear"}; names[K - 1] = "Bull"
    for j in range(1, K - 1): names[j] = "Neutral" if K == 3 else f"Mid{j}"
    hard = post.argmax(1)
    spells = (np.diff(hard[tr_days]) != 0).sum()
    info = dict(K=K, bic=bic, names=names, mean_ret=mu[order], mean_spell=len(tr_days) / max(spells, 1),
                transmat_diag=np.diag(m.transmat_)[order])
    return post.astype(np.float32), info

### 6.2 Directed dependency graph — DYNOTEARS (contemporaneous + lead–lag) and correlation
DYNOTEARS (Pamfil et al., 2020) is fitted on the fold's training returns. Missing returns (before a
listing) are set to 0 after standardisation, **not** forward-filled. Relations are kept separate:
`contemp` (intra-slice W, j→i same day), `leadlag` (inter-slice A, j at t−k → i at t) and `corr`.
Edges are weighted by |coefficient|, capped at `MAX_IN_DEGREE` strongest incoming edges per node, and
row-normalised. The graph is a *statistical dependency* structure; no causal identification is claimed.

In [19]:
import scipy.linalg as slin
import scipy.optimize as sopt

def dynotears(X, Xlags, lambda_w=0.05, lambda_a=0.05, max_iter=20, h_tol=1e-8, rho_max=1e16, w_threshold=0.05):
    n, d = X.shape; pd_ = Xlags.shape[1]; nW, nA = d * d, pd_ * d
    def _split(params):
        wp, wn, ap, an = np.split(params, [nW, 2 * nW, 2 * nW + nA])
        return (wp - wn).reshape(d, d), (ap - an).reshape(pd_, d)
    def _h(Wm):
        E = slin.expm(Wm * Wm); return np.trace(E) - d, E.T * Wm * 2
    def _obj(params, rho, alpha):
        Wm, Am = _split(params); R = X - X @ Wm - Xlags @ Am
        loss = 0.5 / n * (R ** 2).sum(); Gw = -1.0 / n * X.T @ R; Ga = -1.0 / n * Xlags.T @ R
        h, Gh = _h(Wm)
        obj = loss + 0.5 * rho * h * h + alpha * h + lambda_w * params[:2 * nW].sum() + lambda_a * params[2 * nW:].sum()
        Gw_t = Gw + (rho * h + alpha) * Gh
        grad = np.concatenate([(Gw_t + lambda_w).ravel(), (-Gw_t + lambda_w).ravel(),
                               (Ga + lambda_a).ravel(), (-Ga + lambda_a).ravel()])
        return obj, grad
    params = np.zeros(2 * nW + 2 * nA)
    bnds = ([(0, 0) if i == j else (0, None) for _ in range(2) for i in range(d) for j in range(d)] + [(0, None)] * (2 * nA))
    rho, alpha, h = 1.0, 0.0, np.inf
    for _ in range(max_iter):
        while rho < rho_max:
            sol = sopt.minimize(_obj, params, args=(rho, alpha), method="L-BFGS-B", jac=True, bounds=bnds, options={"maxiter": 100})
            h_new, _ = _h(_split(sol.x)[0])
            if h_new > 0.25 * h: rho *= 10
            else: break
        params, h = sol.x, h_new; alpha += rho * h
        if h <= h_tol or rho >= rho_max: break
    Wm, Am = _split(params)
    Wm[np.abs(Wm) < w_threshold] = 0; Am[np.abs(Am) < w_threshold] = 0
    return Wm, Am, h

def _cap_and_normalise(M):
    """M[i, j] = weight of edge j -> i. Keep top incoming edges per row, then row-normalise."""
    M = np.abs(M).astype(np.float32); np.fill_diagonal(M, 0)
    k = CONFIG["MAX_IN_DEGREE"]
    for i in range(M.shape[0]):
        if (M[i] > 0).sum() > k:
            thr = np.sort(M[i])[-k]; M[i][M[i] < thr] = 0
    s = M.sum(1, keepdims=True); return np.divide(M, s, out=np.zeros_like(M), where=s > 0)

def fit_graphs(tr_days, fold_name):
    path = f"{CACHE}/v2/graph_{CONFIG['UNIVERSE_MODE']}_{fold_name}_{N}.npz"
    if os.path.exists(path):
        z = np.load(path); return {k: z[k] for k in z.files}
    R = np.where(ROW[tr_days], RET[tr_days], np.nan)
    mu, sd = np.nanmean(R, 0), np.nanstd(R, 0) + 1e-9
    Z = np.nan_to_num((R - mu) / sd, nan=0.0)                      # missing -> 0, no forward-fill
    # correlation relation
    C = pd.DataFrame(R).corr(min_periods=60).fillna(0).values.copy(); np.fill_diagonal(C, 0)
    Mc = np.zeros((N, N), np.float32)
    for i in range(N):
        nb = np.argsort(-np.abs(C[i]))[:CONFIG["TOP_K_CORR"]]
        Mc[i, nb] = np.abs(C[i, nb]); Mc[nb, i] = np.abs(C[nb, i])
    # DYNOTEARS
    p = CONFIG["DYNOTEARS_P"]; Xc = Z[p:]
    Xl = np.concatenate([Z[p - k - 1: -k - 1] for k in range(p)], axis=1)
    t0 = time.time()
    Wm, Am, h = dynotears(Xc, Xl, CONFIG["DYNOTEARS_LAMBDA"], CONFIG["DYNOTEARS_LAMBDA_A"],
                          CONFIG["DYNOTEARS_MAX_ITER"], w_threshold=CONFIG["DYNOTEARS_W_THRESHOLD"])
    Mw = Wm.T                                                      # W[j, i]: j -> i
    Ma = sum(np.abs(Am[k * N:(k + 1) * N]) for k in range(p)).T    # A_k[j, i]: j(t-k) -> i(t)
    out = dict(contemp=_cap_and_normalise(Mw), leadlag=_cap_and_normalise(Ma), corr=_cap_and_normalise(Mc),
               raw_nnz=np.array([(Wm != 0).sum(), (Am != 0).sum()]), h=np.array(h))
    print(f"  DYNOTEARS {fold_name}: |W|={int((Wm!=0).sum())} |A|={int((Am!=0).sum())} h={h:.2e} ({time.time()-t0:.0f}s)")
    np.savez(path, **out); return out

def random_like(M, rng):
    nnz = (M > 0).sum(); vals = M[M > 0]
    R = np.zeros_like(M); cand = np.array([(i, j) for i in range(N) for j in range(N) if i != j])
    pick = cand[rng.choice(len(cand), nnz, replace=False)]
    R[pick[:, 0], pick[:, 1]] = rng.permutation(vals)
    s = R.sum(1, keepdims=True); return np.divide(R, s, out=np.zeros_like(R), where=s > 0)

### 6.3 Fold preparation: PCA of event vectors, scaling, regime posteriors, graphs

In [20]:
def prepare_fold(f):
    tr = f["tr_days"]
    post, rinfo = fit_regimes(tr)
    graphs = fit_graphs(tr, f["name"])
    feats = {}
    for kind, V in (("struct", V_STRUCT), ("raw", V_RAW)):
        m = (HASNEWS[tr] > 0) & ROW[tr]
        pca = PCA(CONFIG["PCA_COMPONENTS"], random_state=0).fit(V[tr][m].astype(np.float32))
        Zev = np.zeros((T, N, CONFIG["PCA_COMPONENTS"]), np.float32)
        nz = np.abs(V).sum(-1) > 0
        Zev[nz] = pca.transform(V[nz].astype(np.float32))
        Xall = np.concatenate([P, Zev, HASNEWS[..., None], NEWSAGE[..., None]], axis=-1)
        sc = StandardScaler().fit(Xall[tr][ROW[tr]])
        X = ((Xall - sc.mean_) / (sc.scale_ + 1e-9)).astype(np.float32); X[~ROW] = 0.0
        feats[kind] = X
        if kind == "struct": f["pca_var"] = float(pca.explained_variance_ratio_.sum())
    nP = len(PRICE_FEATURES); nE = CONFIG["PCA_COMPONENTS"] + 2
    f.update(post=post, rinfo=rinfo, graphs=graphs, feats=feats,
             idx_price=np.arange(nP), idx_event=np.arange(nP, nP + nE),
             sigma=float(np.nanstd(np.where(YMASK[tr], YT[tr], np.nan))))  # 🔧 [ACL18] YT (raw target under the benchmark)
    names = rinfo["names"]
    print(f"{f['name']}: K={rinfo['K']} {[names[k] for k in range(rinfo['K'])]} mean spell={rinfo['mean_spell']:.1f}d "
          f"| PCA var={f['pca_var']:.1%} | edges contemp={(graphs['contemp']>0).sum()} "
          f"leadlag={(graphs['leadlag']>0).sum()} corr={(graphs['corr']>0).sum()}")
    return f

for f in (FOLDS if RUN_LEGACY else []):
    prepare_fold(f)

F1: K=3 ['Bear', 'Neutral', 'Bull'] mean spell=20.9d | PCA var=40.0% | edges contemp=273 leadlag=837 corr=583


---
## 7 · Models
**SCEG-RC** (full model). For each day τ in the window and each relation r ∈ {contemp, leadlag, corr}:

h_i = ReLU( Θ_self x_i + Σ_r Σ_k π_{τ,k} Σ_j Â_r[i,j] · g_k(e_j) · Θ_{r,k} x_j ),  g_k(e_j) = σ(u_kᵀ e_j + b_k)

π_τ is the filtered regime posterior, e_j the event features of the *source* firm j, Θ_{r,k} a
relation- and regime-specific propagation map. A 2-layer Transformer over the window and an MLP head
follow. The ablation switches remove one mechanism each.

In [21]:
class SCEG(nn.Module):
    def __init__(self, F, idx_event, A, K, H=64, conditioned=True, event_gate=True, use_graph=True,
                 nhead=4, layers=2, dropout=0.2, W=5):
        super().__init__()
        self.K = K if conditioned else 1; self.conditioned = conditioned; self.event_gate = event_gate
        self.use_graph = use_graph and A is not None and A.shape[0] > 0
        self.register_buffer("idx_event", torch.as_tensor(idx_event, dtype=torch.long))
        self.self_lin = nn.Linear(F, H)
        if self.use_graph:
            self.register_buffer("A", torch.as_tensor(A, dtype=torch.float32))      # R x N x N
            self.R = A.shape[0]
            self.rel_lin = nn.Linear(F, self.R * self.K * H, bias=False)
            if event_gate:
                self.gate = nn.Linear(len(idx_event), self.K)
                nn.init.zeros_(self.gate.weight); nn.init.constant_(self.gate.bias, 2.0)
        self.drop = nn.Dropout(dropout)
        self.pos = nn.Parameter(torch.randn(1, W, H) * 0.02)
        layer = nn.TransformerEncoderLayer(H, nhead, H * 4, dropout, activation="gelu", batch_first=True, norm_first=True)
        self.temporal = nn.TransformerEncoder(layer, layers)
        self.norm = nn.LayerNorm(H)
        self.head = nn.Sequential(nn.Linear(H, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 1))
        self.H = H

    def propagate(self, X, PI):                       # X: B,W,N,F   PI: B,W,K
        B, W, Nn, _ = X.shape
        h = self.self_lin(X)
        if self.use_graph:
            M = self.rel_lin(X).view(B, W, Nn, self.R, self.K, self.H)
            if self.event_gate:
                g = torch.sigmoid(self.gate(X[..., self.idx_event]))               # B,W,N,K
                M = M * g[:, :, :, None, :, None]
            agg = torch.einsum("rij,bwjrkh->bwirkh", self.A, M)                  # B,W,N,R,K,H
            if self.conditioned:
                agg = (agg * PI[:, :, None, None, :, None]).sum(4)
            else:
                agg = agg.sum(4)
            h = h + agg.sum(3)
        return torch.relu(h)

    def forward(self, X, PI):
        B, W, Nn, _ = X.shape
        h = self.drop(self.propagate(X, PI))
        z = h.permute(0, 2, 1, 3).reshape(B * Nn, W, self.H) + self.pos[:, :W]
        z = self.norm(self.temporal(z)[:, -1])
        return self.head(z).view(B, Nn)

class MLPBase(nn.Module):
    def __init__(self, F, H=64, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(F, H), nn.ReLU(), nn.Dropout(dropout), nn.Linear(H, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(self, X, PI): return self.net(X[:, -1]).squeeze(-1)

class GRUBase(nn.Module):
    def __init__(self, F, H=64, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(F, H, batch_first=True); self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(H, 1))
    def forward(self, X, PI):
        B, W, Nn, Fd = X.shape
        o, _ = self.gru(X.permute(0, 2, 1, 3).reshape(B * Nn, W, Fd))
        return self.head(o[:, -1]).view(B, Nn)

### 7.1 Variant grid
Every variant appends the regime posterior π_t to the node features unless `regime_feat=False`, so
**SCEG-RC vs "− conditioning"** isolates *regime acting on propagation* from *regime as information*.

In [22]:
REL_ALL = ("contemp", "leadlag", "corr")
VARIANTS = {
    # --- proposed ---
    "SCEG-RC (full)":            dict(model="sceg", rel=REL_ALL, cond=True,  gate=True,  events="struct"),
    # --- mechanism ablations ---
    "− regime conditioning":     dict(model="sceg", rel=REL_ALL, cond=False, gate=True,  events="struct"),
    "− event gate":              dict(model="sceg", rel=REL_ALL, cond=True,  gate=False, events="struct"),
    "− lead–lag relation":       dict(model="sceg", rel=("contemp", "corr"), cond=True, gate=True, events="struct"),
    "− DYNOTEARS (corr only)":   dict(model="sceg", rel=("corr",), cond=True, gate=True, events="struct"),
    # 🔧 [ACL18] v1's "DYNOTEARS graph only" ablation, requested for the benchmark comparison (config only)
    "DYNOTEARS only":            dict(model="sceg", rel=("contemp", "leadlag"), cond=True, gate=True, events="struct"),
    "random graph (matched)":    dict(model="sceg", rel=REL_ALL, cond=True,  gate=True,  events="struct", random_graph=True),
    "regime placebo (permuted)": dict(model="sceg", rel=REL_ALL, cond=True,  gate=True,  events="struct", placebo=True),
    "− regime entirely":         dict(model="sceg", rel=REL_ALL, cond=False, gate=True,  events="struct", regime_feat=False),
    # --- information ablations ---
    "raw text (no structure)":   dict(model="sceg", rel=REL_ALL, cond=True,  gate=True,  events="raw"),
    "price only (no events)":    dict(model="sceg", rel=REL_ALL, cond=True,  gate=False, events=None),
    # --- v1-style design (regime as feature, single untyped graph) ---
    "SCEG-v1 style":             dict(model="sceg", rel=("union",), cond=False, gate=False, events="struct"),
    # --- learned baselines (same features incl. regime posterior) ---
    "MLP":                       dict(model="mlp", events="struct"),
    "GRU":                       dict(model="gru", events="struct"),
    "LightGBM":                  dict(model="lgbm", events="struct"),
}

---
## 8 · Training (batched days, Huber + IC loss, early stopping on validation IC)

In [23]:
def build_inputs(f, cfg, seed):
    X = f["feats"][cfg["events"] or "struct"]
    if cfg.get("events") is None:
        X = X[..., f["idx_price"]]
    post = f["post"]
    if cfg.get("placebo"):
        rng = np.random.default_rng(1234)                 # one fixed permutation of days, applied to all splits
        post = post[rng.permutation(T)]
    if cfg.get("regime_feat", True):
        X = np.concatenate([X, np.broadcast_to(post[:, None, :], (T, N, post.shape[1]))], axis=-1)
    idx_event = f["idx_event"] if cfg.get("events") else np.array([], int)
    A = None
    if cfg["model"] == "sceg":
        G = f["graphs"]
        if cfg["rel"] == ("union",):
            U = ((G["contemp"] > 0) | (G["leadlag"] > 0) | (G["corr"] > 0)).astype(np.float32)
            s = U.sum(1, keepdims=True); A = np.divide(U, s, out=np.zeros_like(U), where=s > 0)[None]
        else:
            mats = [G[r] for r in cfg["rel"]]
            if cfg.get("random_graph"):
                rng = np.random.default_rng(seed + 99); mats = [random_like(M, rng) for M in mats]
            A = np.stack(mats)
    return X.astype(np.float32), post.astype(np.float32), idx_event, A

def batch(X, post, days):
    xb = np.stack([X[t - W_ + 1: t + 1] for t in days]); pb = np.stack([post[t - W_ + 1: t + 1] for t in days])
    return (torch.as_tensor(xb, device=DEVICE), torch.as_tensor(pb, device=DEVICE),
            torch.as_tensor(YT[days], device=DEVICE), torch.as_tensor(YMASK[days], device=DEVICE))  # 🔧 [ACL18] YT

def loss_fn(pred, y, m, sigma, lam):
    y = y / sigma
    hub = (Fnn.huber_loss(pred, y, reduction="none") * m).sum() / m.sum()
    mf = m.float(); cnt = mf.sum(1).clamp(min=2)
    pm = (pred * mf).sum(1, keepdim=True) / cnt[:, None]; ym = (y * mf).sum(1, keepdim=True) / cnt[:, None]
    pc, yc = (pred - pm) * mf, (y - ym) * mf
    ic = (pc * yc).sum(1) / (pc.pow(2).sum(1).sqrt() * yc.pow(2).sum(1).sqrt() + 1e-8)
    return (1 - lam) * hub + lam * (1 - ic.mean())

def daily_ic(P_, days, rank=False):
    out = []
    for r, t in enumerate(days):
        m = YMASK[t]; p, y = P_[r][m], YT[t][m]          # 🔧 [ACL18] YT (IC unchanged)
        if m.sum() < 5 or np.std(p) == 0: out.append(0.0); continue
        if rank: p, y = pd.Series(p).rank().values, pd.Series(y).rank().values
        v = float(np.corrcoef(p, y)[0, 1])
        out.append(v if np.isfinite(v) else 0.0)   # 🔧 [ACL18] numerically constant predictions on small benchmark cross-sections -> 0 (same convention as constant days)
    return np.array(out)

@torch.no_grad()
def predict(model, X, post, days):
    model.eval(); out = []
    for s in range(0, len(days), 64):
        xb, pb, _, _ = batch(X, post, days[s:s + 64]); out.append(model(xb, pb).cpu().numpy())
    return np.concatenate(out)

def train_torch(f, cfg, seed):
    set_seed(seed)
    X, post, idx_event, A = build_inputs(f, cfg, seed)
    Fdim, K = X.shape[-1], post.shape[1]
    if cfg["model"] == "sceg":
        model = SCEG(Fdim, idx_event, A, K, CONFIG["HIDDEN"], cfg.get("cond", True), cfg.get("gate", True) and len(idx_event) > 0,
                     True, CONFIG["NHEAD"], CONFIG["N_TRANSFORMER_LAYERS"], CONFIG["DROPOUT"], W_)
    elif cfg["model"] == "mlp": model = MLPBase(Fdim, CONFIG["HIDDEN"], CONFIG["DROPOUT"])
    else: model = GRUBase(Fdim, CONFIG["HIDDEN"], CONFIG["DROPOUT"])
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=CONFIG["LR"], weight_decay=CONFIG["WEIGHT_DECAY"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CONFIG["EPOCHS"])
    rng = np.random.default_rng(seed); best, best_state, bad = -np.inf, None, 0
    for ep in range(CONFIG["EPOCHS"]):
        model.train()
        order = rng.permutation(f["tr"])
        for s in range(0, len(order), CONFIG["BATCH_DAYS"]):
            xb, pb, yb, mb = batch(X, post, order[s:s + CONFIG["BATCH_DAYS"]])
            opt.zero_grad()
            loss = loss_fn(model(xb, pb), yb, mb, f["sigma"], CONFIG["IC_LOSS_WEIGHT"])
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        sched.step()
        v = daily_ic(predict(model, X, post, f["va"]), f["va"]).mean()
        if v > best + 1e-5:
            best, bad = v, 0; best_state = {k: t.detach().cpu().clone() for k, t in model.state_dict().items()}
        else:
            bad += 1
            if bad >= CONFIG["PATIENCE"]: break
    model.load_state_dict(best_state)
    return predict(model, X, post, f["te"]), best, model

def train_lgbm(f, cfg, seed):
    import lightgbm as lgb
    X, post, _, _ = build_inputs(f, cfg, seed)
    def flat(days):
        cur = X[days]; mean = np.stack([X[t - W_ + 1: t + 1].mean(0) for t in days])
        return np.concatenate([cur, mean], -1)
    def rows(days):
        Z = flat(days); m = YMASK[days]
        return Z[m], (YT[days] / f["sigma"])[m]    # 🔧 [ACL18] YT
    Xtr, ytr = rows(f["tr"]); Xva, yva = rows(f["va"])
    mdl = lgb.LGBMRegressor(n_estimators=3000, learning_rate=0.02, num_leaves=15, min_child_samples=200,
                            subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0,
                            random_state=seed, verbose=-1)
    mdl.fit(Xtr, ytr, eval_set=[(Xva, yva)], callbacks=[lgb.early_stopping(100, verbose=False)])
    Zte = flat(f["te"]); P_ = mdl.predict(Zte.reshape(-1, Zte.shape[-1])).reshape(len(f["te"]), N)
    return P_, float(daily_ic(mdl.predict(flat(f["va"]).reshape(-1, Zte.shape[-1])).reshape(len(f["va"]), N), f["va"]).mean()), mdl

---
## 9 · Run the grid (resumable: predictions cached per variant × fold × seed)

In [24]:
PRED_DIR = f"{CACHE}/v2/preds_{CONFIG['UNIVERSE_MODE']}_{CONFIG.get('NEWS_ALIGNMENT', '')}_{CONFIG['EVENT_ENCODER']}"  # 🔧 [ACL18] + alignment tag
os.makedirs(PRED_DIR, exist_ok=True)   # 🐞 fix: this call had been swallowed by the comment above
MODELS = {}
def slug(s): return re.sub(r"[^A-Za-z0-9]+", "_", s).strip("_")

RUN_VARIANTS = list(VARIANTS)
if os.environ.get("SCEG_VARIANTS"):
    RUN_VARIANTS = [v for v in VARIANTS if slug(v) in os.environ["SCEG_VARIANTS"].split(",")]
t_start = time.time()
for f in (FOLDS if RUN_LEGACY else []):
    for vname in RUN_VARIANTS:
        cfg = VARIANTS[vname]
        for seed in CONFIG["SEEDS"]:
            path = f"{PRED_DIR}/{slug(vname)}__{f['name']}__s{seed}.npz"
            if os.path.exists(path): continue
            t0 = time.time()
            if cfg["model"] == "lgbm": P_, vic, mdl = train_lgbm(f, cfg, seed)
            else:
                P_, vic, mdl = train_torch(f, cfg, seed)
                if vname == "SCEG-RC (full)": MODELS[(f["name"], seed)] = mdl.cpu()
            np.savez(path, pred=P_.astype(np.float32), val_ic=vic)
            print(f"{f['name']} | {vname:28s} | seed {seed} | val IC {vic:+.4f} | {time.time()-t0:5.0f}s", flush=True)
print(f"grid done in {(time.time()-t_start)/60:.1f} min")

grid done in 0.0 min


---
## 10 · Evaluation
All metrics on the **test days of the three folds pooled** (plus per-fold). IC/Rank-IC are daily
cross-sectional correlations (constant-prediction days count as 0). DirAcc is the accuracy of
predicting whether a firm beats the cross-sectional mean (base rate ≈ 50 %). LS = equal-weight
top-minus-bottom decile of raw next-day returns; net Sharpe charges `COST_BPS` per side on turnover.
Newey–West (5 lags) t-statistics; paired tests use the daily IC difference.

In [25]:
def nw_t(x, L=5):
    x = np.asarray(x, float); T_ = len(x); d = x - x.mean()
    g = [np.sum(d[k:] * d[:T_ - k]) / T_ for k in range(L + 1)]
    v = g[0] + 2 * sum((1 - k / (L + 1)) * g[k] for k in range(1, L + 1))
    return x.mean() / np.sqrt(max(v, 1e-18) / T_)

def ls_series(P_, days, cost_bps):
    k = max(1, N // 10); gross, turn = [], []; prevL = prevS = None
    for r, t in enumerate(days):
        m = np.where(YMASK[t])[0]
        if len(m) < 20 or np.std(P_[r][m]) == 0: gross.append(0.0); turn.append(0.0); continue
        o = m[np.argsort(P_[r][m])]; L_, S_ = set(o[-k:]), set(o[:k])
        gross.append(Y0[t][list(L_)].mean() - Y0[t][list(S_)].mean())
        turn.append(1.0 if prevL is None else (len(L_ - prevL) + len(S_ - prevS)) / (2 * k)); prevL, prevS = L_, S_
    gross, turn = np.array(gross), np.array(turn)
    return gross, gross - 4 * cost_bps / 1e4 * turn

def metrics(P_, days):
    ic = daily_ic(P_, days); ric = daily_ic(P_, days, rank=True)
    m = YMASK[days]; pc = P_ - np.nanmean(np.where(m, P_, np.nan), 1, keepdims=True)
    y = YXS[days]
    diracc = ((pc > 0) == (y > 0))[m].mean() * 100; base = max((y > 0)[m].mean(), 1 - (y > 0)[m].mean()) * 100
    gross, net = ls_series(P_, days, CONFIG["COST_BPS"])
    sh = lambda x: x.mean() / (x.std() + 1e-12) * np.sqrt(252)
    return dict(IC=ic.mean(), IC_t=nw_t(ic), RankIC=ric.mean(), RankIC_t=nw_t(ric), ICIR=ic.mean() / (ic.std() + 1e-12),
                coverage=(np.array([np.std(P_[r][YMASK[t]]) > 0 for r, t in enumerate(days)])).mean(),
                DirAcc=diracc, Base=base, LS_gross=sh(gross), LS_net=sh(net), LS_t=nw_t(gross)), ic

# 🔧 [ACL18] benchmark classification metrics. WHY: StockNet-protocol papers report ACC and MCC on the
# benchmark test samples (mean ± sd over runs). Predicted class = 1 if the predicted movement > 0 (the same
# decision rule as the label, movement > 1e-7). Only meaningful with TARGET_MODE="raw".
from sklearn.metrics import matthews_corrcoef
def bench_cls(P_, days):
    m = YMASK[days] & (YCLS[days] >= 0)
    y = YCLS[days][m].astype(int); yhat = (P_[m] > 0).astype(int)
    mcc = matthews_corrcoef(y, yhat) if (y.std() > 0 and yhat.std() > 0) else 0.0
    return dict(ACC=(y == yhat).mean() * 100, MCC=mcc, n=int(m.sum()), pred_up=yhat.mean() * 100)

def load_preds(vname, fold, seeds):
    Ps = [np.load(f"{PRED_DIR}/{slug(vname)}__{fold['name']}__s{s}.npz")["pred"] for s in seeds
          if os.path.exists(f"{PRED_DIR}/{slug(vname)}__{fold['name']}__s{s}.npz")]
    return Ps

def zscore_rows(P_):
    return (P_ - P_.mean(1, keepdims=True)) / (P_.std(1, keepdims=True) + 1e-12)

if RUN_LEGACY:
    rows, ic_series = [], {}
    days_all = np.concatenate([f["te"] for f in FOLDS])
    for vname in VARIANTS:
        per_seed = []
        for s in CONFIG["SEEDS"]:
            Ps = [load_preds(vname, f, [s]) for f in FOLDS]
            if any(len(p) == 0 for p in Ps): continue
            Pcat = np.concatenate([p[0] for p in Ps])
            per_seed.append({**metrics(Pcat, days_all)[0], **bench_cls(Pcat, days_all)})   # 🔧 [ACL18] + ACC/MCC
        if not per_seed: continue
        ens = [load_preds(vname, f, CONFIG["SEEDS"]) for f in FOLDS]
        Pens = np.concatenate([np.mean([zscore_rows(p) for p in e], 0) for e in ens])
        me, ic_series[vname] = metrics(Pens, days_all)
        df = pd.DataFrame(per_seed)
        bc = bench_cls(np.concatenate([np.mean(e, 0) for e in ens]), days_all)  # 🔧 [ACL18] raw-mean ensemble for ACC/MCC
        rows.append({"variant": vname, "seeds": len(per_seed),
                     "ACC (mean±sd)": f"{df.ACC.mean():.2f}±{df.ACC.std(ddof=0):.2f}",
                     "MCC (mean±sd)": f"{df.MCC.mean():.4f}±{df.MCC.std(ddof=0):.4f}",
                     "pred-up %": df.pred_up.mean(), "ens ACC": bc["ACC"], "ens MCC": bc["MCC"], "n test": bc["n"],
                     "IC (mean±sd)": f"{df.IC.mean():.4f}±{df.IC.std(ddof=0):.4f}",
                     "RankIC (mean±sd)": f"{df.RankIC.mean():.4f}±{df.RankIC.std(ddof=0):.4f}",
                     "ens IC": me["IC"], "ens IC t": me["IC_t"], "ens RankIC": me["RankIC"], "ens ICIR": me["ICIR"],
                     "ens DirAcc": me["DirAcc"], "ens LS gross": me["LS_gross"], f"ens LS net {CONFIG['COST_BPS']}bp": me["LS_net"],
                     "coverage": me["coverage"]})

    # naive baselines
    for name, fn in {"lag-1 reversal": lambda t: -RET[t], "lag-1 momentum": lambda t: RET[t]}.items():
        Pn = np.nan_to_num(np.stack([fn(t) for t in days_all]))
        me, ic_series[name] = metrics(Pn, days_all)
        bc = bench_cls(Pn, days_all)                                              # 🔧 [ACL18]
        rows.append({"variant": name, "seeds": 0, "ens ACC": bc["ACC"], "ens MCC": bc["MCC"], "n test": bc["n"],
                     "pred-up %": bc["pred_up"], "ens IC": me["IC"], "ens IC t": me["IC_t"], "ens RankIC": me["RankIC"],
                     "ens ICIR": me["ICIR"], "ens DirAcc": me["DirAcc"], "ens LS gross": me["LS_gross"],
                     f"ens LS net {CONFIG['COST_BPS']}bp": me["LS_net"], "coverage": me["coverage"]})
    # 🔧 [ACL18] majority-class baseline: always predict the majority class of the TRAINING labels
    if CONFIG.get("DATASET") == "acl18":
        tr_all = np.concatenate([f["tr"] for f in FOLDS]); up_tr = YCLS[tr_all][YMASK[tr_all]].mean()
        bc = bench_cls(np.full((len(days_all), N), 1.0 if up_tr > 0.5 else -1.0), days_all)
        rows.append({"variant": f"majority class (train up-rate {up_tr:.3f})", "seeds": 0, "ens ACC": bc["ACC"],
                     "ens MCC": bc["MCC"], "n test": bc["n"], "pred-up %": bc["pred_up"]})
    RESULTS = pd.DataFrame(rows).set_index("variant")
    pd.set_option("display.width", 250)
    print(f"pooled test days: {len(days_all)} | firms: {N}")
    print(RESULTS.round(4).to_string())
    RESULTS.to_csv(f"{CACHE}/v2/results_pooled_{CONFIG.get('NEWS_ALIGNMENT', '')}_{CONFIG['EVENT_ENCODER']}.csv")

pooled test days: 64 | firms: 87
                                      seeds ACC (mean±sd)   MCC (mean±sd)  pred-up %  ens ACC  ens MCC  n test   IC (mean±sd) RankIC (mean±sd)  ens IC  ens IC t  ens RankIC  ens ICIR  ens DirAcc  ens LS gross  ens LS net 5bp  coverage
variant                                                                                                                                                                                                                                   
SCEG-RC (full)                            5    50.82±0.91   0.0157±0.0168    52.5215  51.4516   0.0262    3720  0.0409±0.0239    0.0401±0.0207  0.0643    2.9385      0.0505    0.3199     51.7473        5.0784          2.9322    1.0000
− regime conditioning                     5    51.24±1.99   0.0275±0.0343    45.2742  52.0699   0.0450    3720  0.0589±0.0138    0.0560±0.0161  0.0896    3.5057      0.0805    0.4429     53.0108        7.6476          5.7968    1.0000
− event gate               

### 10.1 Paired tests against SCEG-RC (daily IC difference, Newey–West t)

In [26]:
if RUN_LEGACY:
    ref = "SCEG-RC (full)"; paired = []
    if ref in ic_series:
        for v, s in ic_series.items():
            if v == ref: continue
            d = ic_series[ref] - s
            paired.append({"comparison": f"SCEG-RC − {v}", "ΔIC": d.mean(), "NW t": nw_t(d),
                           "share of days SCEG-RC better": (d > 0).mean()})
    PAIRED = pd.DataFrame(paired).set_index("comparison") if paired else pd.DataFrame()
    print(PAIRED.round(4).to_string())
    PAIRED.to_csv(f"{CACHE}/v2/paired_{CONFIG.get('NEWS_ALIGNMENT', '')}_{CONFIG['EVENT_ENCODER']}.csv")

                                        ΔIC    NW t  share of days SCEG-RC better
comparison                                                                       
SCEG-RC − − regime conditioning     -0.0253 -1.4750                        0.3906
SCEG-RC − − event gate              -0.0107 -0.5256                        0.4531
SCEG-RC − − lead–lag relation        0.0022  0.1275                        0.5469
SCEG-RC − − DYNOTEARS (corr only)    0.0056  0.3463                        0.4531
SCEG-RC − DYNOTEARS only             0.0256  1.8851                        0.6094
SCEG-RC − random graph (matched)    -0.0066 -0.3616                        0.4219
SCEG-RC − regime placebo (permuted)  0.0110  0.6229                        0.5625
SCEG-RC − − regime entirely         -0.0166 -0.8043                        0.4688
SCEG-RC − raw text (no structure)   -0.0211 -1.3775                        0.4062
SCEG-RC − price only (no events)     0.0210  1.1069                        0.5625
SCEG-RC − SCEG-v

### 10.2 Per-fold results (seed-ensemble)

In [27]:
if RUN_LEGACY:
    fold_rows = []
    for vname in VARIANTS:
        for f in FOLDS:
            e = load_preds(vname, f, CONFIG["SEEDS"])
            if not e: continue
            me, _ = metrics(np.mean([zscore_rows(p) for p in e], 0), f["te"])
            fold_rows.append({"variant": vname, "fold": f"{f['name']} ({f['test_year']})", "IC": me["IC"], "IC t": me["IC_t"],
                              "RankIC": me["RankIC"], "LS net": me["LS_net"]})
    FOLD_TABLE = pd.DataFrame(fold_rows).pivot(index="variant", columns="fold", values="IC")
    print(FOLD_TABLE.round(4).to_string())
    pd.DataFrame(fold_rows).to_csv(f"{CACHE}/v2/results_by_fold_{CONFIG.get('NEWS_ALIGNMENT', '')}_{CONFIG['EVENT_ENCODER']}.csv", index=False)

fold                       F1 (2015)
variant                             
DYNOTEARS only                0.0386
GRU                           0.0789
LightGBM                      0.0359
MLP                           0.0553
SCEG-RC (full)                0.0643
SCEG-v1 style                 0.0822
price only (no events)        0.0432
random graph (matched)        0.0709
raw text (no structure)       0.0854
regime placebo (permuted)     0.0533
− DYNOTEARS (corr only)       0.0587
− event gate                  0.0749
− lead–lag relation           0.0621
− regime conditioning         0.0896
− regime entirely             0.0809


---
## 11 · Regime analysis
(a) IC of SCEG-RC vs "− regime conditioning" by dominant regime of the input day. (b) Learned
propagation strength per relation and regime: ‖Θ_{r,k}‖_F averaged over seeds (only from models
trained in this session).

In [28]:
if RUN_LEGACY:
    reg_rows = []
    for vname in ["SCEG-RC (full)", "− regime conditioning", "regime placebo (permuted)"]:
        for f in FOLDS:
            e = load_preds(vname, f, CONFIG["SEEDS"])
            if not e: continue
            Pe = np.mean([zscore_rows(p) for p in e], 0); ic = daily_ic(Pe, f["te"])
            hard = f["post"][f["te"]].argmax(1)
            for k in np.unique(hard):
                reg_rows.append({"variant": vname, "fold": f["name"], "regime": f["rinfo"]["names"][k],
                                 "days": int((hard == k).sum()), "IC_sum": ic[hard == k].sum()})
    if reg_rows:
        R_ = pd.DataFrame(reg_rows).groupby(["variant", "regime"])[["days", "IC_sum"]].sum()
        R_["IC"] = R_["IC_sum"] / R_["days"]
        print(R_[["days", "IC"]].round(4).to_string())

    strength = []
    for (fname, seed), mdl in MODELS.items():
        Wt = mdl.rel_lin.weight.detach().view(mdl.R, mdl.K, mdl.H, -1)
        f = [x for x in FOLDS if x["name"] == fname][0]
        for r, rel in enumerate(REL_ALL):
            for k in range(mdl.K):
                strength.append({"fold": fname, "relation": rel, "regime": f["rinfo"]["names"][k],
                                 "norm": float(Wt[r, k].norm())})
    if strength:
        print(pd.DataFrame(strength).groupby(["relation", "regime"])["norm"].mean().unstack().round(3).to_string())

                                   days      IC
variant                   regime               
SCEG-RC (full)            Bull       56  0.0715
                          Neutral     8  0.0139
regime placebo (permuted) Bull       56  0.0667
                          Neutral     8 -0.0405
− regime conditioning     Bull       56  0.0986
                          Neutral     8  0.0259


### 11.1 Direct, model-free test of the propagation hypothesis
Independent of any neural network: does *news-day information of a firm's graph neighbours* predict
the firm's next-day relative return, and does that depend on the regime? For relation r the signal is
S_i,t = Σ_j Â_r[i,j] · HasNews_j,t · x_j,t, where x_j,t is j's cross-sectionally demeaned return on
its news day (the market's first reaction to j's news). As a control, the same signal is built from
neighbours **without** news that day. Each fold uses its own training-period graph; statistics are on
the test days of all folds pooled (Newey–West t), and split by the filtered regime.

In [29]:
if RUN_LEGACY:
    xs_ret = np.nan_to_num(np.where(ROW, RET, np.nan) - np.nanmean(np.where(ROW, RET, np.nan), 1, keepdims=True))
    def signal_ic(S, days):
        out = []
        for t in days:
            m = YMASK[t]
            if m.sum() < 8 or np.std(S[t][m]) == 0: out.append(0.0); continue
            out.append(np.corrcoef(S[t][m], YXS[t][m])[0, 1])
        return np.array(out)

    prop_rows = []
    for rel in REL_ALL:
        for label, gate in (("neighbours WITH news", HASNEWS), ("neighbours WITHOUT news", 1 - HASNEWS)):
            ics, regs = [], []
            for f in FOLDS:
                S = (xs_ret * gate) @ f["graphs"][rel].T
                ics.append(signal_ic(S, f["te"]))
                regs.append(np.array([f["rinfo"]["names"][k] for k in f["post"][f["te"]].argmax(1)]))
            ic_, rg = np.concatenate(ics), np.concatenate(regs)
            row = {"relation": rel, "signal": label, "IC": ic_.mean(), "NW t": nw_t(ic_)}
            for name in sorted(set(rg)):
                row[f"IC | {name}"] = ic_[rg == name].mean(); row[f"days | {name}"] = int((rg == name).sum())
            prop_rows.append(row)
    PROP = pd.DataFrame(prop_rows).set_index(["relation", "signal"])
    print(PROP.round(4).to_string())
    PROP.to_csv(f"{CACHE}/v2/propagation_test_{CONFIG['UNIVERSE_MODE']}_{CONFIG.get('NEWS_ALIGNMENT', '')}.csv")

                                      IC    NW t  IC | Bull  days | Bull  IC | Neutral  days | Neutral
relation signal                                                                                       
contemp  neighbours WITH news     0.0245  1.2559     0.0227           56        0.0371               8
         neighbours WITHOUT news  0.0439  1.7850     0.0373           56        0.0903               8
leadlag  neighbours WITH news     0.0279  1.7772     0.0298           56        0.0152               8
         neighbours WITHOUT news -0.0030 -0.1527     0.0013           56       -0.0331               8
corr     neighbours WITH news     0.0457  2.1205     0.0438           56        0.0586               8
         neighbours WITHOUT news  0.0553  1.9497     0.0503           56        0.0908               8


---
## 12 · Reading the results honestly
* The **primary claim** is tested by *SCEG-RC vs − regime conditioning* and *SCEG-RC vs regime
  placebo*: if regime-dependent propagation matters, both paired ΔIC should be positive with |t| ≳ 2.
* *− event gate*, *raw text* and *price only* test whether event content, and its structure, drive
  propagation.
* *− lead–lag*, *corr only* and *random graph* test whether the DYNOTEARS directed dependencies matter.
* Report the seed mean ± sd, the ensemble, the paired t, the net-of-cost Sharpe and the per-fold
  stability. A variant that wins in one fold only is not evidence.
* Every component of the grid was fixed before the test folds were inspected. Do not tune
  hyperparameters on these results; if you tune, use the validation IC printed during training.

---
## 13 · Diagnostic analyses (pre-specified: they explain *where* the signal comes from)
These analyses were fixed **before** their results were seen. 13.1–13.3 reuse the cached predictions (seconds);
13.4 retrains five variants at longer horizons (≈1–2 h on a T4, resumable). None of them changes the benchmark
protocol or the §10 numbers.

| # | Question | Test |
|---|---|---|
| 13.1 | Do events add information beyond prices, and does *structure* add beyond raw text? | paired daily ΔIC |
| 13.2 | Is the lead–lag signal specific to neighbours that *had news*? (the propagation hypothesis) | model-free ΔIC, with − without news |
| 13.3 | Is the signal priced **overnight** (close→open) or **intraday** (open→close)? | IC against each part of the target |
| 13.4 | Does the signal, and the lead–lag gain, appear at h = 2, 3, 5 days? | IC and paired ΔIC per horizon, Holm-corrected |
| 13.5 | Benchmark vs strict (16:00 ET) news alignment | side-by-side, once both runs exist |

**Run the strict alignment:** in the config cell set `"NEWS_ALIGNMENT": "strict"`, then Runtime → Run all.
All caches (event shards, embeddings, predictions, result files) are kept separately per alignment.

In [30]:
if RUN_LEGACY:
    # ---- 13.1 / 13.2 · paired tests ----
    from scipy.stats import norm
    def ptest(d, L=5):
        t = nw_t(d, L)
        return dict(dIC=float(np.mean(d)), NW_t=float(t), p=float(2 * (1 - norm.cdf(abs(t)))), share_pos=float((d > 0).mean()))

    diag = []
    for a, b in [("raw text (no structure)", "price only (no events)"),
                 ("SCEG-RC (full)", "price only (no events)"),
                 ("SCEG-RC (full)", "raw text (no structure)")]:
        if a in ic_series and b in ic_series:
            diag.append({"test": f"{a}  −  {b}", **ptest(ic_series[a] - ic_series[b])})
    ll = {}
    for label, gate in (("with", HASNEWS), ("without", 1 - HASNEWS)):
        ll[label] = np.concatenate([signal_ic((xs_ret * gate) @ f["graphs"]["leadlag"].T, f["te"]) for f in FOLDS])
    diag.append({"test": "lead–lag neighbours WITH news  −  WITHOUT news (model-free)", **ptest(ll["with"] - ll["without"])})
    DIAG = pd.DataFrame(diag).set_index("test")
    print(DIAG.round(4).to_string())
    DIAG.to_csv(f"{CACHE}/v2/diag_paired_{CONFIG.get('NEWS_ALIGNMENT', '')}_{CONFIG['EVENT_ENCODER']}.csv")

                                                                dIC    NW_t       p  share_pos
test                                                                                          
raw text (no structure)  −  price only (no events)           0.0422  1.5389  0.1238     0.5938
SCEG-RC (full)  −  price only (no events)                    0.0210  1.1069  0.2683     0.5625
SCEG-RC (full)  −  raw text (no structure)                  -0.0211 -1.3775  0.1684     0.4062
lead–lag neighbours WITH news  −  WITHOUT news (model-free)  0.0309  1.2370  0.2161     0.5938


In [31]:
if RUN_LEGACY:
    # ---- 13.3 · overnight vs intraday decomposition of the target return ----
    # target(t) = adjclose(t+1)/adjclose(t) - 1 = (1 + overnight)(1 + intraday) - 1, with
    # overnight = adjopen(t+1)/adjclose(t) - 1 and intraday = close(t+1)/open(t+1) - 1 (adjopen = open * adjclose/close).
    _ti = px["Date"].map(pos_of_day).values; _ni = px["Stock_symbol"].map(sym_idx).values
    OPENA = np.full((T, N), np.nan); CLOSEA = np.full((T, N), np.nan)
    OPENA[_ti, _ni] = (px["open"] * px["adj_close"] / px["close"]).values; CLOSEA[_ti, _ni] = px["adj_close"].values
    ON = np.full((T, N), np.nan); ID = np.full((T, N), np.nan)
    ON[:-1] = OPENA[1:] / CLOSEA[:-1] - 1
    ID[:-1] = CLOSEA[1:] / OPENA[1:] - 1
    _chk = np.abs(((1 + ON) * (1 + ID) - 1) - Y)[YMASK]
    print(f"check (1+overnight)(1+intraday)-1 = target: median |diff| {np.nanmedian(_chk):.1e} "
          f"| share of firm-days that match (<1e-4): {(_chk < 1e-4).mean():.3f}")

    def ic_vs(P_, days, Ytgt):
        out = []
        for r, t in enumerate(days):
            m = YMASK[t] & np.isfinite(Ytgt[t])
            if m.sum() < 5 or np.std(P_[r][m]) == 0: out.append(0.0); continue
            v = np.corrcoef(P_[r][m], Ytgt[t][m])[0, 1]; out.append(v if np.isfinite(v) else 0.0)
        return np.array(out)

    _m = YMASK[days_all]
    print(f"test variance share: overnight {np.nanvar(ON[days_all][_m]) / np.nanvar(Y[days_all][_m]):.2f} | "
          f"intraday {np.nanvar(ID[days_all][_m]) / np.nanvar(Y[days_all][_m]):.2f}")
    dec_rows = []
    for v in ["SCEG-RC (full)", "− regime conditioning", "raw text (no structure)", "price only (no events)", "GRU", "MLP", "lag-1 momentum"]:
        if v == "lag-1 momentum":
            Pv = np.nan_to_num(np.stack([RET[t] for t in days_all]))
        else:
            ens = [load_preds(v, f, CONFIG["SEEDS"]) for f in FOLDS]
            if any(len(e) == 0 for e in ens): continue
            Pv = np.concatenate([np.mean([zscore_rows(p) for p in e], 0) for e in ens])
        r = {"variant": v}
        for name, Yp in (("full", Y), ("overnight", ON), ("intraday", ID)):
            ic = ic_vs(Pv, days_all, Yp); r[f"IC {name}"] = ic.mean(); r[f"t {name}"] = nw_t(ic)
        dec_rows.append(r)
    DECOMP = pd.DataFrame(dec_rows).set_index("variant")
    print(DECOMP.round(4).to_string())
    DECOMP.to_csv(f"{CACHE}/v2/decomposition_{CONFIG.get('NEWS_ALIGNMENT', '')}_{CONFIG['EVENT_ENCODER']}.csv")

check (1+overnight)(1+intraday)-1 = target: median |diff| 2.2e-10 | share of firm-days that match (<1e-4): 1.000
test variance share: overnight 0.28 | intraday 0.65
                         IC full  t full  IC overnight  t overnight  IC intraday  t intraday
variant                                                                                     
SCEG-RC (full)            0.0643  2.9385        0.0575       1.8681       0.0414      2.3215
− regime conditioning     0.0896  3.5057        0.0717       2.0858       0.0608      4.0283
raw text (no structure)   0.0854  3.5259        0.0500       1.3903       0.0685      3.8377
price only (no events)    0.0432  1.6218        0.0177       0.5764       0.0374      1.5783
GRU                       0.0789  2.7741        0.0494       1.3744       0.0619      2.6852
MLP                       0.0553  1.4854        0.0623       1.9339       0.0242      0.7148
lag-1 momentum            0.0370  1.0061        0.0828       3.8443      -0.0221     -0.584

In [32]:
# ---- 13.4 · horizon decay: cumulative return over t+1..t+h ----
# All firm-days with a complete h-day return are used (the StockNet ±0.5 % filter is defined for h = 1 only),
# so the h = 1 row here differs slightly from §10. Splits purge h days: a sample needs t+1 and t+h in its split.
# Model, loss, features and training are unchanged; only the target and the sample set change.
RUN_DECAY = False   # set True to run the horizon-decay analysis (≈1–2 h on a T4)
DECAY_H = [int(x) for x in os.environ.get("SCEG_DECAY_H", "1,2,3,5").split(",")]
DECAY_VARIANTS = ["SCEG-RC (full)", "− lead–lag relation", "raw text (no structure)", "price only (no events)", "GRU"]
if os.environ.get("SCEG_DECAY_VARIANTS"):
    DECAY_VARIANTS = [v for v in DECAY_VARIANTS if slug(v) in os.environ["SCEG_DECAY_VARIANTS"].split(",")]

def cum_target(h):
    R1 = np.where(ROW, RET, np.nan); Yh = np.full((T, N), np.nan)
    for t in range(T - h):
        Yh[t] = np.prod(1 + R1[t + 1: t + h + 1], axis=0) - 1
    return Yh

def idx_h(day_ids, h, M):
    s = set(day_ids.tolist()); out = []
    embargo = CONFIG.get("SPLIT_RULE", "embargo") != "target_day"
    for t in range(W_ - 1, T - h):
        if (t + 1) not in s or (t + h) not in s: continue
        if embargo and not all((t - k) in s for k in range(W_)): continue
        if M[t].sum() >= MIN_T: out.append(t)
    return np.array(out, int)

def holm(ps):
    ps = np.asarray(ps); o = np.argsort(ps); adj = np.empty_like(ps); run = 0.0
    for r, i in enumerate(o):
        run = max(run, min(1.0, (len(ps) - r) * ps[i])); adj[i] = run
    return adj

if RUN_DECAY:
    _YT0, _YMASK0 = YT, YMASK
    dec_ic, t0d = {}, time.time()
    try:
        for h in DECAY_H:
            Yh = cum_target(h); Mh = ROW & np.isfinite(Yh)
            YT = np.where(Mh, Yh, 0.0).astype(np.float32); YMASK = Mh          # globals used by training/IC code
            fhs = []
            for f in FOLDS:
                fh = dict(f); fh.update(tr=idx_h(f["tr_days"], h, Mh), va=idx_h(f["va_days"], h, Mh),
                                        te=idx_h(f["te_days"], h, Mh),
                                        sigma=float(np.nanstd(np.where(Mh[f["tr_days"]], Yh[f["tr_days"]], np.nan))))
                fhs.append(fh)
            for v in DECAY_VARIANTS:
                cfg = VARIANTS[v]; per_fold = []
                for fh in fhs:
                    Ps = []
                    for seed in CONFIG["SEEDS"]:
                        path = f"{PRED_DIR}/decay_h{h}__{slug(v)}__{fh['name']}__s{seed}.npz"
                        if not os.path.exists(path):
                            P_, vic, _ = (train_lgbm if cfg["model"] == "lgbm" else train_torch)(fh, cfg, seed)
                            np.savez(path, pred=P_.astype(np.float32), val_ic=vic)
                            print(f"h={h} | {v:26s} | {fh['name']} | seed {seed} | val IC {vic:+.4f}", flush=True)
                        Ps.append(np.load(path)["pred"])
                    per_fold.append(daily_ic(np.mean([zscore_rows(p) for p in Ps], 0), fh["te"]))
                dec_ic[(h, v)] = np.concatenate(per_fold)
    finally:
        YT, YMASK = _YT0, _YMASK0                                                # restore the benchmark target
    print(f"decay runs done in {(time.time() - t0d) / 60:.1f} min")
    DECAY = pd.DataFrame({v: {f"h={h}": f"{dec_ic[(h, v)].mean():+.4f} (t {nw_t(dec_ic[(h, v)]):+.2f})"
                              for h in DECAY_H} for v in DECAY_VARIANTS}).T
    print("\nIC of the seed-ensemble by horizon (Newey–West t):"); print(DECAY.to_string())
    prs = []
    for a, b in [("SCEG-RC (full)", "− lead–lag relation"), ("SCEG-RC (full)", "price only (no events)"),
                 ("raw text (no structure)", "price only (no events)")]:
        if a not in DECAY_VARIANTS or b not in DECAY_VARIANTS: continue
        rows_ = [{"comparison": f"{a}  −  {b}", "h": h, **ptest(dec_ic[(h, a)] - dec_ic[(h, b)])} for h in DECAY_H]
        adj = holm([r["p"] for r in rows_])
        for r, q in zip(rows_, adj): r["p Holm (over h)"] = q
        prs += rows_
    if prs:
        DECAY_PAIRED = pd.DataFrame(prs).set_index(["comparison", "h"])
        print("\nPaired daily ΔIC by horizon:"); print(DECAY_PAIRED.round(4).to_string())
        DECAY_PAIRED.to_csv(f"{CACHE}/v2/decay_paired_{CONFIG.get('NEWS_ALIGNMENT', '')}_{CONFIG['EVENT_ENCODER']}.csv")
    DECAY.to_csv(f"{CACHE}/v2/decay_ic_{CONFIG.get('NEWS_ALIGNMENT', '')}_{CONFIG['EVENT_ENCODER']}.csv")

In [33]:
if RUN_LEGACY:
    # ---- 13.5 · benchmark vs strict alignment (needs one run with NEWS_ALIGNMENT = "strict") ----
    enc_ = CONFIG["EVENT_ENCODER"]
    fr = {a: f"{CACHE}/v2/results_pooled_{a}_{enc_}.csv" for a in ("benchmark", "strict")}
    if all(os.path.exists(p) for p in fr.values()):
        cols = ["ens ACC", "ens MCC", "ens IC", "ens IC t"]
        CMP = pd.concat({a: pd.read_csv(p, index_col=0)[cols] for a, p in fr.items()}, axis=1)
        print(CMP.round(4).to_string())
        fd = {a: f"{CACHE}/v2/decomposition_{a}_{enc_}.csv" for a in ("benchmark", "strict")}
        if all(os.path.exists(p) for p in fd.values()):
            print("\nOvernight / intraday IC by alignment:")
            print(pd.concat({a: pd.read_csv(p, index_col=0)[["IC overnight", "IC intraday"]] for a, p in fd.items()},
                            axis=1).round(4).to_string())
    else:
        print("Strict-alignment results not found yet. Set NEWS_ALIGNMENT = 'strict' in the config cell and Run all.")

Strict-alignment results not found yet. Set NEWS_ALIGNMENT = 'strict' in the config cell and Run all.


---
# 14 · SCEG-RC Performance Improvement & Scientific Validation

This section redesigns the parts of SCEG that the audit identified as bottlenecks, then evaluates the
redesigned model (**SCEG-v3**) under a pre-specified protocol. Sections 0–13 above are left unchanged: they
are the *before* results.

### 14.A Audit: why SCEG-RC (v2) under-performed

| # | Component | What was wrong | Evidence |
|---|---|---|---|
| A | **Objective vs metric** | ACL18 scores the *sign of the raw movement*. v2 trained Huber + IC: the IC term ignores the level of the predictions, so the predicted sign was mostly accidental | Predicted-up rate ranged from 45 % to 75 % between variants; ranking IC was significant (0.064, t 2.9) while ACC stayed at chance |
| B | **Event representation** | The six-field schema was written for news articles. On tweets, most fields are empty, and the serialised string (`Event Type: . Magnitude: .`) is mostly template, which dominates its embedding | Empty rates: Event Type 69 %, Magnitude 62 %, Temporal Horizon 89 %; raw text beat structured events (ΔIC −0.021) |
| C | **Graph propagation** | Edge weights were `abs()`-ed, so the direction of a dependency (e.g. negative lead–lag) was lost. Lags 1 and 2 were summed and applied to the same day's state. Messages were added to every node with no gate, so a noisy graph could only add noise | Random graph ≈ DYNOTEARS graph (ΔIC −0.007) |
| D | **Regime conditioning** | K separate propagation matrices per relation, mixed by π_t. The test quarter is almost all "Bull" (56 of 64 days), so the other experts get little data and the model has K× more parameters | Removing conditioning *improved* IC (ΔIC −0.025, t −1.48); placebo ≈ real regimes |
| E | **Price features** | Log-volume entered as a level (a firm-size proxy, not information). No overnight gap, excess return or 20-day horizon. No outlier clipping | Price-only IC 0.043, t 1.6 |
| F | **Training samples** | The ranking loss only saw benchmark samples. The benchmark drops small moves, which is a *label-dependent* filter, so the ranking head never saw a normal cross-section | — |
| G | **Model selection** | Early stopping on validation IC while the benchmark is classification | — |
| H | **Caches** | Embedding caches were validated only by their *length*, so changed texts of the same count could silently reuse old vectors. The graph cache ignored DYNOTEARS parameters | Code inspection |
| I | **Alignment** | Benchmark alignment puts tweets posted after the 16:00 ET close into features for the close→close return that includes the next overnight gap | 9.6 % of tweets are posted after the close |

### 14.B What changed, and how each change is tested

| Change | Why it should help | Preserves the SCEG hypothesis? | Leakage risk | Ablation that tests it |
|---|---|---|---|---|
| **Multi-task head**: logit = market(day) + idiosyncratic(firm); weighted BCE on the benchmark label + IC ranking loss on the idiosyncratic score | Optimises the scored quantity and separates market timing from the cross-sectional signal | Yes (head and loss only) | None: training labels only | λ_rank chosen on validation; the same head is given to every baseline, so the comparison stays fair |
| **Event schema v3** for tweets: closed taxonomy (12 event types incl. `none`), polarity, magnitude, horizon; invalid output → explicit `unknown`/`other` + validity flag; the (subject, predicate, object) triple embedded **without** empty template fields | Missing fields become information ("no event"), not noise; polarity is the dominant signal in stock tweets | Yes, it *strengthens* the "structured event" component (one field, polarity, is added) | None: extraction reads only the tweet text | − structured events (raw text only); − raw text (structured only); price only |
| **Gated fusion** of structured and raw text | Lets the model use structure where it is informative and raw text elsewhere | Yes | None | Same as above |
| **Signed, lag-specific DYNOTEARS relations** (contemp, lag-1, lag-2, corr); lag-2 messages use the t−1 state | Restores the direction and timing of the dependency, which "temporal causal propagation" requires | Yes, it restores it | Graph fitted on training returns only | − lead–lag; random signed graph |
| **Event-conditioned attention inside the prior graph**: α_ij depends on the target's state, the source's state and the source's event | A dynamic effective graph: news decides which prior edges carry information that day | Yes (core idea: events propagate along dependencies) | None: uses day-t information only | − event attention (fixed prior weights) |
| **Event gate** on the source's structured event features | "Firms with events send stronger messages" | Yes | None | − event gate |
| **Residual gated graph update** h = u + σ(g)⊙m, gate starts almost closed | A weak graph cannot destroy stock-level information; the graph has to earn its weight | Yes | None | − graph (= GRU-v3), random graph |
| **Regime FiLM**: regimes scale message channels (shared parameters), initialised to identity | Regime conditioning without K× parameters, so it stays trainable with few days in a regime | Yes (regime changes *propagation*, not just an input) | HMM fitted on training dates; filtered posterior | − regime conditioning; regime placebo |
| **Price features v3**: overnight gap, intraday, excess and cross-sectional returns, 5/20-day momentum and volatility, relative volume; winsorised and standardised with training statistics | Removes firm-size artefacts; adds known short-horizon predictors | Neutral (shared by all models) | All rolling windows look backward; statistics from training days only | Shared by every variant, so it cannot favour SCEG |
| **Ranking loss on all valid firm-days** of the training period (BCE stays on benchmark samples) | More cross-sectional training signal, without the label-dependent filter | Yes | None: training period only | — |
| **Selection on validation AUC** (early stopping and hyperparameters); hyperparameters tuned **per model family** (SCEG, GRU, MLP) | Aligns selection with the benchmark; fair tuning for baselines | Yes | Test predictions are never read during tuning | — |
| **Content-hashed caches** | Stale vectors become impossible | — | Removes a silent contamination route | — |

### 14.C Protocol (fixed before any v3 test number exists)
1. **Primary evaluation = strict 16:00 ET alignment.** Its test set has never been evaluated, by any model.
   Benchmark alignment is reported for comparability with published ACL18 numbers.
2. Splits: the StockNet train / dev / test dates (train 2014-01-01 → 2015-07-31, dev → 2015-09-30, test → 2015-12-31).
3. **Tuning** (14.6) uses training + validation only: grid of dropout × λ_rank, 3 seeds, per family; selection = mean validation AUC.
   Test predictions are not computed in this step.
4. The frozen protocol (configuration + selected hyperparameters) is written to `protocol_frozen.json` **before** the final
   runs. The evaluation cell refuses to run without it and prints its hash.
5. Final runs: 5 seeds for every variant. Report mean ± sd over seeds and the seed ensemble. Primary decision
   threshold 0.5 (fixed); a threshold tuned on validation MCC is reported as a secondary number only.
6. Significance: paired day-block bootstrap (block 5 days, 2,000 resamples) for ΔACC and ΔMCC; Newey–West t for daily ΔIC.

**Disclosure for the paper.** The ACL18 test labels were already evaluated for SCEG v2 under benchmark alignment
(sections 10–11). The v3 changes follow from the code audit above, not from test tuning, but benchmark-alignment test
numbers can no longer count as fully held out. The strict-alignment test set is untouched, which is why it is primary.

> **Final-iteration version (section 14 is now fold-generic).** The same code runs ACL18 (one StockNet split) and
> FNSPID (three walk-forward folds: test 2021, 2022, 2023). On ACL18 it reuses the cached v3 runs unchanged (same
> file names). On FNSPID every fitted object (price-feature winsor/scale, PCA, market-state scaler, HMM, signed
> DYNOTEARS) is refitted **per fold on that fold's training dates**. FNSPID uses the hyperparameters selected on the
> ACL18 validation set; nothing is tuned on FNSPID.

In [34]:
# ===== 14.0 · configuration =====
import hashlib, copy
from sklearn.metrics import roc_auc_score, f1_score, matthews_corrcoef
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

IS_ACL18 = CONFIG.get("DATASET") == "acl18"
V3 = dict(
    ALIGNMENTS=["strict", "benchmark"] if IS_ACL18 else ["strict"],   # FNSPID news is always strict (16:00 ET)
    PRIMARY="strict",
    SCHEMA="v3",                          # closed-taxonomy extraction (one LLM pass, shared by all alignments)
    ENCODER=CONFIG["EVENT_ENCODER"],      # "bge" (paper) | "tfidf" (offline smoke test)
    PCA_RAW=16, PCA_TRIPLE=16,
    WINDOW=CONFIG["WINDOW"], HIDDEN=64, ATT_DIM=32,
    LR=1e-3, WD=1e-4, EPOCHS=60, PATIENCE=10, BATCH_DAYS=16,
    SEEDS=[0, 1, 2, 3, 4], TUNE_SEEDS=[0, 1, 2],
    GRID=[dict(dropout=d, lam_rank=l) for d in (0.1, 0.3) for l in (0.25, 1.0)],
    EXTRACT_BATCH=32, EXTRACT_SHARD=2000, BOOT=2000, BLOCK=5,
)
SMOKE3 = bool(os.environ.get("SCEG_V3_SMOKE"))
if SMOKE3:   # pipeline check only (no LLM / no BGE available): results are NOT reportable
    V3.update(SCHEMA="smoke", ENCODER="tfidf", EPOCHS=3, PATIENCE=2, SEEDS=[0, 1], TUNE_SEEDS=[0], BOOT=200,
              GRID=[dict(dropout=0.1, lam_rank=0.25), dict(dropout=0.3, lam_rank=1.0)])
V3DIR = f"{CACHE}/v3"; os.makedirs(V3DIR, exist_ok=True)
def sha(s): return hashlib.sha1(s.encode("utf-8")).hexdigest()
F0 = FOLDS[0]; TR_END = TRADING_DAYS[F0["tr_days"][-1]]
def fold_tag(fd): return "" if len(FOLDS) == 1 else f"{fd['name']}__"      # single fold -> original ACL18 file names
print(json.dumps({k: str(v) for k, v in V3.items()}, indent=1))
print(f"dataset {CONFIG.get('DATASET', 'fnspid')} | folds {len(FOLDS)} | cache {V3DIR}")

{
 "ALIGNMENTS": "['strict', 'benchmark']",
 "PRIMARY": "strict",
 "SCHEMA": "v3",
 "ENCODER": "bge",
 "PCA_RAW": "16",
 "PCA_TRIPLE": "16",
 "WINDOW": "5",
 "HIDDEN": "64",
 "ATT_DIM": "32",
 "LR": "0.001",
 "WD": "0.0001",
 "EPOCHS": "60",
 "PATIENCE": "10",
 "BATCH_DAYS": "16",
 "SEEDS": "[0, 1, 2, 3, 4]",
 "TUNE_SEEDS": "[0, 1, 2]",
 "GRID": "[{'dropout': 0.1, 'lam_rank': 0.25}, {'dropout': 0.1, 'lam_rank': 1.0}, {'dropout': 0.3, 'lam_rank': 0.25}, {'dropout': 0.3, 'lam_rank': 1.0}]",
 "EXTRACT_BATCH": "32",
 "EXTRACT_SHARD": "2000",
 "BOOT": "2000",
 "BLOCK": "5"
}
dataset acl18 | folds 1 | cache /content/drive/MyDrive/MinorProject/acl18_cache/v3


### 14.1 News/tweets → trading days, with a programmatic leakage audit

In [35]:
def load_items():
    if IS_ACL18:
        tw = pd.read_parquet(TWEET_CACHE)                                   # written by §2
        tw["_ts"] = pd.to_datetime(tw["created_at"], format="%a %b %d %H:%M:%S %z %Y", utc=True)
        tw["file_date"] = pd.to_datetime(tw["file_date"])
        tw["text"] = tw["text"].str.replace(r"\$ (\w)", r"$\1", regex=True).str.slice(0, 1500).str.strip()
        tw = tw[tw["Stock_symbol"].isin(sym_idx) & (tw["text"].str.len() > 0)].copy()
    else:   # FNSPID articles from §2: title + summary, strict 16:00 ET alignment, date-only rows -> next trading day
        tw = articles[articles["Stock_symbol"].isin(sym_idx)].copy()
    tw["key"] = [sha(a + "\t" + b) for a, b in zip(tw["Stock_symbol"], tw["text"])]   # extraction depends on (ticker, text) only
    return tw

def align_items(tw, mode):
    if not IS_ACL18:
        assert mode == "strict"
        a = tw.copy(); a["t"] = a["eff_date"].map(pos_of_day).values
        return a.reset_index(drop=True)
    if mode == "benchmark":            # StockNet rule: calendar file c -> feature day = last trading day <= c
        pos = np.searchsorted(TRADING_DAYS, tw["file_date"].values, side="right") - 1; ok = pos >= 0
    else:                              # strict: tweet time in ET; at/after 16:00 or non-trading day -> next trading day
        et = tw["_ts"].dt.tz_convert("America/New_York")
        day = et.dt.normalize().dt.tz_localize(None)
        cand = day + pd.to_timedelta((et.dt.hour >= CONFIG["NEWS_CUTOFF_HOUR_ET"]).astype(int), unit="D")
        pos = np.searchsorted(TRADING_DAYS, cand.values, side="left"); ok = pos < len(TRADING_DAYS)
    a = tw[ok].copy(); a["t"] = pos[ok]; a["eff_date"] = TRADING_DAYS[a["t"].values]
    a = a.sort_values("_ts").drop_duplicates(["Stock_symbol", "eff_date", "text"])
    a["rank"] = a.groupby(["Stock_symbol", "eff_date"])["_ts"].rank(method="first")
    return a[a["rank"] <= CONFIG["MAX_ARTICLES_PER_TICKER_DAY"]].drop(columns="rank").reset_index(drop=True)

ITEMS = load_items()
ALIGNED = {m: align_items(ITEMS, m) for m in V3["ALIGNMENTS"]}

# ---- targets and masks (defined once, used by every model and every metric) ----
NEXT_ROW = np.zeros((T, N), bool); NEXT_ROW[:-1] = ROW[1:]                 # next trading day has a price row
MR3 = ROW & np.isfinite(Y) & NEXT_ROW & EVAL_MASK                                    # all valid firm-days (ranking loss, IC "all")
if IS_ACL18:
    MC3 = BENCH.copy(); YCLS3 = YCLS.astype(np.int8)                    # StockNet benchmark samples and label
else:
    MC3 = MR3.copy(); YCLS3 = np.where(MR3, (Y > 1e-7).astype(np.int8), -1).astype(np.int8)   # all firm-days, sign of raw return
THR3 = MR3 & ((np.nan_to_num(Y) < -0.005) | (np.nan_to_num(Y) >= 0.0055))  # StockNet-style large-move subset (IC "filtered")
if IS_ACL18:
    assert (MC3 <= MR3).all(), "every benchmark sample must be a valid firm-day"
YC3 = np.where(MC3, YCLS3, 0).astype(np.float32); YR3 = np.where(MR3, Y, 0.0).astype(np.float32)

print("LEAKAGE AUDIT")
for m, a in ALIGNED.items():
    # 🔧 [v9] market close in the market's own time zone (India 15:30 IST); ACL18/FNSPID: 16:00 ET as before
    _tz, _cl = CONFIG.get("MARKET_TZ", "America/New_York"), CONFIG.get("CLOSE_MINUTES", CONFIG["NEWS_CUTOFF_HOUR_ET"] * 60)
    close_et = (a["eff_date"] + pd.Timedelta(minutes=_cl)).dt.tz_localize(_tz)
    late = (a["_ts"] >= close_et.dt.tz_convert("UTC")).mean()
    print(f"  [{m:9s}] items used: {len(a):,} | posted at/after the {_cl // 60:02d}:{_cl % 60:02d} {_tz} close of their feature day: {late:.2%}")
    if m == "strict":
        assert late == 0.0, "strict alignment must not use any item posted after the feature-day close"
_adj = np.full((T, N), np.nan); _adj[px["Date"].map(pos_of_day).values, px["Stock_symbol"].map(sym_idx).values] = px["adj_close"].values
_chk = np.abs(_adj[1:] / _adj[:-1] - 1 - Y[:-1])[MR3[:-1]]
print(f"  target = adjclose(t+1)/adjclose(t) - 1 on all used firm-days: share exact (<1e-6) {np.mean(_chk < 1e-6):.4f}")
for fd in FOLDS:
    _tr, _va, _te = (TRADING_DAYS[fd[k][[0, -1]]] for k in ("tr_days", "va_days", "te_days"))
    assert _tr[1] < _va[0] and _va[1] < _te[0]
    print(f"  {fd['name']}: train {_tr[0].date()}..{_tr[1].date()} | val {_va[0].date()}..{_va[1].date()} | test {_te[0].date()}..{_te[1].date()}")
print("  per fold, fitted on training dates only: price winsor/scale, PCA, market-state scaler, HMM, signed DYNOTEARS,"
      " correlation graph; regime posteriors are filtered (forward algorithm); sector graph is external (no data used)")

LEAKAGE AUDIT
  [strict   ] items used: 85,703 | posted at/after the 16:00 America/New_York close of their feature day: 0.00%
  [benchmark] items used: 85,850 | posted at/after the 16:00 America/New_York close of their feature day: 23.76%
  target = adjclose(t+1)/adjclose(t) - 1 on all used firm-days: share exact (<1e-6) 1.0000
  F1: train 2014-01-02..2015-07-31 | val 2015-08-03..2015-09-30 | test 2015-10-01..2015-12-31
  per fold, fitted on training dates only: price winsor/scale, PCA, market-state scaler, HMM, signed DYNOTEARS, correlation graph; regime posteriors are filtered (forward algorithm); sector graph is external (no data used)


### 14.2 Event extraction, schema v3 (closed taxonomy)
One LLM pass over every distinct (ticker, text) used by any alignment; resumable shards. Values outside the taxonomy
become `other` / `unknown` / `neutral` with `valid = 0`; nothing is silently filled. On FNSPID the prompt is the same
except that "tweet" reads "news item".

In [36]:
EVENT_TYPES = ["earnings", "analyst_rating", "price_target", "merger_acquisition", "product", "legal_regulatory",
               "management", "macro_market", "trading_activity", "opinion", "other", "none"]
POLARITY = ["positive", "negative", "neutral"]
MAGNITUDE = ["small", "medium", "large", "unknown"]
HORIZON = ["past", "immediate", "short_term", "long_term", "unknown"]
PROMPT_V3 = """You annotate one tweet about the stock TICKER.
Return ONLY a JSON object with exactly these keys:
{"subject": "", "predicate": "", "object": "", "event_type": "", "polarity": "", "magnitude": "", "horizon": ""}
Rules:
- subject, predicate, object: the main event as a short triple (who / did what / to what). Use "" when the tweet reports no event.
- event_type: one of EVENT_TYPES. Use "none" when the tweet contains no event (chatter, spam, links only).
- polarity: what the tweet implies for TICKER: positive, negative or neutral.
- magnitude: small, medium, large or unknown.
- horizon: when the event matters: past, immediate, short_term, long_term or unknown.
Tweet: TWEET"""
if not IS_ACL18:   # identical instructions; only the noun changes for news articles
    PROMPT_V3 = (PROMPT_V3.replace("one tweet about", "one news item about").replace("when the tweet", "when the item")
                 .replace("what the tweet implies", "what the item implies").replace("Tweet: TWEET", "News: TWEET"))
if CONFIG.get("DATASET") == "india":   # 🔧 [INDIA] identical instructions; the noun names the item type (exchange filing)
    PROMPT_V3 = (PROMPT_V3.replace("one news item about", "one stock-exchange filing (corporate announcement) of")
                 .replace("when the item", "when the filing").replace("what the item implies", "what the filing implies")
                 .replace("News: TWEET", "Filing: TWEET"))
CAT_FIELDS = [("event_type", EVENT_TYPES, "other"), ("polarity", POLARITY, "neutral"),
              ("magnitude", MAGNITUDE, "unknown"), ("horizon", HORIZON, "unknown")]

def parse_v3(txt):
    out = dict(subject="", predicate="", object="", event_type="other", polarity="neutral",
               magnitude="unknown", horizon="unknown", valid=0)
    m = re.search(r"\{[^{}]*\}", txt, re.S)
    if not m: return out
    try: o = json.loads(m.group())
    except Exception: return out
    ok = 1
    for k in ("subject", "predicate", "object"):
        out[k] = str(o.get(k, "") or "").strip()[:120]
    for k, allowed, default in CAT_FIELDS:
        v = str(o.get(k, "") or "").strip().lower().replace(" ", "_").replace("-", "_")
        if v in allowed: out[k] = v
        else: out[k] = default; ok = 0
    out["valid"] = ok
    return out

_LEX = {"earnings": r"earn|eps|revenue|quarter|guidance", "analyst_rating": r"upgrade|downgrade|rating|overweight|underweight",
        "price_target": r"target", "merger_acquisition": r"acqui|merger|takeover|buyout|deal",
        "product": r"launch|product|iphone|drug|fda|approval", "legal_regulatory": r"lawsuit|sec |court|probe|fine",
        "management": r"ceo|cfo|resign|appoint", "macro_market": r"fed|market|oil|rate|economy",
        "trading_activity": r"call|put|option|volume|breakout|short"}
def smoke_extract(ticker, text):   # SMOKE TEST ONLY: keyword stub used when no LLM can run; never reportable
    t = text.lower(); et = next((k for k, p in _LEX.items() if re.search(p, t)), "opinion")
    pos = len(re.findall(r"\b(up|buy|bull\w*|long|beat|gain\w*|high)\b", t)); neg = len(re.findall(r"\b(down|sell|bear\w*|short|miss|loss|low)\b", t))
    return dict(subject=ticker, predicate="", object="", event_type=et,
                polarity="positive" if pos > neg else "negative" if neg > pos else "neutral",
                magnitude="unknown", horizon="unknown", valid=0)

STORE_DIR = f"{V3DIR}/event_store_{V3['SCHEMA']}"; os.makedirs(STORE_DIR, exist_ok=True)
def load_store():
    fs = sorted(glob.glob(f"{STORE_DIR}/shard_*.parquet"))
    cols = ["key", "subject", "predicate", "object", "event_type", "polarity", "magnitude", "horizon", "valid"]
    return (pd.concat([pd.read_parquet(f) for f in fs], ignore_index=True).drop_duplicates("key")
            if fs else pd.DataFrame(columns=cols))

need = pd.concat([a[["key", "Stock_symbol", "text"]] for a in ALIGNED.values()]).drop_duplicates("key")
STORE = load_store()
todo = need[~need["key"].isin(set(STORE["key"]))]
print(f"distinct (ticker, text) pairs: {len(need):,} | already extracted: {len(need) - len(todo):,} | to extract: {len(todo):,}")
if len(todo):
    shard_no = len(glob.glob(f"{STORE_DIR}/shard_*.parquet")); buf = []
    def flush():
        global shard_no, buf
        if buf:
            pd.DataFrame(buf).to_parquet(f"{STORE_DIR}/shard_{shard_no:05d}.parquet", index=False); shard_no += 1; buf = []
    if V3["SCHEMA"] == "smoke":
        print("WARNING: SMOKE TEST extraction (keyword stub) - results are not SCEG results")
        buf = [{"key": k, **smoke_extract(s, x)} for k, s, x in zip(todo["key"], todo["Stock_symbol"], todo["text"])]; flush()
    else:
        from transformers import AutoTokenizer, AutoModelForCausalLM
        from tqdm.auto import tqdm
        assert DEVICE.type == "cuda", "LLM extraction needs the Colab GPU"
        tok = AutoTokenizer.from_pretrained(CONFIG["EXTRACT_MODEL"]); tok.padding_side = "left"
        if tok.pad_token is None: tok.pad_token = tok.eos_token
        lm = AutoModelForCausalLM.from_pretrained(CONFIG["EXTRACT_MODEL"], torch_dtype=torch.float16).to(DEVICE).eval()
        todo = todo.assign(_len=todo["text"].str.len()).sort_values("_len")          # similar lengths per batch -> less padding
        types_str = ", ".join(EVENT_TYPES)
        B = V3["EXTRACT_BATCH"]
        for s in tqdm(range(0, len(todo), B), desc="event extraction v3", unit="batch"):
            chunk = todo.iloc[s:s + B]
            chats = [tok.apply_chat_template([{"role": "user", "content": PROMPT_V3.replace("EVENT_TYPES", types_str)
                                               .replace("TICKER", "$" + tk).replace("TWEET", tx)}],
                                             tokenize=False, add_generation_prompt=True)
                     for tk, tx in zip(chunk["Stock_symbol"], chunk["text"])]
            enc = tok(chats, return_tensors="pt", padding=True, truncation=True, max_length=512).to(DEVICE)
            with torch.no_grad():
                out = lm.generate(**enc, max_new_tokens=96, do_sample=False, pad_token_id=tok.pad_token_id)
            for k, dec in zip(chunk["key"], tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)):
                buf.append({"key": k, **parse_v3(dec)})
            if len(buf) >= V3["EXTRACT_SHARD"]: flush()
        flush(); del lm, tok; gc.collect(); torch.cuda.empty_cache()
    STORE = load_store()
STORE["valid"] = STORE["valid"].astype(int)
_u = need.merge(STORE, on="key", how="left")
print(f"valid taxonomy outputs: {_u['valid'].mean():.1%} | empty triple: {((_u['subject'] + _u['predicate'] + _u['object']).str.strip() == '').mean():.1%}")
for k, allowed, _ in CAT_FIELDS:
    print(f"  {k:10s}", (_u[k].value_counts(normalize=True).reindex(allowed).fillna(0) * 100).round(1).to_dict())

distinct (ticker, text) pairs: 84,993 | already extracted: 84,993 | to extract: 0
valid taxonomy outputs: 98.2% | empty triple: 4.9%
  event_type {'earnings': 0.9, 'analyst_rating': 2.0, 'price_target': 1.5, 'merger_acquisition': 4.4, 'product': 1.1, 'legal_regulatory': 1.2, 'management': 0.8, 'macro_market': 11.3, 'trading_activity': 17.5, 'opinion': 16.6, 'other': 41.7, 'none': 1.1}
  polarity   {'positive': 24.9, 'negative': 14.2, 'neutral': 60.9}
  magnitude  {'small': 0.5, 'medium': 15.6, 'large': 10.9, 'unknown': 72.9}
  horizon    {'past': 8.2, 'immediate': 18.7, 'short_term': 25.0, 'long_term': 7.4, 'unknown': 40.7}


### 14.3 Representations and per-fold panels (content-hashed embeddings; every statistic from the fold's training dates)

In [37]:
def embed_strings(strings, tag, fit_mask=None):
    """BGE vectors cached per distinct string (SHA-1), so a changed text can never reuse a stale vector.
    Empty strings -> zero vector. TF-IDF (smoke only) is fitted on first-fold training texts."""
    strings = [s if isinstance(s, str) else "" for s in strings]
    D = 768 if V3["ENCODER"] == "bge" else 256
    E = np.zeros((len(strings), D), np.float32)
    nz = np.array([len(s) > 0 for s in strings])
    if not nz.any(): return E
    if V3["ENCODER"] == "bge":
        path = f"{V3DIR}/emb_bge_{tag}.npz"
        keys = np.array([sha(s) if len(s) else "" for s in strings])
        known = {}
        if os.path.exists(path):
            z = np.load(path, allow_pickle=False); M = z["M"]; known = {k: i for i, k in enumerate(z["keys"])}
        else:
            M = np.zeros((0, D), np.float16)
        miss = {}
        for k, s in zip(keys[nz], np.array(strings, dtype=object)[nz]):
            if k not in known and k not in miss: miss[k] = s
        if miss:
            from sentence_transformers import SentenceTransformer
            enc = SentenceTransformer(CONFIG["EMBED_MODEL"], device=str(DEVICE))
            newM = enc.encode(list(miss.values()), batch_size=CONFIG["EMBED_BATCH_SIZE"], convert_to_numpy=True,
                              show_progress_bar=True, normalize_embeddings=True).astype(np.float16)
            del enc; gc.collect(); torch.cuda.empty_cache()
            allk = np.concatenate([np.array(list(known.keys()), dtype="<U40"), np.array(list(miss.keys()), dtype="<U40")])
            M = np.concatenate([M, newM]); np.savez(path, keys=allk, M=M)
            known = {k: i for i, k in enumerate(allk)}
        E[nz] = M[[known[k] for k in keys[nz]]].astype(np.float32)
    else:
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.decomposition import TruncatedSVD
        fm = nz & (fit_mask if fit_mask is not None else True)
        vec = TfidfVectorizer(max_features=30000, ngram_range=(1, 2), min_df=3, sublinear_tf=True)
        Xtr = vec.fit_transform([s for s, m in zip(strings, fm) if m])
        D = min(D, Xtr.shape[1] - 1); E = np.zeros((len(strings), D), np.float32)
        svd = TruncatedSVD(D, random_state=0).fit(Xtr)
        Z = svd.transform(vec.transform([s for s, m in zip(strings, nz) if m]))
        E[nz] = Z / (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-9)
    return E

def cat_matrix(df):
    """Per-item structured features (explicit categories, incl. 'none'/'unknown'; nothing is imputed)."""
    cols, names = [], []
    for k, allowed, _ in CAT_FIELDS:
        if k == "polarity": continue
        for v in allowed:
            cols.append((df[k] == v).values.astype(np.float32)); names.append(f"{k}={v}")
    pol = df["polarity"].map({"positive": 1.0, "negative": -1.0}).fillna(0.0).values.astype(np.float32)
    cols += [pol, (df["polarity"] == "positive").values.astype(np.float32), (df["polarity"] == "negative").values.astype(np.float32),
             df["valid"].values.astype(np.float32)]
    names += ["polarity", "pos_share", "neg_share", "valid_share"]
    return np.stack(cols, 1), names

def ffill_panel(V, has, limit):
    last = np.full(N, -10**9); age = np.zeros((T, N), np.float32)
    for t in range(T):
        last = np.where(has[t], t, last); a = t - last
        age[t] = np.minimum(a, V3.get("AGE_CAP", 10))
        fill = (~has[t]) & (a <= limit) & (last >= 0)
        if fill.any(): V[t, fill] = V[last[fill], np.where(fill)[0]]
    return V, age / V3.get("AGE_CAP", 10)

def build_news_panel(mode):
    a = ALIGNED[mode].merge(STORE, on="key", how="left")
    for k, _, default in CAT_FIELDS: a[k] = a[k].fillna(default)
    for k in ("subject", "predicate", "object"): a[k] = a[k].fillna("")
    a["valid"] = a["valid"].fillna(0)
    triple = [" | ".join(x for x in (s, p, o) if x) for s, p, o in zip(a["subject"], a["predicate"], a["object"])]
    fit_mask = (a["eff_date"] <= TR_END).values
    E_raw = embed_strings(a["text"].tolist(), "raw", fit_mask)
    E_tri = embed_strings(triple, "triple", fit_mask)
    C, cnames = cat_matrix(a)
    ti_, ni_ = a["t"].values, a["Stock_symbol"].map(sym_idx).values
    cnt = np.zeros((T, N), np.float32); np.add.at(cnt, (ti_, ni_), 1.0)
    has = cnt > 0
    def pool(M, w=None):
        S = np.zeros((T, N, M.shape[1]), np.float32); c = np.zeros((T, N), np.float32)
        ww = np.ones(len(M), np.float32) if w is None else w.astype(np.float32)
        np.add.at(S, (ti_, ni_), M * ww[:, None]); np.add.at(c, (ti_, ni_), ww)
        S[c > 0] /= c[c > 0][:, None]; return S
    RAWp = pool(E_raw); TRIp = pool(E_tri, np.array([len(x) > 0 for x in triple])); CATp = pool(C)
    del E_raw, E_tri; gc.collect()
    RAWp, age = ffill_panel(RAWp, has, CONFIG["FFILL_LIMIT"])
    TRIp, _ = ffill_panel(TRIp, has, CONFIG["FFILL_LIMIT"]); CATp, _ = ffill_panel(CATp, has, CONFIG["FFILL_LIMIT"])
    NEWS = np.stack([has.astype(np.float32), age, np.log1p(cnt)], -1)
    return dict(RAW=RAWp, TRI=TRIp, CAT=CATp, NEWS=NEWS, cat_names=cnames, n_items=len(a))

def trmask(tr_days):
    m = np.zeros(T, bool); m[tr_days] = True; return m

def fit_pca_train(V, has_any, tr_days):
    m = has_any[tr_days] & ROW[tr_days]
    pca = PCA(V3["PCA_RAW"], random_state=0).fit(V[tr_days][m])
    Z = np.zeros(V.shape[:2] + (pca.n_components_,), np.float32); nz = np.abs(V).sum(-1) > 0
    Z[nz] = pca.transform(V[nz]); return Z

def price_panel_v3(tr_days):
    q = px.sort_values(["Stock_symbol", "Date"]).copy(); g = q.groupby("Stock_symbol", group_keys=False)
    q["adj_open"] = q["open"] * q["adj_close"] / q["close"]
    feats = {"r1": q["Return"], "gap": q["adj_open"] / g["adj_close"].shift(1) - 1,
             "intra": q["close"] / q["open"] - 1, "range": (q["high"] - q["low"]) / q["close"],
             "r5": g["adj_close"].pct_change(5, fill_method=None), "r20": g["adj_close"].pct_change(20, fill_method=None),
             "vol5": g["Return"].transform(lambda x: x.rolling(5).std()),
             "vol20": g["Return"].transform(lambda x: x.rolling(20, min_periods=10).std()),
             "vratio": np.log1p(q["volume"]) - g["volume"].transform(lambda x: np.log1p(x).rolling(20, min_periods=5).mean()),
             "ma5": q["close"] / g["close"].transform(lambda x: x.rolling(5).mean()) - 1,
             "ma20": q["close"] / g["close"].transform(lambda x: x.rolling(20, min_periods=10).mean()) - 1}
    ti_, ni_ = q["Date"].map(pos_of_day).values, q["Stock_symbol"].map(sym_idx).values
    Fm = np.full((T, N, len(feats)), np.nan, np.float32)
    for j, v in enumerate(feats.values()): Fm[ti_, ni_, j] = v.values
    names = list(feats)
    xs = []                                             # same-day cross-sectional (relative) versions
    for nm in ("r1", "r5", "gap"):
        v = np.where(ROW & EVAL_MASK, Fm[..., names.index(nm)], np.nan)   # 🔧 [v9.1] A1: members only (India)
        xs.append((v - np.nanmean(v, 1, keepdims=True)) / (np.nanstd(v, 1, keepdims=True) + 1e-9)); names.append(nm + "_xs")
    Fm = np.concatenate([Fm, np.stack(xs, -1)], -1)
    trm = trmask(tr_days)
    for j in range(Fm.shape[-1]):                       # winsorise + standardise with TRAINING statistics only
        v = Fm[trm][..., j][ROW[trm]]; v = v[np.isfinite(v)]
        lo, hi = np.quantile(v, [0.01, 0.99]); c = np.clip(Fm[..., j], lo, hi)
        mu, sd = np.clip(v, lo, hi).mean(), np.clip(v, lo, hi).std() + 1e-9
        Fm[..., j] = np.nan_to_num((c - mu) / sd)
    Fm[~ROW] = 0.0
    return Fm, names

def std_train(Z, mask_tn, tr_days):
    trm = trmask(tr_days)
    v = Z[trm][mask_tn[trm]]; mu, sd = v.mean(0), v.std(0) + 1e-9
    out = ((Z - mu) / sd).astype(np.float32); out[~mask_tn] = 0.0; return out

def split_samples(day_ids):
    """Sample day t (inputs t-W+1..t, target t+1). ACL18: split of the TARGET day (StockNet). FNSPID: v2's walk-forward
    rule, i.e. the whole input window and the target lie inside the split (embargo + purge)."""
    s = set(day_ids.tolist()); W = V3["WINDOW"]; out = []
    for t in range(W - 1, T - 1):
        if (t + 1) not in s: continue
        if CONFIG.get("SPLIT_RULE", "embargo") != "target_day" and not all((t - k) in s for k in range(W)): continue
        if MC3[t].any() or MR3[t].sum() >= 5: out.append(t)
    return np.array(out, int)

POOLED = {m: build_news_panel(m) for m in V3["ALIGNMENTS"]}
FOLD3 = []
for fd in FOLDS:
    tr = fd["tr_days"]
    PRICE, PRICE_NAMES = price_panel_v3(tr)
    if "post" in fd: PI, rinfo = fd["post"], fd["rinfo"]        # ACL18: same posterior as sections 6/14 (cached runs)
    else: PI, rinfo = fit_regimes(tr)                           # FNSPID: sticky HMM, K by BIC, filtered, this fold only
    MK = StandardScaler().fit(MARKET_STATE[tr]).transform(MARKET_STATE).astype(np.float32)
    Xm, sl = {}, None
    for m in V3["ALIGNMENTS"]:
        p = POOLED[m]; has_any = p["NEWS"][..., 0] > 0; nz = np.abs(p["RAW"]).sum(-1) > 0
        RAWz = std_train(fit_pca_train(p["RAW"], has_any, tr), nz, tr)
        TRIz = std_train(fit_pca_train(p["TRI"], has_any, tr), np.abs(p["TRI"]).sum(-1) > 0, tr)
        CATz = std_train(p["CAT"], nz, tr)
        NEWSz = p["NEWS"].copy(); NEWSz[..., 2] = NEWSz[..., 2] / max(1.0, NEWSz[tr][..., 2].max())   # training max
        Xm[m] = np.concatenate([PRICE, RAWz, TRIz, CATz, NEWSz], -1).astype(np.float32)
        sl = {}; o = 0
        for name, d in (("price", PRICE.shape[-1]), ("raw", RAWz.shape[-1]), ("tri", TRIz.shape[-1]),
                        ("cat", CATz.shape[-1]), ("news", 3)):
            sl[name] = (o, o + d); o += d
    TRd, VAd, TEd = split_samples(fd["tr_days"]), split_samples(fd["va_days"]), split_samples(fd["te_days"])
    posw = float((MC3[TRd] & (YCLS3[TRd] == 0)).sum() / max(1, (MC3[TRd] & (YCLS3[TRd] == 1)).sum()))
    sig = float(np.nanstd(np.where(MC3[TRd], Y[TRd], np.nan)))
    FOLD3.append(dict(name=fd["name"], fd=fd, X=Xm, sl=sl, PI=PI.astype(np.float32), rinfo=rinfo, MK=MK,
                      TR=TRd, VA=VAd, TE=TEd, POSW=posw, SIGMA=sig, K=PI.shape[1]))
    print(f"{fd['name']}: regimes K={rinfo['K']} {[rinfo['names'][k] for k in range(rinfo['K'])]} | sample days train {len(TRd)} "
          f"val {len(VAd)} test {len(TEd)} | classification samples test {int(MC3[TEd].sum()):,} | node features {Xm[V3['PRIMARY']].shape[-1]}")
for m, p in POOLED.items():
    print(f"[{m}] items {p['n_items']:,} | firm-days with same-day items {(p['NEWS'][..., 0] > 0)[ROW].mean():.1%}")
CAT_NAMES = POOLED[V3["PRIMARY"]]["cat_names"]
del POOLED; gc.collect()
TE_ALL = np.concatenate([f3["TE"] for f3 in FOLD3])
# backward-compatible single-fold names used by earlier cells/analyses
PI3, MK3, TR3, VA3, TE3, K3 = FOLD3[0]["PI"], FOLD3[0]["MK"], FOLD3[0]["TR"], FOLD3[0]["VA"], FOLD3[0]["TE"], FOLD3[0]["K"]
PANELS = {m: dict(X=FOLD3[0]["X"][m], sl=FOLD3[0]["sl"], cat_names=CAT_NAMES) for m in V3["ALIGNMENTS"]}

F1: regimes K=3 ['Bear', 'Neutral', 'Bull'] | sample days train 398 val 42 test 64 | classification samples test 3,720 | node features 74
[strict] items 85,703 | firm-days with same-day items 37.0%
[benchmark] items 85,850 | firm-days with same-day items 37.1%


### 14.4 Relations: signed, lag-specific DYNOTEARS (per fold, training returns only) + external sector graph

In [38]:
def signed_cap(M, k):
    M = M.astype(np.float32).copy(); np.fill_diagonal(M, 0)
    for i in range(M.shape[0]):
        if (M[i] != 0).sum() > k:
            thr = np.sort(np.abs(M[i]))[-k]; M[i][np.abs(M[i]) < thr] = 0
    s = np.abs(M).sum(1, keepdims=True); return np.divide(M, s, out=np.zeros_like(M), where=s > 0)

def fit_graphs_v3(fd):
    tag = sha(json.dumps([CONFIG[k] for k in ("DYNOTEARS_P", "DYNOTEARS_LAMBDA", "DYNOTEARS_LAMBDA_A", "DYNOTEARS_MAX_ITER",
                                                "DYNOTEARS_W_THRESHOLD", "TOP_K_CORR", "MAX_IN_DEGREE")])
              + str(TRADING_DAYS[fd["tr_days"][0]]) + str(TRADING_DAYS[fd["tr_days"][-1]]) + str(N)
              + ("|pit_members" if CONFIG.get("DATASET") == "india" else ""))[:12]   # 🔧 [v9.1] A1 (India cache key only)
    path = f"{V3DIR}/graph_signed_{tag}.npz"
    if os.path.exists(path):
        z = np.load(path); return {k: z[k] for k in z.files}
    R = np.where((ROW & EVAL_MASK)[fd["tr_days"]], RET[fd["tr_days"]], np.nan)   # 🔧 [v9.1] A1: member-days only (India)
    Z = np.nan_to_num((R - np.nanmean(R, 0)) / (np.nanstd(R, 0) + 1e-9), nan=0.0)
    C = pd.DataFrame(R).corr(min_periods=60).fillna(0).values.copy(); np.fill_diagonal(C, 0)
    Mc = np.zeros((N, N), np.float32)
    for i in range(N):
        nb = np.argsort(-np.abs(C[i]))[:CONFIG["TOP_K_CORR"]]; Mc[i, nb] = C[i, nb]
    p = CONFIG["DYNOTEARS_P"]; Xc = Z[p:]; Xl = np.concatenate([Z[p - k - 1: -k - 1] for k in range(p)], axis=1)
    t0 = time.time()
    Wm, Am, h = dynotears(Xc, Xl, CONFIG["DYNOTEARS_LAMBDA"], CONFIG["DYNOTEARS_LAMBDA_A"], CONFIG["DYNOTEARS_MAX_ITER"],
                          w_threshold=CONFIG["DYNOTEARS_W_THRESHOLD"])
    k = CONFIG["MAX_IN_DEGREE"]
    out = dict(contemp=signed_cap(Wm.T, k), lag1=signed_cap(Am[:N].T, k), lag2=signed_cap(Am[N:2 * N].T, k),
               corr=signed_cap(Mc, k))
    print(f"  DYNOTEARS (signed) {fd['name']}: |W|={int((Wm != 0).sum())} |A|={int((Am != 0).sum())} ({time.time() - t0:.0f}s)")
    np.savez(path, **out); return out

# External sector relation (no data from any split is used).
# ACL18: the StockNet release's own StockTable. FNSPID: GICS sectors of the NASDAQ-100 constituents, assigned from
# public classifications [AUTHOR VERIFICATION REQUIRED: check against the GICS list used in the paper].
GICS_NDX = {
 "Information Technology": "AAPL ADBE ADI ADSK AMAT AMD ANSS ASML AVGO CDNS CDW CRWD CSCO CTSH DDOG ENPH FTNT GFS INTC INTU KLAC LRCX MCHP MRVL MSFT MU NVDA NXPI ON PANW QCOM SNPS TEAM TTD TXN WDAY ZM ZS",
 "Communication Services": "CHTR CMCSA EA GOOG GOOGL META NFLX SIRI TMUS TTWO WBD",
 "Consumer Discretionary": "ABNB AMZN BKNG EBAY JD LCID LULU MAR MELI ORLY PDD ROST SBUX TSLA",
 "Consumer Staples": "COST DLTR KDP KHC MDLZ MNST PEP WBA",
 "Health Care": "AMGN AZN BIIB DXCM GEHC GILD IDXX ILMN ISRG MRNA REGN SGEN VRTX",
 "Industrials": "ADP CPRT CSX CTAS FAST HON ODFL PAYX PCAR VRSK",
 "Utilities": "AEP CEG EXC XEL", "Energy": "BKR FANG", "Financials": "PYPL", "Real Estate": "CSGP"}
def sector_map():
    if CONFIG.get("DATASET") == "india": return dict(INDIA_SECTOR)   # 🔧 [INDIA] sectors.csv (external)
    if IS_ACL18:
        st = pd.read_csv(f"{CONFIG['ACL18_ROOT']}/StockTable", sep="\t")
        return {str(s).strip().lstrip("$"): str(sec).strip() for sec, s in zip(st["Sector"], st["Symbol"])}
    return {t: sec for sec, ts in GICS_NDX.items() for t in ts.split()}
SECTOR = sector_map()
def sector_graph():
    sec = [SECTOR.get(s) for s in SYMBOLS]
    M = np.array([[1.0 if (a is not None and a == b and i != j) else 0.0 for j, b in enumerate(sec)] for i, a in enumerate(sec)], np.float32)
    s = M.sum(1, keepdims=True); return np.divide(M, s, out=np.zeros_like(M), where=s > 0)
_SG = sector_graph()
print(f"sector graph: {int((_SG > 0).sum())} edges | firms without a sector entry: {[s for s in SYMBOLS if s not in SECTOR]}")
for f3 in FOLD3:
    f3["G"] = {**fit_graphs_v3(f3["fd"]), "sector": _SG}
    for r, M in f3["G"].items():
        nzv = M[M != 0]
        print(f"  {f3['name']} {r:8s} edges {len(nzv):5d} | negative share {np.mean(nzv < 0) if len(nzv) else 0:.1%}")
G3 = FOLD3[0]["G"]

def random_signed(M, rng):
    vals = M[M != 0]; R = np.zeros_like(M)
    cand = np.array([(i, j) for i in range(N) for j in range(N) if i != j])
    pick = cand[rng.choice(len(cand), len(vals), replace=False)]; R[pick[:, 0], pick[:, 1]] = rng.permutation(vals)
    s = np.abs(R).sum(1, keepdims=True); return np.divide(R, s, out=np.zeros_like(R), where=s > 0)

sector graph: 762 edges | firms without a sector entry: []
  F1 contemp  edges   273 | negative share 0.7%
  F1 lag1     edges   791 | negative share 47.9%
  F1 lag2     edges   774 | negative share 51.0%
  F1 corr     edges   435 | negative share 0.2%
  F1 sector   edges   762 | negative share 0.0%


### 14.5 SCEG-v3 model
For firm *i* on day *t* (the window covers t−W+1..t):

1. **Node encoder:** price → h_p; structured events (triple embedding + categorical event features) and raw text are
   fused by a learned gate; news presence/age/volume enter additively. A GRU over the window gives u_i(t), u_i(t−1).
2. **Event-conditioned propagation** for relation r with signed prior S_r:
   α_ij = softmax_j( a_rᵀ tanh(Q u_i + K[u_j; e_j]) + log|S_r,ij| ) over prior edges only;
   m_i = Σ_r Σ_j α_ij · sign(S_r,ij) · g(e_j) · (1 + γ_r(π_t)) ⊙ V_r u_j, where lag2 uses u_j(t−1).
3. **Residual update:** h_i = u_i + σ(G[u_i; m_i]) ⊙ m_i; the gate bias starts at −2 (graph almost off).
   `zero_out=True` (nested experiment) also starts the message projection at 0, so the model starts *exactly* at the
   graph-free model.
4. **Heads:** logit_i = market(mean_j h_j, market state, π_t) + idio(h_i) (`market=False` drops the market term).

In [39]:
class SCEG3(nn.Module):
    def __init__(self, sl, K, M, H=64, dropout=0.1, rels=("contemp", "lag1", "lag2", "corr"), S=None,
                 use_graph=True, attention=True, event_gate=True, film=True, regime_feat=True,
                 use_raw=True, use_struct=True, temporal="gru", A=32, zero_out=False, market=True):
        super().__init__()
        self.sl, self.K, self.H = sl, K, H
        self.use_graph, self.attention, self.film_on, self.regime_feat = use_graph, attention, film, regime_feat
        self.use_raw, self.use_struct, self.temporal, self.market = use_raw, use_struct, temporal, market
        self.event_gate = event_gate and use_struct
        n = lambda k: sl[k][1] - sl[k][0]
        self.price = nn.Linear(n("price"), H)
        if use_struct: self.struct = nn.Linear(n("tri") + n("cat"), H)
        if use_raw: self.raw = nn.Linear(n("raw"), H)
        if use_struct and use_raw: self.fuse = nn.Linear(2 * H, H)
        self.has_text = use_struct or use_raw
        if self.has_text: self.news = nn.Linear(3, H)
        d_in = H * (2 if self.has_text else 1) + (K if regime_feat else 0)
        self.inp = nn.Sequential(nn.Linear(d_in, H), nn.LayerNorm(H), nn.GELU(), nn.Dropout(dropout))
        if temporal == "gru": self.gru = nn.GRU(H, H, batch_first=True)
        self.rels = list(rels) if use_graph else []
        if use_graph:
            self.register_buffer("S", torch.as_tensor(np.stack([S[r] for r in self.rels]), dtype=torch.float32))
            R = len(self.rels)
            self.V = nn.ModuleList([nn.Linear(H, H, bias=False) for _ in range(R)])
            if attention:
                self.Q = nn.ModuleList([nn.Linear(H, A) for _ in range(R)])
                self.Kp = nn.ModuleList([nn.Linear(2 * H, A, bias=False) for _ in range(R)])
                self.att = nn.ModuleList([nn.Linear(A, 1, bias=False) for _ in range(R)])
            if self.event_gate:
                self.gate = nn.Linear(n("cat") + 3, 1); nn.init.zeros_(self.gate.weight); nn.init.constant_(self.gate.bias, 2.0)
            if film:
                self.filmL = nn.Linear(K, R * H); nn.init.zeros_(self.filmL.weight); nn.init.zeros_(self.filmL.bias)
            self.out = nn.Linear(H, H)
            if zero_out: nn.init.zeros_(self.out.weight); nn.init.zeros_(self.out.bias)
            self.res = nn.Linear(2 * H, H); nn.init.constant_(self.res.bias, -2.0)
        self.drop = nn.Dropout(dropout)
        self.idio = nn.Sequential(nn.Linear(H, H // 2), nn.GELU(), nn.Dropout(dropout), nn.Linear(H // 2, 1))
        self.mkt = nn.Sequential(nn.Linear(H + M + K, 32), nn.GELU(), nn.Linear(32, 1))
        self.stats = {}

    def _x(self, X, k): return X[..., self.sl[k][0]:self.sl[k][1]]

    def forward(self, X, PI, MK):                      # X: B,W,N,F  PI: B,W,K  MK: B,W,M
        B, W, Nn, _ = X.shape
        parts = [Fnn.gelu(self.price(self._x(X, "price")))]
        txt = None
        if self.has_text:
            if self.use_struct: s_ = self.struct(torch.cat([self._x(X, "tri"), self._x(X, "cat")], -1))
            if self.use_raw: r_ = self.raw(self._x(X, "raw"))
            if self.use_struct and self.use_raw:
                g = torch.sigmoid(self.fuse(torch.cat([s_, r_], -1))); txt = g * s_ + (1 - g) * r_
                self.stats["fusion_struct_weight"] = float(g.detach().mean())
            else:
                txt = s_ if self.use_struct else r_
            txt = Fnn.gelu(txt + self.news(self._x(X, "news")))
            parts.append(txt)
        if self.regime_feat: parts.append(PI[:, :, None, :].expand(B, W, Nn, PI.shape[-1]))
        x = self.inp(torch.cat(parts, -1))                                      # B,W,N,H
        if self.temporal == "gru":
            o, _ = self.gru(x.permute(0, 2, 1, 3).reshape(B * Nn, W, self.H)); o = o.view(B, Nn, W, self.H)
            u, u1 = o[:, :, -1], o[:, :, -2]
        else:
            u, u1 = x[:, -1], x[:, -2]
        h = u
        if self.use_graph:
            e_src = txt[:, -1] if txt is not None else torch.zeros_like(u)
            gsrc = (torch.sigmoid(self.gate(torch.cat([self._x(X, "cat")[:, -1], self._x(X, "news")[:, -1]], -1)))
                    if self.event_gate else 1.0)
            fg = self.filmL(PI[:, -1]).view(B, len(self.rels), 1, self.H) if self.film_on else None
            agg = 0.0
            for r, name in enumerate(self.rels):
                S = self.S[r]; src = u1 if name == "lag2" else u
                V = self.V[r](src) * gsrc
                if fg is not None: V = V * (1 + fg[:, r])
                if self.attention:
                    q = self.Q[r](u); k = self.Kp[r](torch.cat([src, e_src], -1))
                    e = self.att[r](torch.tanh(q[:, :, None, :] + k[:, None, :, :])).squeeze(-1)   # B,N,N
                    mask = (S != 0)
                    e = (e + torch.log(S.abs() + 1e-9)).masked_fill(~mask, -1e9)
                    Aw = torch.softmax(e, -1) * mask * torch.sign(S)
                else:
                    Aw = S.expand(B, Nn, Nn)
                agg = agg + torch.einsum("bij,bjh->bih", Aw, V)
            agg = self.out(agg)
            gr = torch.sigmoid(self.res(torch.cat([u, agg], -1)))
            h = u + gr * agg
            self.stats["residual_gate"] = float(gr.detach().mean())
        s = self.idio(self.drop(h)).squeeze(-1)                                 # B,N (stock-specific score)
        if not self.market:
            return s, s
        m = self.mkt(torch.cat([h.mean(1), MK[:, -1], PI[:, -1]], -1))          # B,1 (market component)
        return s + m, s

### 14.6 Training, variants, and hyperparameters (ACL18: selected on validation; FNSPID: transferred, not re-tuned)

In [40]:
VARIANTS3 = {
    "SCEG-v3 (full)":                  dict(family="sceg"),
    "− graph (= GRU-v3)":              dict(family="gru", use_graph=False),
    "− regime conditioning":           dict(family="sceg", film=False),
    "regime placebo (shuffled)":       dict(family="sceg", placebo=True),
    "random graph (signed, matched)":  dict(family="sceg", random_graph=True),
    "− lead–lag (lag1, lag2)":         dict(family="sceg", rels=("contemp", "corr")),
    "− event attention":               dict(family="sceg", attention=False),
    "− event gate":                    dict(family="sceg", event_gate=False),
    "− structured events (raw only)":  dict(family="sceg", use_struct=False),
    "− raw text (structured only)":    dict(family="sceg", use_raw=False),
    "price only":                      dict(family="sceg", use_struct=False, use_raw=False),
    "MLP-v3":                          dict(family="mlp", use_graph=False, temporal="none"),
    "LightGBM-v3":                     dict(family="lgbm"),
    "sector graph only":               dict(family="sceg", rels=("sector",)),      # final iteration (P3)
}
FAMILY_BASE = {"sceg": "SCEG-v3 (full)", "gru": "− graph (= GRU-v3)", "mlp": "MLP-v3"}
_tn = lambda a: torch.as_tensor(a, device=DEVICE)
YC_t, MC_t, YR_t, MR_t = _tn(YC3), _tn(MC3), _tn(YR3), _tn(MR3)

def daily_ic_mask(score, days, mask, rank=False):
    """Daily cross-sectional correlation of a score with the next-day return over `mask` firms; <5 firms or a
    constant score -> 0 (same convention as v2)."""
    out = []
    for r, t in enumerate(days):
        m = mask[t]
        if m.sum() < 5 or np.std(score[r][m]) == 0: out.append(0.0); continue
        a, b = score[r][m], Y[t][m]
        if rank: a, b = pd.Series(a).rank().values, pd.Series(b).rank().values
        v = np.corrcoef(a, b)[0, 1]; out.append(v if np.isfinite(v) else 0.0)
    return np.array(out)
def ic_days(score, days, rank=False): return daily_ic_mask(score, days, MR3, rank)

_TENS = {}
def price_v2_block(f3):
    """Legacy v2 price features (§1.1), standardised with this fold's training statistics (attribution experiment)."""
    tr = f3["fd"]["tr_days"]; sc = StandardScaler().fit(P[tr][ROW[tr]])
    Z = ((P - sc.mean_) / (sc.scale_ + 1e-9)).astype(np.float32); Z[~ROW] = 0.0; return Z

def tensors_for(f3, mode, cfg):
    key = (f3["name"], mode, bool(cfg.get("placebo")), cfg.get("price", "v3"))
    if key not in _TENS:
        X, sl = f3["X"][mode], f3["sl"]
        if cfg.get("price", "v3") == "v2":
            Zp = price_v2_block(f3); d = Zp.shape[-1] - (sl["price"][1] - sl["price"][0])
            X = np.concatenate([Zp, X[..., sl["price"][1]:]], -1)
            sl = {k: (0, Zp.shape[-1]) if k == "price" else (a + d, b + d) for k, (a, b) in sl.items()}
        PI = f3["PI"][np.random.default_rng(1234).permutation(T)] if cfg.get("placebo") else f3["PI"]
        _TENS[key] = (_tn(X), _tn(PI), _tn(f3["MK"]), sl)
    return _TENS[key]

_MEMBER_T = torch.as_tensor(EVAL_MASK, device=DEVICE) if CONFIG.get("DATASET") == "india" else None
def windows(Xt, PIt, MKt, days):
    idx = torch.as_tensor(np.asarray(days)[:, None] + np.arange(-V3["WINDOW"] + 1, 1)[None], device=DEVICE)
    Xw = Xt[idx]
    if _MEMBER_T is not None:   # 🔧 [v9.1] A1: the node set of a sample dated t = index members at t (whole window)
        mem = _MEMBER_T[torch.as_tensor(np.asarray(days), device=DEVICE)]
        Xw = Xw * mem[:, None, :, None].to(Xw.dtype)
    return Xw, PIt[idx], MKt[idx]

def ic_loss(s, y, m):
    mf = m.float(); cnt = mf.sum(1).clamp(min=2)
    ps = (s * mf).sum(1, keepdim=True) / cnt[:, None]; ys = (y * mf).sum(1, keepdim=True) / cnt[:, None]
    pc, yc = (s - ps) * mf, (y - ys) * mf
    ic = (pc * yc).sum(1) / (pc.pow(2).sum(1).sqrt() * yc.pow(2).sum(1).sqrt() + 1e-8)
    ok = mf.sum(1) >= 5
    return (1 - ic[ok]).mean() if ok.any() else s.sum() * 0

@torch.no_grad()
def predict3(model, Xt, PIt, MKt, days):
    model.eval(); L, S_ = [], []
    for s in range(0, len(days), 32):
        xb, pb, mb = windows(Xt, PIt, MKt, days[s:s + 32]); lo, sc = model(xb, pb, mb)
        L.append(lo.float().cpu().numpy()); S_.append(sc.float().cpu().numpy())
    return np.concatenate(L), np.concatenate(S_)

def auc_on(logit, days):
    m = MC3[days]; y = YCLS3[days][m]
    return roc_auc_score(y, logit[m]) if 0 < y.mean() < 1 else 0.5

def build_model(f3, mode, cfg, hp, seed, sl=None):
    S = f3["G"]
    if cfg.get("random_graph"):   # same draw order as the original section 14 (reproduces cached ACL18 runs)
        rng = np.random.default_rng(seed + 99)
        S = {**{r: random_signed(M, rng) for r, M in f3["G"].items() if r != "sector"}, "sector": f3["G"]["sector"]}
    rels = cfg.get("rels", ("contemp", "lag1", "lag2", "corr"))
    return SCEG3(sl or f3["sl"], f3["K"], f3["MK"].shape[1], V3["HIDDEN"], hp["dropout"], rels, S,
                 use_graph=cfg.get("use_graph", True), attention=cfg.get("attention", True),
                 event_gate=cfg.get("event_gate", True), film=cfg.get("film", True),
                 use_raw=cfg.get("use_raw", True), use_struct=cfg.get("use_struct", True),
                 temporal=cfg.get("temporal", "gru"), A=V3["ATT_DIM"], zero_out=cfg.get("zero_out", False),
                 market=cfg.get("market", True)).to(DEVICE)

ENCODER_MODULES = ("price", "struct", "raw", "fuse", "news", "inp", "gru")
def train3(f3, mode, cfg, hp, seed, with_test, init_state=None, freeze=()):
    """cfg options: placebo, random_graph, rels, use_graph, attention, event_gate, film, use_raw, use_struct, temporal,
    zero_out, market (bool), loss ('v3' | 'v2'), price ('v3' | 'v2'), select ('auc' | 'ic' | 'composite').
    init_state/freeze: nested experiment (start from a trained model; frozen modules stay in eval mode)."""
    set_seed(seed)
    Xt, PIt, MKt, sl = tensors_for(f3, mode, cfg)
    model = build_model(f3, mode, cfg, hp, seed, sl)
    if init_state is not None:
        model.load_state_dict(init_state, strict=False)
    frozen = [getattr(model, n) for n in freeze if hasattr(model, n)]
    for mod in frozen:
        for p_ in mod.parameters(): p_.requires_grad_(False)
    opt = torch.optim.AdamW([p_ for p_ in model.parameters() if p_.requires_grad], lr=V3["LR"], weight_decay=V3["WD"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=V3["EPOCHS"])
    posw = torch.tensor(f3["POSW"], device=DEVICE); rng = np.random.default_rng(seed)
    TRd, VAd, TEd = f3["TR"], f3["VA"], f3["TE"]
    def val_sel():
        vl, vs = predict3(model, Xt, PIt, MKt, VAd)
        sel = cfg.get("select", "auc")
        if sel == "composite":        # section 16 (pre-registered): excess AUC + IC, one criterion for both heads
            return (auc_on(vl, VAd) - 0.5) + float(ic_days(vs, VAd).mean())
        return auc_on(vl, VAd) if sel == "auc" else float(ic_days(vs, VAd).mean())
    best, best_state, bad = -np.inf, None, 0
    if init_state is not None:        # the starting model is itself a candidate: stage 2 is kept only if validation improves
        best = val_sel(); best_state = {k: t.detach().cpu().clone() for k, t in model.state_dict().items()}
    for ep in range(V3["EPOCHS"]):
        model.train()
        for mod in frozen: mod.eval()
        order = rng.permutation(TRd)
        for s in range(0, len(order), V3["BATCH_DAYS"]):
            days = order[s:s + V3["BATCH_DAYS"]]
            xb, pb, mb = windows(Xt, PIt, MKt, days); dt = torch.as_tensor(days, device=DEVICE)
            logit, sc = model(xb, pb, mb)
            if cfg.get("loss", "v3") == "v2":          # v2 objective: Huber + IC on the (benchmark) samples, as in §8
                loss = loss_fn(logit, YR_t[dt], MC_t[dt], f3["SIGMA"], CONFIG["IC_LOSS_WEIGHT"])
            else:
                mc = MC_t[dt]
                loss = Fnn.binary_cross_entropy_with_logits(logit[mc], YC_t[dt][mc], pos_weight=posw)
                loss = loss + hp["lam_rank"] * ic_loss(sc, YR_t[dt], MR_t[dt])
            opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        sched.step()
        v = val_sel()
        if v > best + 1e-5:
            best, bad = v, 0; best_state = {k: t.detach().cpu().clone() for k, t in model.state_dict().items()}
        else:
            bad += 1
            if bad >= V3["PATIENCE"]: break
    model.load_state_dict(best_state)
    vl, vs = predict3(model, Xt, PIt, MKt, VAd)
    out = dict(val_auc=auc_on(vl, VAd), val_sel=best, val_logit=vl, val_score=vs)
    if with_test:
        out["te_logit"], out["te_score"] = predict3(model, Xt, PIt, MKt, TEd)
    return out, model

def lgbm_features(f3, mode):
    X = f3["X"][mode]; W = V3["WINDOW"]
    def f(days):
        cur = X[days]; mean = np.stack([X[t - W + 1:t + 1].mean(0) for t in days])
        glob_ = np.concatenate([f3["PI"][days], f3["MK"][days]], -1)[:, None, :].repeat(N, 1)
        return np.concatenate([cur, mean, glob_], -1)
    return f

def train_lgbm3(f3, mode, seed, with_test, return_model=False):
    import lightgbm as lgb
    f = lgbm_features(f3, mode)
    TRd, VAd, TEd = f3["TR"], f3["VA"], f3["TE"]
    Ztr, Zva = f(TRd), f(VAd)
    mtr, mva = MC3[TRd], MC3[VAd]
    clf = lgb.LGBMClassifier(n_estimators=3000, learning_rate=0.02, num_leaves=15, min_child_samples=200, subsample=0.8,
                             subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, random_state=seed, verbose=-1)
    clf.fit(Ztr[mtr], YCLS3[TRd][mtr], eval_set=[(Zva[mva], YCLS3[VAd][mva])], callbacks=[lgb.early_stopping(100, verbose=False)])
    def lg(Z):
        pr = np.clip(clf.predict_proba(Z.reshape(-1, Z.shape[-1]))[:, 1], 1e-6, 1 - 1e-6).reshape(len(Z), N)
        return np.log(pr / (1 - pr))
    vl = lg(Zva); out = dict(val_auc=auc_on(vl, VAd), val_logit=vl, val_score=vl)
    if with_test:
        tl = lg(f(TEd)); out.update(te_logit=tl, te_score=tl)
    return (out, clf) if return_model else out

# ---- hyperparameters ----
SEL_PATH = f"{V3DIR}/selected_hparams_{V3['PRIMARY']}_{V3['ENCODER']}_{V3['SCHEMA']}.json"
ACL18_SELECTED = {"sceg": {"dropout": 0.3, "lam_rank": 0.25}, "gru": {"dropout": 0.3, "lam_rank": 0.25},
                  "mlp": {"dropout": 0.3, "lam_rank": 1.0}}         # selected on ACL18 validation (section 14.6, sha 0a60ab442ab1)
if not IS_ACL18:
    SELECTED = ACL18_SELECTED if not SMOKE3 else {k: dict(dropout=0.3, lam_rank=0.25) for k in FAMILY_BASE}
    _acl_sel = os.environ.get("SCEG_ACL18_SELECTION", "/content/drive/MyDrive/MinorProject/acl18_cache/v3/selected_hparams_strict_bge_v3.json")
    if os.path.exists(_acl_sel) and not SMOKE3:
        assert json.load(open(_acl_sel)) == ACL18_SELECTED, "ACL18 selection file differs from the frozen values"
    print("FNSPID: hyperparameters transferred from ACL18 validation selection (no FNSPID tuning):", SELECTED)
elif os.path.exists(SEL_PATH):
    SELECTED = json.load(open(SEL_PATH)); print("loaded selection:", SELECTED)
else:
    rows_sel, SELECTED = [], {}
    for fam, vname in FAMILY_BASE.items():
        for gi, hp in enumerate(V3["GRID"]):
            aucs = []
            for seed in V3["TUNE_SEEDS"]:
                p = f"{V3DIR}/tune_{V3['PRIMARY']}_{V3['ENCODER']}_{V3['SCHEMA']}_{fam}_g{gi}_s{seed}.json"
                if os.path.exists(p): a = json.load(open(p))["val_auc"]
                else:
                    a = train3(FOLD3[0], V3["PRIMARY"], VARIANTS3[vname], hp, seed, with_test=False)[0]["val_auc"]
                    json.dump({"val_auc": a}, open(p, "w"))
                aucs.append(a)
            rows_sel.append({"family": fam, **hp, "val AUC mean": np.mean(aucs), "val AUC sd": np.std(aucs)})
            print(f"  {fam:5s} {hp} | val AUC {np.mean(aucs):.4f} ± {np.std(aucs):.4f}", flush=True)
        best = max((r for r in rows_sel if r["family"] == fam), key=lambda r: r["val AUC mean"])
        SELECTED[fam] = {k: best[k] for k in ("dropout", "lam_rank")}
    print(pd.DataFrame(rows_sel).round(4).to_string(index=False))
    json.dump(SELECTED, open(SEL_PATH, "w"))
print("selected:", SELECTED)

# ---- freeze the protocol BEFORE any test prediction is produced (a second file freezes the final-iteration additions) ----
FROZEN = f"{V3DIR}/protocol_frozen_{V3['ENCODER']}_{V3['SCHEMA']}.json"
if not os.path.exists(FROZEN):
    json.dump({"V3": {k: str(v) for k, v in V3.items()}, "selected": SELECTED, "variants": list(VARIANTS3),
               "primary_threshold": 0.5, "dataset": CONFIG.get("DATASET", "fnspid"),
               "note": "" if IS_ACL18 else "hyperparameters transferred from ACL18; no FNSPID tuning",
               "frozen_at": time.strftime("%Y-%m-%d %H:%M:%S")}, open(FROZEN, "w"), indent=1)
print("protocol frozen:", FROZEN, "| sha1", sha(open(FROZEN).read())[:12])

# ---- 🔧 [v9.1] A1 audit: in every input window, firms that are not index members at the sample date carry no data ----
if _MEMBER_T is not None:
    _Xt, _PIt, _MKt, _ = tensors_for(FOLD3[0], V3["PRIMARY"], {})
    _d = np.asarray(FOLD3[0]["TR"][::25]); _xw = windows(_Xt, _PIt, _MKt, _d)[0]
    _nm = ~torch.as_tensor(EVAL_MASK[_d], device=DEVICE)
    _leak = float(_xw.abs().sum(-1).sum(1)[_nm].max()) if bool(_nm.any()) else 0.0
    assert _leak == 0.0, "A1 violated: a non-member firm carries input data"
    print(f"A1 audit passed: {int(_nm.sum())} non-member firm-samples checked, all inputs zero")


loaded selection: {'sceg': {'dropout': 0.3, 'lam_rank': 0.25}, 'gru': {'dropout': 0.3, 'lam_rank': 0.25}, 'mlp': {'dropout': 0.3, 'lam_rank': 1.0}}
selected: {'sceg': {'dropout': 0.3, 'lam_rank': 0.25}, 'gru': {'dropout': 0.3, 'lam_rank': 0.25}, 'mlp': {'dropout': 0.3, 'lam_rank': 1.0}}
protocol frozen: /content/drive/MyDrive/MinorProject/acl18_cache/v3/protocol_frozen_bge_v3.json | sha1 0a60ab442ab1


### 14.7 Final runs (frozen configuration, 5 seeds, every fold and alignment; resumable)

In [41]:
if RUN_PHASE == "prep":   # 🔧 [v9] stop before the first test prediction
    raise PhaseComplete("Phase 'prep' finished: extraction, embeddings, graphs and regimes are cached. "
                        "Next: set RUN_PHASE = 'all' and Run all (India also needs the FNSPID shake-down file).")
require_shakedown("14.7 v3 test predictions")   # 🔧 [v9] no-op except India
PRED3 = f"{V3DIR}/preds_{V3['ENCODER']}_{V3['SCHEMA']}"; os.makedirs(PRED3, exist_ok=True)
def pred_path(f3, mode, vname, seed): return f"{PRED3}/{mode}__{fold_tag(f3)}{slug(vname)}__s{seed}.npz"
RUN3 = list(VARIANTS3)
if os.environ.get("SCEG_V3_VARIANTS"):          # smoke tests only
    RUN3 = [v for v in RUN3 if slug(v) in os.environ["SCEG_V3_VARIANTS"].split(",")]
t0 = time.time()
for f3 in FOLD3:
    for mode in V3["ALIGNMENTS"]:
        for vname in RUN3:
            cfg = VARIANTS3[vname]
            for seed in V3["SEEDS"]:
                path = pred_path(f3, mode, vname, seed)
                if os.path.exists(path): continue
                t1 = time.time()
                if cfg["family"] == "lgbm":
                    out = train_lgbm3(f3, mode, seed, True)
                else:
                    hp = SELECTED[cfg["family"] if cfg["family"] in SELECTED else "sceg"]
                    out, model = train3(f3, mode, cfg, hp, seed, with_test=True)
                    if vname == "SCEG-v3 (full)":
                        torch.save(model.state_dict(), f"{PRED3}/{mode}__{fold_tag(f3)}full__s{seed}.pt")
                    if vname == "− graph (= GRU-v3)":
                        torch.save(model.state_dict(), f"{PRED3}/{mode}__{fold_tag(f3)}gru_state__s{seed}.pt")
                np.savez(path, **{k: np.asarray(v) for k, v in out.items()})
                print(f"[{mode}] {f3['name']} {vname:34s} seed {seed} | val AUC {out['val_auc']:.4f} | {time.time() - t1:4.0f}s", flush=True)
print(f"final runs done in {(time.time() - t0) / 60:.1f} min")

final runs done in 0.0 min


---
# 15 · Final experimental iteration (pre-registered, then model development stops)

| # | Experiment | Research question | Exact change | Control | Metric / test |
|---|---|---|---|---|---|
| P1 | SCEG-v3 on FNSPID (run this notebook with `DATASET="fnspid"`) | Do the graph / regime / event mechanisms help on a longer test set with a bear year? | §14 pipeline, 3 walk-forward folds, strict 16:00 ET, ACL18 hyperparameters | Same ablation family as ACL18 + sector graph | per-seed mean ± sd; paired day-block bootstrap; Holm |
| P2 | Nested SCEG | Does graph propagation add information beyond an already-trained GRU-v3? | Train GRU-v3 → freeze its encoder → add the graph module (message projection starts at 0, so stage 2 starts exactly at GRU-v3; validation decides whether it is kept) | Same stage 2 with **no** graph (heads retrained only), same seeds | paired ΔACC / ΔMCC / ΔIC, Holm |
| P3 | Graph sanity check (inside P2) | Is the problem propagation itself or the learned causal graph? | none · random (matched to DYNOTEARS) · DYNOTEARS (contemp + lag1 + lag2) · DYNOTEARS + corr (= SCEG relations) · **sector** | no graph; random graph | as P2 |
| P4 | Attribution of the v2 → v3 gain (ACL18, both alignments) | Which shared change produced the gain? | GRU-v3 with one change reverted: v2 loss · v2 price features · no market head · early stopping on val IC; all four reverted; SCEG-v3 with v2 loss | GRU-v3 / SCEG-v3 | as P2 |
| P5 | Evaluation fixes | — | one IC definition on two sample sets; per-seed primary; majority test; market vs stock-specific split; LightGBM diagnostic; v2 IC recomputed | — | bootstrap CIs, Holm per family |

**Decision rule (fixed before results).** Evaluated on FNSPID (primary, strict) and checked on ACL18 strict:
**A** nested DYNOTEARS (or SCEG relations) beats *both* nested no-graph and nested random (Holm p < 0.05, positive) →
narrow positive graph claim; **B** sector beats nested no-graph and random while DYNOTEARS does not → relational
propagation helps but the learned causal graph is too noisy; **C** otherwise → freeze the model, the paper reports
the controlled negative result.

In [42]:
require_shakedown("15.1 nested test predictions")   # 🔧 [v9] no-op except India
# ===== 15.1 · P2 + P3: nested SCEG (GRU-v3 -> + graph) =====
NESTED = {   # name -> relations (None = no graph); "random" uses signed random graphs matched to the DYNOTEARS relations
    "nested: no graph (heads retrained)": None,
    "nested: + random (matched to DYNOTEARS)": ("contemp", "lag1", "lag2"),
    "nested: + DYNOTEARS": ("contemp", "lag1", "lag2"),
    "nested: + DYNOTEARS + corr (SCEG relations)": ("contemp", "lag1", "lag2", "corr"),
    "nested: + sector": ("sector",),
}
def nested_base(f3, mode, seed):
    """Trained GRU-v3 state (reused from §14.7 if saved there; otherwise trained here with the same seed and settings)."""
    p = f"{PRED3}/{mode}__{fold_tag(f3)}gru_state__s{seed}.pt"
    if not os.path.exists(p):
        _, mdl = train3(f3, mode, VARIANTS3["− graph (= GRU-v3)"], SELECTED["gru"], seed, with_test=False)
        torch.save(mdl.state_dict(), p)
    return torch.load(p, map_location="cpu")

t0 = time.time()
for f3 in FOLD3:
    for mode in [V3["PRIMARY"]]:
        for seed in V3["SEEDS"]:
            base = None
            for vname, rels in NESTED.items():
                path = pred_path(f3, mode, vname, seed)
                if os.path.exists(path): continue
                base = base if base is not None else nested_base(f3, mode, seed)
                if rels is None:
                    cfg = dict(family="gru", use_graph=False)
                else:
                    cfg = dict(family="sceg", rels=rels, zero_out=True, random_graph="random" in vname)
                out, _ = train3(f3, mode, cfg, SELECTED["gru"], seed, with_test=True, init_state=base, freeze=ENCODER_MODULES)
                np.savez(path, **{k: np.asarray(v) for k, v in out.items()})
                print(f"[{mode}] {f3['name']} {vname:44s} seed {seed} | val AUC {out['val_auc']:.4f}", flush=True)
            # the base itself (no stage 2) for reference
            path = pred_path(f3, mode, "nested: GRU-v3 base (no stage 2)", seed)
            if not os.path.exists(path):
                base = base if base is not None else nested_base(f3, mode, seed)
                Xt, PIt, MKt, sl = tensors_for(f3, mode, {})
                mdl = build_model(f3, mode, dict(use_graph=False), SELECTED["gru"], seed); mdl.load_state_dict(base)
                vl, vs = predict3(mdl, Xt, PIt, MKt, f3["VA"]); tl, ts = predict3(mdl, Xt, PIt, MKt, f3["TE"])
                np.savez(path, val_auc=auc_on(vl, f3["VA"]), val_logit=vl, val_score=vs, te_logit=tl, te_score=ts)
print(f"nested runs done in {(time.time() - t0) / 60:.1f} min")

nested runs done in 0.0 min


In [43]:
# ===== 15.2 · P4: attribution of the v2 -> v3 improvement (ACL18 only; both alignments) =====
ATTRIB = {
    "GRU-v3 | v2 loss (Huber + IC)":        dict(family="gru", use_graph=False, loss="v2"),
    "GRU-v3 | v2 price features":           dict(family="gru", use_graph=False, price="v2"),
    "GRU-v3 | no market head":              dict(family="gru", use_graph=False, market=False),
    "GRU-v3 | early stop on val IC":        dict(family="gru", use_graph=False, select="ic"),
    "GRU-v3 | all four reverted":           dict(family="gru", use_graph=False, loss="v2", price="v2", market=False, select="ic"),
    "SCEG-v3 | v2 loss (Huber + IC)":       dict(family="sceg", loss="v2"),
}
if IS_ACL18:
    t0 = time.time()
    for f3 in FOLD3:
        for mode in V3["ALIGNMENTS"]:
            for vname, cfg in ATTRIB.items():
                for seed in V3["SEEDS"]:
                    path = pred_path(f3, mode, vname, seed)
                    if os.path.exists(path): continue
                    out, _ = train3(f3, mode, cfg, SELECTED[cfg["family"]], seed, with_test=True)
                    np.savez(path, **{k: np.asarray(v) for k, v in out.items()})
                    print(f"[{mode}] {vname:36s} seed {seed} | val AUC {out['val_auc']:.4f}", flush=True)
    print(f"attribution runs done in {(time.time() - t0) / 60:.1f} min")
else:
    print("P4 runs on ACL18 only (the v2 -> v3 comparison exists there)")

attribution runs done in 0.0 min


### 15.3 Corrected evaluation (P5)
* **Primary result = per-seed mean ± sd** over the seeds (pooled over folds). The seed ensemble is secondary.
* Classification metrics on the classification set (ACL18: the 3,720 StockNet test samples; FNSPID: every valid
  firm-day) at the fixed 0.5 threshold. **IC / RankIC use one definition** (daily cross-sectional Pearson / Spearman
  of the stock-specific score with the next-day return) on two sample sets: **all firm-days** and the
  **large-move subset** (|move| outside [−0.5 %, +0.55 %], the StockNet filter).
* **Market vs stock-specific:** each day's logits are split into their cross-sectional mean (common part) and the
  deviation. *Market ACC* scores the common part alone; *stock DirAcc* scores the deviation against the return's
  deviation from the day's mean (base rate ≈ 50 %).
* **Paired tests are seed-paired:** the statistic is the mean over seeds of the per-seed difference. CIs and p-values
  come from a day-block bootstrap (block 5); ΔIC also gets a Newey–West t. Two-sided p-values are Holm-corrected
  within each family. Every model is also tested against the majority class.

In [44]:
def load_runs(mode, vname):
    """Per-seed pooled predictions (folds concatenated in order) -> {seed: dict(te_logit, te_score, val_logit)}."""
    runs = {}
    for seed in V3["SEEDS"]:
        paths = [pred_path(f3, mode, vname, seed) for f3 in FOLD3]
        if not all(os.path.exists(p) for p in paths): continue
        zs = [np.load(p) for p in paths]
        runs[seed] = {k: np.concatenate([z[k] for z in zs]) for k in ("te_logit", "te_score")}
    return runs

def cls_metrics(logit, days, thr=0.0):
    m = MC3[days]; y = YCLS3[days][m].astype(int); p = logit[m]; yh = (p > thr).astype(int)
    return dict(ACC=(y == yh).mean() * 100, MCC=matthews_corrcoef(y, yh) if yh.std() > 0 else 0.0,
                F1=f1_score(y, yh, zero_division=0) * 100, AUC=roc_auc_score(y, p) * 100 if 0 < y.mean() < 1 else 50.0,
                up=yh.mean() * 100)

def split_metrics(logit, days):
    """Market (common) vs stock-specific parts of the prediction."""
    mk_hit = mk_n = st_hit = st_n = 0
    for r, t in enumerate(days):
        mr, mc = MR3[t], MC3[t]
        if mr.sum() < 5: continue
        c = logit[r][mr].mean()
        y = YCLS3[t][mc]; mk_hit += ((c > 0) == (y == 1)).sum(); mk_n += len(y)
        dev = logit[r][mr] - c; ydev = Y[t][mr] - Y[t][mr].mean(); ok = (dev != 0) & (ydev != 0)
        st_hit += ((dev[ok] > 0) == (ydev[ok] > 0)).sum(); st_n += ok.sum()
    return dict(ACC_market=mk_hit / max(mk_n, 1) * 100, DirAcc_stock=st_hit / max(st_n, 1) * 100)

def run_metrics(logit, score, days):
    c = cls_metrics(logit, days)
    all_m = MR3[days]; y_all = (Y[days][all_m] > 1e-7).astype(int)
    return {**c, "ACC_all_firmdays": ((logit[all_m] > 0).astype(int) == y_all).mean() * 100,
            "IC_all": daily_ic_mask(score, days, MR3).mean(), "RankIC_all": daily_ic_mask(score, days, MR3, True).mean(),
            "IC_filtered": daily_ic_mask(score, days, THR3).mean(), "RankIC_filtered": daily_ic_mask(score, days, THR3, True).mean(),
            **split_metrics(logit, days)}

METRIC_COLS = ["ACC", "MCC", "F1", "AUC", "ACC_all_firmdays", "ACC_market", "DirAcc_stock", "IC_all", "RankIC_all",
               "IC_filtered", "RankIC_filtered", "up"]
def summary_row(name, runs, days):
    per = pd.DataFrame([run_metrics(r["te_logit"], r["te_score"], days) for r in runs.values()])
    row = {"model": name, "seeds": len(runs)}
    for c in METRIC_COLS:
        row[c] = f"{per[c].mean():.2f}±{per[c].std(ddof=0):.2f}" if c in ("ACC", "F1", "AUC", "ACC_all_firmdays",
                 "ACC_market", "DirAcc_stock", "up") else f"{per[c].mean():.4f}±{per[c].std(ddof=0):.4f}"
    pr = np.mean([1 / (1 + np.exp(-r["te_logit"])) for r in runs.values()], 0); el = np.log(np.clip(pr, 1e-6, 1 - 1e-6) / np.clip(1 - pr, 1e-6, 1))
    ce = cls_metrics(el, days)
    row.update({"ens ACC (secondary)": round(ce["ACC"], 2), "ens MCC (secondary)": round(ce["MCC"], 4)})
    return row, per

def _counts(logit, days):
    """Per day: tp, fp, tn, fn on the classification set."""
    out = np.zeros((len(days), 4))
    for r, t in enumerate(days):
        m = MC3[t]; y = YCLS3[t][m].astype(int); a = (logit[r][m] > 0).astype(int)
        out[r] = [((a == 1) & (y == 1)).sum(), ((a == 1) & (y == 0)).sum(), ((a == 0) & (y == 0)).sum(), ((a == 0) & (y == 1)).sum()]
    return out

def _mcc(c):
    tp, fp, tn, fn = c[..., 0], c[..., 1], c[..., 2], c[..., 3]
    d = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    return np.where(d > 0, (tp * tn - fp * fn) / np.maximum(d, 1e-12), 0.0)

def paired_test(runsA, runsB, days, B=None, block=None, seed=0):
    """Seed-paired comparison A − B. runsB may hold a single run (e.g. majority) that is used for every seed."""
    B = B or V3["BOOT"]; block = block or V3["BLOCK"]
    seeds = sorted(set(runsA) & set(runsB)) if len(runsB) > 1 else sorted(runsA)
    getB = (lambda s: runsB[s]) if len(runsB) > 1 else (lambda s: next(iter(runsB.values())))
    CA = np.stack([_counts(runsA[s]["te_logit"], days) for s in seeds]); CB = np.stack([_counts(getB(s)["te_logit"], days) for s in seeds])
    ICA = np.stack([ic_days(runsA[s]["te_score"], days) for s in seeds]); ICB = np.stack([ic_days(getB(s)["te_score"], days) for s in seeds])
    def stat(idx):
        a, b = CA[:, idx].sum(1), CB[:, idx].sum(1)                     # seeds x 4
        acc = lambda c: (c[:, 0] + c[:, 2]) / c.sum(1) * 100
        return np.mean(acc(a) - acc(b)), np.mean(_mcc(a) - _mcc(b))
    nd = len(days); pt = stat(np.arange(nd)); rng = np.random.default_rng(seed); nb = int(np.ceil(nd / block)); bs = []
    for _ in range(B):
        st = rng.integers(0, nd, nb); idx = ((st[:, None] + np.arange(block)[None]) % nd).ravel()[:nd]; bs.append(stat(idx))
    bs = np.array(bs)
    p2 = lambda v: float(min(1.0, 2 * min(np.mean(v <= 0), np.mean(v >= 0))))
    dic = (ICA - ICB).mean(0); t_ic = nw_t(dic)
    from scipy.stats import norm as _norm
    return dict(dACC=pt[0], dACC_lo=np.percentile(bs[:, 0], 2.5), dACC_hi=np.percentile(bs[:, 0], 97.5), p_ACC=p2(bs[:, 0]),
                dMCC=pt[1], dMCC_lo=np.percentile(bs[:, 1], 2.5), dMCC_hi=np.percentile(bs[:, 1], 97.5), p_MCC=p2(bs[:, 1]),
                dIC=float(dic.mean()), t_IC=float(t_ic), p_IC=float(2 * (1 - _norm.cdf(abs(t_ic)))), seeds=len(seeds))

def holm(ps):
    ps = np.asarray(ps, float); o = np.argsort(ps); adj = np.empty_like(ps); run = 0.0
    for r, i in enumerate(o):
        run = max(run, min(1.0, (len(ps) - r) * ps[i])); adj[i] = run
    return adj

def family_table(pairs, mode, title):
    rows_ = []
    for a, b in pairs:
        ra, rb = load_runs(mode, a), (MAJ if b == "majority class" else load_runs(mode, b))
        if not ra or not rb: continue
        rows_.append({"A": a, "B": b, **paired_test(ra, rb, TE_ALL)})
    if not rows_: return pd.DataFrame()
    df = pd.DataFrame(rows_)
    for c in ("p_ACC", "p_MCC", "p_IC"): df[c + " (Holm)"] = holm(df[c].values)
    df["ΔACC 95% CI"] = [f"[{l:+.2f}, {h:+.2f}]" for l, h in zip(df.dACC_lo, df.dACC_hi)]
    df["ΔMCC 95% CI"] = [f"[{l:+.3f}, {h:+.3f}]" for l, h in zip(df.dMCC_lo, df.dMCC_hi)]
    show = ["A", "B", "seeds", "dACC", "ΔACC 95% CI", "p_ACC (Holm)", "dMCC", "ΔMCC 95% CI", "p_MCC (Holm)", "dIC", "t_IC", "p_IC (Holm)"]
    print(f"\n--- [{mode}] {title} (seed-paired; day-block bootstrap; Holm within this table) ---")
    print(df[show].round(4).to_string(index=False))
    df.to_csv(f"{V3DIR}/final_{slug(title)}_{mode}_{V3['ENCODER']}_{V3['SCHEMA']}.csv", index=False)
    return df

up_tr = np.mean(np.concatenate([YCLS3[f3["TR"]][MC3[f3["TR"]]] for f3 in FOLD3]))
MAJ = {"maj": dict(te_logit=np.full((len(TE_ALL), N), 1.0 if up_tr > 0.5 else -1.0), te_score=np.zeros((len(TE_ALL), N)))}
print(f"test: {len(TE_ALL)} days over {len(FOLD3)} fold(s) | classification samples {int(MC3[TE_ALL].sum()):,} | "
      f"majority class = {'up' if up_tr > 0.5 else 'down'} (training up-rate {up_tr:.3f})")

test: 64 days over 1 fold(s) | classification samples 3,720 | majority class = up (training up-rate 0.507)


In [45]:
# ===== 15.4 · main tables: per-seed mean ± sd (primary) =====
ALL_MODELS = list(VARIANTS3) + list(NESTED) + ["nested: GRU-v3 base (no stage 2)"] + (list(ATTRIB) if IS_ACL18 else [])
FINAL = {}
for mode in V3["ALIGNMENTS"]:
    rows_ = []
    for vname in ALL_MODELS:
        runs = load_runs(mode, vname)
        if runs: rows_.append(summary_row(vname, runs, TE_ALL)[0])
    c = cls_metrics(MAJ["maj"]["te_logit"], TE_ALL)
    rows_.append({"model": "majority class", "seeds": 0, "ACC": f"{c['ACC']:.2f}", "MCC": "0", "up": f"{c['up']:.1f}"})
    TAB = pd.DataFrame(rows_).set_index("model")
    print(f"\n=== [{mode}] {CONFIG.get('DATASET', 'fnspid')} test: per-seed mean ± sd (primary); ensemble secondary ===")
    print(TAB.to_string())
    TAB.to_csv(f"{V3DIR}/final_table_{mode}_{V3['ENCODER']}_{V3['SCHEMA']}.csv")
    FINAL[mode] = TAB


=== [strict] acl18 test: per-seed mean ± sd (primary); ensemble secondary ===
                                             seeds         ACC             MCC          F1         AUC ACC_all_firmdays  ACC_market DirAcc_stock          IC_all      RankIC_all     IC_filtered RankIC_filtered           up  ens ACC (secondary)  ens MCC (secondary)
model                                                                                                                                                                                                                                                                  
SCEG-v3 (full)                                   5  54.12±3.12   0.0850±0.0646  59.33±6.99  57.15±5.07       52.77±2.12  52.84±3.73   51.34±0.94   0.0354±0.0311   0.0426±0.0295   0.0455±0.0375   0.0510±0.0316  63.59±12.55                53.63               0.0679
− graph (= GRU-v3)                               5  53.98±2.22   0.0747±0.0496  60.07±2.76  57.94±2.22       53.04±1.74  54.06±3.

In [46]:
# ===== 15.5 · paired families with Holm correction =====
FAM = {}
for mode in V3["ALIGNMENTS"]:
    ref = "SCEG-v3 (full)"
    FAM[(mode, "full-model ablations")] = family_table(
        [(ref, v) for v in VARIANTS3 if v != ref], mode, "full-model ablations")
    FAM[(mode, "nested graph")] = family_table(
        [("nested: + DYNOTEARS", "nested: no graph (heads retrained)"),
         ("nested: + DYNOTEARS + corr (SCEG relations)", "nested: no graph (heads retrained)"),
         ("nested: + sector", "nested: no graph (heads retrained)"),
         ("nested: + random (matched to DYNOTEARS)", "nested: no graph (heads retrained)"),
         ("nested: + DYNOTEARS", "nested: + random (matched to DYNOTEARS)"),
         ("nested: + DYNOTEARS + corr (SCEG relations)", "nested: + random (matched to DYNOTEARS)"),
         ("nested: + sector", "nested: + random (matched to DYNOTEARS)"),
         ("nested: no graph (heads retrained)", "nested: GRU-v3 base (no stage 2)")], mode, "nested graph")
    if IS_ACL18:
        FAM[(mode, "attribution")] = family_table(
            [("− graph (= GRU-v3)", v) for v in ATTRIB if v.startswith("GRU")] +
            [("SCEG-v3 (full)", "SCEG-v3 | v2 loss (Huber + IC)")], mode, "attribution")
    FAM[(mode, "vs majority")] = family_table(
        [(v, "majority class") for v in ["SCEG-v3 (full)", "− graph (= GRU-v3)", "MLP-v3", "price only",
                                         "− structured events (raw only)", "random graph (signed, matched)",
                                         "nested: + DYNOTEARS", "nested: + sector"]], mode, "vs majority")


--- [strict] full-model ablations (seed-paired; day-block bootstrap; Holm within this table) ---
             A                              B  seeds    dACC     ΔACC 95% CI  p_ACC (Holm)    dMCC      ΔMCC 95% CI  p_MCC (Holm)     dIC    t_IC  p_IC (Holm)
SCEG-v3 (full)             − graph (= GRU-v3)      5  0.1452  [-2.75, +2.64]           1.0  0.0102 [-0.046, +0.062]           1.0 -0.0022 -0.1892       1.0000
SCEG-v3 (full)          − regime conditioning      5 -0.2849  [-2.80, +2.36]           1.0 -0.0027 [-0.047, +0.041]           1.0  0.0060  1.2353       1.0000
SCEG-v3 (full)      regime placebo (shuffled)      5  0.7366  [-0.67, +2.28]           1.0  0.0176 [-0.004, +0.039]           1.0  0.0001  0.0655       1.0000
SCEG-v3 (full) random graph (signed, matched)      5 -1.4247  [-4.46, +1.09]           1.0 -0.0270 [-0.087, +0.028]           1.0 -0.0099 -1.2043       1.0000
SCEG-v3 (full)        − lead–lag (lag1, lag2)      5  0.6935  [-4.05, +5.30]           1.0  0.0122 [-0.067,

In [47]:
# ===== 15.6 · regime analysis (FNSPID includes the 2022 bear market) =====
for mode in [V3["PRIMARY"]]:
    reg_day = np.concatenate([np.array([f3["rinfo"]["names"][k] for k in f3["PI"][f3["TE"]].argmax(1)]) for f3 in FOLD3])
    print(f"\n[{mode}] test-day regime occupancy:", pd.Series(reg_day).value_counts().to_dict())
    rows_ = []
    for vname in ["SCEG-v3 (full)", "− regime conditioning", "regime placebo (shuffled)", "− graph (= GRU-v3)"]:
        runs = load_runs(mode, vname)
        if not runs: continue
        for reg in sorted(set(reg_day)):
            sel = reg_day == reg; d = TE_ALL[sel]
            per = [run_metrics(r["te_logit"][sel], r["te_score"][sel], d) for r in runs.values()]
            rows_.append({"model": vname, "regime": reg, "days": int(sel.sum()),
                          "ACC": np.mean([p_["ACC"] for p_ in per]), "IC_all": np.mean([p_["IC_all"] for p_ in per]),
                          "DirAcc_stock": np.mean([p_["DirAcc_stock"] for p_ in per])})
    if rows_:
        REG = pd.DataFrame(rows_).set_index(["regime", "model"]).sort_index()
        print(REG.round(4).to_string())
        REG.to_csv(f"{V3DIR}/final_regimes_{mode}_{V3['ENCODER']}_{V3['SCHEMA']}.csv")
        for reg in sorted(set(reg_day)):
            sel = reg_day == reg
            if sel.sum() < 20: continue
            for b in ("− regime conditioning", "regime placebo (shuffled)"):
                ra, rb = load_runs(mode, "SCEG-v3 (full)"), load_runs(mode, b)
                if not ra or not rb: continue
                sub = lambda R: {s: {k: v[sel] for k, v in r.items()} for s, r in R.items()}
                t_ = paired_test(sub(ra), sub(rb), TE_ALL[sel])
                print(f"  {reg:8s} ({int(sel.sum())} days): full − {b}: ΔACC {t_['dACC']:+.2f} [{t_['dACC_lo']:+.2f}, {t_['dACC_hi']:+.2f}] "
                      f"p {t_['p_ACC']:.3f} | ΔIC {t_['dIC']:+.4f} (t {t_['t_IC']:+.2f})  [exploratory, uncorrected]")

# By test fold (FNSPID: 2021 / 2022 bear market / 2023). Uses calendar years, not HMM labels. Exploratory, uncorrected.
if len(FOLD3) > 1:
    fold_of = np.concatenate([np.full(len(f3["TE"]), i) for i, f3 in enumerate(FOLD3)])
    rows_ = []
    for i, f3 in enumerate(FOLD3):
        sel = fold_of == i; yr = f3["fd"]["test_year"]
        sub = lambda R: {s: {k: v[sel] for k, v in r.items()} for s, r in R.items()}
        ra = load_runs(V3["PRIMARY"], "SCEG-v3 (full)")
        for b in ("− graph (= GRU-v3)", "− regime conditioning", "regime placebo (shuffled)"):
            rb = load_runs(V3["PRIMARY"], b)
            if not ra or not rb: continue
            t_ = paired_test(sub(ra), sub(rb), TE_ALL[sel])
            rows_.append({"test year": yr, "comparison": f"full − {b}", "ΔACC": t_["dACC"],
                          "ΔACC 95% CI": f"[{t_['dACC_lo']:+.2f}, {t_['dACC_hi']:+.2f}]", "p": t_["p_ACC"],
                          "ΔIC": t_["dIC"], "t_IC": t_["t_IC"]})
    if rows_:
        BYYEAR = pd.DataFrame(rows_)
        print("\nBy test year (exploratory, uncorrected; 2022 = bear-market year):")
        print(BYYEAR.round(4).to_string(index=False))
        BYYEAR.to_csv(f"{V3DIR}/final_by_test_year_{V3['ENCODER']}_{V3['SCHEMA']}.csv", index=False)


[strict] test-day regime occupancy: {'Bull': 56, 'Neutral': 8}
                                   days      ACC  IC_all  DirAcc_stock
regime  model                                                         
Bull    SCEG-v3 (full)               56  55.0920  0.0399       51.5476
        regime placebo (shuffled)    56  54.3998  0.0399       51.3465
        − graph (= GRU-v3)           56  54.8987  0.0427       51.3136
        − regime conditioning        56  56.3268  0.0305       51.1248
Neutral SCEG-v3 (full)                8  48.0702  0.0037       49.8851
        regime placebo (shuffled)     8  47.0565  0.0027       48.9368
        − graph (= GRU-v3)            8  48.2261  0.0017       48.2184
        − regime conditioning         8  42.4172  0.0216       51.2644
  Bull     (56 days): full − − regime conditioning: ΔACC -1.23 [-3.91, +2.01] p 0.429 | ΔIC +0.0094 (t +1.94)  [exploratory, uncorrected]
  Bull     (56 days): full − regime placebo (shuffled): ΔACC +0.69 [-0.96, +2.27] p 0.39

In [48]:
# ===== 15.7 · diagnostics: v2 vs v3 IC under one definition (ACL18); LightGBM identical-output check =====
if IS_ACL18 and RUN_LEGACY:
    rows_ = []
    lf = FOLDS[0]; ldays = lf["te"]
    if not (len(ldays) == len(TE_ALL) and (ldays == TE_ALL).all()):
        print("WARNING: v2 and v3 test-day sets differ; each generation is scored on its own test days")
    cols = ("ACC", "MCC", "IC_filtered", "IC_all", "RankIC_filtered", "RankIC_all", "ACC_market", "DirAcc_stock")
    for v2name, v3name in [("SCEG-RC (full)", "SCEG-v3 (full)"), ("GRU", "− graph (= GRU-v3)"), ("MLP", "MLP-v3"),
                           ("raw text (no structure)", "− structured events (raw only)"), ("price only (no events)", "price only")]:
        per = [run_metrics(P_[0], P_[0], ldays) for P_ in (load_preds(v2name, lf, [s]) for s in CONFIG["SEEDS"]) if P_]
        if per:
            rows_.append({"model": f"v2 {v2name}", **{c: np.mean([p_[c] for p_ in per]) for c in cols}})
        runs = load_runs("benchmark", v3name)
        if runs:
            per = [run_metrics(r["te_logit"], r["te_score"], TE_ALL) for r in runs.values()]
            rows_.append({"model": f"v3 {v3name}", **{c: np.mean([p_[c] for p_ in per]) for c in cols}})
    V2V3 = pd.DataFrame(rows_).set_index("model")
    print("v2 vs v3 under ONE IC definition (per-seed means; benchmark alignment, the only alignment v2 was run with).")
    print("IC_filtered = the definition v2 reported (large-move benchmark samples); IC_all = all firm-days (used in §14.8 of the")
    print("previous run). ACC_market = accuracy of the day-level common part only; DirAcc_stock = stock-specific direction.")
    print(V2V3.round(4).to_string())
    V2V3.to_csv(f"{V3DIR}/final_v2_vs_v3_ic.csv")

# LightGBM: why were ACL18 outputs identical across alignments?
if len(V3["ALIGNMENTS"]) > 1:
    f3 = FOLD3[0]; rep = {}
    for mode in V3["ALIGNMENTS"]:
        out, clf = train_lgbm3(f3, mode, 0, True, return_model=True)
        imp = clf.booster_.feature_importance(importance_type="split"); F_ = f3["X"][mode].shape[-1]
        text_idx = [j for blk in ("raw", "tri", "cat", "news") for j in range(*f3["sl"][blk])]
        text_idx = text_idx + [F_ + j for j in text_idx]                     # current day + window mean
        rep[mode] = dict(best_iteration=clf.best_iteration_, splits=int(imp.sum()),
                         text_split_share=float(imp[text_idx].sum() / max(imp.sum(), 1)), te=out["te_logit"])
    a, b = V3["ALIGNMENTS"][:2]
    print(f"LightGBM seed 0 | best_iteration {rep[a]['best_iteration']} / {rep[b]['best_iteration']} | total splits "
          f"{rep[a]['splits']} / {rep[b]['splits']} | share of splits on text features {rep[a]['text_split_share']:.1%} / "
          f"{rep[b]['text_split_share']:.1%} | max |Δ test logit| between alignments {np.abs(rep[a]['te'] - rep[b]['te']).max():.2e}")
    print("Interpretation: if no split uses a text feature (share 0 %), the text inputs are the only thing that differs between "
          "alignments, so identical outputs are expected, not a bug: the early-stopped booster is a price/market-only model.")

v2 vs v3 under ONE IC definition (per-seed means; benchmark alignment, the only alignment v2 was run with).
IC_filtered = the definition v2 reported (large-move benchmark samples); IC_all = all firm-days (used in §14.8 of the
previous run). ACC_market = accuracy of the day-level common part only; DirAcc_stock = stock-specific direction.
                                       ACC     MCC  IC_filtered  IC_all  RankIC_filtered  RankIC_all  ACC_market  DirAcc_stock
model                                                                                                                         
v2 SCEG-RC (full)                  50.8226  0.0157       0.0409  0.0345           0.0401      0.0343     48.3817       51.4727
v3 SCEG-v3 (full)                  55.2796  0.1069       0.0702  0.0560           0.0697      0.0568     55.5376       51.9253
v2 GRU                             53.7258  0.0732       0.0718  0.0574           0.0731      0.0597     52.2204       52.0402
v3 − graph (= GRU-v3)     

In [49]:
# ===== 15.8 · claims summary for this dataset (read by the cross-dataset matrix) =====
def _get(mode, fam, a, b):
    df = FAM.get((mode, fam))
    if df is None or df.empty: return None
    r = df[(df.A == a) & (df.B == b)]
    return None if r.empty else r.iloc[0]
CLAIMS = [
    ("C1 full SCEG-v3 beats GRU-v3 (graph module, full model)", "full-model ablations", "SCEG-v3 (full)", "− graph (= GRU-v3)"),
    ("C2 nested: DYNOTEARS graph adds to trained GRU", "nested graph", "nested: + DYNOTEARS", "nested: no graph (heads retrained)"),
    ("C3 nested: DYNOTEARS beats matched random graph", "nested graph", "nested: + DYNOTEARS", "nested: + random (matched to DYNOTEARS)"),
    ("C4 nested: sector graph adds to trained GRU", "nested graph", "nested: + sector", "nested: no graph (heads retrained)"),
    ("C5 nested: sector beats matched random graph", "nested graph", "nested: + sector", "nested: + random (matched to DYNOTEARS)"),
    ("C6 regime conditioning helps (vs none)", "full-model ablations", "SCEG-v3 (full)", "− regime conditioning"),
    ("C7 real regimes beat shuffled regimes", "full-model ablations", "SCEG-v3 (full)", "regime placebo (shuffled)"),
    ("C8 structured events add to raw text", "full-model ablations", "SCEG-v3 (full)", "− structured events (raw only)"),
    ("C9 text adds to price only", "full-model ablations", "SCEG-v3 (full)", "price only"),
    ("C10 event attention helps", "full-model ablations", "SCEG-v3 (full)", "− event attention"),
    ("C11 event gate helps", "full-model ablations", "SCEG-v3 (full)", "− event gate"),
    ("C12 lead–lag relations help", "full-model ablations", "SCEG-v3 (full)", "− lead–lag (lag1, lag2)"),
    ("C13 SCEG-v3 beats majority class", "vs majority", "SCEG-v3 (full)", "majority class"),
    ("C14 GRU-v3 beats majority class", "vs majority", "− graph (= GRU-v3)", "majority class"),
    ("C15 v3 loss beats v2 loss (GRU-v3)", "attribution", "− graph (= GRU-v3)", "GRU-v3 | v2 loss (Huber + IC)"),
    ("C16 v3 price features beat v2 (GRU-v3)", "attribution", "− graph (= GRU-v3)", "GRU-v3 | v2 price features"),
    ("C17 market head helps (GRU-v3)", "attribution", "− graph (= GRU-v3)", "GRU-v3 | no market head"),
    ("C18 AUC selection beats IC selection (GRU-v3)", "attribution", "− graph (= GRU-v3)", "GRU-v3 | early stop on val IC"),
]
rows_ = []
for mode in V3["ALIGNMENTS"]:
    for cid, fam, a, b in CLAIMS:
        r = _get(mode, fam, a, b)
        if r is None: continue
        pos = [(r[f"p_{m} (Holm)"] < 0.05) and (r[f"d{m}"] > 0) for m in ("ACC", "MCC", "IC")]
        neg = [(r[f"p_{m} (Holm)"] < 0.05) and (r[f"d{m}"] < 0) for m in ("ACC", "MCC", "IC")]
        verdict = ("mixed (significant in opposite directions)" if any(pos) and any(neg) else "supported" if any(pos)
                   else "contradicted" if any(neg) else "not supported (n.s.)")
        rows_.append({"dataset": CONFIG.get("DATASET", "fnspid"), "alignment": mode, "claim": cid,
                      "ΔACC [95% CI] (Holm p)": f"{r.dACC:+.2f} {r['ΔACC 95% CI']} ({r['p_ACC (Holm)']:.3f})",
                      "ΔMCC (Holm p)": f"{r.dMCC:+.3f} ({r['p_MCC (Holm)']:.3f})",
                      "ΔIC (t, Holm p)": f"{r.dIC:+.4f} ({r.t_IC:+.2f}, {r['p_IC (Holm)']:.3f})", "verdict": verdict})
CLAIMS_DF = pd.DataFrame(rows_)
CLAIMS_DF.to_csv(f"{V3DIR}/claims_{CONFIG.get('DATASET', 'fnspid')}_{V3['ENCODER']}_{V3['SCHEMA']}.csv", index=False)
print(CLAIMS_DF.to_string(index=False))

dataset alignment                                                   claim        ΔACC [95% CI] (Holm p)  ΔMCC (Holm p)        ΔIC (t, Holm p)              verdict
  acl18    strict C1 full SCEG-v3 beats GRU-v3 (graph module, full model)  +0.15 [-2.75, +2.64] (1.000) +0.010 (1.000) -0.0022 (-0.19, 1.000) not supported (n.s.)
  acl18    strict          C2 nested: DYNOTEARS graph adds to trained GRU  -0.55 [-1.40, +0.28] (1.000) -0.011 (1.000) +0.0040 (+2.09, 0.183) not supported (n.s.)
  acl18    strict         C3 nested: DYNOTEARS beats matched random graph  +0.10 [-0.93, +1.19] (1.000) +0.002 (1.000) +0.0055 (+2.48, 0.105) not supported (n.s.)
  acl18    strict             C4 nested: sector graph adds to trained GRU  -0.47 [-1.46, +0.29] (1.000) -0.010 (1.000) -0.0022 (-0.79, 0.876) not supported (n.s.)
  acl18    strict            C5 nested: sector beats matched random graph  +0.18 [-1.30, +1.68] (1.000) +0.003 (1.000) -0.0006 (-0.21, 0.876) not supported (n.s.)
  acl18    strict     

In [50]:
# ===== 15.9 · cross-dataset claim–evidence matrix and the pre-registered decision =====
# 🔧 [v9] generic over the three dataset caches (v8 knew only acl18 <-> fnspid). Logic and decision rule unchanged.
_root9 = os.path.dirname(CACHE.rstrip("/"))
_CACHES9 = {"acl18": "acl18_cache", "fnspid": "fnspid_cache", "india": "india_cache"}
_this = CONFIG.get("DATASET", "fnspid")
OTHER_V3 = {ds: os.path.join(_root9, d, "v3") for ds, d in _CACHES9.items() if ds != _this}
parts = [CLAIMS_DF]
for ds, d in OTHER_V3.items():
    _op = f"{d}/claims_{ds}_{V3['ENCODER']}_{V3['SCHEMA']}.csv"
    if os.path.exists(_op) and not SYNTH: parts.append(pd.read_csv(_op))
    else: print(f"({ds} claims not found at {_op}; run that notebook, then re-run this cell)")
ALLC = pd.concat(parts, ignore_index=True)
ALLC = ALLC[ALLC.alignment == "strict"]
MATRIX = ALLC.pivot_table(index="claim", columns="dataset", values=["ΔACC [95% CI] (Holm p)", "ΔIC (t, Holm p)", "verdict"], aggfunc="first")
print(MATRIX.to_string())
MATRIX.to_csv(f"{V3DIR}/claim_evidence_matrix_{V3['ENCODER']}_{V3['SCHEMA']}.csv")

def verdict_of(ds, claim):
    r = ALLC[(ALLC.dataset == ds) & (ALLC.claim.str.startswith(claim))]
    return None if r.empty else r.iloc[0]["verdict"]
for primary in [d for d in ("fnspid", "india") if (ALLC.dataset == d).any()] or [_this]:
    A_ = verdict_of(primary, "C2") == "supported" and verdict_of(primary, "C3") == "supported"
    B_ = (verdict_of(primary, "C4") == "supported" and verdict_of(primary, "C5") == "supported" and not A_)
    outcome = "A" if A_ else "B" if B_ else "C"
    print(f"\nPRE-REGISTERED DECISION (dataset: {primary}, strict): outcome {outcome}")
    print({"A": "narrow positive graph claim: report exactly which relation/component carries the gain",
           "B": "relational propagation helps (sector) but the learned causal graph is too noisy",
           "C": "freeze the model; the paper reports the controlled cross-dataset negative result"}[outcome])
    for c in ("C2", "C3", "C4", "C5"):
        print(f"  {c}: " + " | ".join(f"{ds} = {verdict_of(ds, c)}" for ds in ("acl18", "fnspid", "india") if (ALLC.dataset == ds).any()))


(india claims not found at /content/drive/MyDrive/MinorProject/india_cache/v3/claims_india_bge_v3.csv; run that notebook, then re-run this cell)
                                                                      verdict                              ΔACC [95% CI] (Holm p)                                       ΔIC (t, Holm p)                        
dataset                                                                 acl18                fnspid                         acl18                        fnspid                   acl18                  fnspid
claim                                                                                                                                                                                                          
C1 full SCEG-v3 beats GRU-v3 (graph module, full model)  not supported (n.s.)  not supported (n.s.)  +0.15 [-2.75, +2.64] (1.000)  +0.85 [+0.05, +1.64] (0.481)  -0.0022 (-0.19, 1.000)  -0.0029 (-0.45, 1.000)
C10 event attention hel

---
# 16 · SCEG-R: the pre-registered final model (frozen on validation before any of its test predictions exist)

**Why this section exists.** Sections 14–15 showed on ACL18 (strict) that no SCEG-v3 mechanism is significant after
Holm correction; the nested stage 2 often fell back to the graph-free model *on validation*; structured events never
beat raw text; the fusion gate stayed at ≈0.5; regime and event-attention effects appear only under the leaky
*benchmark* alignment; and the v3 objective raised market-direction accuracy while the cross-sectional IC fell
(GRU with all four v2 choices reverted: significantly higher IC). SCEG-R is the **one** consolidation these facts
justify. It removes or simplifies components and changes one objective detail; it adds no new mechanism.

| Change vs SCEG-v3 | Evidence / theory | Validation experiment (reported before the test) | Leakage risk |
|---|---|---|---|
| Raw text is the primary text channel; structured events enter as a **zero-initialised residual** (txt = raw + a·struct, a = 0 at start) instead of a sigmoid fusion gate | fusion weight ≈ 0.50 (inert); raw-only ≥ full ≥ structured-only; a residual starts at the raw-only model and can only move away from it if training finds signal | SCEG-R vs "SCEG-R \| raw only" (val AUC, val IC, paired over seeds × folds) | none (same features) |
| **Stock-relative head is demeaned**: logit = market(t) + [s_i − mean_valid(s)] | ranking is a cross-sectional task; with a non-demeaned s the BCE loss can move s's day mean and trade ranking for market timing (v3 lowered IC while raising market ACC) | SCEG-R vs "SCEG-R \| joint head" | none (the mean is over the same day's firms, all at time t) |
| Regime is **context only** (posterior as encoder + market-head input); FiLM on messages removed | "− regime conditioning" n.s. under strict; FNSPID HMM never assigns *Bear* on a test day, so a regime-gated mechanism is barely exercised | not re-tested (already tested in §14/§15; removal is a simplification) | HMM fitted on training dates only (unchanged) |
| Event attention and event gate **removed** from propagation; messages are prior-weighted, zero-initialised and residual-gated | "− event attention"/"− event gate" n.s. under strict; attention's IC benefit exists only under benchmark alignment (post-close tweets), i.e. it is the component most exposed to timing leakage | graph arms below | lower than v3 |
| Graph is an **optional residual stage** on a frozen encoder, adopted only if validation says so; DYNOTEARS is a tested prior, not an assumption | nested stage 2 fell back to the base on validation for several seeds; DYNOTEARS vs no-graph / random ΔIC positive but n.s. | stage 2 with DYNOTEARS, sector, random (matched), and a no-graph heads-retrained control | priors fitted on training dates only |
| Price features (v2 vs v3) **chosen on validation** | v2 price was better on the ACL18 *test* (MCC) — that cannot decide anything; validation decides | stage-1 arms price = v3 / v2 | per-fold training statistics |
| Early stopping on **(val AUC − 0.5) + val IC** | the model has two outputs; AUC-only stopping selects market timing, IC-only selects ranking; the sum weights both excess-over-chance measures equally | SCEG-R vs "SCEG-R \| AUC stopping" | validation only |

**Unchanged and not re-tuned:** window, hidden size, optimiser, epochs/patience, dropout and λ_rank = the ACL18
validation selection for the GRU family (FNSPID inherits it). Seeds 0–4. Strict alignment only.

**Freeze protocol.** (1) The specification below is written to disk and hashed; a later run with a different
specification stops. (2) Every arm is trained with `with_test=False`; model states are saved. (3) Two choices are made
from validation only — price set, then graph prior — by the rules printed below. (4) The selection file is written.
(5) Only then are test predictions produced for the final model and for all pre-registered arms (so that the
ablations are reported, not hidden). **Disclosure:** on ACL18 the strict test set was already inspected for SCEG-v3
variants, so SCEG-R's ACL18 test result is *not* fully held out. FNSPID is the independent confirmation, and SCEG-R
was specified before any FNSPID v3 result was seen.

In [51]:
# ===== 16.0 · SCEG-R model + pre-registration =====
class SCEGR(SCEG3):
    """SCEG-v3 encoder with (i) raw-primary text + zero-initialised structured residual, (ii) no FiLM, no event
    attention, no event gate, (iii) optional zero-initialised residual graph stage, (iv) demeaned stock-relative head."""
    def __init__(self, *a, joint_head=False, **k):
        k.update(film=False, attention=False, event_gate=False)
        super().__init__(*a, **k)
        self.joint_head = joint_head
        if hasattr(self, "fuse"): del self.fuse
        if self.use_struct and self.use_raw:
            self.s_mix = nn.ParameterDict({"a": nn.Parameter(torch.zeros(1))})

    def forward(self, X, PI, MK):
        B, W, Nn, _ = X.shape
        parts, txt = [Fnn.gelu(self.price(self._x(X, "price")))], None
        if self.has_text:
            if self.use_raw and self.use_struct:
                txt = self.raw(self._x(X, "raw")) + self.s_mix["a"] * self.struct(torch.cat([self._x(X, "tri"), self._x(X, "cat")], -1))
                self.stats["struct_residual_scale"] = float(self.s_mix["a"].detach())
            elif self.use_raw:
                txt = self.raw(self._x(X, "raw"))
            else:
                txt = self.struct(torch.cat([self._x(X, "tri"), self._x(X, "cat")], -1))
            parts.append(Fnn.gelu(txt + self.news(self._x(X, "news"))))
        if self.regime_feat: parts.append(PI[:, :, None, :].expand(B, W, Nn, PI.shape[-1]))
        x = self.inp(torch.cat(parts, -1))
        o, _ = self.gru(x.permute(0, 2, 1, 3).reshape(B * Nn, W, self.H)); o = o.view(B, Nn, W, self.H)
        u, u1 = o[:, :, -1], o[:, :, -2]
        valid = (self._x(X, "price")[:, -1].abs().sum(-1) > 0).float()               # B,N firms present on day t
        h = u
        if self.use_graph:
            agg = 0.0
            for r, name in enumerate(self.rels):
                src = u1 if name == "lag2" else u
                agg = agg + torch.einsum("ij,bjh->bih", self.S[r], self.V[r](src) * valid[..., None])
            agg = self.out(agg)
            gr = torch.sigmoid(self.res(torch.cat([u, agg], -1)))
            h = u + gr * agg
            self.stats["residual_gate"] = float(gr.detach().mean())
        s = self.idio(self.drop(h)).squeeze(-1)
        cnt = valid.sum(1, keepdim=True).clamp(min=1)
        hbar = (h * valid[..., None]).sum(1) / cnt
        m = self.mkt(torch.cat([hbar, MK[:, -1], PI[:, -1]], -1))
        if self.joint_head:
            return s + m, s
        s_c = s - (s * valid).sum(1, keepdim=True) / cnt                              # stock-relative part only
        return m + s_c, s

_build_model_v3 = build_model
def build_model(f3, mode, cfg, hp, seed, sl=None):
    if cfg.get("family") != "scegr":
        return _build_model_v3(f3, mode, cfg, hp, seed, sl)
    S = f3["G"]
    if cfg.get("random_graph"):
        rng = np.random.default_rng(seed + 99)
        S = {**{r: random_signed(M, rng) for r, M in f3["G"].items() if r != "sector"}, "sector": f3["G"]["sector"]}
    rels = cfg.get("rels") or ("contemp",)
    return SCEGR(sl or f3["sl"], f3["K"], f3["MK"].shape[1], V3["HIDDEN"], hp["dropout"], rels, S,
                 use_graph=cfg.get("use_graph", False), use_raw=cfg.get("use_raw", True), use_struct=cfg.get("use_struct", True),
                 A=V3["ATT_DIM"], zero_out=True, joint_head=cfg.get("joint_head", False)).to(DEVICE)

HP_R = SELECTED["gru"]
R_BASE = dict(family="scegr", use_graph=False, select="composite")
R_SPEC = {
    "hp": HP_R, "seeds": V3["SEEDS"], "mode": V3["PRIMARY"],
    "stage1_price_arms": {"v3": {**R_BASE, "price": "v3"}, "v2": {**R_BASE, "price": "v2"}},
    "stage1_ablations": {"raw only": dict(use_struct=False), "joint head": dict(joint_head=True), "AUC stopping": dict(select="auc")},
    "stage2_arms": {"heads retrained (no graph)": None, "+ DYNOTEARS": ("contemp", "lag1", "lag2"),
                    "+ sector": ("sector",), "+ random (matched)": ("contemp", "lag1", "lag2")},
    "rule_price": "argmax of the pooled (folds x seeds) mean validation composite; tie (|diff| < 1e-4) -> v3 (no change)",
    "rule_graph": "adopt prior g if mean paired val composite(g - heads retrained) > 2*SE and mean paired val composite(g - random) > 0; "
                  "if several pass, the larger mean vs heads retrained; otherwise the final model has no graph (stage-1 base)",
    "selection_criterion": "(val AUC - 0.5) + val IC",
}
R_DIR = f"{PRED3}/scegr"; os.makedirs(R_DIR, exist_ok=True)
R_PREREG = f"{V3DIR}/scegr_prereg_{V3['ENCODER']}_{V3['SCHEMA']}.json"
_spec_txt = json.dumps(R_SPEC, sort_keys=True, default=str)
if os.path.exists(R_PREREG):
    assert json.load(open(R_PREREG))["spec"] == _spec_txt, "SCEG-R specification changed after pre-registration: refuse to run"
else:
    json.dump({"spec": _spec_txt, "sha": sha(_spec_txt)[:12], "at": time.strftime("%Y-%m-%d %H:%M:%S"),
               "dataset": CONFIG.get("DATASET", "fnspid")}, open(R_PREREG, "w"), indent=1)
print("SCEG-R pre-registered:", R_PREREG, "| spec sha", sha(_spec_txt)[:12])
# 🔧 [v9] SCEG-R must be byte-identical to the v8 pre-registration on every dataset (no India-specific change)
_prod = json.dumps({**R_SPEC, "hp": ACL18_SELECTED["gru"], "seeds": [0, 1, 2, 3, 4], "mode": "strict"}, sort_keys=True, default=str)
assert sha(_prod)[:12] == EXPECTED_SCEGR_SPEC_SHA, "SCEG-R specification code differs from v8"
if not SMOKE3:
    assert sha(_spec_txt)[:12] == EXPECTED_SCEGR_SPEC_SHA, "SCEG-R specification differs from the v8 pre-registration"
print("SCEG-R specification identical to v8:", EXPECTED_SCEGR_SPEC_SHA)

def r_state(f3, arm, seed): return f"{R_DIR}/{R_SPEC['mode']}__{fold_tag(f3)}{slug(arm)}__s{seed}.pt"
def r_val(f3, arm, seed):   return f"{R_DIR}/{R_SPEC['mode']}__{fold_tag(f3)}{slug(arm)}__s{seed}__val.npz"
R_CFG = {}                                              # arm name -> (cfg, stage-2 base arm or None)

def r_train(f3, arm, cfg, seed, base_arm=None):
    """Train one arm WITHOUT test predictions; save its state and validation outputs."""
    if os.path.exists(r_state(f3, arm, seed)) and os.path.exists(r_val(f3, arm, seed)): return
    init = torch.load(r_state(f3, base_arm, seed), map_location="cpu") if base_arm else None
    out, mdl = train3(f3, R_SPEC["mode"], cfg, HP_R, seed, with_test=False, init_state=init,
                      freeze=(ENCODER_MODULES + ("s_mix",)) if base_arm else ())
    torch.save(mdl.state_dict(), r_state(f3, arm, seed))
    np.savez(r_val(f3, arm, seed), val_logit=out["val_logit"], val_score=out["val_score"])
    print(f"  {f3['name']} {arm:38s} seed {seed} | val AUC {out['val_auc']:.4f} | val sel {out['val_sel']:.4f} | "
          f"{ {k: round(v, 3) for k, v in mdl.stats.items()} }", flush=True)

def r_valstats(arm):
    """Per (fold, seed): val AUC, val IC, composite — computed from saved validation outputs only."""
    rows_ = []
    for f3 in FOLD3:
        for seed in R_SPEC["seeds"]:
            z = np.load(r_val(f3, arm, seed)); a = auc_on(z["val_logit"], f3["VA"]); ic = float(ic_days(z["val_score"], f3["VA"]).mean())
            rows_.append(dict(arm=arm, fold=f3["name"], seed=seed, val_AUC=a, val_IC=ic, composite=(a - 0.5) + ic))
    return pd.DataFrame(rows_)

def paired_val(a, b):
    d = r_valstats(a).composite.values - r_valstats(b).composite.values
    return float(d.mean()), float(d.std(ddof=1) / np.sqrt(len(d))) if len(d) > 1 else float("nan")

SCEG-R pre-registered: /content/drive/MyDrive/MinorProject/acl18_cache/v3/scegr_prereg_bge_v3.json | spec sha 0bfe3098df8c
SCEG-R specification identical to v8: 0bfe3098df8c


In [52]:
# ===== 16.1 · stage 1 (validation only): price-set arms, then choice of price set =====
t0 = time.time()
for p_, cfg in R_SPEC["stage1_price_arms"].items():
    R_CFG[f"SCEG-R base | price {p_}"] = (cfg, None)
for f3 in FOLD3:
    for seed in R_SPEC["seeds"]:
        for arm, (cfg, _) in list(R_CFG.items()):
            r_train(f3, arm, cfg, seed)
VS = {p_: r_valstats(f"SCEG-R base | price {p_}") for p_ in R_SPEC["stage1_price_arms"]}
for p_, d in VS.items():
    print(f"price {p_}: val AUC {d.val_AUC.mean():.4f}±{d.val_AUC.std(ddof=0):.4f} | val IC {d.val_IC.mean():.4f}±{d.val_IC.std(ddof=0):.4f} "
          f"| composite {d.composite.mean():.4f}")
P_STAR = "v2" if VS["v2"].composite.mean() - VS["v3"].composite.mean() > 1e-4 else "v3"
BASE_ARM = f"SCEG-R base | price {P_STAR}"
print(f"stage-1 choice (validation): price {P_STAR} | {(time.time() - t0) / 60:.1f} min")

price v3: val AUC 0.6157±0.0168 | val IC 0.0631±0.0170 | composite 0.1788
price v2: val AUC 0.6085±0.0085 | val IC 0.0583±0.0128 | composite 0.1668
stage-1 choice (validation): price v3 | 0.0 min


In [53]:
# ===== 16.2 · component ablations and stage 2 graph arms (validation only) =====
base_cfg = R_SPEC["stage1_price_arms"][P_STAR]
for name, delta in R_SPEC["stage1_ablations"].items():
    R_CFG[f"SCEG-R | {name}"] = ({**base_cfg, **delta}, None)
for name, rels in R_SPEC["stage2_arms"].items():
    cfg = ({**base_cfg} if rels is None else {**base_cfg, "use_graph": True, "rels": rels, "random_graph": "random" in name})
    R_CFG[f"SCEG-R {name}"] = (cfg, BASE_ARM)
t0 = time.time()
for f3 in FOLD3:
    for seed in R_SPEC["seeds"]:
        for arm, (cfg, b) in R_CFG.items():
            r_train(f3, arm, cfg, seed, base_arm=b)
print(f"validation arms done in {(time.time() - t0) / 60:.1f} min")

VAL_TAB = pd.concat([r_valstats(a) for a in R_CFG]).groupby("arm")[["val_AUC", "val_IC", "composite"]].agg(["mean", "std"])
print("\n=== SCEG-R validation table (no test data used) ===")
print(VAL_TAB.round(4).to_string())
print("\nmodification checks on validation (paired over folds x seeds; composite difference, mean ± SE):")
for name in R_SPEC["stage1_ablations"]:
    mu, se = paired_val(BASE_ARM, f"SCEG-R | {name}")
    print(f"  SCEG-R base − {name:14s}: {mu:+.4f} ± {se:.4f}")

# ---- graph adoption rule (pre-registered) ----
ctrl, rnd = "SCEG-R heads retrained (no graph)", "SCEG-R + random (matched)"
cands = []
for g in ("SCEG-R + DYNOTEARS", "SCEG-R + sector"):
    mu, se = paired_val(g, ctrl); mr, _ = paired_val(g, rnd)
    ok = (mu > 2 * se) and (mr > 0)
    print(f"  {g:22s}: vs no graph {mu:+.4f} ± {se:.4f} | vs random {mr:+.4f} -> {'ADOPT' if ok else 'reject'}")
    if ok: cands.append((mu, g))
FINAL_ARM = max(cands)[1] if cands else BASE_ARM

# ---- freeze ----
R_SEL = f"{V3DIR}/scegr_selection_{V3['ENCODER']}_{V3['SCHEMA']}.json"
if os.path.exists(R_SEL):
    _sel = json.load(open(R_SEL))
    assert _sel["final_arm"] == FINAL_ARM and _sel["price"] == P_STAR, "selection differs from the frozen file"
else:
    json.dump({"final_arm": FINAL_ARM, "price": P_STAR, "prereg_sha": sha(_spec_txt)[:12],
               "val_table": VAL_TAB.round(5).reset_index().astype(str).values.tolist(),
               "frozen_at": time.strftime("%Y-%m-%d %H:%M:%S")}, open(R_SEL, "w"), indent=1)
print(f"\nFROZEN final model: {FINAL_ARM} (price {P_STAR}) -> {R_SEL} | sha1 {sha(open(R_SEL).read())[:12]}")

validation arms done in 0.0 min

=== SCEG-R validation table (no test data used) ===
                                  val_AUC          val_IC         composite        
                                     mean     std    mean     std      mean     std
arm                                                                                
SCEG-R + DYNOTEARS                 0.6157  0.0188  0.0631  0.0190    0.1788  0.0106
SCEG-R + random (matched)          0.6206  0.0292  0.0615  0.0224    0.1821  0.0125
SCEG-R + sector                    0.6212  0.0304  0.0613  0.0229    0.1824  0.0129
SCEG-R base | price v2             0.6085  0.0095  0.0583  0.0143    0.1668  0.0098
SCEG-R base | price v3             0.6157  0.0188  0.0631  0.0190    0.1788  0.0106
SCEG-R heads retrained (no graph)  0.6157  0.0188  0.0631  0.0190    0.1788  0.0106
SCEG-R | AUC stopping              0.6426  0.0182  0.0290  0.0144    0.1716  0.0144
SCEG-R | joint head                0.6272  0.0185  0.0613  0.0189    0.1885

In [54]:
require_shakedown("16.3 SCEG-R test predictions")   # 🔧 [v9] no-op except India
# ===== 16.3 · test predictions — produced only now, for the frozen final model and every pre-registered arm =====
R_FINAL_NAME = "SCEG-R (final, frozen)"
assert os.path.exists(R_SEL)
def r_test(arm, out_name):
    cfg, _ = R_CFG[arm] if arm in R_CFG else (R_SPEC["stage1_price_arms"][arm.split()[-1]], None)
    for f3 in FOLD3:
        Xt, PIt, MKt, sl = tensors_for(f3, R_SPEC["mode"], cfg)
        for seed in R_SPEC["seeds"]:
            path = pred_path(f3, R_SPEC["mode"], out_name, seed)
            if os.path.exists(path): continue
            mdl = build_model(f3, R_SPEC["mode"], cfg, HP_R, seed, sl); mdl.load_state_dict(torch.load(r_state(f3, arm, seed), map_location="cpu"))
            z = np.load(r_val(f3, arm, seed)); tl, ts = predict3(mdl, Xt, PIt, MKt, f3["TE"])
            np.savez(path, val_logit=z["val_logit"], val_score=z["val_score"], te_logit=tl, te_score=ts)
r_test(FINAL_ARM, R_FINAL_NAME)
for arm in R_CFG: r_test(arm, arm)
print("test predictions written")

test predictions written


In [55]:
# ===== 16.4 · results: SCEG-R vs the v3 models, and its own pre-registered ablations =====
mode = R_SPEC["mode"]
R_MODELS = [R_FINAL_NAME] + list(R_CFG) + ["SCEG-v3 (full)", "− graph (= GRU-v3)", "MLP-v3"]
rows_ = [summary_row(v, load_runs(mode, v), TE_ALL)[0] for v in R_MODELS if load_runs(mode, v)]
c = cls_metrics(MAJ["maj"]["te_logit"], TE_ALL)
rows_.append({"model": "majority class", "seeds": 0, "ACC": f"{c['ACC']:.2f}", "MCC": "0", "up": f"{c['up']:.1f}"})
R_TAB = pd.DataFrame(rows_).set_index("model")
print(f"=== [{mode}] {CONFIG.get('DATASET', 'fnspid')} test: SCEG-R (per-seed mean ± sd) ===")
print(R_TAB.to_string()); R_TAB.to_csv(f"{V3DIR}/scegr_table_{mode}_{V3['ENCODER']}_{V3['SCHEMA']}.csv")

R_FAM = {}
R_FAM["headline"] = family_table([(R_FINAL_NAME, "majority class"), (R_FINAL_NAME, "SCEG-v3 (full)"),
                                  (R_FINAL_NAME, "− graph (= GRU-v3)"), (R_FINAL_NAME, "MLP-v3")], mode, "SCEG-R headline")
R_FAM["components"] = family_table([(BASE_ARM, f"SCEG-R | {n}") for n in R_SPEC["stage1_ablations"]] +
                                   [(BASE_ARM, f"SCEG-R base | price {'v2' if P_STAR == 'v3' else 'v3'}")], mode, "SCEG-R components")
R_FAM["graph"] = family_table([("SCEG-R + DYNOTEARS", ctrl), ("SCEG-R + sector", ctrl), (rnd, ctrl),
                               ("SCEG-R + DYNOTEARS", rnd), ("SCEG-R + sector", rnd)], mode, "SCEG-R graph")
print("\nReading guide: the headline family is the paper's final comparison. On ACL18 it is only partially held out "
      "(the strict test was inspected for SCEG-v3 variants); FNSPID is the independent confirmation. Components and "
      "graph families test each SCEG-R modification; they do not change the frozen model.")

=== [strict] acl18 test: SCEG-R (per-seed mean ± sd) ===
                                   seeds         ACC            MCC          F1         AUC ACC_all_firmdays  ACC_market DirAcc_stock         IC_all     RankIC_all    IC_filtered RankIC_filtered           up  ens ACC (secondary)  ens MCC (secondary)
model                                                                                                                                                                                                                                                    
SCEG-R (final, frozen)                 5  55.77±1.07  0.1254±0.0263  52.70±7.35  58.20±1.11       54.45±0.74  56.30±2.98   50.88±0.23  0.0242±0.0046  0.0286±0.0049  0.0234±0.0074   0.0221±0.0071  44.60±15.98                56.80               0.1451
SCEG-R base | price v3                 5  55.77±1.07  0.1254±0.0263  52.70±7.35  58.20±1.11       54.45±0.74  56.30±2.98   50.88±0.23  0.0242±0.0046  0.0286±0.0049  0.0234±0.0074   0.0221±0.007

In [56]:
# ===== 16.5 · cross-dataset confirmation (criterion fixed before the FNSPID results) =====
# SCEG-R is CONFIRMED on a dataset if, in its headline family (Holm), it beats the majority class on MCC or IC
# (p < 0.05) AND is not significantly worse than GRU-v3 on IC. It is REFUTED otherwise. FNSPID and India are the
# confirmation datasets; ACL18 (partially inspected test) is reported but cannot confirm on its own.
# 🔧 [v9] reads all three caches (v8: acl18 <-> fnspid only). Criterion unchanged.
_h = R_FAM["headline"].assign(dataset=CONFIG.get("DATASET", "fnspid"))
_h.to_csv(f"{V3DIR}/scegr_headline_{CONFIG.get('DATASET', 'fnspid')}_{V3['ENCODER']}_{V3['SCHEMA']}.csv", index=False)
_hs = [_h]
for ds, d in OTHER_V3.items():
    _oth = f"{d}/scegr_headline_{ds}_{V3['ENCODER']}_{V3['SCHEMA']}.csv"
    if os.path.exists(_oth) and not SYNTH: _hs.append(pd.read_csv(_oth))
    else: print(f"(other dataset's SCEG-R file not found at {_oth})")
H = pd.concat(_hs, ignore_index=True)
def r_verdict(d):
    maj = d[d.B == "majority class"]; gru = d[d.B == "− graph (= GRU-v3)"]
    if maj.empty or gru.empty: return "incomplete"
    beats = (maj["p_MCC (Holm)"].iloc[0] < 0.05 and maj.dMCC.iloc[0] > 0) or (maj["p_IC (Holm)"].iloc[0] < 0.05 and maj.dIC.iloc[0] > 0)
    worse = gru["p_IC (Holm)"].iloc[0] < 0.05 and gru.dIC.iloc[0] < 0
    return "CONFIRMED" if beats and not worse else "REFUTED"
for ds, d in H.groupby("dataset"):
    print(f"{ds:7s}: SCEG-R {r_verdict(d)}" + ("  (not independent: test previously inspected)" if ds == "acl18" else ""))


(other dataset's SCEG-R file not found at /content/drive/MyDrive/MinorProject/fnspid_cache/v3/scegr_headline_fnspid_bge_v3.csv)
(other dataset's SCEG-R file not found at /content/drive/MyDrive/MinorProject/india_cache/v3/scegr_headline_india_bge_v3.csv)
acl18  : SCEG-R CONFIRMED  (not independent: test previously inspected)


---
# 17 · Cross-market confirmation (pre-registered in §0.1; identical code on ACL18, FNSPID and India)

**Question.** Do structured events, inter-firm propagation and regime information add *stock-specific* predictive
value, and do the ACL18 / FNSPID conclusions carry over to an untouched Indian (NSE) confirmation set?

**What is fixed before any test number of this section exists** (protocol `PROTOCOL_CORE`, hash printed in §0.1):
the frozen model (SCEG-R, v8 spec `0bfe3098df8c`), the transferred hyperparameters, the validation-selection rules,
the evaluation arms (XM arms below), 8 primary + 6 secondary hypotheses, the primary metric (daily stock-specific IC),
the tests (seed-paired; Newey–West t for ΔIC, day-block bootstrap for ΔACC/ΔMCC; Holm within family), an equivalence
margin (TOST, |ΔIC| < 0.01) and the rule that classifies each mechanism as *generalizes / does not transfer /
consistent null / fails matched placebo / inconclusive*.

**XM arms** (each changes exactly one thing relative to the validation-selected SCEG-R base, `<BASE>`):

| Arm | Stage | Change | Tests |
|---|---|---|---|
| XM \| price only | 1 | no text | H2 (text adds), S1 (raw text adds) |
| XM \| structured only | 1 | structured events, no raw text | S6 |
| XM \| regime shuffled | 1 | HMM posterior permuted across days (placebo) | H5 |
| XM \| no regime | 1 | HMM posterior replaced by a constant | S2 |
| XM + lead-lag (lag1, lag2) | 2 | DYNOTEARS lag-1/lag-2 relations only | S3, S4 |
| XM + random (matched lead-lag) | 2 | signed random graph, same edge count and weights | S4 |
| XM + random (matched sector) | 2 | random graph matched to the sector graph | H4d |

Stage 2 = the v8 nested design: the trained base encoder is frozen and a zero-initialised residual graph stage is added
(so every graph arm starts exactly at the graph-free model). The v8 arms "SCEG-R + DYNOTEARS / + sector / + random
(matched) / heads retrained (no graph)" and "SCEG-R | raw only" complete the design.


In [57]:
# ===== 17.1 · XM arms: validation-only training, then (gated) test predictions =====
XM = PROTOCOL_CORE["xm_arms"]
XM_SHA = sha(json.dumps(XM, sort_keys=True))[:12]
if "_tensors_for_v3" not in globals(): _tensors_for_v3 = tensors_for
def tensors_for(f3, mode, cfg):
    """v8 tensors, plus regime='none': the HMM posterior is replaced by the uniform constant 1/K (no regime information)."""
    if cfg.get("regime") == "none":
        key = (f3["name"], mode, "noregime", cfg.get("price", "v3"))
        if key not in _TENS:
            Xt, PIt, MKt, sl = _tensors_for_v3(f3, mode, {k: v for k, v in cfg.items() if k not in ("regime", "placebo")})
            _TENS[key] = (Xt, torch.full_like(PIt, 1.0 / PIt.shape[-1]), MKt, sl)
        return _TENS[key]
    return _tensors_for_v3(f3, mode, cfg)

if "_build_model_r" not in globals(): _build_model_r = build_model
def build_model(f3, mode, cfg, hp, seed, sl=None):
    """v8 builder, plus random_sector: the sector relation is replaced by a signed random graph with the same number
    of edges and the same weights (matched placebo for the sector prior)."""
    if cfg.get("random_sector"):
        rng = np.random.default_rng(seed + 199)
        f3b = dict(f3); f3b["G"] = {**f3["G"], "sector": random_signed(f3["G"]["sector"], rng)}
        return _build_model_r(f3b, mode, {k: v for k, v in cfg.items() if k != "random_sector"}, hp, seed, sl)
    return _build_model_r(f3, mode, cfg, hp, seed, sl)

for name, delta in XM["stage1"].items():
    R_CFG[name] = ({**base_cfg, **delta}, None)
for name, delta in XM["stage2"].items():
    d = dict(delta); d["rels"] = tuple(d["rels"])
    R_CFG[name] = ({**base_cfg, "use_graph": True, **d}, BASE_ARM)
XM_ARMS = list(XM["stage1"]) + list(XM["stage2"])
t0 = time.time()
for f3 in FOLD3:
    for seed in R_SPEC["seeds"]:
        for arm in XM_ARMS:
            cfg, b = R_CFG[arm]
            r_train(f3, arm, cfg, seed, base_arm=b)
print(f"XM arms (spec sha {XM_SHA}) trained on training data, selected on validation: {(time.time() - t0) / 60:.1f} min")
XM_VAL = pd.concat([r_valstats(a) for a in XM_ARMS + [BASE_ARM]]).groupby("arm")[["val_AUC", "val_IC", "composite"]].mean()
print(XM_VAL.round(4).to_string())
require_shakedown("17.1 XM test predictions")
for arm in XM_ARMS: r_test(arm, arm)
print("XM test predictions written")


XM arms (spec sha e271dd356798) trained on training data, selected on validation: 0.0 min
                                val_AUC  val_IC  composite
arm                                                       
SCEG-R base | price v3           0.6157  0.0631     0.1788
XM + lead-lag (lag1, lag2)       0.6157  0.0631     0.1788
XM + random (matched lead-lag)   0.6157  0.0631     0.1788
XM + random (matched sector)     0.6216  0.0613     0.1829
XM | no regime                   0.6134  0.0657     0.1791
XM | price only                  0.6253  0.0607     0.1860
XM | regime shuffled             0.6257  0.0666     0.1923
XM | structured only             0.6226  0.0450     0.1676
XM test predictions written


In [58]:
# ===== 17.2 · pre-registered hypothesis families: difference tests + equivalence (TOST) + classification =====
from scipy.stats import norm as _N
mode = R_SPEC["mode"]; DELTA = 0.01
def _resolve(x): return {"<BASE>": BASE_ARM, "<FINAL>": R_FINAL_NAME}.get(x, x)
def _runs(x): return MAJ if x == "majority class" else load_runs(mode, x)
def nw_se(x, L=5):
    x = np.asarray(x, float); T_ = len(x); d = x - x.mean()
    g = [np.sum(d[k:] * d[:T_ - k]) / T_ for k in range(L + 1)]
    return float(np.sqrt(max(g[0] + 2 * sum((1 - k / (L + 1)) * g[k] for k in range(1, L + 1)), 1e-18) / T_))
def daily_dic(ra, rb, days=None):
    days = TE_ALL if days is None else days
    seeds = sorted(set(ra) & set(rb)) if len(rb) > 1 else sorted(ra)
    getB = (lambda s: rb[s]) if len(rb) > 1 else (lambda s: next(iter(rb.values())))
    return np.mean([ic_days(ra[s]["te_score"], days) - ic_days(getB(s)["te_score"], days) for s in seeds], 0)

def run_family(key, title):
    rows = []
    for hid, label, a, b in PROTOCOL_CORE["hypotheses"][key]:
        A, B = _resolve(a), _resolve(b); ra, rb = _runs(A), _runs(B)
        if not ra or not rb:
            rows.append(dict(id=hid, hypothesis=label, A=A, B=B, status="MISSING RUNS")); continue
        pt = paired_test(ra, rb, TE_ALL)
        d = daily_dic(ra, rb); m, se = float(d.mean()), nw_se(d)
        p_tost = max(1 - _N.cdf((m + DELTA) / se), _N.cdf((m - DELTA) / se))
        rows.append(dict(id=hid, hypothesis=label, A=A, B=B, seeds=pt["seeds"], days=len(d), dIC=m, se_IC=se,
                         IC90_lo=m - 1.645 * se, IC90_hi=m + 1.645 * se, t_IC=pt["t_IC"], p_IC=pt["p_IC"], p_TOST=p_tost,
                         dACC=pt["dACC"], ACC95=f"[{pt['dACC_lo']:+.2f}, {pt['dACC_hi']:+.2f}]", p_ACC=pt["p_ACC"],
                         dMCC=pt["dMCC"], MCC95=f"[{pt['dMCC_lo']:+.3f}, {pt['dMCC_hi']:+.3f}]", p_MCC=pt["p_MCC"], status="ok"))
    df = pd.DataFrame(rows); ok = df["status"] == "ok"
    for c in ("p_IC", "p_TOST", "p_ACC", "p_MCC"):
        df.loc[ok, c + " (Holm)"] = holm(df.loc[ok, c].values)
    def cls(r):
        if r["status"] != "ok": return "missing"
        if r["p_IC (Holm)"] < 0.05: return "positive" if r["dIC"] > 0 else "negative"
        return "equivalent" if r["p_TOST (Holm)"] < 0.05 else "inconclusive"
    df["class"] = df.apply(cls, axis=1); df["family"] = key
    show = ["id", "hypothesis", "seeds", "days", "dIC", "IC90_lo", "IC90_hi", "t_IC", "p_IC (Holm)", "p_TOST (Holm)",
            "dACC", "ACC95", "p_ACC (Holm)", "dMCC", "p_MCC (Holm)", "class"]
    print(f"\n=== [{CONFIG['DATASET']}] {title} (seed-paired; Holm within family; TOST margin |dIC| < {DELTA}) ===")
    print(df[[c for c in show if c in df]].round(4).to_string(index=False))
    return df

XM_RESULTS = pd.concat([run_family("primary", "PRIMARY hypotheses H1-H5"),
                        run_family("secondary", "SECONDARY hypotheses S1-S6")], ignore_index=True)
print("\nReading guide: 'equivalent' is a positive finding of absence (effect within ±0.01 IC); 'inconclusive' means the "
      "test set cannot tell (typical for ACL18's 64 test days).")



=== [acl18] PRIMARY hypotheses H1-H5 (seed-paired; Holm within family; TOST margin |dIC| < 0.01) ===
 id                           hypothesis  seeds  days     dIC  IC90_lo  IC90_hi    t_IC  p_IC (Holm)  p_TOST (Holm)    dACC           ACC95  p_ACC (Holm)    dMCC  p_MCC (Holm)        class
 H1    SCEG-R final beats majority class      5    64  0.0242  -0.0117   0.0600  1.1100       1.0000         1.0000  4.4785 [-3.88, +13.26]           1.0  0.1254         0.080 inconclusive
 H2                   text adds to price      5    64  0.0045  -0.0236   0.0326  0.2635       1.0000         1.0000  1.9785  [-1.20, +5.16]           1.0  0.0506         0.112 inconclusive
 H3    structured events add to raw text      5    64 -0.0089  -0.0190   0.0011 -1.4656       0.9993         1.0000 -0.8011  [-1.88, +0.21]           1.0 -0.0123         1.000 inconclusive
H4a     DYNOTEARS graph adds to no graph      5    64  0.0000  -0.0000   0.0000  0.0000       1.0000         0.0000  0.0000  [+0.00, +0.00]   

In [59]:
# ===== 17.3 · market-direction vs stock-specific decomposition (so market timing cannot pose as stock selection) =====
def day_parts(logit, days):
    """Per day: market hits / n (common part of the logits vs each firm's label) and stock-specific hits / n."""
    out = np.zeros((len(days), 4))
    for r, t in enumerate(days):
        mr, mc = MR3[t], MC3[t]
        if mr.sum() < 5: continue
        c = logit[r][mr].mean(); y = YCLS3[t][mc]
        dev = logit[r][mr] - c; ydev = Y[t][mr] - Y[t][mr].mean(); ok = (dev != 0) & (ydev != 0)
        out[r] = [((c > 0) == (y == 1)).sum(), len(y), ((dev[ok] > 0) == (ydev[ok] > 0)).sum(), ok.sum()]
    return out
def boot_block(stat_fn, nd, B=None, block=None, seed=0):
    B = B or V3["BOOT"]; block = block or V3["BLOCK"]; rng = np.random.default_rng(seed); nb = int(np.ceil(nd / block)); out = []
    for _ in range(B):
        st = rng.integers(0, nd, nb); out.append(stat_fn(((st[:, None] + np.arange(block)[None]) % nd).ravel()[:nd]))
    return np.array(out)

DECOMP_MODELS = [R_FINAL_NAME, BASE_ARM, "XM | price only", "SCEG-R | raw only", "XM | structured only",
                 "SCEG-R heads retrained (no graph)", "SCEG-R + DYNOTEARS", "SCEG-R + sector", "XM + lead-lag (lag1, lag2)",
                 "XM | regime shuffled", "XM | no regime", "− graph (= GRU-v3)", "MLP-v3", "LightGBM-v3"]
rows_ = []
for v in dict.fromkeys(DECOMP_MODELS):
    runs = load_runs(mode, v)
    if not runs: continue
    per = pd.DataFrame([run_metrics(r["te_logit"], r["te_score"], TE_ALL) for r in runs.values()])
    ic_d = np.mean([ic_days(r["te_score"], TE_ALL) for r in runs.values()], 0)
    bs = boot_block(lambda idx: ic_d[idx].mean(), len(ic_d))
    rows_.append({"model": v, "seeds": len(runs), **{f"{c}": f"{per[c].mean():.2f}±{per[c].std(ddof=0):.2f}"
                  for c in ("ACC", "AUC", "ACC_market", "DirAcc_stock")},
                  "MCC": f"{per['MCC'].mean():.4f}±{per['MCC'].std(ddof=0):.4f}",
                  "IC": f"{per['IC_all'].mean():.4f}±{per['IC_all'].std(ddof=0):.4f}",
                  "IC 95% CI (seed-avg, block bootstrap)": f"[{np.percentile(bs, 2.5):+.4f}, {np.percentile(bs, 97.5):+.4f}]",
                  "RankIC": f"{per['RankIC_all'].mean():.4f}±{per['RankIC_all'].std(ddof=0):.4f}"})
_mj = run_metrics(MAJ["maj"]["te_logit"], MAJ["maj"]["te_score"], TE_ALL)
rows_.append({"model": "majority class (reference)", "seeds": 0, "ACC": f"{_mj['ACC']:.2f}", "AUC": "50.00",
              "ACC_market": f"{_mj['ACC_market']:.2f}", "DirAcc_stock": "n/a (constant)", "MCC": "0", "IC": "0"})
DECOMP = pd.DataFrame(rows_).set_index("model")
print(f"=== [{CONFIG['DATASET']}] per-seed mean ± sd; ACC_market = the day-level common part only; DirAcc_stock = "
      "stock-specific direction vs the cross-sectional mean (base ≈ 50 %) ===")
print(DECOMP.to_string())

drows = []
for _, r in XM_RESULTS[XM_RESULTS.status == "ok"].iterrows():
    ra, rb = _runs(r.A), _runs(r.B)
    seeds = sorted(set(ra) & set(rb)) if len(rb) > 1 else sorted(ra)
    getB = (lambda s: rb[s]) if len(rb) > 1 else (lambda s: next(iter(rb.values())))
    PA = np.stack([day_parts(ra[s]["te_logit"], TE_ALL) for s in seeds]); PB = np.stack([day_parts(getB(s)["te_logit"], TE_ALL) for s in seeds])
    def st(idx):
        a, b = PA[:, idx].sum(1), PB[:, idx].sum(1)
        return [np.mean(a[:, 0] / a[:, 1] - b[:, 0] / b[:, 1]) * 100, np.mean(a[:, 2] / a[:, 3] - b[:, 2] / b[:, 3]) * 100]
    pt = st(np.arange(len(TE_ALL))); bs = boot_block(st, len(TE_ALL))
    p2 = lambda v: float(min(1.0, 2 * min(np.mean(v <= 0), np.mean(v >= 0))))
    drows.append(dict(id=r.id, dACC_market=pt[0], p_market=p2(bs[:, 0]), dDirAcc_stock=pt[1], p_stock=p2(bs[:, 1]),
                      dIC=r.dIC, cls=r["class"]))
DDEC = pd.DataFrame(drows)
if len(DDEC):
    DDEC["reading"] = np.where((DDEC.p_market < 0.05) & (DDEC.dACC_market > 0) & (DDEC.cls != "positive"),
                               "market timing only: NOT evidence of stock selection", "")
    print("\nPaired decomposition of every hypothesis (uncorrected bootstrap p; the IC class from 17.2 is the verdict):")
    print(DDEC.round(4).to_string(index=False))


=== [acl18] per-seed mean ± sd; ACC_market = the day-level common part only; DirAcc_stock = stock-specific direction vs the cross-sectional mean (base ≈ 50 %) ===
                                   seeds         ACC         AUC  ACC_market    DirAcc_stock             MCC              IC IC 95% CI (seed-avg, block bootstrap)          RankIC
model                                                                                                                                                                             
SCEG-R (final, frozen)                 5  55.77±1.07  58.20±1.11  56.30±2.98      50.88±0.23   0.1254±0.0263   0.0242±0.0046                    [-0.0183, +0.0634]   0.0286±0.0049
SCEG-R base | price v3                 5  55.77±1.07  58.20±1.11  56.30±2.98      50.88±0.23   0.1254±0.0263   0.0242±0.0046                    [-0.0183, +0.0634]   0.0286±0.0049
XM | price only                        5  53.79±0.63  56.35±1.92  54.34±1.49      51.22±0.56   0.0748±0.0140   0.0197±0.0

In [60]:
# ===== 17.4 · EXPLORATORY (uncorrected): is graph / regime usefulness conditional on the market regime or year? =====
reg_day = np.concatenate([np.array([f3["rinfo"]["names"][k] for k in f3["PI"][f3["TE"]].argmax(1)]) for f3 in FOLD3])
yr_day = np.concatenate([np.full(len(f3["TE"]), f3["fd"]["test_year"]) for f3 in FOLD3])
print(f"[{CONFIG['DATASET']}] test-day regime occupancy (filtered HMM, per-fold labels):", pd.Series(reg_day).value_counts().to_dict())
erows = []
for hid in ("H4a", "H4c", "S3", "H5", "S2"):
    r = XM_RESULTS[(XM_RESULTS.id == hid) & (XM_RESULTS.status == "ok")]
    if r.empty: continue
    r = r.iloc[0]; d = daily_dic(_runs(r.A), _runs(r.B))
    for kind, lab in (("regime", reg_day), ("year", yr_day)):
        for g in sorted(set(lab)):
            x = d[lab == g]
            if len(x) < 10: continue
            erows.append(dict(id=hid, split=kind, group=str(g), days=len(x), dIC=x.mean(), t=x.mean() / nw_se(x)))
    if {"Bear", "Bull"} <= set(reg_day):
        a, b = d[reg_day == "Bear"], d[reg_day == "Bull"]
        if len(a) >= 10 and len(b) >= 10:
            erows.append(dict(id=hid, split="contrast", group="Bear − Bull", days=len(a) + len(b), dIC=a.mean() - b.mean(),
                              t=(a.mean() - b.mean()) / np.sqrt(nw_se(a) ** 2 + nw_se(b) ** 2)))
EXPL = pd.DataFrame(erows)
if len(EXPL): print(EXPL.round(4).to_string(index=False))
print("EXPLORATORY: not corrected for multiplicity; a regime-conditional effect is reported as conditional usefulness only "
      "if it also appears in another dataset.")


[acl18] test-day regime occupancy (filtered HMM, per-fold labels): {'Bull': 56, 'Neutral': 8}
 id  split group  days     dIC       t
H4a regime  Bull    56  0.0000  0.0000
H4a   year  2015    64  0.0000  0.0000
H4c regime  Bull    56 -0.0002 -0.0564
H4c   year  2015    64 -0.0015 -0.5282
 S3 regime  Bull    56  0.0000  0.0000
 S3   year  2015    64  0.0000  0.0000
 H5 regime  Bull    56 -0.0080 -1.5409
 H5   year  2015    64 -0.0097 -1.9996
 S2 regime  Bull    56 -0.0059 -1.3237
 S2   year  2015    64 -0.0065 -1.5728
EXPLORATORY: not corrected for multiplicity; a regime-conditional effect is reported as conditional usefulness only if it also appears in another dataset.


In [61]:
# ===== 17.5 · export this dataset's confirmation results (read by 17.6 in every notebook) =====
XM_FILE = f"{V3DIR}/xmarket_{CONFIG['DATASET']}_{V3['ENCODER']}_{V3['SCHEMA']}.csv"
_exp = XM_RESULTS.assign(dataset=CONFIG["DATASET"], protocol_core=CORE_SHA, protocol=PROTOCOL_SHA,
                         partially_inspected=CONFIG["DATASET"] == "acl18")
if len(DDEC): _exp = _exp.merge(DDEC[["id", "dACC_market", "p_market", "dDirAcc_stock", "p_stock"]], on="id", how="left")
_exp.to_csv(XM_FILE, index=False)
DECOMP.to_csv(f"{V3DIR}/xmarket_decomposition_{CONFIG['DATASET']}_{V3['ENCODER']}_{V3['SCHEMA']}.csv")
if len(EXPL): EXPL.to_csv(f"{V3DIR}/xmarket_exploratory_{CONFIG['DATASET']}_{V3['ENCODER']}_{V3['SCHEMA']}.csv", index=False)
print("written:", XM_FILE)


written: /content/drive/MyDrive/MinorProject/acl18_cache/v3/xmarket_acl18_bge_v3.csv


In [62]:
# ===== 17.6 · CROSS-MARKET TABLE: which mechanisms generalize, which are dataset-dependent, which fail placebos =====
_root = os.path.dirname(CACHE.rstrip("/")); _caches = {"acl18": "acl18_cache", "fnspid": "fnspid_cache", "india": "india_cache"}
if SYNTH: _caches = {"india": os.path.basename(CACHE.rstrip("/"))}
XR, CL = [], []
for ds, cdir in _caches.items():
    p = f"{_root}/{cdir}/v3/xmarket_{ds}_{V3['ENCODER']}_{V3['SCHEMA']}.csv"
    q = f"{_root}/{cdir}/v3/claims_{ds}_{V3['ENCODER']}_{V3['SCHEMA']}.csv"
    if os.path.exists(p):
        x = pd.read_csv(p)
        if (x["protocol_core"] != CORE_SHA).any(): print(f"WARNING: {ds} was run under a different protocol core")
        XR.append(x)
    else: print(f"({ds}: no confirmation file yet at {p})")
    if os.path.exists(q): CL.append(pd.read_csv(q))
XR = pd.concat(XR, ignore_index=True) if XR else pd.DataFrame()
CL = pd.concat(CL, ignore_index=True) if CL else pd.DataFrame()
_v3map = {"supported": "positive", "contradicted": "negative", "not supported (n.s.)": "inconclusive",
          "mixed (significant in opposite directions)": "conflicting"}
MECH = [("Raw text (vs price only)", "S1"), ("Structured events (vs raw text)", "H3"), ("Text overall (vs price only)", "H2"),
        ("DYNOTEARS graph (vs none)", "H4a"), ("DYNOTEARS vs matched random", "H4b"),
        ("Sector graph (vs none)", "H4c"), ("Sector vs matched random", "H4d"), ("Sector vs DYNOTEARS", "S5"),
        ("Lead-lag propagation (vs none)", "S3"), ("Lead-lag vs matched random", "S4"),
        ("Regime information (vs none)", "S2"), ("Real vs shuffled regimes", "H5"),
        ("Regime-conditioned propagation, v3 FiLM (vs none)", "v3:C6"), ("Regime-conditioned propagation vs shuffled (v3)", "v3:C7"),
        ("SCEG-R vs majority (stock-specific IC)", "H1")]
def _cls(ds, key):
    if key.startswith("v3:"):
        if CL.empty: return None
        r = CL[(CL.dataset == ds) & (CL.alignment == "strict") & CL.claim.str.startswith(key[3:] + " ")]
        return None if r.empty else _v3map.get(r.iloc[0]["verdict"], "inconclusive")
    if XR.empty: return None
    r = XR[(XR.dataset == ds) & (XR.id == key)]
    return None if r.empty else r.iloc[0]["class"]
def _cell(ds, key):
    if key.startswith("v3:") or XR.empty: return _cls(ds, key) or "—"
    r = XR[(XR.dataset == ds) & (XR.id == key)]
    if r.empty: return "—"
    r = r.iloc[0]
    return f"{r['class']} ({r['dIC']:+.4f})" if r["class"] != "missing" else "missing"
def generalization(key):
    c = {ds: _cls(ds, key) for ds in ("acl18", "fnspid", "india")}; c = {k: v for k, v in c.items() if v}
    ind, oth = c.get("india"), [c[d] for d in ("acl18", "fnspid") if d in c]
    if ind is None: return "India not run yet"
    if not oth: return f"India {ind}; ACL18/FNSPID not run yet"
    if "positive" in c.values() and "negative" in c.values(): return "conflicting"
    if ind == "negative": return "harmful in India"
    if ind == "positive": return "GENERALIZES (+)" if "positive" in oth else "India-specific (+)"
    if "positive" in oth: return "does not transfer"
    if ind == "equivalent" and c.get("fnspid") == "equivalent": return "consistent null"
    return "inconclusive"
_placebo = {"H4a": "H4b", "H4c": "H4d", "S3": "S4", "S2": "H5", "v3:C6": "v3:C7"}
rows_ = []
for label, key in MECH:
    g = generalization(key)
    if key in _placebo:
        pl = [ds for ds in ("acl18", "fnspid", "india") if _cls(ds, key) == "positive" and _cls(ds, _placebo[key]) != "positive"]
        if pl: g += f" | fails matched placebo in {', '.join(pl)}"
    rows_.append({"Mechanism": label, "ACL18 (partially inspected)": _cell("acl18", key), "FNSPID": _cell("fnspid", key),
                  "India (confirmation)": _cell("india", key), "Generalizes?": g})
CROSS = pd.DataFrame(rows_)
_dec_row = {"Mechanism": "Market direction vs stock-specific signal (SCEG-R final: ACC_market [majority] · DirAcc_stock · IC)"}
for ds, cdir, col in (("acl18", "acl18_cache", "ACL18 (partially inspected)"), ("fnspid", "fnspid_cache", "FNSPID"),
                      ("india", "india_cache", "India (confirmation)")):
    if SYNTH and ds != "india": _dec_row[col] = "—"; continue
    p = f"{_root}/{cdir if not SYNTH else os.path.basename(CACHE.rstrip('/'))}/v3/xmarket_decomposition_{ds}_{V3['ENCODER']}_{V3['SCHEMA']}.csv"
    if not os.path.exists(p): _dec_row[col] = "—"; continue
    d = pd.read_csv(p, index_col=0); f_ = d.index[0]; mj = d.loc["majority class (reference)"]
    _dec_row[col] = f"{d.loc[f_, 'ACC_market']} [{mj['ACC_market']}] · {d.loc[f_, 'DirAcc_stock']} · {d.loc[f_, 'IC']}"
_dec_row["Generalizes?"] = "read: timing skill = ACC_market above [majority]; selection skill = DirAcc_stock > 50 and IC > 0"
CROSS.loc[len(CROSS)] = _dec_row
pd.set_option("display.max_colwidth", 120)
print(CROSS.to_string(index=False))
CROSS.to_csv(f"{V3DIR}/cross_market_table_{V3['ENCODER']}_{V3['SCHEMA']}.csv", index=False)
print("\nClasses: positive / negative (Holm p_IC < 0.05) · equivalent (TOST |ΔIC| < 0.01, Holm) · inconclusive. "
      "Datasets differ in market AND text source (tweets / news / exchange filings): read 'generalizes' as "
      "'holds across these dataset environments', not as a pure market effect.")


(fnspid: no confirmation file yet at /content/drive/MyDrive/MinorProject/fnspid_cache/v3/xmarket_fnspid_bge_v3.csv)
(india: no confirmation file yet at /content/drive/MyDrive/MinorProject/india_cache/v3/xmarket_india_bge_v3.csv)
                                                                                          Mechanism                     ACL18 (partially inspected)       FNSPID India (confirmation)                                                                                     Generalizes?
                                                                           Raw text (vs price only)                          inconclusive (+0.0134)            —                    —                                                                                India not run yet
                                                                    Structured events (vs raw text)                          inconclusive (-0.0089)            —                    —                                

In [63]:
# ===== 17.7 · reproducibility manifest (versions, hashes of protocol, selections and data) =====
import platform, importlib
_vers = {}
for m in ("numpy", "pandas", "torch", "sklearn", "scipy", "hmmlearn", "lightgbm", "transformers", "sentence_transformers", "yfinance"):
    try: _vers[m] = importlib.import_module(m).__version__
    except Exception: _vers[m] = "n/a"
MANIFEST = {"dataset": CONFIG["DATASET"], "synthetic": SYNTH, "protocol_sha": PROTOCOL_SHA, "protocol_core_sha": CORE_SHA,
            "scegr_spec_sha": sha(_spec_txt)[:12], "scegr_selection_sha": sha(open(R_SEL).read())[:12],
            "scegr_final_arm": FINAL_ARM, "price_set": P_STAR, "xm_spec_sha": XM_SHA,
            "v3_protocol_sha": sha(open(FROZEN).read())[:12], "data_files": globals().get("DATA_MANIFEST", {}),
            "data_choices": globals().get("DATA_CHOICES", {}), "python": platform.python_version(), "packages": _vers,
            "device": str(DEVICE), "written_at": time.strftime("%Y-%m-%d %H:%M:%S")}
json.dump(MANIFEST, open(f"{V3DIR}/run_manifest_{CONFIG['DATASET']}.json", "w"), indent=1, default=str)
print(json.dumps(MANIFEST, indent=1, default=str))


{
 "dataset": "acl18",
 "synthetic": false,
 "protocol_sha": "5d21ac233a966618",
 "protocol_core_sha": "b92d3e6827d1f8e9",
 "scegr_spec_sha": "0bfe3098df8c",
 "scegr_selection_sha": "87e9f0bce585",
 "scegr_final_arm": "SCEG-R base | price v3",
 "price_set": "v3",
 "xm_spec_sha": "e271dd356798",
 "v3_protocol_sha": "0a60ab442ab1",
 "data_files": {},
 "data_choices": {},
 "python": "3.13.15",
 "packages": {
  "numpy": "2.1.3",
  "pandas": "2.2.3",
  "torch": "2.11.0+cu128",
  "sklearn": "1.6.1",
  "scipy": "1.16.3",
  "hmmlearn": "0.3.3",
  "lightgbm": "4.6.0",
  "transformers": "5.16.1",
  "sentence_transformers": "5.7.0",
  "yfinance": "0.2.66"
 },
 "device": "cuda",
 "written_at": "2026-09-29 16:55:34"
}


---
# 18 · Supplementary baselines, full classification metrics, calibration and a trading check (v9.1)

**Status of this section.** Added in v9.1, *after* the confirmation protocol (§0.1) was frozen and *before* any
India data were loaded successfully. Nothing here changes the frozen model, the §17 hypotheses or their Holm
families. It answers reviewer requests that the pre-registered design did not cover:

| Addition | Why a reviewer asks for it | How it is kept honest |
|---|---|---|
| Logistic regression | "Is a linear model on the same features enough?" | same features as LightGBM-v3; C from a fixed grid, chosen on validation AUC; deterministic |
| Transformer temporal baseline | "Is the GRU the bottleneck?" | SCEG-R base with the GRU swapped for a 2-layer causal Transformer; same inputs, loss, selection rule, seeds |
| Balanced accuracy, precision, recall, PR-AUC, confusion matrix | accuracy alone hides class imbalance and one-sided predictors | fixed 0.5 threshold (logit 0); no threshold tuning |
| Brier score, ECE (10 bins) | calibration of the predicted probabilities | reported, never used for selection |
| Long–short quintile portfolio | "does the ranking signal survive costs?" | stock-specific score, members only, `COST_BPS` per side, Newey–West t; *not* evidence of tradability |
| Per-seed table, median, best seed | seeds measure stability, not a hyperparameter | the primary number is the mean ± sd; the best seed is labelled supplementary |

The specification is hashed and frozen (18.0) before any of these test predictions exist; India test predictions are
gated by `require_shakedown`, as everywhere else.

In [64]:
# ===== 18.0 · supplementary specification (frozen before its test predictions) =====
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (balanced_accuracy_score, precision_score, recall_score, average_precision_score,
                             brier_score_loss, confusion_matrix)
SUPP = {
    "logreg": {"features": "LightGBM-v3 features: day t + window mean + regime posterior + market state",
               "C_grid": [0.01, 0.1, 1.0], "selection": "validation AUC", "fit": "classification set of the training days"},
    "transformer": {"arm": "SCEG-R base (validation-selected price set, no graph) with the GRU replaced by a 2-layer causal "
                           "Transformer encoder (4 heads, hidden 64, learned positions)", "selection": "(val AUC - 0.5) + val IC",
                    "hp": "SCEG-R hp (transferred)", "seeds": [0, 1, 2, 3, 4]},
    "metrics": ["ACC", "BalAcc", "Precision", "Recall", "F1", "AUC", "PR-AUC", "MCC", "Brier", "ECE10", "confusion matrix"],
    "threshold": "logit 0 (probability 0.5), fixed",
    "portfolio": "daily long top quintile / short bottom quintile of the stock-specific score among valid (member) firms, "
                 "equal weight, next-day close-to-close return, cost COST_BPS per side on turnover; Sharpe (252 d), NW t",
    "family": "supplementary: SCEG-R final vs LogReg, vs Transformer, vs LightGBM-v3 (Holm within this family only)",
    "status": "added in v9.1 after the protocol freeze, before any India data; outside the pre-registered hypotheses",
}
SUPP_SHA = sha(json.dumps(SUPP, sort_keys=True))[:12]
SUPP_PATH = f"{V3DIR}/supplementary_spec_frozen.json"
if os.path.exists(SUPP_PATH):
    assert json.load(open(SUPP_PATH))["sha"] == SUPP_SHA, "supplementary specification changed after freezing"
else:
    json.dump({"sha": SUPP_SHA, "spec": SUPP, "frozen_at": time.strftime("%Y-%m-%d %H:%M:%S")}, open(SUPP_PATH, "w"), indent=1)
print("supplementary spec sha", SUPP_SHA)
mode = R_SPEC["mode"]
LR_NAME, TF_NAME = "LogReg (supplementary)", "Transformer temporal (supplementary)"

# ---- Transformer temporal encoder: drop-in replacement for the GRU (returns (outputs, None) like nn.GRU) ----
class _CausalTF(nn.Module):
    def __init__(self, H, W, heads=4, layers=2, dropout=0.1):
        super().__init__()
        layer = nn.TransformerEncoderLayer(H, heads, 2 * H, dropout, activation="gelu", batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.pos = nn.Parameter(torch.randn(1, W, H) * 0.02)
        self.register_buffer("mask", torch.triu(torch.full((W, W), float("-inf")), 1))   # day k sees days <= k only
    def forward(self, x):
        w = x.shape[1]
        return self.enc(x + self.pos[:, :w], mask=self.mask[:w, :w]), None

if "_build_model_xm" not in globals(): _build_model_xm = build_model
def build_model(f3, mode, cfg, hp, seed, sl=None):
    if cfg.get("temporal_tf"):
        m = _build_model_xm(f3, mode, {k: v for k, v in cfg.items() if k != "temporal_tf"}, hp, seed, sl)
        m.gru = _CausalTF(V3["HIDDEN"], V3["WINDOW"], dropout=hp["dropout"]).to(DEVICE)
        return m
    return _build_model_xm(f3, mode, cfg, hp, seed, sl)

supplementary spec sha ea7b06bbfc75


In [65]:
# ===== 18.1 · train the supplementary baselines (validation only), then (gated) test predictions =====
R_CFG[TF_NAME] = ({**base_cfg, "temporal_tf": True}, None)
t0 = time.time()
for f3 in FOLD3:
    for seed in R_SPEC["seeds"]:
        r_train(f3, TF_NAME, R_CFG[TF_NAME][0], seed)
print(f"Transformer baseline trained (validation selection only): {(time.time() - t0) / 60:.1f} min")
print(r_valstats(TF_NAME)[["val_AUC", "val_IC", "composite"]].mean().round(4).to_dict(), "| SCEG-R base:",
      r_valstats(BASE_ARM)[["val_AUC", "val_IC", "composite"]].mean().round(4).to_dict())

def logreg_fold(f3):
    """Fit on training days, choose C on validation AUC. Returns (C, val_auc, model, scaler, feature fn)."""
    f = lgbm_features(f3, mode)
    Ztr, Zva = f(f3["TR"]), f(f3["VA"]); mtr = MC3[f3["TR"]]
    sc = StandardScaler().fit(Ztr[mtr]); best = None
    for C in SUPP["logreg"]["C_grid"]:
        clf = LogisticRegression(C=C, max_iter=3000).fit(sc.transform(Ztr[mtr]), YCLS3[f3["TR"]][mtr])
        vl = clf.decision_function(sc.transform(Zva.reshape(-1, Zva.shape[-1]))).reshape(len(f3["VA"]), N)
        a = auc_on(vl, f3["VA"])
        if best is None or a > best[1]: best = (C, a, clf, vl)
    return best, sc, f

require_shakedown("18.1 supplementary test predictions")
LR_CHOICE = {}
for f3 in FOLD3:
    path = pred_path(f3, mode, LR_NAME, 0)
    if os.path.exists(path): continue
    (C, a, clf, vl), sc, f = logreg_fold(f3)
    Zte = f(f3["TE"]); tl = clf.decision_function(sc.transform(Zte.reshape(-1, Zte.shape[-1]))).reshape(len(f3["TE"]), N)
    np.savez(path, val_auc=a, val_logit=vl, val_score=vl, te_logit=tl, te_score=tl)
    LR_CHOICE[f3["name"]] = dict(C=C, val_auc=round(a, 4))
if LR_CHOICE:
    json.dump(LR_CHOICE, open(f"{V3DIR}/supplementary_logreg_choice.json", "w"), indent=1)
    print("LogReg C chosen on validation AUC:", LR_CHOICE)
r_test(TF_NAME, TF_NAME)
print("supplementary test predictions written")

Transformer baseline trained (validation selection only): 0.0 min
{'val_AUC': 0.6692, 'val_IC': 0.0531, 'composite': 0.2224} | SCEG-R base: {'val_AUC': 0.6157, 'val_IC': 0.0631, 'composite': 0.1788}
supplementary test predictions written


In [66]:
# ===== 18.2 · full classification metrics, calibration, trading check (per-seed mean ± sd; fixed threshold) =====
def full_cls(logit, days):
    m = MC3[days]; y = YCLS3[days][m].astype(int); z = logit[m].astype(float)
    p = 1 / (1 + np.exp(-np.clip(z, -30, 30))); yh = (z > 0).astype(int)
    b = np.minimum((p * 10).astype(int), 9)
    ece = float(sum(abs(p[b == k].mean() - y[b == k].mean()) * (b == k).mean() for k in range(10) if (b == k).any()))
    tn, fp, fn, tp = confusion_matrix(y, yh, labels=[0, 1]).ravel()
    return dict(ACC=(y == yh).mean() * 100, BalAcc=balanced_accuracy_score(y, yh) * 100,
                Precision=precision_score(y, yh, zero_division=0) * 100, Recall=recall_score(y, yh, zero_division=0) * 100,
                F1=f1_score(y, yh, zero_division=0) * 100, AUC=roc_auc_score(y, p) * 100 if 0 < y.mean() < 1 else 50.0,
                PR_AUC=average_precision_score(y, p) * 100, MCC=matthews_corrcoef(y, yh) if yh.std() > 0 else 0.0,
                Brier=brier_score_loss(y, p), ECE10=ece, TN=tn, FP=fp, FN=fn, TP=tp)

def ls_portfolio(score, days, q=0.2):
    gross, turn, prev = [], [], None
    for r, t in enumerate(days):
        idx = np.where(MR3[t])[0]
        if len(idx) < 10 or np.std(score[r][idx]) == 0: gross.append(0.0); turn.append(0.0); continue
        k = max(1, int(round(q * len(idx)))); o = idx[np.argsort(score[r][idx])]; L_, S_ = set(o[-k:]), set(o[:k])
        gross.append(float(Y[t][list(L_)].mean() - Y[t][list(S_)].mean()))
        turn.append(1.0 if prev is None else (len(L_ - prev[0]) + len(S_ - prev[1])) / (2 * k)); prev = (L_, S_)
    gross, turn = np.array(gross), np.array(turn); net = gross - 4 * CONFIG["COST_BPS"] / 1e4 * turn
    sh = lambda x: float(x.mean() / (x.std() + 1e-12) * np.sqrt(252))
    return dict(LS_Sharpe_gross=sh(gross), LS_Sharpe_net=sh(net), LS_t=float(nw_t(gross)), LS_ann_ret_net=float(net.mean() * 252 * 100),
                turnover=float(turn.mean()))

SUPP_MODELS = [R_FINAL_NAME, BASE_ARM, TF_NAME, "SCEG-v3 (full)", "− graph (= GRU-v3)", "MLP-v3", "LightGBM-v3", LR_NAME]
rows_, PER_SEED = [], []
for v in dict.fromkeys(SUPP_MODELS):
    runs = load_runs(mode, v)
    if not runs: print(f"({v}: no test predictions)"); continue
    per = pd.DataFrame([{**full_cls(r["te_logit"], TE_ALL), "IC": ic_days(r["te_score"], TE_ALL).mean(),
                         **ls_portfolio(r["te_score"], TE_ALL)} for r in runs.values()], index=list(runs))
    PER_SEED.append(per.assign(model=v, seed=list(runs)))
    row = {"model": v, "seeds": len(runs)}
    for c in ("ACC", "BalAcc", "Precision", "Recall", "F1", "AUC", "PR_AUC"):
        row[c] = f"{per[c].mean():.2f}±{per[c].std(ddof=0):.2f}"
    for c in ("MCC", "Brier", "ECE10", "IC"):
        row[c] = f"{per[c].mean():.4f}±{per[c].std(ddof=0):.4f}"
    for c in ("LS_Sharpe_gross", "LS_Sharpe_net", "LS_t"):
        row[c] = f"{per[c].mean():.2f}±{per[c].std(ddof=0):.2f}"
    row["confusion (mean TN/FP/FN/TP)"] = "/".join(f"{per[c].mean():.0f}" for c in ("TN", "FP", "FN", "TP"))
    rows_.append(row)
_mj = full_cls(MAJ["maj"]["te_logit"], TE_ALL)
rows_.append({"model": "majority class", "seeds": 0, **{c: f"{_mj[c]:.2f}" for c in ("ACC", "BalAcc", "Precision", "Recall", "F1", "AUC", "PR_AUC")},
              "MCC": "0", "Brier": f"{_mj['Brier']:.4f}", "IC": "0",
              "confusion (mean TN/FP/FN/TP)": "/".join(str(int(_mj[c])) for c in ("TN", "FP", "FN", "TP"))})
SUPP_TAB = pd.DataFrame(rows_).set_index("model")
pd.set_option("display.width", 250)
print(f"=== [{CONFIG['DATASET']}] test, per-seed mean ± sd, fixed threshold 0.5; PR-AUC base rate = "
      f"{YCLS3[TE_ALL][MC3[TE_ALL]].mean() * 100:.1f} % ===")
print(SUPP_TAB.to_string())
SUPP_TAB.to_csv(f"{V3DIR}/supplementary_metrics_{CONFIG['DATASET']}_{V3['ENCODER']}_{V3['SCHEMA']}.csv")
PER_SEED = pd.concat(PER_SEED)
PER_SEED.to_csv(f"{V3DIR}/supplementary_per_seed_{CONFIG['DATASET']}_{V3['ENCODER']}_{V3['SCHEMA']}.csv")
print("Brier/ECE: SCEG models are trained with a class-weighted BCE (pos_weight), so their probabilities are not "
      "calibrated by construction; ECE is reported, never used for selection. The portfolio is a diagnostic of the "
      "ranking signal, not a trading result (no market impact, no borrow constraints, close-to-close execution).")

=== [acl18] test, per-seed mean ± sd, fixed threshold 0.5; PR-AUC base rate = 51.3 % ===
                                      seeds         ACC      BalAcc   Precision       Recall          F1         AUC      PR_AUC             MCC          Brier          ECE10              IC LS_Sharpe_gross LS_Sharpe_net       LS_t confusion (mean TN/FP/FN/TP)
model                                                                                                                                                                                                                                                               
SCEG-R (final, frozen)                    5  55.77±1.07  55.91±1.34  59.12±3.37  50.36±15.05  52.70±7.35  58.20±1.11  60.75±1.21   0.1254±0.0263  0.2475±0.0050  0.0444±0.0254   0.0242±0.0046       1.72±0.24    -0.02±0.34  0.79±0.10             1114/698/947/961
SCEG-R base | price v3                    5  55.77±1.07  55.91±1.34  59.12±3.37  50.36±15.05  52.70±7.35  58.20±1.11  60.75±1.21

In [67]:
# ===== 18.3 · seeds: individual runs, aggregate, and the best seed (supplementary only) =====
fin = PER_SEED[PER_SEED.model == R_FINAL_NAME]
if len(fin):
    cols = ["ACC", "BalAcc", "MCC", "AUC", "IC", "LS_Sharpe_net"]
    agg = pd.DataFrame({"mean": fin[cols].mean(), "sd": fin[cols].std(ddof=0), "median": fin[cols].median(),
                        "min": fin[cols].min(), "max": fin[cols].max()}).T
    print(f"SCEG-R final, individual seeds (test):\n{fin.set_index('seed')[cols].round(4).to_string()}\n")
    print(f"aggregate (PRIMARY = mean ± sd):\n{agg.round(4).to_string()}")
    b = fin.loc[fin["MCC"].idxmax()]
    print(f"\nbest seed by test MCC (SUPPLEMENTARY ONLY, selected on test, not a result): seed {int(b['seed'])} | "
          f"ACC {b['ACC']:.2f} | MCC {b['MCC']:.4f} | IC {b['IC']:.4f}")
    spread = fin["ACC"].max() - fin["ACC"].min()
    if spread > 3 * max(fin["ACC"].std(ddof=0), 1e-9) or spread > 3.0:
        print(f"NOTE: seed spread {spread:.2f} ACC points: investigate (up-rate per seed below) before interpreting any seed.")
        for s, r in load_runs(mode, R_FINAL_NAME).items():
            print(f"  seed {s}: predicted-up rate {(r['te_logit'][MC3[TE_ALL]] > 0).mean() * 100:.1f} %")

# ===== 18.4 · supplementary paired family (Holm within this family only) =====
SUPP_FAM = family_table([(R_FINAL_NAME, LR_NAME), (R_FINAL_NAME, TF_NAME), (R_FINAL_NAME, "LightGBM-v3"),
                         (TF_NAME, BASE_ARM)], mode, "supplementary baselines")
MANIFEST.update(supplementary_spec_sha=SUPP_SHA)
json.dump(MANIFEST, open(f"{V3DIR}/run_manifest_{CONFIG['DATASET']}.json", "w"), indent=1, default=str)

SCEG-R final, individual seeds (test):
          ACC   BalAcc     MCC      AUC      IC  LS_Sharpe_net
seed                                                          
0     54.0054  53.4534  0.0765  57.1906  0.0166        -0.0964
1     56.4785  56.8445  0.1426  58.1772  0.0282        -0.1410
2     56.0484  56.0267  0.1205  58.3670  0.0285        -0.2567
3     55.2151  55.8891  0.1378  57.0802  0.0211        -0.2594
4     57.0968  57.3472  0.1496  60.1612  0.0265         0.6539

aggregate (PRIMARY = mean ± sd):
            ACC   BalAcc     MCC      AUC      IC  LS_Sharpe_net
mean    55.7688  55.9122  0.1254  58.1952  0.0242        -0.0199
sd       1.0735   1.3407  0.0263   1.1088  0.0046         0.3429
median  56.0484  56.0267  0.1378  58.1772  0.0265        -0.1410
min     54.0054  53.4534  0.0765  57.0802  0.0166        -0.2594
max     57.0968  57.3472  0.1496  60.1612  0.0285         0.6539

best seed by test MCC (SUPPLEMENTARY ONLY, selected on test, not a result): seed 4 | ACC 57.10 

---
# 19 · Evidence audit and model-free propagation study (v9.2)

**Status of this section.** Added in v9.2 after the executed ACL18 and FNSPID runs of §14–§15 had been inspected.
It changes nothing that already exists: SCEG-v3, the §15 families, SCEG-R (§16, spec `0bfe3098df8c`), §17 and §18 are
untouched and every cached prediction is reused as is. It adds four things, each with an explicit evidential status:

| Part | What it does | New training? | Status on ACL18 / FNSPID | Status on India |
|---|---|---|---|---|
| 19.1 | fold × seed validation statistics for the graph arms; share of nested runs that kept stage 2 | no (reads caches) | descriptive (post hoc) | descriptive |
| 19.2 | which *validation* quantity tracks test skill? AUC split into a market-timing and a cross-sectional part | no | diagnostic (post hoc) | diagnostic |
| 19.3 | precision of the key test contrasts: SE, minimum detectable effect, equivalence (TOST) at a fixed bound | no | descriptive (post hoc) | descriptive |
| 19.4 | P4 replication: "GRU-v3 \| all four reverted" on FNSPID | 15 runs, FNSPID only | **confirmatory on FNSPID** (hypothesis formed on ACL18 only; this arm has never been run on FNSPID) | not run |
| 19.5 | model-free propagation event study: news shock at A → abnormal return of graph neighbours B at t+1…t+5, against node-permutation, matched-random and placebo-date nulls | no | validation: development; **test: exploratory** (related quantities were seen in §11.1 and §15) | **test: confirmatory** (frozen here, before any India test prediction; gated) |

Every setting of 19.3–19.5 is fixed in the specification 19.0 and hashed *before* the first quantity it governs is
computed. The spec cannot be changed silently: a changed spec stops the notebook.

In [68]:
# ===== 19.0 · frozen specification of §19 =====
from scipy.stats import spearmanr, wilcoxon, ttest_1samp, norm as _norm19
SPEC19 = {
    "version": "v9.2-s19",
    "val_pairs": [["sector graph only", "SCEG-v3 (full)"], ["sector graph only", "random graph (signed, matched)"],
                  ["sector graph only", "− graph (= GRU-v3)"], ["SCEG-v3 (full)", "− graph (= GRU-v3)"],
                  ["random graph (signed, matched)", "− graph (= GRU-v3)"], ["SCEG-v3 (full)", "random graph (signed, matched)"],
                  ["SCEG-v3 (full)", "regime placebo (shuffled)"], ["SCEG-v3 (full)", "− regime conditioning"],
                  ["nested: + sector", "nested: no graph (heads retrained)"],
                  ["nested: + DYNOTEARS", "nested: no graph (heads retrained)"],
                  ["nested: + random (matched to DYNOTEARS)", "nested: no graph (heads retrained)"],
                  ["nested: + sector", "nested: + DYNOTEARS"], ["nested: + sector", "nested: + random (matched to DYNOTEARS)"],
                  ["nested: + DYNOTEARS", "nested: + random (matched to DYNOTEARS)"]],
    "val_tests": "paired over (fold, seed) units: mean difference, wins, one-sample t, Wilcoxon signed-rank; no Holm (descriptive)",
    "test_pairs_precision": [["sector graph only", "SCEG-v3 (full)"], ["SCEG-v3 (full)", "random graph (signed, matched)"],
                             ["SCEG-v3 (full)", "− graph (= GRU-v3)"], ["SCEG-v3 (full)", "regime placebo (shuffled)"],
                             ["nested: + DYNOTEARS", "nested: + random (matched to DYNOTEARS)"],
                             ["nested: + sector", "nested: + random (matched to DYNOTEARS)"],
                             ["nested: + DYNOTEARS", "nested: no graph (heads retrained)"]],
    "equivalence_bound_IC": 0.005,
    "mde": "(z_0.975 + z_0.80) * SE(ΔIC), SE from the Newey–West t of §15 (5 lags)",
    "p4_replication": {
        "dataset": "fnspid", "arm": "GRU-v3 | all four reverted",
        "config": "v2 loss (Huber + IC), v2 price features, no market head, early stopping on validation IC",
        "hypothesis": "test IC_all(all four reverted) − IC_all(GRU-v3) > 0",
        "test": "seed-paired daily ΔIC, Newey–West t (5 lags), two-sided α = 0.05; a single test (no family)",
        "secondary": ["DirAcc_stock", "ACC", "MCC"], "origin": "ACL18 §15.2: ΔIC −0.0273 (GRU-v3 − reverted), Holm p 0.007",
    },
    "event_study": {
        "abnormal_return": "AR_i,t = r_i,t − β_i,t m_t; m_t = equal-weighted mean return of valid firms; β_i,t = OLS slope "
                           "over days [t−250, t−30] (≥ 120 obs, else β = 1); strictly past data",
        "source_shock": "news_A,t × (AR_A,t−1 + AR_A,t): [−1, 0] event window, known at the close of t",
        "signal": "S_B,t = Σ_A W[B, A] × shock_A,t (W row = target), W from the fold's training-period graph",
        "target": "CAR_B(t+1 … t+h) of abnormal returns; days whose window leaves the split are dropped",
        "horizons": [1, 2, 3, 5], "primary_horizon": 1,
        "statistic": "primary: pooled propagation slope β = Σ_t Σ_B S_B,t c̃_B,t / Σ_t Σ_B S_B,t², c̃ = CAR demeaned across valid "
                     "firms on day t (day fixed effect), all valid firms (S = 0 included, so sparse graphs are not dropped); "
                     "SE from Newey–West (5 lags) on the daily contributions. Secondary: mean daily Pearson IC over firms with "
                     "S ≠ 0 (≥ 5 firms, else the day is skipped). Valid (member) firms only",
        "families": {"DYNOTEARS lag1": "primary", "DYNOTEARS lag1+lag2": "secondary", "sector": "secondary",
                     "sector-masked DYNOTEARS lag1": "secondary (within-industry lead–lag, Hou 2007)",
                     "correlation top-k": "secondary", "DYNOTEARS contemporaneous": "secondary"},
        "controls": {"no-news shock": "(1 − news_A,t) × (AR_A,t−1 + AR_A,t), same graph",
                     "node permutation": "W relabelled by a random node permutation (degree and weights kept), 200 draws",
                     "matched random": "random_signed(W) (same number and values of edges), 50 draws",
                     "placebo dates": "each firm's news indicator circularly shifted by a random 60 … T−60 days, 100 draws",
                     "own-shock": "S_B = B's own shock (no relation): reversal/momentum reference"},
        "inference": "NW t (5 lags) of β vs 0; one-sided empirical p of β vs each null ((1 + #null ≥ obs)/(1 + draws)); "
                     "news − no-news Δβ with NW t on the paired daily contributions",
        "confirmatory_rule_india": "propagation supported on India test iff DYNOTEARS lag1, h = 1: β > 0 with NW p < 0.05 "
                                   "AND node-permutation p < 0.05 AND placebo-date p < 0.05 (all three; no correction needed)",
        "regime_split": "days split by argmax of the filtered regime posterior at t (descriptive)",
        "status": {"acl18": "validation descriptive; test exploratory", "fnspid": "validation descriptive; test exploratory",
                   "india": "validation descriptive; test confirmatory (gated by require_shakedown)"},
        "co_mention": "not included: the items are single-ticker (StockNet cashtag files, FNSPID per-symbol rows), so co-mention "
                      "edges cannot be built without a new extraction pass",
    },
    "rejected": {"hybrid sector + DYNOTEARS layer": "validation cannot tell relation types apart (19.1), so a hybrid would be "
                                                    "chosen on noise; not added",
                 "new graph arms with IC selection": "SCEG-R stage 2 (§16) already tests a graph on an IC-aware base; more arms "
                                                     "= more forking paths"},
}
S19_SHA = sha(json.dumps(SPEC19, sort_keys=True))[:12]
S19_PATH = f"{V3DIR}/section19_spec_frozen.json"
if os.path.exists(S19_PATH):
    assert json.load(open(S19_PATH))["sha"] == S19_SHA, "§19 specification changed after freezing: refuse to run"
else:
    json.dump({"sha": S19_SHA, "spec": SPEC19, "frozen_at": time.strftime("%Y-%m-%d %H:%M:%S"),
               "dataset": CONFIG["DATASET"]}, open(S19_PATH, "w"), indent=1)
print("§19 spec sha", S19_SHA, "| frozen file", S19_PATH)
MODE19 = V3["PRIMARY"]

def _val_unit(vname, f3, seed):
    p = pred_path(f3, MODE19, vname, seed)
    return np.load(p) if os.path.exists(p) else None

§19 spec sha 97a329f6221b | frozen file /content/drive/MyDrive/MinorProject/acl18_cache/v3/section19_spec_frozen.json


In [69]:
# ===== 19.1 · validation statistics over (fold, seed) units — no test data =====
def val_table(vname):
    rows_ = []
    for f3 in FOLD3:
        for seed in V3["SEEDS"]:
            z = _val_unit(vname, f3, seed)
            if z is None: continue
            rows_.append(dict(model=vname, fold=f3["name"], seed=seed, val_AUC=float(z["val_auc"]),
                              val_IC=float(ic_days(z["val_score"], f3["VA"]).mean())))
    return pd.DataFrame(rows_)

V19_MODELS = [m for m in dict.fromkeys(list(VARIANTS3) + list(NESTED) + ["nested: GRU-v3 base (no stage 2)"] + list(ATTRIB))]
VAL19 = pd.concat([val_table(m) for m in V19_MODELS], ignore_index=True)
VAL19.to_csv(f"{V3DIR}/s19_validation_units_{MODE19}.csv", index=False)
print(f"validation units: {len(VAL19)} (model × fold × seed)\n")
print("--- per-fold mean validation AUC over all models (non-stationarity check) ---")
_pf = VAL19.groupby("fold").val_AUC.agg(["mean", "std", "count"])
_pf["share < 0.5"] = VAL19.assign(b=VAL19.val_AUC < 0.5).groupby("fold").b.mean()
print(_pf.round(4).to_string())
print("\n--- per-model validation AUC / IC (mean, sd over fold × seed; min, max) ---")
print(VAL19.groupby("model")[["val_AUC", "val_IC"]].agg(["mean", "std", "min", "max"]).round(4)
      .sort_values(("val_AUC", "mean"), ascending=False).to_string())

rows_ = []
for a, b in SPEC19["val_pairs"]:
    A, B = VAL19[VAL19.model == a], VAL19[VAL19.model == b]
    m = A.merge(B, on=["fold", "seed"], suffixes=("_a", "_b"))
    if len(m) < 3: continue
    for col in ("val_AUC", "val_IC"):
        d = (m[f"{col}_a"] - m[f"{col}_b"]).values
        w = wilcoxon(d).pvalue if np.any(d != 0) else 1.0
        rows_.append({"A": a, "B": b, "metric": col, "units": len(d), "mean Δ": d.mean(), "sd Δ": d.std(ddof=1),
                      "wins A": f"{int((d > 0).sum())}/{len(d)}", "t p": ttest_1samp(d, 0).pvalue, "Wilcoxon p": w,
                      "folds with Δ>0": f"{int((m.assign(d=d).groupby('fold').d.mean() > 0).sum())}/{m.fold.nunique()}"})
VALPAIR19 = pd.DataFrame(rows_)
print("\n--- paired validation comparisons over (fold, seed) units (descriptive; no multiplicity correction) ---")
print(VALPAIR19.round(4).to_string(index=False))
VALPAIR19.to_csv(f"{V3DIR}/s19_validation_pairs_{MODE19}.csv", index=False)

# nested experiment: how often did stage 2 beat the frozen base on validation? (fallback keeps the base -> upward bias)
rows_ = []
for vname in NESTED:
    for f3 in FOLD3:
        for seed in V3["SEEDS"]:
            z, zb = _val_unit(vname, f3, seed), _val_unit("nested: GRU-v3 base (no stage 2)", f3, seed)
            if z is None or zb is None: continue
            rows_.append(dict(model=vname, kept_base=abs(float(z["val_auc"]) - float(zb["val_auc"])) < 1e-6,
                              gain=float(z["val_auc"]) - float(zb["val_auc"])))
if rows_:
    NK = pd.DataFrame(rows_).groupby("model").agg(units=("gain", "size"), share_kept_base=("kept_base", "mean"),
                                                  mean_val_gain=("gain", "mean"))
    print("\n--- nested runs: share where stage 2 never beat the frozen base on validation (gain ≥ 0 by construction) ---")
    print(NK.round(4).to_string())

validation units: 130 (model × fold × seed)

--- per-fold mean validation AUC over all models (non-stationarity check) ---
        mean     std  count  share < 0.5
fold                                    
F1    0.6159  0.0226    130          0.0

--- per-model validation AUC / IC (mean, sd over fold × seed; min, max) ---
                                            val_AUC                          val_IC                        
                                               mean     std     min     max    mean     std     min     max
model                                                                                                      
− lead–lag (lag1, lag2)                      0.6450  0.0192  0.6286  0.6747  0.0189  0.0277 -0.0212  0.0411
nested: + random (matched to DYNOTEARS)      0.6327  0.0065  0.6230  0.6384  0.0320  0.0189 -0.0007  0.0462
nested: + sector                             0.6322  0.0045  0.6271  0.6384  0.0274  0.0163 -0.0007  0.0421
GRU-v3 | v2 loss (Huber + IC)

In [70]:
# ===== 19.2 · which validation quantity tracks test skill? (diagnostic; test used only for this rank check) =====
def auc_split(logit, days):
    """Pooled AUC of the day-mean logit (market timing) and of the within-day demeaned logit (cross-section)."""
    m = MC3[days]; y = YCLS3[days][m]
    if not (0 < y.mean() < 1): return 0.5, 0.5
    L = logit.astype(float); c = np.array([L[r][MR3[t]].mean() if MR3[t].sum() else 0.0 for r, t in enumerate(days)])
    Lc = np.broadcast_to(c[:, None], L.shape)
    return roc_auc_score(y, Lc[m]), roc_auc_score(y, (L - Lc)[m])

rows_ = []
for vname in V19_MODELS:
    vals, tests = [], []
    for f3 in FOLD3:
        for seed in V3["SEEDS"]:
            z = _val_unit(vname, f3, seed)
            if z is None or "te_logit" not in z.files: continue
            am, ax = auc_split(z["val_logit"], f3["VA"])
            vals.append(dict(val_AUC=float(z["val_auc"]), val_AUC_market=am, val_AUC_xs=ax,
                             val_IC=float(ic_days(z["val_score"], f3["VA"]).mean())))
            tests.append(dict(test_AUC=auc_on(z["te_logit"], f3["TE"]), test_IC=float(ic_days(z["te_score"], f3["TE"]).mean()),
                              **{k: v for k, v in split_metrics(z["te_logit"], f3["TE"]).items()}))
    if vals: rows_.append({"model": vname, **pd.DataFrame(vals).mean().to_dict(), **pd.DataFrame(tests).mean().to_dict()})
VT19 = pd.DataFrame(rows_).set_index("model")
print("--- per model: validation quantities (mean over fold × seed) and test quantities (mean over fold × seed) ---")
print(VT19.round(4).sort_values("val_AUC", ascending=False).to_string())
rows_ = []
for v in ("val_AUC", "val_AUC_market", "val_AUC_xs", "val_IC"):
    for t_ in ("test_AUC", "test_IC", "ACC_market", "DirAcc_stock"):
        r, p = spearmanr(VT19[v], VT19[t_]); rows_.append(dict(validation=v, test=t_, spearman=r, p=p, models=len(VT19)))
RANK19 = pd.DataFrame(rows_)
print("\n--- rank agreement across models (Spearman; models are not independent: descriptive) ---")
print(RANK19.pivot(index="validation", columns="test", values="spearman").round(3).to_string())
VT19.to_csv(f"{V3DIR}/s19_val_vs_test_{MODE19}.csv"); RANK19.to_csv(f"{V3DIR}/s19_rank_agreement_{MODE19}.csv", index=False)

--- per model: validation quantities (mean over fold × seed) and test quantities (mean over fold × seed) ---
                                             val_AUC  val_AUC_market  val_AUC_xs  val_IC  test_AUC  test_IC  ACC_market  DirAcc_stock
model                                                                                                                                
− lead–lag (lag1, lag2)                       0.6450          0.6648      0.5001  0.0189    0.5421   0.0202     55.1828       50.3556
nested: + random (matched to DYNOTEARS)       0.6327          0.6564      0.5026  0.0320    0.5738   0.0334     53.4194       51.0022
nested: + sector                              0.6322          0.6556      0.5007  0.0274    0.5735   0.0328     52.1398       50.7830
GRU-v3 | v2 loss (Huber + IC)                 0.6292          0.6514      0.4998  0.0208    0.5791   0.0440     53.7043       51.4583
nested: no graph (heads retrained)            0.6289          0.6489      0.5019  0.032

In [71]:
# ===== 19.3 · precision of the key test contrasts: SE, MDE, equivalence at ±δ (descriptive) =====
d19 = SPEC19["equivalence_bound_IC"]; z80 = _norm19.ppf(0.975) + _norm19.ppf(0.80)
rows_ = []
for a, b in SPEC19["test_pairs_precision"]:
    ra, rb = load_runs(MODE19, a), load_runs(MODE19, b)
    if not ra or not rb: continue
    pt = paired_test(ra, rb, TE_ALL)
    se = abs(pt["dIC"] / pt["t_IC"]) if pt["t_IC"] != 0 else np.nan
    lo90, hi90 = pt["dIC"] - 1.645 * se, pt["dIC"] + 1.645 * se
    rows_.append({"A": a, "B": b, "ΔIC": pt["dIC"], "SE": se, "95% CI": f"[{pt['dIC'] - 1.96 * se:+.4f}, {pt['dIC'] + 1.96 * se:+.4f}]",
                  "MDE (80% power)": z80 * se, f"equivalent within ±{d19}": bool(lo90 > -d19 and hi90 < d19),
                  "ΔACC": pt["dACC"], "ΔMCC": pt["dMCC"], "test days": len(TE_ALL)})
PREC19 = pd.DataFrame(rows_)
print(f"--- test contrasts: what the data can and cannot resolve (TOST at ±{d19} IC, 90% CI) ---")
print(PREC19.round(4).to_string(index=False))
PREC19.to_csv(f"{V3DIR}/s19_precision_{MODE19}.csv", index=False)
print("\nReading: a contrast that is neither significant nor equivalent is *unresolved* (not 'no effect').")
print("\n--- market timing vs stock-specific skill (test; per-seed mean ± sd, from §15) ---")
_ms = [m for m in ["SCEG-v3 (full)", "sector graph only", "random graph (signed, matched)", "− graph (= GRU-v3)", "MLP-v3",
                   "LightGBM-v3", "nested: + DYNOTEARS", "nested: + sector", "GRU-v3 | all four reverted"] if load_runs(MODE19, m)]
print(pd.DataFrame([summary_row(m, load_runs(MODE19, m), TE_ALL)[0] for m in _ms]).set_index("model")
      [["ACC", "MCC", "AUC", "ACC_market", "DirAcc_stock", "IC_all", "RankIC_all"]].to_string())

--- test contrasts: what the data can and cannot resolve (TOST at ±0.005 IC, 90% CI) ---
                  A                                       B     ΔIC     SE             95% CI  MDE (80% power)  equivalent within ±0.005    ΔACC    ΔMCC  test days
  sector graph only                          SCEG-v3 (full) -0.0007 0.0086 [-0.0175, +0.0160]           0.0240                     False  0.7419  0.0094         64
     SCEG-v3 (full)          random graph (signed, matched) -0.0099 0.0082 [-0.0260, +0.0062]           0.0230                     False -1.4247 -0.0270         64
     SCEG-v3 (full)                      − graph (= GRU-v3) -0.0022 0.0116 [-0.0249, +0.0205]           0.0325                     False  0.1452  0.0102         64
     SCEG-v3 (full)               regime placebo (shuffled)  0.0001 0.0017 [-0.0032, +0.0034]           0.0047                      True  0.7366  0.0176         64
nested: + DYNOTEARS nested: + random (matched to DYNOTEARS)  0.0055 0.0022 [+0.0012, +0.009

In [72]:
# ===== 19.4 · P4 replication on FNSPID (confirmatory; hypothesis fixed on ACL18, frozen in 19.0) =====
P4 = SPEC19["p4_replication"]; P4_ARM = P4["arm"]
if CONFIG["DATASET"] == P4["dataset"]:
    t0 = time.time()
    for f3 in FOLD3:
        for seed in V3["SEEDS"]:
            path = pred_path(f3, MODE19, P4_ARM, seed)
            if os.path.exists(path): continue
            out, _ = train3(f3, MODE19, ATTRIB[P4_ARM], SELECTED["gru"], seed, with_test=True)
            np.savez(path, **{k: np.asarray(v) for k, v in out.items()})
            print(f"[{MODE19}] {f3['name']} {P4_ARM} seed {seed} | val IC-selected, val AUC {out['val_auc']:.4f}", flush=True)
    print(f"P4 replication runs: {(time.time() - t0) / 60:.1f} min")
if load_runs(MODE19, P4_ARM) and load_runs(MODE19, "− graph (= GRU-v3)"):
    pt = paired_test(load_runs(MODE19, P4_ARM), load_runs(MODE19, "− graph (= GRU-v3)"), TE_ALL)
    P4_RES = {"dataset": CONFIG["DATASET"], "ΔIC": pt["dIC"], "NW t": pt["t_IC"], "p (two-sided)": pt["p_IC"],
              "ΔACC": pt["dACC"], "ΔMCC": pt["dMCC"], "seeds": pt["seeds"],
              "status": "confirmatory" if CONFIG["DATASET"] == P4["dataset"] else "origin of the hypothesis (not a test)",
              "verdict": ("replicated" if pt["dIC"] > 0 and pt["p_IC"] < 0.05 else "not replicated")
                         if CONFIG["DATASET"] == P4["dataset"] else "—"}
    print("--- P4: GRU-v3 | all four reverted  −  GRU-v3 ---"); print(pd.Series(P4_RES).to_string())
    json.dump({k: (float(v) if isinstance(v, (np.floating, float)) else v) for k, v in P4_RES.items()},
              open(f"{V3DIR}/s19_p4_{CONFIG['DATASET']}.json", "w"), indent=1)
    print(pd.DataFrame([summary_row(m, load_runs(MODE19, m), TE_ALL)[0] for m in (P4_ARM, "− graph (= GRU-v3)")]).set_index("model")
          [["ACC", "MCC", "AUC", "ACC_market", "DirAcc_stock", "IC_all", "RankIC_all"]].to_string())
else:
    print(f"P4 replication runs on {P4['dataset'].upper()} only (ACL18 is where the hypothesis came from; India: not registered)")

--- P4: GRU-v3 | all four reverted  −  GRU-v3 ---
dataset                                          acl18
ΔIC                                            0.02734
NW t                                          3.251031
p (two-sided)                                  0.00115
ΔACC                                         -0.145161
ΔMCC                                         -0.001463
seeds                                                5
status           origin of the hypothesis (not a test)
verdict                                              —
                                   ACC            MCC         AUC  ACC_market DirAcc_stock         IC_all     RankIC_all
model                                                                                                                   
GRU-v3 | all four reverted  53.83±0.98  0.0733±0.0201  55.68±1.18  52.08±1.31   52.36±0.46  0.0649±0.0074  0.0657±0.0097
− graph (= GRU-v3)          53.98±2.22  0.0747±0.0496  57.94±2.22  54.06±3.64   50.93±0.55  

In [73]:
# ===== 19.5 · model-free propagation event study (no model, no training) =====
ES = SPEC19["event_study"]
VALID19 = ROW & EVAL_MASK & np.isfinite(RET)
_R = np.where(VALID19, RET, np.nan).astype(float)
MKT19 = np.nanmean(np.where(VALID19, _R, np.nan), 1); MKT19 = np.nan_to_num(MKT19)
def rolling_beta(R, m, lo=250, hi=30, minobs=120):
    """β_i,t from days [t−lo, t−hi]: strictly past data only."""
    dfR, sm = pd.DataFrame(R).shift(hi), pd.Series(m).shift(hi)
    w = lo - hi + 1
    mR = dfR.rolling(w, min_periods=minobs).mean(); mm = sm.rolling(w, min_periods=minobs).mean()
    mRm = dfR.mul(sm, axis=0).rolling(w, min_periods=minobs).mean(); mm2 = (sm ** 2).rolling(w, min_periods=minobs).mean()
    b = (mRm.sub(mR.mul(mm, axis=0))).div(mm2 - mm ** 2, axis=0)
    return b.fillna(1.0).values
BETA19 = rolling_beta(_R, MKT19)
AR19 = np.where(VALID19, _R - BETA19 * MKT19[:, None], np.nan)
_ARz = np.nan_to_num(AR19)
NEWS19 = (FOLD3[0]["X"][MODE19][..., FOLD3[0]["sl"]["news"][0]] > 0) & VALID19
SHOCK_BASE = np.zeros_like(_ARz); SHOCK_BASE[1:] = _ARz[1:] + _ARz[:-1]            # AR(t−1) + AR(t)
print(f"abnormal returns: β median {np.nanmedian(BETA19[VALID19]):.2f} | news firm-days {NEWS19.sum():,} "
      f"({NEWS19[VALID19].mean():.1%} of valid firm-days)")

def car(h):
    C = np.full((T, N), np.nan)
    for t in range(T - h):
        C[t] = np.nansum(AR19[t + 1:t + h + 1], 0)
        C[t][~VALID19[t + 1:t + h + 1].all(0)] = np.nan
    return C
CAR19 = {h: car(h) for h in ES["horizons"]}

def ev_days(days, h):
    d = np.asarray(days); return d[d + h <= d.max()]          # the target window stays inside the split

def ic_series(S, days, h):
    """Daily cross-sectional Pearson IC over firms with a non-zero signal (vectorised; NaN if < 5 firms)."""
    d = ev_days(days, h); s, c = S[d], CAR19[h][d]
    ok = VALID19[d] & np.isfinite(c) & (s != 0); n = ok.sum(1)
    s0, c0 = np.where(ok, s, 0.0), np.where(ok, c, 0.0)
    ms, mc = s0.sum(1) / np.maximum(n, 1), c0.sum(1) / np.maximum(n, 1)
    ds, dc = np.where(ok, s - ms[:, None], 0.0), np.where(ok, c - mc[:, None], 0.0)
    den = np.sqrt((ds ** 2).sum(1) * (dc ** 2).sum(1))
    with np.errstate(invalid="ignore", divide="ignore"):
        r = (ds * dc).sum(1) / den
    r[(n < 5) | ~(den > 1e-12)] = np.nan
    return r

def slope_parts(S, days, h):
    """Daily numerator/denominator of the pooled day-demeaned slope (all valid firms; zero signals included)."""
    d = ev_days(days, h); s, c = S[d], CAR19[h][d]
    ok = VALID19[d] & np.isfinite(c); n = ok.sum(1)
    c0 = np.where(ok, c, 0.0); ct = np.where(ok, c - (c0.sum(1) / np.maximum(n, 1))[:, None], 0.0)
    s0 = np.where(ok, s, 0.0)
    return (s0 * ct).sum(1), (s0 ** 2).sum(1)

def slope_stat(parts):
    """parts: list of (num, den) per fold -> (β, NW t, days with signal, daily contributions u)."""
    num = np.concatenate([p[0] for p in parts]); den = np.concatenate([p[1] for p in parts])
    if den.sum() <= 0: return np.nan, np.nan, 0, (num, den)
    b = num.sum() / den.sum(); u = (num - b * den) / den.mean()
    return b, b / nw_se_mean(u), int((den > 0).sum()), (num, den)

def nw_se_mean(u, L=5):
    """Newey–West standard error of the mean of u."""
    u = u - u.mean(); n = len(u); g = (u @ u) / n
    for l in range(1, L + 1): g += 2 * (1 - l / (L + 1)) * (u[l:] @ u[:-l]) / n
    return np.sqrt(max(g, 1e-30) / n)

def _rownorm(M):
    s = np.abs(M).sum(1, keepdims=True); return np.divide(M, s, out=np.zeros_like(M), where=s > 0)
def families(f3):
    G = f3["G"]; smask = (G["sector"] > 0).astype(np.float32)
    return {"DYNOTEARS lag1": G["lag1"], "DYNOTEARS lag1+lag2": _rownorm(G["lag1"] + G["lag2"]), "sector": G["sector"],
            "sector-masked DYNOTEARS lag1": _rownorm(G["lag1"] * smask), "correlation top-k": G["corr"],
            "DYNOTEARS contemporaneous": G["contemp"]}

_PLAC = {}
def placebo_news(b):
    """Each firm's news indicator circularly shifted by its own random offset in [60, T − 60) (same seed b everywhere)."""
    if b not in _PLAC:
        sh = np.random.default_rng(3000 + b).integers(60, T - 60, N)
        _PLAC[b] = np.stack([np.roll(NEWS19[:, i], sh[i]) for i in range(N)], 1) & VALID19
    return _PLAC[b]

def signal(W, news):
    return (np.where(news, SHOCK_BASE, 0.0)) @ W.T

def nwt(x):
    x = x[np.isfinite(x)]
    return (nw_t(x) if len(x) > 10 else np.nan), len(x)

def event_study(split, B_perm=200, B_rand=50, B_plac=100):
    """split in {'VA', 'TE'}; one row per family × horizon, folds pooled (each fold uses its own training graph)."""
    rows_, regrows = [], []
    for fam in families(FOLD3[0]):
        for h in ES["horizons"]:
            P_news, P_non, P_own, ic_o, cover = [], [], [], [], []
            for f3 in FOLD3:
                W = families(f3)[fam]; days = f3[split]; S = signal(W, NEWS19)
                P_news.append(slope_parts(S, days, h)); P_non.append(slope_parts(signal(W, ~NEWS19 & VALID19), days, h))
                P_own.append(slope_parts(np.where(NEWS19, SHOCK_BASE, 0.0), days, h))
                ic_o.append(ic_series(S, days, h)); cover.append(float((S[ev_days(days, h)] != 0).mean()))
            b, t_b, nd, (num, den) = slope_stat(P_news)
            b0, _, _, (num0, den0) = slope_stat(P_non)
            dd = (num - b * den) / den.mean() - (num0 - b0 * den0) / den0.mean() if np.isfinite(b) and np.isfinite(b0) else None
            t_d = (b - b0) / nw_se_mean(dd) if dd is not None else np.nan
            ic = np.concatenate(ic_o)
            row = {"split": split, "family": fam, "h": h, "status": ES["families"][fam], "days with signal": nd,
                   "coverage (S≠0)": np.mean(cover), "β news": b, "NW t": t_b,
                   "p (NW, 2-sided)": 2 * (1 - _norm19.cdf(abs(t_b))) if np.isfinite(t_b) else np.nan,
                   "β no-news": b0, "Δβ news−no-news": b - b0, "t Δβ": t_d, "β own shock": slope_stat(P_own)[0],
                   "IC(S≠0) news": np.nanmean(ic) if np.isfinite(ic).any() else np.nan, "IC days": int(np.isfinite(ic).sum())}
            if h == ES["primary_horizon"]:        # nulls only at the primary horizon (cost)
                def mk_perm(f3, k):
                    p = np.random.default_rng(1000 + k).permutation(N); return signal(families(f3)[fam][np.ix_(p, p)], NEWS19)
                def mk_rand(f3, k):
                    return signal(random_signed(families(f3)[fam], np.random.default_rng(2000 + k)), NEWS19)
                def mk_plac(f3, k):
                    return signal(families(f3)[fam], placebo_news(k))
                for name, fn, B in (("node-perm", mk_perm, B_perm), ("matched-random", mk_rand, B_rand), ("placebo-date", mk_plac, B_plac)):
                    nv = np.array([slope_stat([slope_parts(fn(f3, k), f3[split], h) for f3 in FOLD3])[0] for k in range(B)])
                    row[f"null {name} mean"] = np.nanmean(nv) if np.isfinite(nv).any() else np.nan
                    row[f"p {name}"] = ((1 + np.sum(nv[np.isfinite(nv)] >= b)) / (1 + np.isfinite(nv).sum())) if np.isfinite(b) else np.nan
                for f3 in FOLD3:                  # regime split (descriptive)
                    d = ev_days(f3[split], h); reg = f3["PI"][d].argmax(1); nu, de = slope_parts(signal(families(f3)[fam], NEWS19), f3[split], h)
                    for k in np.unique(reg):
                        sel = reg == k
                        regrows.append({"split": split, "family": fam, "fold": f3["name"], "regime": f3["rinfo"]["names"][k],
                                        "days": int(sel.sum()), "β news": nu[sel].sum() / de[sel].sum() if de[sel].sum() > 0 else np.nan})
            rows_.append(row)
    return pd.DataFrame(rows_), pd.DataFrame(regrows)

t0 = time.time()
ES_VAL, ES_VAL_REG = event_study("VA")
print(f"--- validation (development; descriptive) | {(time.time() - t0) / 60:.1f} min ---")
_cols = ["family", "h", "status", "days with signal", "coverage (S≠0)", "β news", "NW t", "β no-news", "Δβ news−no-news", "t Δβ",
         "β own shock", "IC(S≠0) news", "IC days"]
print(ES_VAL[_cols].round(4).to_string(index=False))
_pc = [c for c in ES_VAL.columns if c.startswith("p ") or c.startswith("null ")]
print(ES_VAL[ES_VAL.h == ES["primary_horizon"]][["family", "β news"] + _pc].round(4).to_string(index=False))

require_shakedown("19.5 event study on test days")
ES_TE, ES_TE_REG = event_study("TE")
_st = ES["status"][CONFIG["DATASET"]]
print(f"\n--- test | status: {_st} ---")
print(ES_TE[_cols].round(4).to_string(index=False))
print(ES_TE[ES_TE.h == ES["primary_horizon"]][["family", "β news"] + _pc].round(4).to_string(index=False))
_prim = ES_TE[(ES_TE.family == "DYNOTEARS lag1") & (ES_TE.h == ES["primary_horizon"])].iloc[0]
_ok = bool(_prim["β news"] > 0 and _prim["p (NW, 2-sided)"] < 0.05 and _prim["p node-perm"] < 0.05 and _prim["p placebo-date"] < 0.05)
print(f"\nPRIMARY (DYNOTEARS lag1, h = 1): β {_prim['β news']:+.4f}, NW p {_prim['p (NW, 2-sided)']:.3f}, "
      f"node-perm p {_prim['p node-perm']:.3f}, placebo-date p {_prim['p placebo-date']:.3f} -> "
      + (("SUPPORTED" if _ok else "NOT SUPPORTED") + " (confirmatory rule)" if CONFIG["DATASET"] == "india"
         else ("rule met" if _ok else "rule not met") + " — exploratory on this dataset, not a confirmation"))
print("\nregime split (test, primary horizon; descriptive):")
print(ES_TE_REG[ES_TE_REG.family == "DYNOTEARS lag1"].round(4).to_string(index=False))
for nm, df_ in (("val", ES_VAL), ("test", ES_TE), ("val_regime", ES_VAL_REG), ("test_regime", ES_TE_REG)):
    df_.to_csv(f"{V3DIR}/s19_event_study_{nm}.csv", index=False)
json.dump({"spec_sha": S19_SHA, "dataset": CONFIG["DATASET"], "status": _st, "primary": {k: (float(v) if isinstance(v, (float, np.floating)) else v)
           for k, v in _prim.items()}, "rule_met": _ok, "synthetic": SYNTH}, open(f"{V3DIR}/s19_event_study_primary.json", "w"), indent=1,
          default=lambda o: o.item() if hasattr(o, "item") else str(o))
print("§19 written to", V3DIR)
MANIFEST.update(section19_spec_sha=S19_SHA)
json.dump(MANIFEST, open(f"{V3DIR}/run_manifest_{CONFIG['DATASET']}.json", "w"), indent=1, default=str)

abnormal returns: β median 1.00 | news firm-days 20,774 (37.0% of valid firm-days)
--- validation (development; descriptive) | 0.1 min ---
                      family  h                                         status  days with signal  coverage (S≠0)  β news    NW t  β no-news  Δβ news−no-news    t Δβ  β own shock  IC(S≠0) news  IC days
              DYNOTEARS lag1  1                                        primary                39          0.9232 -0.0797 -1.5522    -0.0809           0.0013  0.0157      -0.0076       -0.0252       39
              DYNOTEARS lag1  2                                        primary                38          0.9218 -0.1198 -1.5361    -0.0598          -0.0600 -0.3848      -0.0834       -0.0269       38
              DYNOTEARS lag1  3                                        primary                37          0.9204 -0.1711 -1.3268     0.0079          -0.1789 -0.7540      -0.1300       -0.0355       37
              DYNOTEARS lag1  5                          

---
# 20 · Supplementary (post hoc): magnitude of next-day returns — reporting-only, no retraining

**What SCEG predicts (main task):** next-day **direction** (ACC / MCC / AUC) and **cross-sectional ranking** (IC / RankIC).
SCEG-v3 was **not** designed or trained to predict the size of returns: its main head uses BCE (direction) plus an IC loss
(ranking, which ignores scale).

**What this section adds:** a secondary check of how close *predicted returns* are to *realised returns*, only for cached runs
that were trained with a regression (Huber) loss on returns and therefore have a return scale: the ACL18 legacy v2 grid, the
v3 arms with the v2 loss (ACL18 §15.2) and, on FNSPID, "GRU-v3 | all four reverted" (§19.4). Metrics: MAE, RMSE, out-of-sample
R² against a zero-return forecast, Diebold–Mariano test. It was added after test results were seen, trains nothing, and is
not a hypothesis test of SCEG.


In [74]:
# ===== 20 · SUPPLEMENTARY (post hoc, reporting-only): magnitude of next-day returns — no retraining =====
# SCEG's main task is DIRECTION (ACC/MCC/AUC) and CROSS-SECTIONAL RANKING (IC/RankIC). SCEG-v3 was NOT designed or
# trained to predict return magnitude: its main head is trained with BCE (direction) + an IC loss (ranking, scale-free).
# This cell asks a different, secondary question, only of cached runs that happened to be trained with a REGRESSION loss
# on returns (Huber on return / sigma), so that their output has a return scale:
#   * v3 arms with loss="v2" (Huber + IC): "GRU-v3 | v2 loss", "GRU-v3 | all four reverted", "SCEG-v3 | v2 loss"
#     (ACL18 §15.2; on FNSPID only "all four reverted", after §19.4 has run)
#   * ACL18 legacy v2 grid (§9, Huber + IC; LightGBM = L2 regression), benchmark alignment only
# BCE-trained runs (SCEG-v3 full, GRU-v3, MLP-v3, LightGBM-v3, nested, SCEG-R) output logits, not returns: excluded.
# Predicted return = model output × the fold's training sigma (the scale the loss divided by). Nothing is trained or tuned;
# test predictions are only read. Added after test results were seen -> descriptive, not a hypothesis test.
from scipy.stats import norm as _norm20
MAG_SPEC = {
    "status": "supplementary, post hoc, reporting-only; not a claim about SCEG's main task",
    "prediction": "model output x training sigma of the fold (the scale used by the Huber loss)",
    "baselines": {"zero": "predict 0 return (primary reference)", "hist_mean": "training-period mean return of the fold"},
    "metrics": "MAE, RMSE (in % return), R2_OS = 1 - SSE_model / SSE_zero, pred_sd / realised_sd, "
               "Diebold-Mariano on daily mean squared-error differences (Newey-West, 5 lags), two-sided, uncorrected",
    "sets": {"large_moves": "classification set MC3 (ACL18: StockNet samples |move| outside [-0.5%, +0.55%]; FNSPID: all firm-days)",
             "all_firmdays": "MR3, every valid firm-day"},
    "summary": "per-seed mean ± sd (primary); seed-ensemble (mean prediction) secondary",
}
MAG_SHA = sha(json.dumps(MAG_SPEC, sort_keys=True))[:12]
json.dump({"sha": MAG_SHA, "spec": MAG_SPEC, "written_at": time.strftime("%Y-%m-%d %H:%M:%S")},
          open(f"{V3DIR}/s20_magnitude_spec_{CONFIG['DATASET']}.json", "w"), indent=1)

def mag_metrics(pred, days, mask):
    """pred: (days, N) predicted next-day return. Returns metrics on the firm-days in mask."""
    y = Y[days]; m = mask[days] & np.isfinite(y) & np.isfinite(pred)
    e, y0 = (pred - y)[m], y[m]
    sse, sse0 = float((e ** 2).sum()), float((y0 ** 2).sum())
    dse = []                                                    # daily mean(sq error model - sq error zero)
    for r in range(len(days)):
        mm = m[r]
        if mm.sum() >= 1: dse.append(np.mean((pred[r][mm] - y[r][mm]) ** 2 - y[r][mm] ** 2))
    dse = np.array(dse); t = nw_t(dse) if len(dse) > 10 else np.nan
    return dict(n=int(m.sum()), MAE=np.abs(e).mean() * 100, RMSE=np.sqrt((e ** 2).mean()) * 100,
                MAE_zero=np.abs(y0).mean() * 100, RMSE_zero=np.sqrt((y0 ** 2).mean()) * 100,
                R2_OS=1 - sse / sse0, pred_sd_ratio=float(pred[m].std() / (y0.std() + 1e-12)),
                DM_t=t, DM_p=2 * (1 - _norm20.cdf(abs(t))) if np.isfinite(t) else np.nan)

def _v3_preds(mode, vname, seed):
    """Concatenated test predictions (return scale) and test days over folds, or None if any fold is missing."""
    P_, D_ = [], []
    for f3 in FOLD3:
        p = pred_path(f3, mode, vname, seed)
        if not os.path.exists(p): return None
        P_.append(np.load(p)["te_logit"].astype(float) * f3["SIGMA"]); D_.append(f3["TE"])
    return np.concatenate(P_), np.concatenate(D_)

def _legacy_preds(vname, seed):
    if not (RUN_LEGACY and all("sigma" in f for f in FOLDS)): return None
    P_, D_ = [], []
    for f in FOLDS:
        e = load_preds(vname, f, [seed])
        if not e: return None
        P_.append(e[0].astype(float) * f["sigma"]); D_.append(f["te"])
    return np.concatenate(P_), np.concatenate(D_)

SOURCES = []                                                    # (label, alignment, loader(seed))
_REG_ARMS = [v for v, c in ATTRIB.items() if c.get("loss") == "v2"]
for mode in V3["ALIGNMENTS"]:
    for v in _REG_ARMS:
        SOURCES.append((f"v3 {v}", mode, (lambda s, mode=mode, v=v: _v3_preds(mode, v, s))))
if RUN_LEGACY:
    for v in VARIANTS:
        SOURCES.append((f"v2 legacy {v}", CONFIG.get("NEWS_ALIGNMENT", ""), (lambda s, v=v: _legacy_preds(v, s))))

rows_, SEEDS20 = [], sorted(set(V3["SEEDS"]) | set(CONFIG["SEEDS"]))
for label, align, loader in SOURCES:
    runs = {s: loader(s) for s in SEEDS20}; runs = {s: r for s, r in runs.items() if r is not None}
    if not runs: continue
    days = next(iter(runs.values()))[1]
    ens = np.mean([r[0] for r in runs.values()], 0)
    for set_name, mask in (("large_moves", MC3), ("all_firmdays", MR3)):
        per = pd.DataFrame([mag_metrics(r[0], r[1], mask) for r in runs.values()])
        e = mag_metrics(ens, days, mask)
        rows_.append({"model": label, "alignment": align, "set": set_name, "seeds": len(runs), "n firm-days": e["n"],
                      "MAE %": f"{per.MAE.mean():.4f}±{per.MAE.std(ddof=0):.4f}",
                      "RMSE %": f"{per.RMSE.mean():.4f}±{per.RMSE.std(ddof=0):.4f}",
                      "R2_OS vs zero": f"{per.R2_OS.mean():+.4f}±{per.R2_OS.std(ddof=0):.4f}",
                      "pred_sd / realised_sd": round(per.pred_sd_ratio.mean(), 3),
                      "DM t (neg = better than zero)": round(per.DM_t.mean(), 2),
                      "ens R2_OS (secondary)": round(e["R2_OS"], 4), "ens DM p (uncorr.)": round(e["DM_p"], 3),
                      "MAE % zero": round(e["MAE_zero"], 4), "RMSE % zero": round(e["RMSE_zero"], 4)})
# reference rows: zero forecast (by definition R2_OS = 0) and the training-period historical mean
for align in ["—"]:                                             # baselines do not depend on the news alignment
    for set_name, mask in (("large_moves", MC3), ("all_firmdays", MR3)):
        z = mag_metrics(np.zeros((len(TE_ALL), N)), TE_ALL, mask)
        hm = np.concatenate([np.full((len(f3["TE"]), N), float(np.nanmean(np.where(MR3[f3["TR"]], Y[f3["TR"]], np.nan))))
                             for f3 in FOLD3])
        h = mag_metrics(hm, TE_ALL, mask)
        for name, r in (("baseline: zero return", z), ("baseline: training mean return", h)):
            rows_.append({"model": name, "alignment": align, "set": set_name, "seeds": 0, "n firm-days": r["n"],
                          "MAE %": f"{r['MAE']:.4f}", "RMSE %": f"{r['RMSE']:.4f}", "R2_OS vs zero": f"{r['R2_OS']:+.4f}",
                          "pred_sd / realised_sd": round(r["pred_sd_ratio"], 3),
                          "DM t (neg = better than zero)": (round(r["DM_t"], 2) if name != "baseline: zero return"
                                                            and np.isfinite(r["DM_t"]) else np.nan)})
MAG = pd.DataFrame(rows_)
print(f"=== §20 SUPPLEMENTARY magnitude evaluation ({CONFIG['DATASET']}, spec {MAG_SHA}) — post hoc, reporting-only ===")
print("Main SCEG task = direction + ranking (see §15/§16). These rows ask only how close predicted returns are to realised ones.")
if MAG.empty or not (MAG.seeds > 0).any():
    print("No cached regression-loss runs found for this dataset (FNSPID: run §19.4 first). Nothing to report.")
else:
    pd.set_option("display.width", 250)
    print(MAG.to_string(index=False))
    print("\nReading: R2_OS > 0 means lower squared error than always predicting 0 %; R2_OS <= 0 means no magnitude skill."
          "\npred_sd / realised_sd near 0 means the model predicts almost no movement (a near-zero R2_OS is then expected).")
MAG.to_csv(f"{V3DIR}/s20_magnitude_{CONFIG['DATASET']}.csv", index=False)
print("saved:", f"{V3DIR}/s20_magnitude_{CONFIG['DATASET']}.csv")


=== §20 SUPPLEMENTARY magnitude evaluation (acl18, spec 0fcea84984de) — post hoc, reporting-only ===
Main SCEG task = direction + ranking (see §15/§16). These rows ask only how close predicted returns are to realised ones.
                              model alignment          set  seeds  n firm-days         MAE %        RMSE %  R2_OS vs zero  pred_sd / realised_sd  DM t (neg = better than zero)  ens R2_OS (secondary)  ens DM p (uncorr.)  MAE % zero  RMSE % zero
   v3 GRU-v3 | v2 loss (Huber + IC)    strict  large_moves      5         3720 1.5454±0.0243 1.9749±0.0598 -0.0183±0.0625                  0.227                           0.33                 0.0202               0.581      1.5774       1.9580
   v3 GRU-v3 | v2 loss (Huber + IC)    strict all_firmdays      5         5568 1.1745±0.0617 1.6472±0.0717 -0.0494±0.0930                  0.271                           0.87                 0.0043               0.864      1.1384       1.6095
      v3 GRU-v3 | all four reverted    strict

---
# 21 · Ranking study (v10): why SCEG ranks poorly, and a controlled, validation-only correction

**Question.** Why does SCEG's cross-sectional RankIC stay low? What is the bottleneck, and does a theoretically motivated fix
improve RankIC under the same leakage-controlled protocol?

**Nothing above this section changes.** §0–§20 and their cached predictions are only read. Every new arm here is trained on training
dates, early-stopped and selected on validation dates only, and written to disk **without test predictions**. Test predictions for
*all* §21 arms are produced once, in §21.8, after the stage decisions and the final configuration have been frozen to a hashed file.

| Part | What it does | Uses test data? |
|---|---|---|
| 21.1 | Audit of the RankIC implementation and a set of leakage checks | reads cached test predictions only (no training) |
| 21.2 | Baseline table of the original models + model-free heuristics, RankIC stability over time | reads cached test predictions |
| 21.3 | Bottleneck diagnostics (objective, target, graph, events, regime, power) | training / validation data only |
| 21.4–21.5 | SCEG-Rank model + pre-registration of stages, arms and the adoption rule (hashed before any §21 training) | no |
| 21.6 | Sequential one-factor-at-a-time stages S, A, B, W, M, C, C2, D, E, F (validation only) | **no** |
| 21.7 | Freeze the final configuration; ablations and ranking baselines (validation only) | **no** |
| 21.8 | Test predictions for every pre-registered arm (gated on the frozen file) | yes, once |
| 21.9 | Reports A–F | yes (reporting only) |

**Evidential status.** ACL18: the SCEG test set has been inspected in earlier sections, so ACL18 §21 results are *partially held out*.
Run the identical cells in the FNSPID notebook: FNSPID (737 test days) is where a RankIC improvement can be resolved statistically.
On ACL18 (64 test days, 87 stocks) the standard error of a mean RankIC is ≈ 0.013 (computed in 21.3), so differences below ≈ 0.02 are not
resolvable there whatever the model.

In [75]:
# ===== 21.0 · configuration of the ranking study =====
from scipy.stats import rankdata as _rankdata, spearmanr as _spearmanr, norm as _N21
RK = dict(
    version="v10-s21",
    mode=V3["PRIMARY"],                  # strict alignment only
    seeds=list(V3["SEEDS"]),             # predetermined: 0..4
    epochs=V3["EPOCHS"], patience=V3["PATIENCE"], batch_days=V3["BATCH_DAYS"], lr=V3["LR"], wd=V3["WD"],
    hidden=V3["HIDDEN"], dropout=HP_R["dropout"],
    adopt_t=2.0,                          # adoption rule (validation): NW t of the seed-averaged daily ΔRankIC >= 2.0 ...
    adopt_min=0.002,                      # ... and mean ΔRankIC >= 0.002 ...
    adopt_units=0.6,                      # ... and the arm wins in >= 60 % of (fold, seed) units
    listnet_tau=1.0,
    ls_quantile=0.2,
)
RK_DIR = f"{V3DIR}/rank21"; os.makedirs(RK_DIR, exist_ok=True)
DS21 = CONFIG.get("DATASET", "fnspid")
print(json.dumps(RK, indent=1)); print("dataset", DS21, "| cache", RK_DIR)

{
 "version": "v10-s21",
 "mode": "strict",
 "seeds": [
  0,
  1,
  2,
  3,
  4
 ],
 "epochs": 60,
 "patience": 10,
 "batch_days": 16,
 "lr": 0.001,
 "wd": 0.0001,
 "hidden": 64,
 "dropout": 0.3,
 "adopt_t": 2.0,
 "adopt_min": 0.002,
 "adopt_units": 0.6,
 "listnet_tau": 1.0,
 "ls_quantile": 0.2
}
dataset acl18 | cache /content/drive/MyDrive/MinorProject/acl18_cache/v3/rank21


### 21.1 · Audit of the RankIC implementation and of leakage
Checks, each printed with PASS / FLAG:
1. **What is ranked**: the stock-specific score `te_score` (pre-sigmoid, continuous), not a thresholded probability.
2. **Against what**: the next-day close-to-close adjusted return `Y[t] = adjclose(t+1)/adjclose(t) − 1` (raw; ranking is invariant to
   subtracting the day mean, so raw vs market-adjusted makes no difference to RankIC).
3. **Within each day**: correlation is computed per trading day over `MR3[t]` (price row on t and t+1, finite target), then averaged.
4. **Spearman correctness**: re-computed with `scipy.stats.spearmanr` on every test day of the cached SCEG-R runs.
5. **Ties** (average ranks), **days with < 5 firms or constant scores** (set to 0, counted), **missing stocks** (excluded that day).
6. **Metric validity**: `RankIC_all` (valid), `RankIC_filtered` (conditions on the *realised* |return| → not implementable ex ante; diagnostic
   only), ensemble RankIC (valid only as a separately declared model), v2 legacy IC (computed on the StockNet large-move subset → filtered).
7. **Leakage**: strict alignment, split disjointness, a *future-perturbation test* of the price features, graph/PCA/HMM fit windows,
   and universe selection.

In [76]:
# ===== 21.1 · audit =====
AUD = []
def _aud(check, ok, detail): AUD.append(dict(check=check, status="PASS" if ok is True else ("FLAG" if ok is False else ok), detail=detail))

_r0 = load_runs(RK["mode"], R_FINAL_NAME)
_s0 = next(iter(_r0.values()))["te_score"] if _r0 else None
if _s0 is not None:
    mine = daily_ic_mask(_s0, TE_ALL, MR3, rank=True)
    ref, nfirm, ties_s, ties_y, skip = [], [], [], [], 0
    for r, t in enumerate(TE_ALL):
        m = MR3[t]; nfirm.append(int(m.sum()))
        a, b = _s0[r][m], Y[t][m]
        ties_s.append(1 - len(np.unique(a)) / max(len(a), 1)); ties_y.append(1 - len(np.unique(b)) / max(len(b), 1))
        if m.sum() < 5 or np.std(a) == 0: ref.append(0.0); skip += 1; continue
        v = _spearmanr(a, b).correlation; ref.append(v if np.isfinite(v) else 0.0)
    ref = np.array(ref)
    _aud("Spearman implementation == scipy.spearmanr (every test day)", bool(np.allclose(mine, ref, atol=1e-10)),
         f"max |diff| {np.abs(mine - ref).max():.1e} over {len(ref)} days")
    _aud("firms per test day (MR3)", "INFO", f"min {min(nfirm)} | median {int(np.median(nfirm))} | max {max(nfirm)} of N={N}")
    _aud("days set to 0 (<5 firms or constant score)", skip == 0, f"{skip} of {len(TE_ALL)}")
    _aud("ties in the ranked score", "INFO", f"mean share of tied values per day {np.mean(ties_s):.2%} (average ranks used)")
    _aud("ties in the target", "INFO", f"mean share of tied returns per day {np.mean(ties_y):.2%} (average ranks used)")
_aud("score that is ranked", True, "te_score = stock-specific head output s (continuous, pre-sigmoid); the direction logit is not ranked")
_aud("RankIC_all (all valid firm-days)", True, "ex-ante implementable: the evaluated set depends only on price availability")
_aud("RankIC_filtered (|move| outside [-0.5%, +0.55%])", False,
     "conditions on the realised target: not implementable ex ante, inflates ranking metrics -> diagnostic only, never a headline")
_aud("ensemble RankIC", "INFO", "valid only as a separately declared model (a seed ensemble), never compared with per-seed means")
_aud("v2 legacy IC/RankIC (§10)", False, "computed on StockNet benchmark samples (outcome-filtered) -> not comparable with RankIC_all")

# leakage: strict alignment and split disjointness
if "ALIGNED" in globals() and RK["mode"] in ALIGNED:
    _a = ALIGNED[RK["mode"]]
    if "_ts" in _a:
        _tz, _cl = CONFIG.get("MARKET_TZ", "America/New_York"), CONFIG.get("CLOSE_MINUTES", CONFIG["NEWS_CUTOFF_HOUR_ET"] * 60)
        _late = float((_a["_ts"] >= (_a["eff_date"] + pd.Timedelta(minutes=_cl)).dt.tz_localize(_tz).dt.tz_convert("UTC")).mean())
        _aud("strict alignment: items at/after the close of their feature day", _late == 0.0, f"{_late:.4%}")
for f3 in FOLD3:
    tr, va, te = set(f3["TR"] + 1), set(f3["VA"] + 1), set(f3["TE"] + 1)       # target days
    _aud(f"{f3['name']}: train/val/test target days disjoint", not (tr & va or tr & te or va & te),
         f"{len(tr)}/{len(va)}/{len(te)} target days")
    _aud(f"{f3['name']}: target days ordered train < validation < test", max(tr) < min(va) and max(va) < min(te),
         "ACL18 target-day rule lets early test INPUT windows reach back into validation days (features only, no labels)")

# leakage: future-perturbation test of the price features (perturb every price after a cut-off; features up to it must not move)
try:
    _cut = int(FOLD3[0]["VA"][-1]); _tr0 = FOLD3[0]["fd"]["tr_days"]
    _F1, _ = price_panel_v3(_tr0)
    _px_keep = px
    _q = px.copy(); _fut = _q["Date"] > TRADING_DAYS[_cut]
    _rng = np.random.default_rng(0)
    for c in ("open", "high", "low", "close", "adj_close"):
        if c in _q: _q.loc[_fut, c] = _q.loc[_fut, c] * np.exp(_rng.normal(0, 0.05, _fut.sum()))
    if "volume" in _q: _q.loc[_fut, "volume"] = _q.loc[_fut, "volume"] * np.exp(_rng.normal(0, 0.5, _fut.sum()))
    _q = _q.sort_values(["Stock_symbol", "Date"]); _q["Return"] = _q.groupby("Stock_symbol")["adj_close"].pct_change(fill_method=None)
    px = _q
    try: _F2, _ = price_panel_v3(_tr0)
    finally: px = _px_keep
    _d = float(np.abs(_F1[:_cut + 1] - _F2[:_cut + 1]).max())
    _aud("future-perturbation test: price features on days <= cut-off unchanged when all later prices are randomised", _d < 1e-6,
         f"max |Δfeature| on days <= {TRADING_DAYS[_cut].date()}: {_d:.2e}")
except Exception as e:
    _aud("future-perturbation test", "NOT RUN", repr(e)[:120])

_aud("graphs, PCA, scalers, HMM fitted on training dates only (per fold)", True,
     "fit_graphs_v3 / fit_pca_train / std_train / price_panel_v3 / fit_regimes take fd['tr_days']; HMM posteriors are filtered")
_aud("graph leakage via target days", True, "DYNOTEARS/correlation use RET on training days; RET[t] only uses prices up to t")
if IS_ACL18:
    _aud("universe selection", "INFO", "StockNet's fixed 88-stock list (chosen ex post by the dataset authors): mild survivorship, shared by all models")
else:
    _aud("universe selection", False, "coverage filter keeps tickers whose series reach the sample end (uses future information): "
         "survivorship bias, shared by all models; disclose")
_aud("test-set tuning (original SCEG)", False if IS_ACL18 else True,
     "ACL18: SCEG-R was specified after the SCEG-v3 test was inspected (partial hold-out)" if IS_ACL18 else "FNSPID: none")
_aud("LLM / encoder look-ahead", False, "Qwen2.5 and BGE were trained after the sample period; a positive text effect would need a pre-period extractor")
AUDIT21 = pd.DataFrame(AUD); pd.set_option("display.max_colwidth", 140)
print(AUDIT21.to_string(index=False)); AUDIT21.to_csv(f"{RK_DIR}/audit_{DS21}.csv", index=False)

                                                                                                     check status                                                                                                                      detail
                                               Spearman implementation == scipy.spearmanr (every test day)   PASS                                                                                             max |diff| 1.4e-17 over 64 days
                                                                                  firms per test day (MR3)   INFO                                                                                         min 87 | median 87 | max 87 of N=87
                                                                days set to 0 (<5 firms or constant score)   PASS                                                                                                                     0 of 64
                                                

### 21.2 · Clean baseline: original models, model-free heuristics, and RankIC stability over time
Primary statistic: per-seed mean ± sd of the test RankIC (all valid firm-days). Also: the day-level sd of RankIC, ICIR (mean / day-sd),
the Newey–West t of the seed-averaged daily series, the share of positive days, and RankIC per calendar month (ACL18) or quarter (FNSPID).
Heuristics need no training: their sign is fixed from the **training-period** RankIC, never from test data.

In [77]:
# ===== 21.2 · baseline table =====
def rank_series(score, days): return daily_ic_mask(score, days, MR3, rank=True)
def rank_stats(runs, days=None, name=""):
    days = TE_ALL if days is None else days
    per = np.stack([rank_series(r["te_score"], days) for r in runs.values()])            # seeds x days
    ic = np.stack([daily_ic_mask(r["te_score"], days, MR3) for r in runs.values()])
    avg = per.mean(0)
    row = dict(model=name, seeds=len(runs), RankIC=per.mean(1).mean(), RankIC_sd_seeds=per.mean(1).std(ddof=0),
               RankIC_sd_days=avg.std(ddof=0), ICIR=avg.mean() / (avg.std(ddof=0) + 1e-12), NW_t=nw_t(avg),
               pos_days=float((avg > 0).mean() * 100), IC=ic.mean(1).mean(), IC_sd_seeds=ic.mean(1).std(ddof=0))
    return row, avg

PRICE_IDX = {n: i for i, n in enumerate(PRICE_NAMES)}
def heuristic_runs(feat, days_tr, name):
    """Score = ±(a price feature on day t). Sign from the training-period mean RankIC (no test data)."""
    out = {}
    te = np.concatenate([f3["X"][RK["mode"]][f3["TE"]][..., f3["sl"]["price"][0] + PRICE_IDX[feat]] for f3 in FOLD3])
    tr_sign = np.sign(np.mean(np.concatenate([rank_series(f3["X"][RK["mode"]][f3["TR"]][..., f3["sl"]["price"][0] + PRICE_IDX[feat]],
                                                          f3["TR"]) for f3 in FOLD3])) + 1e-12)
    out[0] = dict(te_score=tr_sign * te, te_logit=np.zeros_like(te))
    return out, tr_sign

BASE_ROWS, BASE_SERIES = [], {}
ORIG = {"SCEG-R (final, frozen)": R_FINAL_NAME, "SCEG-R base": BASE_ARM, "SCEG-v3 (full)": "SCEG-v3 (full)",
        "GRU-v3 (no graph)": "− graph (= GRU-v3)", "MLP-v3": "MLP-v3", "LightGBM-v3": "LightGBM-v3"}
for lab, v in ORIG.items():
    runs = load_runs(RK["mode"], v)
    if not runs: continue
    row, s = rank_stats(runs, name=lab); BASE_SERIES[lab] = s
    per = pd.DataFrame([{**cls_metrics(r["te_logit"], TE_ALL)} for r in runs.values()])
    row.update(ACC=per.ACC.mean(), MCC=per.MCC.mean(), AUC=per.AUC.mean()); BASE_ROWS.append(row)
HEUR = {}
for feat, lab in (("r1", "1-day reversal/momentum (sign from training)"), ("r5", "5-day reversal/momentum (sign from training)"),
                  ("r20", "20-day momentum/reversal (sign from training)")):
    if feat in PRICE_IDX:
        runs, sg = heuristic_runs(feat, None, lab); HEUR[lab] = runs
        row, s = rank_stats(runs, name=f"{lab} [sign {'+' if sg > 0 else '−'}]"); BASE_SERIES[lab] = s; BASE_ROWS.append(row)
BASE21 = pd.DataFrame(BASE_ROWS).set_index("model")
print(f"=== [{DS21}] test RankIC baseline (all valid firm-days; per-seed mean ± sd; day-level statistics of the seed-averaged series) ===")
print(BASE21.round(4).to_string()); BASE21.to_csv(f"{RK_DIR}/baseline_{DS21}.csv")

# stability over time
_per = pd.Series(pd.to_datetime(TRADING_DAYS[TE_ALL + 1])).dt.to_period("M" if IS_ACL18 else "Q").astype(str).values
STAB = pd.DataFrame({k: pd.Series(v).groupby(_per).mean() for k, v in BASE_SERIES.items()})
STAB["days"] = pd.Series(np.ones(len(_per))).groupby(_per).sum().values
print(f"\nRankIC by {'month' if IS_ACL18 else 'quarter'} of the target day (seed-averaged):"); print(STAB.round(4).to_string())
STAB.to_csv(f"{RK_DIR}/stability_{DS21}.csv")

=== [acl18] test RankIC baseline (all valid firm-days; per-seed mean ± sd; day-level statistics of the seed-averaged series) ===
                                                        seeds  RankIC  RankIC_sd_seeds  RankIC_sd_days    ICIR    NW_t  pos_days      IC  IC_sd_seeds      ACC     MCC      AUC
model                                                                                                                                                                          
SCEG-R (final, frozen)                                      5  0.0286           0.0049          0.1439  0.1986  1.4563   57.8125  0.0242       0.0046  55.7688  0.1254  58.1952
SCEG-R base                                                 5  0.0286           0.0049          0.1439  0.1986  1.4563   57.8125  0.0242       0.0046  55.7688  0.1254  58.1952
SCEG-v3 (full)                                              5  0.0426           0.0295          0.1249  0.3407  2.9618   60.9375  0.0354       0.0311  54.1237  0.0850 

### 21.3 · Bottleneck diagnostics (training and validation data only)
Each diagnostic tests one link of *events → representations → graph → propagation → temporal → head → ranking*.

| ID | Question | Measurement |
|---|---|---|
| D1 | Is training / selection dominated by the market direction? | share of logit variance that is common to all firms on a day; validation AUC split into market and cross-sectional parts |
| D2 | Does the score have enough cross-sectional resolution? | distinct values per day, cross-sectional vs time-series spread of the score |
| D3 | Is the ranking loss aligned with RankIC? | heavy tails of daily returns; how much of a day's Pearson IC is carried by the 10 % most extreme movers |
| D4 | Is the graph a meaningful company–company structure? | density, sign balance, stability across two halves of the training period, market-factor content, smoothing ratio |
| D5 | Do events differentiate companies? | same-day coverage, share of default categories, training-period RankIC of event polarity |
| D6 | Can regime affect ranking at all? | regime occupancy by split; a regime feature is identical for all firms on a day |
| D7 | Can the test set resolve an improvement? | SE of mean RankIC and the paired MDE at 80 % power |

In [78]:
# ===== 21.3 · diagnostics =====
DIAG = []
def _dg(d, k, v, note=""): DIAG.append(dict(id=d, quantity=k, value=v, note=note))

# D1 · common vs cross-sectional part of the SCEG-R direction logit, and validation AUC split (validation outputs only)
for arm, lab in ((BASE_ARM, "SCEG-R base"),):
    shares, am_, ax_ = [], [], []
    for f3 in FOLD3:
        for seed in R_SPEC["seeds"]:
            p_ = r_val(f3, arm, seed)
            if not os.path.exists(p_): continue
            z = np.load(p_); L = z["val_logit"].astype(float); days = f3["VA"]
            c = np.array([L[r][MR3[t]].mean() if MR3[t].sum() else 0 for r, t in enumerate(days)])
            dev = np.concatenate([L[r][MR3[t]] - c[r] for r, t in enumerate(days)])
            shares.append(np.var(c) / (np.var(c) + np.var(dev) + 1e-12))
            a1, a2 = auc_split(L, days); am_.append(a1); ax_.append(a2)
    if shares:
        _dg("D1", f"{lab}: share of validation logit variance common to all firms on a day", float(np.mean(shares)),
            "BCE on the raw sign rewards this part; it carries no cross-sectional information")
        _dg("D1", f"{lab}: validation AUC of the day-common logit (market timing)", float(np.mean(am_)))
        _dg("D1", f"{lab}: validation AUC of the within-day demeaned logit (stock selection)", float(np.mean(ax_)),
            "≈ 0.5 means the direction head does not rank stocks; selection by AUC then selects market timing")
_dg("D1", "original early-stopping / selection criterion", "val AUC (v3) | (val AUC − 0.5) + val IC (SCEG-R)",
    "neither is the evaluation metric (RankIC)")
_dg("D1", "original ranking-loss weight", f"λ_rank = {HP_R['lam_rank']} (chosen by validation AUC in §14.6)", "")

# D2 · resolution of the ranked score (validation)
res_u, res_ratio = [], []
for f3 in FOLD3:
    for seed in R_SPEC["seeds"]:
        p_ = r_val(f3, BASE_ARM, seed)
        if not os.path.exists(p_): continue
        S_ = np.load(p_)["val_score"].astype(float)
        xs = np.mean([np.std(S_[r][MR3[t]]) for r, t in enumerate(f3["VA"]) if MR3[t].sum() > 1])
        ts = np.std([S_[r][MR3[t]].mean() for r, t in enumerate(f3["VA"]) if MR3[t].sum() > 1])
        res_u.append(np.mean([len(np.unique(np.round(S_[r][MR3[t]], 6))) / max(MR3[t].sum(), 1) for r, t in enumerate(f3["VA"])]))
        res_ratio.append(xs / (ts + 1e-12))
if res_u:
    _dg("D2", "distinct score values per day / firms (validation)", float(np.mean(res_u)), "1.0 = no ties; the score is continuous")
    _dg("D2", "cross-sectional sd / time-series sd of the daily mean score", float(np.mean(res_ratio)),
        "the ranked score is s (pre-sigmoid): probability saturation around 0.5 cannot cause ties")

# D3 · heavy tails of the ranking target (training days)
kurt, share10, gap = [], [], []
for f3 in FOLD3:
    for t in f3["TR"]:
        m = MR3[t]
        if m.sum() < 10: continue
        y = Y[t][m].astype(float); z = (y - y.mean()) / (y.std() + 1e-12)
        kurt.append(np.mean(z ** 4)); k = max(1, int(round(0.1 * len(z)))); o = np.argsort(np.abs(z))[-k:]
        share10.append(np.sum(z[o] ** 2) / np.sum(z ** 2))
        g = np.random.default_rng(int(t)).normal(size=len(y)) + 0.2 * z          # a synthetic signal with fixed rank quality
        pe = np.corrcoef(g, y)[0, 1]; sp = _spearmanr(g, y).correlation; gap.append(pe - sp)
_dg("D3", "mean daily cross-sectional kurtosis of next-day returns (training)", float(np.mean(kurt)), "3 = Gaussian")
_dg("D3", "share of daily return variance carried by the 10 % most extreme movers", float(np.mean(share10)),
    "Pearson IC loss on raw returns is dominated by these firms; RankIC weighs every firm equally")
_dg("D3", "sd of (Pearson − Spearman) for one fixed-quality synthetic signal across training days", float(np.std(gap)),
    "day-to-day disagreement between the training loss (Pearson on raw returns) and the metric (Spearman)")

# D4 · graph quality
def _jacc(A, B): a, b = A != 0, B != 0; return float((a & b).sum() / max((a | b).sum(), 1))
def resid_corr_graph(tr_days, k=None):
    k = k or CONFIG["TOP_K_CORR"]
    R = np.where((ROW & EVAL_MASK)[tr_days], RET[tr_days], np.nan); mk = np.nanmean(R, 1, keepdims=True)
    Rc = np.nan_to_num(R - np.nanmean(R, 0)); mc = np.nan_to_num(mk - np.nanmean(mk))
    beta = (Rc * mc).sum(0) / ((mc ** 2).sum() + 1e-12)
    E = np.where(np.isfinite(R), R - beta[None] * mk, np.nan)                     # market-residual returns (training betas)
    C = pd.DataFrame(E).corr(min_periods=60).fillna(0).values.copy(); np.fill_diagonal(C, 0)
    M = np.zeros((N, N), np.float32)
    for i in range(N):
        nb = np.argsort(-np.abs(C[i]))[:k]; M[i, nb] = C[i, nb]
    return signed_cap(M, CONFIG["MAX_IN_DEGREE"])
for f3 in FOLD3:
    f3["G"]["resid_corr"] = resid_corr_graph(f3["fd"]["tr_days"])
    for r in ("contemp", "lag1", "lag2", "corr", "resid_corr", "sector"):
        M = f3["G"][r]; nz = M[M != 0]
        _dg("D4", f"{f3['name']} {r}: edges | density | negative share", f"{len(nz)} | {len(nz) / (N * (N - 1)):.3f} | {np.mean(nz < 0) if len(nz) else 0:.2f}",
            "a lead–lag relation with ~50 % negative signs is what pure noise produces" if r in ("lag1", "lag2") else "")
    # overlap of raw-return correlation neighbours with market-residual neighbours: how much is market factor
    _dg("D4", f"{f3['name']}: Jaccard(corr graph on raw returns, corr graph on market-residual returns)", _jacc(f3["G"]["corr"], f3["G"]["resid_corr"]),
        "low overlap = raw-return neighbours are largely driven by the common market factor")
    # smoothing ratio of each relation on a cross-sectional signal (training days): sd of A·x / sd of x
    j = f3["sl"]["price"][0] + PRICE_IDX["r1_xs"] if "r1_xs" in PRICE_IDX else None
    if j is not None:
        Xr = f3["X"][RK["mode"]][f3["TR"]][..., j]
        for r in ("contemp", "corr", "sector", "resid_corr"):
            A = f3["G"][r]; ratio = np.mean([np.std(A @ x) / (np.std(x) + 1e-12) for x in Xr])
            _dg("D4", f"{f3['name']} {r}: smoothing ratio sd(A·x)/sd(x) of r1_xs", float(ratio),
                "<1: propagation averages away cross-sectional differences (the signal ranking needs)")
# stability of DYNOTEARS lead–lag edges across the two halves of training (first fold; cached; set RUN21_DYNO_STAB=False to skip)
RUN21_DYNO_STAB = bool(int(os.environ.get("RUN21_DYNO_STAB", "1")))
if RUN21_DYNO_STAB:
    f3 = FOLD3[0]; tr = f3["fd"]["tr_days"]; half = len(tr) // 2
    pth = f"{RK_DIR}/dyno_halves_{DS21}_{f3['name']}.npz"
    if os.path.exists(pth): z = np.load(pth); A1, A2 = z["A1"], z["A2"]
    else:
        def _fit(days):
            R = np.where((ROW & EVAL_MASK)[days], RET[days], np.nan)
            Z = np.nan_to_num((R - np.nanmean(R, 0)) / (np.nanstd(R, 0) + 1e-9), nan=0.0)
            p = CONFIG["DYNOTEARS_P"]; Xc = Z[p:]; Xl = np.concatenate([Z[p - k - 1: -k - 1] for k in range(p)], axis=1)
            _, Am, _ = dynotears(Xc, Xl, CONFIG["DYNOTEARS_LAMBDA"], CONFIG["DYNOTEARS_LAMBDA_A"], CONFIG["DYNOTEARS_MAX_ITER"],
                                 w_threshold=CONFIG["DYNOTEARS_W_THRESHOLD"])
            return signed_cap(Am[:N].T, CONFIG["MAX_IN_DEGREE"])
        A1, A2 = _fit(tr[:half]), _fit(tr[half:]); np.savez(pth, A1=A1, A2=A2)
    same_sign = np.mean(np.sign(A1[(A1 != 0) & (A2 != 0)]) == np.sign(A2[(A1 != 0) & (A2 != 0)])) if ((A1 != 0) & (A2 != 0)).any() else np.nan
    _dg("D4", f"{f3['name']}: DYNOTEARS lag-1 edge Jaccard, first vs second half of training", _jacc(A1, A2),
        "near 0 = the learned lead–lag structure is not stable, i.e. mostly estimation noise")
    _dg("D4", f"{f3['name']}: sign agreement on edges present in both halves", float(same_sign), "0.5 = coin flip")
    _dg("D4", "DYNOTEARS parameters vs observations (first fold)", f"{N * N * (CONFIG['DYNOTEARS_P'] + 1):,} vs {len(tr)} days × {N} firms", "")

# D5 · events
p_has = FOLD3[0]["X"][RK["mode"]][..., FOLD3[0]["sl"]["news"][0]] > 0
_dg("D5", "share of valid firm-days with a same-day text item", float(p_has[MR3].mean()), "the rest carry forward-filled (≤5 days) or zero text")
if "CAT_NAMES" in globals():
    ci = {n: i for i, n in enumerate(CAT_NAMES)}
    if "polarity" in ci:
        j = FOLD3[0]["sl"]["cat"][0] + ci["polarity"]
        vals = []
        for f3 in FOLD3:
            Xf = f3["X"][RK["mode"]]
            for t in f3["TR"]:
                m = MR3[t] & (Xf[t, :, f3["sl"]["news"][0]] > 0)
                if m.sum() >= 5 and np.std(Xf[t, m, j]) > 0: vals.append(_spearmanr(Xf[t, m, j], Y[t][m]).correlation)
        vals = np.nan_to_num(np.array(vals))
        _dg("D5", "training-period mean daily RankIC of LLM event polarity (firms with a same-day item, days with >= 5 such firms)",
            float(vals.mean()) if len(vals) else np.nan, f"{len(vals)} days; SE {vals.std(ddof=1) / np.sqrt(max(len(vals), 2)):.4f}")
if "STORE" in globals():
    _dg("D5", "share of items with event_type in {other, none}", float(STORE["event_type"].isin(["other", "none"]).mean()), "")
    _dg("D5", "share of items with magnitude = unknown", float((STORE["magnitude"] == "unknown").mean()), "")

# D6 · regime
for f3 in FOLD3:
    for split in ("TR", "VA", "TE"):
        occ = pd.Series([f3["rinfo"]["names"][k] for k in f3["PI"][f3[split]].argmax(1)]).value_counts(normalize=True).round(2).to_dict()
        _dg("D6", f"{f3['name']} regime occupancy ({split})", str(occ), "one dominant regime -> regime effects are not testable in that split" if split == "TE" else "")
_dg("D6", "regime as an input feature", "identical for all firms on a day",
    "it can change a ranking only through interactions with firm features (conditioned scoring), never additively")

# D7 · power of the test set for RankIC
avg0 = BASE_SERIES.get("SCEG-R (final, frozen)")
if avg0 is not None:
    nd = len(avg0); se = float(np.std(avg0, ddof=1) / np.sqrt(nd))
    _dg("D7", "SE of the mean test RankIC (i.i.d. days)", se, f"{nd} test days, ~{int(np.median(MR3[TE_ALL].sum(1)))} firms/day")
    ra = load_runs(RK["mode"], R_FINAL_NAME); rb = load_runs(RK["mode"], "− graph (= GRU-v3)")
    if ra and rb:
        d = np.mean([rank_series(ra[s]["te_score"], TE_ALL) - rank_series(rb[s]["te_score"], TE_ALL) for s in sorted(set(ra) & set(rb))], 0)
        se_d = float(np.std(d, ddof=1) / np.sqrt(nd))
        _dg("D7", "SE of a paired ΔRankIC between two different models (SCEG-R vs GRU-v3)", se_d, "")
        _dg("D7", "minimum detectable ΔRankIC at 80 % power (two-sided 5 %)", (1.96 + 0.84) * se_d, "")
DIAG21 = pd.DataFrame(DIAG)
print(DIAG21.to_string(index=False)); DIAG21.to_csv(f"{RK_DIR}/diagnostics_{DS21}.csv", index=False)

id                                                                                                        quantity                                             value                                                                                                         note
D1                                    SCEG-R base: share of validation logit variance common to all firms on a day                                          0.856977                             BCE on the raw sign rewards this part; it carries no cross-sectional information
D1                                             SCEG-R base: validation AUC of the day-common logit (market timing)                                          0.622224                                                                                                             
D1                                  SCEG-R base: validation AUC of the within-day demeaned logit (stock selection)                                           0.51321             ≈

### 21.4 · SCEG-Rank: the same SCEG pipeline, with the ranking objective made explicit
SCEG-Rank keeps every SCEG stage — **structured events → inter-firm graph → propagation → temporal context → regime → heads** — and
makes each one a switch, so that one factor can change at a time:

* `text`: `raw+struct` (SCEG-R: raw primary + zero-initialised structured residual) · `raw` · `struct` · `none`; `same_day` masks forward-filled text.
* `graph`: `none` · `dynotears` (contemp + lag1 + lag2) · `sector` · `resid_corr` (top-k correlation of **market-residual** training returns);
  `random` = signed random graph matched in edge count and weights; `msg`: `avg` (neighbour average) · `contrast` (neighbour average − self),
  a high-pass message that keeps relative information.
* `temporal`: `gru` over the 5-day window · `none` (day t only).
* `regime`: `none` (uniform posterior) · `feature` (SCEG-R) · `film_head` (regime modulates the *scoring* map, the only way a regime common
  to all firms can change a cross-sectional ranking) · `film_prop` (regime modulates propagation, SCEG-v3's hypothesis).
* heads: direction logit `m_t + (s − mean s)` (SCEG-R), stock score `s` (ranked), magnitude `r̂` (return / σ).
* loss `L = w_dir·BCE + w_rank·L_rank(s, target) + w_mag·Huber(r̂, r/σ)`, with `L_rank` ∈ Pearson IC · pairwise RankNet ·
  LambdaRankIC-weighted pairwise (Lin et al. 2026: pair weight ∝ |Δrank(score)|·|Δrank(target)|, the change in Spearman from a swap) ·
  ListNet; target ∈ raw return · Gaussian-rank (cross-sectional) · Gaussian-rank on large-move training firm-days.
* early stopping / selection: `composite` (SCEG-R) or `rankic` (validation RankIC of `s`, the evaluation metric).

With the defaults below (`B0`), SCEG-Rank is SCEG-R base: raw+struct text, no graph, GRU, regime feature, BCE + 0.25·Pearson IC on raw
returns, composite selection. B0 is re-trained here so every comparison uses one code path; 21.9 also reports the cached original.

In [79]:
# ===== 21.4 · SCEG-Rank model, losses and trainer =====
B0 = dict(text="raw+struct", same_day=False, graph="none", random=False, msg="avg", temporal="gru", regime="feature",
          target="raw", rank_loss="pearson", w_dir=1.0, w_rank=float(HP_R["lam_rank"]), w_mag=0.0,
          select="composite")
def cfg_key(c): return sha(json.dumps({k: c[k] for k in sorted(B0)}, sort_keys=True))[:10]
GRAPH_RELS = {"none": (), "dynotears": ("contemp", "lag1", "lag2"), "sector": ("sector",), "resid_corr": ("resid_corr",)}

class SCEGRank(nn.Module):
    def __init__(self, sl, K, M, c, S=None, H=64, dropout=0.3):
        super().__init__()
        self.sl, self.c, self.H, self.K = sl, c, H, K
        n = lambda k: sl[k][1] - sl[k][0]
        self.use_raw = c["text"] in ("raw+struct", "raw"); self.use_struct = c["text"] in ("raw+struct", "struct")
        self.price = nn.Linear(n("price"), H)
        if self.use_raw: self.raw = nn.Linear(n("raw"), H)
        if self.use_struct: self.struct = nn.Linear(n("tri") + n("cat"), H)
        if self.use_raw and self.use_struct: self.s_mix = nn.Parameter(torch.zeros(1))     # SCEG-R: zero-initialised structured residual
        self.has_text = self.use_raw or self.use_struct
        if self.has_text: self.news = nn.Linear(3, H)
        d_in = H * (2 if self.has_text else 1) + (K if c["regime"] == "feature" else 0)
        self.inp = nn.Sequential(nn.Linear(d_in, H), nn.LayerNorm(H), nn.GELU(), nn.Dropout(dropout))
        if c["temporal"] == "gru": self.gru = nn.GRU(H, H, batch_first=True)
        self.rels = list(GRAPH_RELS[c["graph"]])
        if self.rels:
            self.register_buffer("S", torch.as_tensor(np.stack([S[r] for r in self.rels]), dtype=torch.float32))
            self.V = nn.ModuleList([nn.Linear(H, H, bias=False) for _ in self.rels])
            self.out = nn.Linear(H, H); nn.init.zeros_(self.out.weight); nn.init.zeros_(self.out.bias)   # starts as the no-graph model
            self.res = nn.Linear(2 * H, H); nn.init.constant_(self.res.bias, -2.0)
            if c["regime"] == "film_prop":
                self.filmP = nn.Linear(K, len(self.rels) * H); nn.init.zeros_(self.filmP.weight); nn.init.zeros_(self.filmP.bias)
        if c["regime"] == "film_head":
            self.filmH = nn.Linear(K, 2 * H); nn.init.zeros_(self.filmH.weight); nn.init.zeros_(self.filmH.bias)
        self.drop = nn.Dropout(dropout)
        self.idio = nn.Sequential(nn.Linear(H, H // 2), nn.GELU(), nn.Dropout(dropout), nn.Linear(H // 2, 1))
        self.mag = nn.Sequential(nn.Linear(H, H // 2), nn.GELU(), nn.Linear(H // 2, 1))
        self.mkt = nn.Sequential(nn.Linear(H + M + K, 32), nn.GELU(), nn.Linear(32, 1))
        self.stats = {}

    def _x(self, X, k): return X[..., self.sl[k][0]:self.sl[k][1]]

    def forward(self, X, PI, MK):
        B, W, Nn, _ = X.shape
        parts = [Fnn.gelu(self.price(self._x(X, "price")))]
        if self.has_text:
            if self.use_raw and self.use_struct:
                txt = self.raw(self._x(X, "raw")) + self.s_mix * self.struct(torch.cat([self._x(X, "tri"), self._x(X, "cat")], -1))
            elif self.use_raw: txt = self.raw(self._x(X, "raw"))
            else: txt = self.struct(torch.cat([self._x(X, "tri"), self._x(X, "cat")], -1))
            parts.append(Fnn.gelu(txt + self.news(self._x(X, "news"))))
        if self.c["regime"] == "feature": parts.append(PI[:, :, None, :].expand(B, W, Nn, PI.shape[-1]))
        x = self.inp(torch.cat(parts, -1))
        if self.c["temporal"] == "gru":
            o, _ = self.gru(x.permute(0, 2, 1, 3).reshape(B * Nn, W, self.H)); o = o.view(B, Nn, W, self.H)
            u, u1 = o[:, :, -1], o[:, :, -2]
        else:
            u, u1 = x[:, -1], x[:, -2]
        valid = (self._x(X, "price")[:, -1].abs().sum(-1) > 0).float()
        h = u
        if self.rels:
            fp = self.filmP(PI[:, -1]).view(B, len(self.rels), 1, self.H) if self.c["regime"] == "film_prop" else None
            agg = 0.0
            for r, name in enumerate(self.rels):
                src = u1 if name == "lag2" else u
                Vs = self.V[r](src) * valid[..., None]
                m_r = torch.einsum("ij,bjh->bih", self.S[r], Vs)
                if self.c["msg"] == "contrast":
                    m_r = m_r - self.S[r].abs().sum(1)[None, :, None] * self.V[r](u)
                if fp is not None: m_r = m_r * (1 + fp[:, r])
                agg = agg + m_r
            agg = self.out(agg); gr = torch.sigmoid(self.res(torch.cat([u, agg], -1)))
            h = u + gr * agg; self.stats["residual_gate"] = float(gr.detach().mean())
        if self.c["regime"] == "film_head":
            g, b = self.filmH(PI[:, -1]).view(B, 1, 2, self.H).unbind(2); h = h * (1 + g) + b
        s = self.idio(self.drop(h)).squeeze(-1)
        mag = self.mag(h).squeeze(-1)
        cnt = valid.sum(1, keepdim=True).clamp(min=1)
        m = self.mkt(torch.cat([(h * valid[..., None]).sum(1) / cnt, MK[:, -1], PI[:, -1]], -1))
        s_c = s - (s * valid).sum(1, keepdim=True) / cnt
        if self.use_raw and self.use_struct: self.stats["struct_residual_scale"] = float(self.s_mix.detach())
        return m + s_c, s, mag

# ---- ranking targets (label transforms of day t+1 returns; no leakage) ----
GR3 = np.zeros((T, N), np.float32)
for t in range(T):
    idx = np.where(MR3[t])[0]
    if len(idx) >= 5: GR3[t, idx] = _N21.ppf((_rankdata(Y[t, idx]) - 0.5) / len(idx))
GR_t = _tn(GR3); THR_t = _tn(THR3)

def pearson_loss(s, y, m): return ic_loss(s, y, m)
def pairwise_loss(s, y, m, weighted=False):
    ds = s[:, :, None] - s[:, None, :]; dy = y[:, :, None] - y[:, None, :]
    pm = (m[:, :, None] & m[:, None, :] & (dy > 0)).float()
    l = Fnn.softplus(-ds)
    if weighted:                                          # LambdaRankIC: |Δrank(score)| · |Δrank(target)| (detached)
        with torch.no_grad():
            big = torch.finfo(s.dtype).max / 4
            rs = torch.argsort(torch.argsort(torch.where(m, s, torch.full_like(s, big)), 1), 1).float()
            ry = torch.argsort(torch.argsort(torch.where(m, y, torch.full_like(y, big)), 1), 1).float()
            w = (rs[:, :, None] - rs[:, None, :]).abs() * (ry[:, :, None] - ry[:, None, :]).abs()
            w = w * pm; w = w / (w.sum() / pm.sum().clamp(min=1)).clamp(min=1e-8)
        pm = pm * w
    return (l * pm).sum() / pm.sum().clamp(min=1)
def listnet_loss(s, y, m, tau=1.0):
    mf = m.float(); cnt = mf.sum(1, keepdim=True).clamp(min=2)                  # per-day z-score so the target scale is fixed
    mu = (y * mf).sum(1, keepdim=True) / cnt; sd = (((y - mu) * mf) ** 2).sum(1, keepdim=True).div(cnt).sqrt().clamp(min=1e-8)
    y = (y - mu) / sd
    neg = torch.finfo(s.dtype).min / 4
    ls = torch.log_softmax(torch.where(m, s, torch.full_like(s, neg)), 1)
    py = torch.softmax(torch.where(m, y / tau, torch.full_like(y, neg)), 1)
    ok = m.sum(1) >= 5
    return -(py * torch.where(m, ls, torch.zeros_like(ls))).sum(1)[ok].mean() if ok.any() else s.sum() * 0

def rank_loss(c, s, dt):
    y = GR_t[dt] if c["target"].startswith("gauss") else YR_t[dt]
    m = MR_t[dt]
    if c["target"] == "gauss_large": m = m & THR_t[dt]
    if c["rank_loss"] == "pearson": return pearson_loss(s, y, m)
    if c["rank_loss"] == "pairwise": return pairwise_loss(s, y, m)
    if c["rank_loss"] == "lambda_pairwise": return pairwise_loss(s, y, m, weighted=True)
    if c["rank_loss"] == "listnet": return listnet_loss(s, y, m, RK["listnet_tau"])
    raise ValueError(c["rank_loss"])

# ---- inputs ----
_TENS21 = {}
def tensors21(f3, c):
    key = (f3["name"], c["regime"] == "none", c["same_day"])
    if key not in _TENS21:
        Xt, PIt, MKt, sl = tensors_for(f3, RK["mode"], {"price": "v3", **({"regime": "none"} if c["regime"] == "none" else {})})
        if c["same_day"]:                                   # text only on days with a same-day item (no forward fill)
            has = (Xt[..., sl["news"][0]] > 0).float()[..., None]; Xt = Xt.clone()
            for blk in ("raw", "tri", "cat"): Xt[..., sl[blk][0]:sl[blk][1]] *= has
        _TENS21[key] = (Xt, PIt, MKt, sl)
    return _TENS21[key]

def graphs21(f3, c, seed):
    S = dict(f3["G"])
    if c["random"]:
        rng = np.random.default_rng(seed + 99)
        S = {r: random_signed(M, rng) for r, M in S.items()}
    return S

def build21(f3, c, seed):
    Xt, PIt, MKt, sl = tensors21(f3, c)
    return SCEGRank(sl, f3["K"], f3["MK"].shape[1], c, graphs21(f3, c, seed), RK["hidden"], RK["dropout"]).to(DEVICE)

@torch.no_grad()
def predict21(model, Xt, PIt, MKt, days):
    model.eval(); L, S_, G_ = [], [], []
    for s in range(0, len(days), 32):
        xb, pb, mb = windows(Xt, PIt, MKt, days[s:s + 32]); lo, sc, mg = model(xb, pb, mb)
        L.append(lo.float().cpu().numpy()); S_.append(sc.float().cpu().numpy()); G_.append(mg.float().cpu().numpy())
    return np.concatenate(L), np.concatenate(S_), np.concatenate(G_)

def val_criterion(c, vl, vs, days):
    if c["select"] == "rankic": return float(rank_series(vs, days).mean())
    return (auc_on(vl, days) - 0.5) + float(ic_days(vs, days).mean())

def p21(f3, c, seed, kind): return f"{RK_DIR}/{RK['mode']}__{fold_tag(f3)}{cfg_key(c)}__s{seed}.{kind}"

def train21(f3, c, seed):
    """Train on training days, early-stop on validation. Writes the state and VALIDATION outputs only (no test)."""
    if os.path.exists(p21(f3, c, seed, "pt")) and os.path.exists(p21(f3, c, seed, "val.npz")): return
    set_seed(seed)
    Xt, PIt, MKt, sl = tensors21(f3, c); model = build21(f3, c, seed)
    opt = torch.optim.AdamW(model.parameters(), lr=RK["lr"], weight_decay=RK["wd"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=RK["epochs"])
    posw = torch.tensor(f3["POSW"], device=DEVICE); rng = np.random.default_rng(seed); sig = float(f3["SIGMA"])
    best, best_state, bad = -np.inf, None, 0
    for ep in range(RK["epochs"]):
        model.train(); order = rng.permutation(f3["TR"])
        for s0 in range(0, len(order), RK["batch_days"]):
            days = order[s0:s0 + RK["batch_days"]]; xb, pb, mb = windows(Xt, PIt, MKt, days); dt = torch.as_tensor(days, device=DEVICE)
            logit, sc, mg = model(xb, pb, mb)
            loss = 0.0
            if c["w_dir"] > 0:
                mc = MC_t[dt]; loss = loss + c["w_dir"] * Fnn.binary_cross_entropy_with_logits(logit[mc], YC_t[dt][mc], pos_weight=posw)
            if c["w_rank"] > 0: loss = loss + c["w_rank"] * rank_loss(c, sc, dt)
            if c["w_mag"] > 0:
                mr = MR_t[dt]; loss = loss + c["w_mag"] * Fnn.huber_loss(mg[mr], YR_t[dt][mr] / sig)
            opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        sched.step()
        vl, vs, _ = predict21(model, Xt, PIt, MKt, f3["VA"]); v = val_criterion(c, vl, vs, f3["VA"])
        if v > best + 1e-5: best, bad, best_state = v, 0, {k: t.detach().cpu().clone() for k, t in model.state_dict().items()}
        else:
            bad += 1
            if bad >= RK["patience"]: break
    model.load_state_dict(best_state)
    vl, vs, vm = predict21(model, Xt, PIt, MKt, f3["VA"])
    torch.save(best_state, p21(f3, c, seed, "pt"))
    np.savez(p21(f3, c, seed, "val.npz"), val_logit=vl, val_score=vs, val_mag=vm, val_sel=best, epochs=ep + 1,
             **{f"stat_{k}": v for k, v in model.stats.items()})

def val_units(c):
    """(fold, seed) validation RankIC units + the seed-averaged daily series pooled over folds."""
    units, daily = [], []
    for f3 in FOLD3:
        per = []
        for seed in RK["seeds"]:
            z = np.load(p21(f3, c, seed, "val.npz")); d = rank_series(z["val_score"], f3["VA"])
            units.append(dict(fold=f3["name"], seed=seed, val_RankIC=float(d.mean()), val_AUC=auc_on(z["val_logit"], f3["VA"]))); per.append(d)
        daily.append(np.mean(per, 0))
    return pd.DataFrame(units), np.concatenate(daily)

def compare_val(c_new, c_inc):
    ua, da = val_units(c_new); ub, db = val_units(c_inc)
    d = da - db; wins = float((ua.val_RankIC.values > ub.val_RankIC.values).mean())
    return dict(val_before=ub.val_RankIC.mean(), val_before_sd=ub.groupby("seed").val_RankIC.mean().std(ddof=0),
                val_after=ua.val_RankIC.mean(), val_after_sd=ua.groupby("seed").val_RankIC.mean().std(ddof=0),
                dval=float(d.mean()), t_val=float(nw_t(d)), wins=wins,
                adopt=bool(nw_t(d) >= RK["adopt_t"] and d.mean() >= RK["adopt_min"] and wins >= RK["adopt_units"]))

### 21.5 · Pre-registration (written and hashed before any §21 model is trained)
**Order of stages** (each changes one factor, starting from the incumbent of the previous stage):

| Stage | Factor | Arms (incumbent first) | Hypothesis |
|---|---|---|---|
| S | model selection | composite (SCEG-R) · validation RankIC | selecting on a market-dominated AUC does not select ranking skill (D1) |
| A | ranking target | raw return · Gaussian rank · Gaussian rank on large-move training days | Pearson on heavy-tailed raw returns ≠ Spearman (D3); low-|move| days are label noise |
| B | ranking loss form | Pearson IC · pairwise RankNet · LambdaRankIC-weighted pairwise · ListNet | pairwise/listwise losses optimise order directly |
| W | loss-weight ratio w_rank / w_dir | 0.25 · 1 · 4 · 16 (only the ratio matters under AdamW) | the ranking term is under-weighted relative to BCE |
| M | magnitude auxiliary | w_mag 0 · 0.5 | a return-scale head regularises the shared representation |
| C | graph source | none · DYNOTEARS · sector · market-residual correlation; winner must also beat its matched random graph | relations carry stock-specific information only after the market factor is removed (D4) |
| C2 | message form (only if a graph is adopted) | neighbour average · contrast | averaging smooths away cross-sectional differences (D4) |
| D | event representation | raw+struct · raw · struct · raw+struct same-day only | structured fields add information / stale text dilutes it |
| E | temporal context | GRU (W=5) · none | the window adds ranking information |
| F | regime | feature · none · FiLM on scoring head · FiLM on propagation (only with a graph) | a common regime can change rankings only by modulating the scoring map |

**Adoption rule (validation only):** an arm replaces the incumbent iff NW t of the seed-averaged daily validation ΔRankIC ≥ 2.0, the mean
ΔRankIC ≥ 0.002, and it wins in ≥ 60 % of (fold, seed) units; among several, the largest mean Δ. A graph source must additionally beat a
signed random graph matched to it (validation ΔRankIC > 0). Otherwise the incumbent stays. Test data are not read until 21.8.

**Afterwards (21.7, validation-only training):** ablations of the final model (− structured events, − graph, − temporal, − regime,
− ranking loss, ± magnitude, − all text), an *identity-complete* SCEG-Rank (every SCEG component switched on, graph = best validation
source), and ranking baselines (ridge on the Gaussian-rank target, LightGBM LambdaRank, model-free heuristics).

In [80]:
# ===== 21.5 · pre-registration =====
STAGES = [
    ("S", "model selection criterion", [dict(select="composite"), dict(select="rankic")]),
    ("A", "ranking target", [dict(target="raw"), dict(target="gauss"), dict(target="gauss_large")]),
    ("B", "ranking loss form", [dict(rank_loss="pearson"), dict(rank_loss="pairwise"), dict(rank_loss="lambda_pairwise"), dict(rank_loss="listnet")]),
    # AdamW is (up to epsilon, weight decay and clipping) invariant to a common loss scale, so only the ratio w_rank / w_dir matters:
    ("W", "loss weights (w_rank, w_dir)", [dict(w_rank=float(HP_R["lam_rank"]), w_dir=1.0), dict(w_rank=1.0, w_dir=1.0),
                                           dict(w_rank=4.0, w_dir=1.0), dict(w_rank=16.0, w_dir=1.0)]),
    ("M", "magnitude auxiliary task", [dict(w_mag=0.0), dict(w_mag=0.5)]),
    ("C", "graph source", [dict(graph="none"), dict(graph="dynotears"), dict(graph="sector"), dict(graph="resid_corr")]),
    ("C2", "message form (only if a graph was adopted)", [dict(msg="avg"), dict(msg="contrast")]),
    ("D", "event representation", [dict(text="raw+struct", same_day=False), dict(text="raw", same_day=False),
                                   dict(text="struct", same_day=False), dict(text="raw+struct", same_day=True)]),
    ("E", "temporal context", [dict(temporal="gru"), dict(temporal="none")]),
    ("F", "regime", [dict(regime="feature"), dict(regime="none"), dict(regime="film_head"), dict(regime="film_prop")]),
]
PREREG21 = dict(RK=RK, B0=B0, stages=[(s, d, a) for s, d, a in STAGES], rule="see 21.5 markdown",
                ablations=["- structured events", "- graph", "- temporal", "- regime", "- ranking loss", "± magnitude", "- all text",
                           "identity-complete"], baselines=["ridge (gauss-rank)", "LightGBM LambdaRank", "heuristics"])
_pr_txt = json.dumps(PREREG21, sort_keys=True, default=str)
PR_PATH = f"{RK_DIR}/prereg_{DS21}.json"
if os.path.exists(PR_PATH):
    assert json.load(open(PR_PATH))["spec"] == _pr_txt, "§21 pre-registration changed after it was frozen: refuse to run"
else:
    json.dump({"spec": _pr_txt, "sha": sha(_pr_txt)[:12], "at": time.strftime("%Y-%m-%d %H:%M:%S")}, open(PR_PATH, "w"), indent=1)
print("§21 pre-registered:", PR_PATH, "| sha", sha(_pr_txt)[:12])

§21 pre-registered: /content/drive/MyDrive/MinorProject/acl18_cache/v3/rank21/prereg_acl18.json | sha 644adf556467


### 21.6 · Sequential stages (validation only)

In [81]:
# ===== 21.6 · run the stages =====
def run_arms(cfgs):
    for f3 in FOLD3:
        for seed in RK["seeds"]:
            for c in cfgs:
                t1 = time.time(); fresh = not os.path.exists(p21(f3, c, seed, "val.npz"))
                train21(f3, c, seed)
                if fresh: print(f"  {f3['name']} {cfg_key(c)} seed {seed} | {time.time() - t1:4.0f}s", flush=True)

INC = dict(B0); DECISIONS = []
t0 = time.time()
for sid, desc, arms in STAGES:
    graph_on = INC["graph"] != "none"
    if sid == "C2" and not graph_on:
        DECISIONS.append(dict(stage=sid, factor=desc, skipped="no graph adopted in stage C")); continue
    arms_ = [a for a in arms if not (sid == "F" and a.get("regime") == "film_prop" and not graph_on)]
    inc = dict(INC)
    cands = [c for c in ({**INC, **a} for a in arms_) if cfg_key(c) != cfg_key(inc)]
    extra = [{**c, "random": True} for c in cands if c["graph"] != "none"] if sid == "C" else []
    print(f"\nstage {sid} · {desc}: {len(cands)} challengers vs incumbent {cfg_key(inc)}", flush=True)
    run_arms([inc] + cands + extra)
    rows = []
    for c in cands:
        r = compare_val(c, inc)
        if sid == "C":
            rr = compare_val(c, {**c, "random": True}); r["dval_vs_random"] = rr["dval"]; r["adopt"] = bool(r["adopt"] and rr["dval"] > 0)
        rows.append(dict(arm={k: c[k] for k in arms_[0]}, key=cfg_key(c), cfg=c, **r))
    passed = [r for r in rows if r["adopt"]]
    win = max(passed, key=lambda r: r["dval"]) if passed else None
    if win is not None: INC = dict(win["cfg"])
    DECISIONS.append(dict(stage=sid, factor=desc, incumbent=cfg_key(inc), inc_cfg=inc, arms=rows,
                          extra_cfgs=extra, chosen=cfg_key(INC), changed=win is not None))
    print(pd.DataFrame(rows).drop(columns=["key", "cfg"]).round(4).to_string(index=False))
    print(f"  -> {'ADOPT ' + str(win['arm']) if win else 'keep incumbent'}", flush=True)
FINAL21 = dict(INC)
json.dump(DECISIONS, open(f"{RK_DIR}/decisions_{DS21}.json", "w"), indent=1, default=str)
print(f"\nstages done in {(time.time() - t0) / 60:.1f} min | final configuration: {FINAL21}")


stage S · model selection criterion: 1 challengers vs incumbent 350606dbd0
  F1 350606dbd0 seed 0 |   43s
  F1 6b71bb75e6 seed 0 |   14s
  F1 350606dbd0 seed 1 |   13s
  F1 6b71bb75e6 seed 1 |   11s
  F1 350606dbd0 seed 2 |   14s
  F1 6b71bb75e6 seed 2 |   10s
  F1 350606dbd0 seed 3 |    6s
  F1 6b71bb75e6 seed 3 |    6s
  F1 350606dbd0 seed 4 |    6s
  F1 6b71bb75e6 seed 4 |    8s
                 arm  val_before  val_before_sd  val_after  val_after_sd   dval  t_val  wins  adopt
{'select': 'rankic'}      0.0536         0.0238     0.0637        0.0113 0.0101  1.609   0.4  False
  -> keep incumbent

stage A · ranking target: 2 challengers vs incumbent 350606dbd0
  F1 3de37e2051 seed 0 |    5s
  F1 313d7cce2b seed 0 |    4s
  F1 3de37e2051 seed 1 |    7s
  F1 313d7cce2b seed 1 |    6s
  F1 3de37e2051 seed 2 |    6s
  F1 313d7cce2b seed 2 |    5s
  F1 3de37e2051 seed 3 |   11s
  F1 313d7cce2b seed 3 |    6s
  F1 3de37e2051 seed 4 |    7s
  F1 313d7cce2b seed 4 |    6s
                   

### 21.7 · Freeze the final configuration; ablations, identity-complete model and ranking baselines (validation only)

In [82]:
# ===== 21.7 · freeze, then ablations and baselines (validation-only training) =====
FIN_PATH = f"{RK_DIR}/final_{DS21}.json"
_fin_txt = json.dumps({"final": FINAL21, "decisions": DECISIONS, "prereg": sha(_pr_txt)[:12]}, sort_keys=True, default=str)
if os.path.exists(FIN_PATH):
    assert json.load(open(FIN_PATH))["spec"] == _fin_txt, "final §21 configuration differs from the frozen file"
else:
    json.dump({"spec": _fin_txt, "sha": sha(_fin_txt)[:12], "at": time.strftime("%Y-%m-%d %H:%M:%S")}, open(FIN_PATH, "w"), indent=1)
print("FROZEN §21 final configuration:", FINAL21, "| sha", sha(_fin_txt)[:12])

# best validation graph source (stage C results; validation only)
_cs = next(d for d in DECISIONS if d["stage"] == "C")
_g_rows = [r for r in _cs["arms"] if r["cfg"]["graph"] != "none"]
_g_best = max(_g_rows, key=lambda r: r["val_after"])["cfg"]["graph"] if _g_rows else "dynotears"
F_ = FINAL21
ABL = {}
ABL["− structured events" if F_["text"] in ("raw+struct", "struct") else "+ structured events"] = (
    {**F_, "text": "raw" if F_["text"] == "raw+struct" else ("none" if F_["text"] == "struct" else "raw+struct")})
ABL["− graph" if F_["graph"] != "none" else f"+ graph ({_g_best})"] = (
    {**F_, "graph": "none", "msg": "avg", "regime": "feature" if F_["regime"] == "film_prop" else F_["regime"]} if F_["graph"] != "none"
    else {**F_, "graph": _g_best})
ABL["− temporal" if F_["temporal"] == "gru" else "+ temporal"] = {**F_, "temporal": "none" if F_["temporal"] == "gru" else "gru"}
ABL["− regime" if F_["regime"] != "none" else "+ regime feature"] = {**F_, "regime": "none" if F_["regime"] != "none" else "feature"}
ABL["− ranking loss"] = {**F_, "w_rank": 0.0}
ABL["− magnitude" if F_["w_mag"] > 0 else "+ magnitude"] = {**F_, "w_mag": 0.0 if F_["w_mag"] > 0 else 0.5}
ABL["− all text"] = {**F_, "text": "none", "same_day": False}
ABL = {k: v for k, v in ABL.items() if cfg_key(v) != cfg_key(F_)}
IDENT = {**F_, "text": "raw+struct", "graph": _g_best, "temporal": "gru", "regime": "film_prop", "w_mag": 0.5}
ARMS_EXTRA = {**ABL, "identity-complete SCEG-Rank": IDENT, "original re-trained (B0)": dict(B0)}
run_arms(list(ARMS_EXTRA.values()))

# ranking baselines on the same inputs (validation-only selection)
def flat_feats(f3, days):
    X = f3["X"][RK["mode"]]; W = V3["WINDOW"]
    cur = X[days]; mean = np.stack([X[t - W + 1:t + 1].mean(0) for t in days])
    glob_ = np.concatenate([f3["PI"][days], f3["MK"][days]], -1)[:, None, :].repeat(N, 1)
    return np.concatenate([cur, mean, glob_], -1)
def train_ridge21(f3):
    from sklearn.linear_model import Ridge
    pth = f"{RK_DIR}/{RK['mode']}__{fold_tag(f3)}ridge.npz"
    if os.path.exists(pth): return
    Ztr, Zva = flat_feats(f3, f3["TR"]), flat_feats(f3, f3["VA"]); mtr = MR3[f3["TR"]]
    best = (-np.inf, None)
    for a in (1.0, 10.0, 100.0, 1000.0):
        mdl = Ridge(alpha=a).fit(Ztr[mtr], GR3[f3["TR"]][mtr])
        v = rank_series(mdl.predict(Zva.reshape(-1, Zva.shape[-1])).reshape(len(f3["VA"]), N), f3["VA"]).mean()
        if v > best[0]: best = (v, a)
    np.savez(pth, alpha=best[1], val_RankIC=best[0])
def train_lgbrank21(f3, seed):
    import lightgbm as lgb
    pth = f"{RK_DIR}/{RK['mode']}__{fold_tag(f3)}lgbrank__s{seed}.txt"
    if os.path.exists(pth): return
    def grp(days):
        Z = flat_feats(f3, days); m = MR3[days]
        lab = np.zeros((len(days), N), int)
        for r, t in enumerate(days):
            idx = np.where(m[r])[0]
            if len(idx): lab[r, idx] = np.minimum((_rankdata(Y[t, idx]) - 1) * 10 // len(idx), 9).astype(int)
        return Z[m], lab[m], m.sum(1)
    Xa, ya, ga = grp(f3["TR"]); Xv, yv, gv = grp(f3["VA"])
    rk = lgb.LGBMRanker(objective="lambdarank", n_estimators=2000, learning_rate=0.02, num_leaves=15, min_child_samples=200,
                        subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=seed, verbose=-1)
    rk.fit(Xa, ya, group=ga[ga > 0], eval_set=[(Xv, yv)], eval_group=[gv[gv > 0]], eval_at=[10],
           callbacks=[lgb.early_stopping(100, verbose=False)])
    rk.booster_.save_model(pth)
for f3 in FOLD3:
    train_ridge21(f3)
    for seed in RK["seeds"]: train_lgbrank21(f3, seed)
print("ablations, identity-complete model and baselines trained (validation only)")

FROZEN §21 final configuration: {'text': 'raw+struct', 'same_day': False, 'graph': 'none', 'random': False, 'msg': 'avg', 'temporal': 'gru', 'regime': 'none', 'target': 'raw', 'rank_loss': 'pearson', 'w_dir': 1.0, 'w_rank': 0.25, 'w_mag': 0.0, 'select': 'composite'} | sha 803af8b354eb
  F1 d26d22e50d seed 0 |    6s
  F1 ef3cc3f4f8 seed 0 |    6s
  F1 a30fec8aad seed 0 |   10s
  F1 a1f24bb266 seed 0 |    6s
  F1 bdbc2a8bb1 seed 0 |   11s
  F1 6796e5fbbe seed 0 |    9s
  F1 b620dc87ca seed 0 |    9s
  F1 d26d22e50d seed 1 |    8s
  F1 ef3cc3f4f8 seed 1 |    8s
  F1 a30fec8aad seed 1 |    3s
  F1 a1f24bb266 seed 1 |    7s
  F1 bdbc2a8bb1 seed 1 |    7s
  F1 6796e5fbbe seed 1 |    4s
  F1 b620dc87ca seed 1 |    9s
  F1 d26d22e50d seed 2 |    5s
  F1 ef3cc3f4f8 seed 2 |    9s
  F1 a30fec8aad seed 2 |    9s
  F1 a1f24bb266 seed 2 |    6s
  F1 bdbc2a8bb1 seed 2 |    5s
  F1 6796e5fbbe seed 2 |    5s
  F1 b620dc87ca seed 2 |    6s
  F1 d26d22e50d seed 3 |    8s
  F1 ef3cc3f4f8 seed 3 |   11s
 

### 21.8 · Test predictions (produced once, after the freeze)

In [83]:
# ===== 21.8 · test predictions for every pre-registered §21 arm =====
require_shakedown("21.8 ranking-study test predictions")
assert os.path.exists(FIN_PATH) and os.path.exists(PR_PATH), "freeze files missing: test predictions are locked"
def test21(c):
    for f3 in FOLD3:
        Xt, PIt, MKt, sl = tensors21(f3, c)
        for seed in RK["seeds"]:
            out = p21(f3, c, seed, "test.npz")
            if os.path.exists(out): continue
            mdl = build21(f3, c, seed); mdl.load_state_dict(torch.load(p21(f3, c, seed, "pt"), map_location="cpu"))
            tl, ts, tm = predict21(mdl, Xt, PIt, MKt, f3["TE"]); np.savez(out, te_logit=tl, te_score=ts, te_mag=tm)
def runs21(c):
    runs = {}
    for seed in RK["seeds"]:
        ps = [p21(f3, c, seed, "test.npz") for f3 in FOLD3]
        if all(os.path.exists(p) for p in ps):
            zs = [np.load(p) for p in ps]; runs[seed] = {k: np.concatenate([z[k] for z in zs]) for k in ("te_logit", "te_score", "te_mag")}
    return runs
stage_cfgs = [dict(FINAL21)]
for d in DECISIONS:
    if "arms" not in d: continue
    stage_cfgs += [d["inc_cfg"]] + [r["cfg"] for r in d["arms"]] + list(d["extra_cfgs"])
ALL_TEST_CFGS = {cfg_key(c): c for c in stage_cfgs + list(ARMS_EXTRA.values())}
for c in ALL_TEST_CFGS.values(): test21(c)
# baselines
BASE_RUNS21 = {}
_r = {}
for seed in RK["seeds"]:
    ts_all = []
    for f3 in FOLD3:
        import lightgbm as lgb
        bst = lgb.Booster(model_file=f"{RK_DIR}/{RK['mode']}__{fold_tag(f3)}lgbrank__s{seed}.txt")
        Z = flat_feats(f3, f3["TE"]); ts_all.append(bst.predict(Z.reshape(-1, Z.shape[-1])).reshape(len(f3["TE"]), N))
    ts_all = np.concatenate(ts_all); _r[seed] = dict(te_score=ts_all, te_logit=np.zeros_like(ts_all))
BASE_RUNS21["LightGBM LambdaRank"] = _r
from sklearn.linear_model import Ridge
_ts = []
for f3 in FOLD3:
    a = float(np.load(f"{RK_DIR}/{RK['mode']}__{fold_tag(f3)}ridge.npz")["alpha"])
    mdl = Ridge(alpha=a).fit(flat_feats(f3, f3["TR"])[MR3[f3["TR"]]], GR3[f3["TR"]][MR3[f3["TR"]]])
    Z = flat_feats(f3, f3["TE"]); _ts.append(mdl.predict(Z.reshape(-1, Z.shape[-1])).reshape(len(f3["TE"]), N))
_ts = np.concatenate(_ts); BASE_RUNS21["ridge (Gaussian-rank target)"] = {0: dict(te_score=_ts, te_logit=np.zeros_like(_ts))}
BASE_RUNS21.update(HEUR)
print("test predictions written for", len(ALL_TEST_CFGS), "SCEG-Rank configurations + baselines")

test predictions written for 30 SCEG-Rank configurations + baselines


### 21.9 · Reports A–F
All test numbers below are **reported, not used**: every decision was taken in 21.6–21.7 on validation data.
ΔRankIC tests: seed-paired daily differences (seed-averaged), Newey–West t (5 lags), Holm within each table.

In [84]:
# ===== 21.9 · reports =====
def rank_row(name, runs, mag=False):
    row, avg = rank_stats(runs, name=name)
    per = [dict(**cls_metrics(r["te_logit"], TE_ALL), **split_metrics(r["te_logit"], TE_ALL), **ls_portfolio(r["te_score"], TE_ALL))
           for r in runs.values()]
    P = pd.DataFrame(per)
    row.update(ACC=P.ACC.mean(), ACC_sd=P.ACC.std(ddof=0), MCC=P.MCC.mean(), AUC=P.AUC.mean(), DirAcc_stock=P.DirAcc_stock.mean(),
               LS_Sharpe_gross=P.LS_Sharpe_gross.mean(), LS_Sharpe_net=P.LS_Sharpe_net.mean())
    if mag and all("te_mag" in r for r in runs.values()):
        mae, rmse, r2 = [], [], []
        for r in runs.values():
            pr = []
            o = 0
            for f3 in FOLD3:
                n_ = len(f3["TE"]); pr.append(r["te_mag"][o:o + n_] * f3["SIGMA"]); o += n_
            pr = np.concatenate(pr); m = MR3[TE_ALL]; e = pr[m] - Y[TE_ALL][m]
            mae.append(np.abs(e).mean() * 100); rmse.append(np.sqrt((e ** 2).mean()) * 100)
            r2.append(1 - (e ** 2).sum() / (Y[TE_ALL][m] ** 2).sum())
        row.update(MAE_pct=np.mean(mae), RMSE_pct=np.mean(rmse), R2_OS_vs_zero=np.mean(r2))
    return row, avg

def dtest(runsA, runsB):
    sa = sorted(runsA); sb = sorted(runsB)
    a = np.mean([rank_series(runsA[s]["te_score"], TE_ALL) for s in sa], 0)
    b = np.mean([rank_series(runsB[s]["te_score"], TE_ALL) for s in sb], 0)
    d = a - b; t = nw_t(d)
    return dict(dRankIC=float(d.mean()), t=float(t), p=float(2 * (1 - _N21.cdf(abs(t)))), lo=float(d.mean() - 1.96 * nw_se(d)),
                hi=float(d.mean() + 1.96 * nw_se(d)))

pd.set_option("display.width", 260)
# ---- A. original SCEG ----
orig_runs = load_runs(RK["mode"], R_FINAL_NAME)
REP_A = pd.DataFrame([rank_row("original SCEG-R (cached, §16)", orig_runs)[0], rank_row("original re-trained with §21 code (B0)", runs21(B0))[0]]).set_index("model")
print("=== A. Original SCEG (test) ==="); print(REP_A.round(4).to_string())

# ---- B. diagnostics (summary of 21.3) ----
print("\n=== B. Diagnostic findings (training / validation data; see 21.3) ==="); print(DIAG21.to_string(index=False))

# ---- C. controlled improvements (stage by stage) ----
rowsC = []
HYP = {"S": "selection on a market-dominated criterion does not select ranking skill",
       "A": "Pearson on heavy-tailed raw returns is misaligned with Spearman; small moves are label noise",
       "B": "pairwise / listwise losses optimise the order directly", "W": "the ranking term is under-weighted",
       "M": "a return-scale auxiliary head regularises the representation",
       "C": "relations carry stock-specific information once the market factor is removed",
       "C2": "neighbour averaging smooths away cross-sectional differences", "D": "structured fields / fresh text add information",
       "E": "the 5-day window adds ranking information", "F": "a regime common to all firms matters only by modulating the scoring map"}
for d in DECISIONS:
    if "arms" not in d:
        rowsC.append(dict(stage=d["stage"], modification=d["factor"], decision="skipped: " + d.get("skipped", ""))); continue
    rb = runs21(d["inc_cfg"])
    for r in d["arms"]:
        ra = runs21(r["cfg"]); tr = dtest(ra, rb) if ra and rb else {}
        rowsC.append(dict(stage=d["stage"], modification=f"{d['factor']}: {r['arm']}", hypothesis=HYP.get(d["stage"], ""),
                          val_before=r["val_before"], val_after=r["val_after"], dval=r["dval"], t_val=r["t_val"],
                          wins=r["wins"], vs_random=r.get("dval_vs_random", np.nan),
                          test_before=rank_stats(rb)[0]["RankIC"] if rb else np.nan,
                          test_after=rank_stats(ra)[0]["RankIC"] if ra else np.nan,
                          test_after_sd=rank_stats(ra)[0]["RankIC_sd_seeds"] if ra else np.nan,
                          dtest=tr.get("dRankIC", np.nan), t_test=tr.get("t", np.nan), p_test=tr.get("p", np.nan),
                          decision="ADOPTED (validation)" if (d["changed"] and r["key"] == d["chosen"]) else "not adopted"))
REP_C = pd.DataFrame(rowsC)
if "p_test" in REP_C:
    ok = REP_C["p_test"].notna(); REP_C.loc[ok, "p_test (Holm)"] = holm(REP_C.loc[ok, "p_test"].values)
print("\n=== C. Controlled modifications: validation decides, test reports (RankIC, all firm-days) ===")
print(REP_C.round(4).to_string(index=False)); REP_C.to_csv(f"{RK_DIR}/report_C_{DS21}.csv", index=False)

# ---- D. final SCEG-Rank ----
fin_runs = runs21(FINAL21)
rowD, _ = rank_row("final SCEG-Rank (§21, frozen)", fin_runs, mag=FINAL21["w_mag"] > 0)
rowI, _ = rank_row("identity-complete SCEG-Rank", runs21(IDENT), mag=True)
REP_D = pd.DataFrame([rowD, rowI, REP_A.reset_index().iloc[0].to_dict()]).set_index("model")
print("\n=== D. Final model (test; per-seed mean ± sd where marked) ==="); print(REP_D.round(4).to_string())
tt = dtest(fin_runs, orig_runs)
print(f"final − original SCEG-R: ΔRankIC {tt['dRankIC']:+.4f} [{tt['lo']:+.4f}, {tt['hi']:+.4f}] | NW t {tt['t']:+.2f} | p {tt['p']:.3f}")

# ---- E. ablations ----
rowsE = []
for name, c in ABL.items():
    ra = runs21(c)
    if not ra: continue
    r = dtest(fin_runs, ra); rowsE.append(dict(ablation=name, RankIC=rank_stats(ra)[0]["RankIC"], **{f"final − ablation {k}": v for k, v in r.items()}))
REP_E = pd.DataFrame(rowsE)
if len(REP_E): REP_E["p (Holm)"] = holm(REP_E["final − ablation p"].values)
print("\n=== E. Ablations of the final model (positive Δ = the component helps) ==="); print(REP_E.round(4).to_string(index=False))
REP_E.to_csv(f"{RK_DIR}/report_E_{DS21}.csv", index=False)

# ---- baselines ----
rowsB = []
for name, runs in {**BASE_RUNS21, **{k: load_runs(RK["mode"], v) for k, v in ORIG.items()}}.items():
    if not runs: continue
    r = dtest(fin_runs, runs); rowsB.append(dict(baseline=name, RankIC=rank_stats(runs)[0]["RankIC"], seeds=len(runs),
                                                  **{f"final − baseline {k}": v for k, v in r.items()}))
REP_BL = pd.DataFrame(rowsB); REP_BL["p (Holm)"] = holm(REP_BL["final − baseline p"].values)
print("\n=== Baselines (does SCEG-Rank beat simpler rankers?) ==="); print(REP_BL.round(4).to_string(index=False))
REP_BL.to_csv(f"{RK_DIR}/report_baselines_{DS21}.csv", index=False)

# ---- stability of the final model over time ----
_sf = rank_stats(fin_runs)[1]; _so = rank_stats(orig_runs)[1]
STAB21 = pd.DataFrame({"final SCEG-Rank": pd.Series(_sf).groupby(_per).mean(), "original SCEG-R": pd.Series(_so).groupby(_per).mean(),
                       "days": pd.Series(np.ones(len(_per))).groupby(_per).sum()})
print(f"\nRankIC by {'month' if IS_ACL18 else 'quarter'} (seed-averaged):"); print(STAB21.round(4).to_string())

# ---- F. automatic verdicts (the interpretation must follow these numbers) ----
adopted = [d["stage"] + ": " + str(next(r["arm"] for r in d["arms"] if r["key"] == d["chosen"])) for d in DECISIONS if d.get("changed")]
sig_base = REP_BL[(REP_BL["p (Holm)"] < 0.05)]
VERD = {
    "modifications adopted on validation": adopted or "none (the original configuration was kept at every stage)",
    "final vs original: ΔRankIC [95% CI], p": f"{tt['dRankIC']:+.4f} [{tt['lo']:+.4f}, {tt['hi']:+.4f}], p = {tt['p']:.3f}",
    "improvement statistically resolved on this test set": bool(tt["p"] < 0.05),
    "baselines significantly WORSE than final (Holm)": sig_base[sig_base["final − baseline dRankIC"] > 0].baseline.tolist(),
    "baselines significantly BETTER than final (Holm)": sig_base[sig_base["final − baseline dRankIC"] < 0].baseline.tolist(),
    "components that help (ablation Δ > 0, Holm p < 0.05)": REP_E[(REP_E.get("p (Holm)", 1) < 0.05) & (REP_E["final − ablation dRankIC"] > 0)].ablation.tolist() if len(REP_E) else [],
    "evidential status": "partially held out (ACL18 SCEG test previously inspected)" if IS_ACL18 else "held out (confirmation)",
}
for k, v in VERD.items(): print(f"F · {k}: {v}")
json.dump({k: (v if isinstance(v, (list, str, bool)) else str(v)) for k, v in VERD.items()}, open(f"{RK_DIR}/verdicts_{DS21}.json", "w"), indent=1)

=== A. Original SCEG (test) ===
                                        seeds  RankIC  RankIC_sd_seeds  RankIC_sd_days    ICIR    NW_t  pos_days      IC  IC_sd_seeds      ACC  ACC_sd     MCC      AUC  DirAcc_stock  LS_Sharpe_gross  LS_Sharpe_net
model                                                                                                                                                                                                                
original SCEG-R (cached, §16)               5  0.0286           0.0049          0.1439  0.1986  1.4563   57.8125  0.0242       0.0046  55.7688  1.0735  0.1254  58.1952       50.8836           1.7215        -0.0199
original re-trained with §21 code (B0)      5  0.0391           0.0161          0.1462  0.2678  1.9712   59.3750  0.0390       0.0148  55.5376  0.6217  0.1141  58.5811       51.2608           3.2607         1.5870

=== B. Diagnostic findings (training / validation data; see 21.3) ===
id                                       